# Mathematical Progress for Station v1.5

This notebook lists the recent progress made by Station on construction-based mathematical open problems. All of the solutions were discovered independently by AI agents (Gemini 3.1 Pro and GPT-5.5) within the Station system ([https://github.com/dualverse-ai/station](https://github.com/dualverse-ai/station)), without external hints or expert input unless otherwise stated.

Note that these results may not be final, as most Station instances are still running. We will update this notebook in the future.

Last update: 2026 May 25

## 1. Finiteness Problem for Diophantine Equations

Problem source: [Epoch AI](https://epoch.ai/frontiermath/open-problems/small-diophantine)

This section contains a standalone payload and exact verification code for three large-$x$ Diophantine equations. The solved equations are 3, 4, and 9 from the nine-equation list below.

For each solved equation, the payload gives three integer triples $(x,y,z)$ with pairwise distinct $x$ values and $|x| > 10^{50}$. All checks below use exact Python integer arithmetic.

### Results

The Station was able to solve three equations without human intervention or expert input: Eq3, Eq4, and Eq9. Although Eq4 and Eq9 can be solved by GPT-5.4 Pro alone, Eq3 has not, as far as we are aware, been shown to be solved by any AI system other than by the problem authors themselves. To our knowledge, the method for Eq3 has not been publicly disclosed, and we did not know it beforehand. 

Note that, to save computational cost, a single Station instance was given the problem statement for solving all nine equations.

### Method

Each construction rewrites the quadratic part in $z$ using a product identity. For equations 3 and 4, write the equation as

$$
z^2 + y^2z + f(x) = 0
$$

and set

$W = 2z + y^2$.

Then the equation is equivalent to

$W^2 = y^4 - 4f(x)$.

For equation 9, set

$W = 2z + y^2 - 1$.

Then equation 9 is equivalent to

$W^2 = (y^2 - 1)^2 - 4(x^3 + 2)$.

The families below choose a rational or integer parameter so that the right-hand side is a square by construction. After the square identity is enforced, $z$ is recovered from $W$, and congruence conditions are imposed so that all rational expressions become integers.

For equation 3, use the parabolic square-return branch

$$
x = q - \frac{9q^2}{16},
$$

$$
y = \frac{aq}{4},
$$

$$
W = 2 - q + \frac{5q^2}{16} - \frac{27q^3}{32},
$$

with $a = 155 + 1674t$ and $q = (a^4 + 1271)/837$. This gives $W^2 = y^4 - 4(x^3 + x - 1)$. The congruence class $a = 155 + 1674t$ makes $x$, $y$, and $z = (W - y^2)/2$ integral for all integer $t$.

For equation 4, take $p = 6t + 2$ and $n = (p^2 + 2)/6$. The congruence $p^2 \equiv 4 \pmod{6}$ makes $n$ integral. Then

$$
x = -3n^2 - 2n - 1,
$$

$$
y = np,
$$

$$
z = 3n^3 + 5n^2 + 4n + 1
$$

satisfy $z^2 + y^2z + x^3 + x + 1 = 0$ identically.

For equation 9, take $s = 5t + 1$ and $n = (6s^2 - 1)/5$. The congruence $s^2 \equiv 1 \pmod{5}$ makes $n$ integral. Then

$$
x = 6n^2 - 2,
$$

$$
y = 6ns,
$$

$$
z = -2(6n^3 - 6n^2 + 1)
$$

satisfy $z^2 + y^2z - z + x^3 + 2 = 0$ identically.

Because $|x|$ grows polynomially with the parameter, choosing large values of $t$ immediately gives $|x| > 10^{50}$. The code cells verify both the specific payload and a wider enumeration of the same families using exact arithmetic.


In [2]:
#@title Verification functions
from __future__ import annotations

import json
import math
from fractions import Fraction

X_THRESHOLD = 10**50
TARGET_SOLUTIONS_PER_EQUATION = 3

EQUATION_TEXT = {
    1: "z^2 + y^2*z + x^3 - 2 = 0",
    2: "z^2 + y^2*z + x^3 - x - 1 = 0",
    3: "z^2 + y^2*z + x^3 + x - 1 = 0",
    4: "z^2 + y^2*z + x^3 + x + 1 = 0",
    5: "z^2 + y^2*z + x^3 - 3 = 0",
    6: "z^2 + y^2*z + x^3 + 3 = 0",
    7: "z^2 + y^2*z + x^3 - x - 2 = 0",
    8: "z^2 + y^2*z + x^3 - x + 2 = 0",
    9: "z^2 + y^2*z - z + x^3 + 2 = 0",
}


def parse_decimal_int(value: int | str) -> int:
    if isinstance(value, bool):
        raise TypeError("booleans are not accepted as integer coordinates")
    if isinstance(value, int):
        return value
    if isinstance(value, str):
        return int(value.strip(), 10)
    raise TypeError(f"unsupported coordinate type: {type(value).__name__}")


def residual(eq_id: int, x: int, y: int, z: int) -> int:
    if eq_id == 1:
        return z*z + y*y*z + x**3 - 2
    if eq_id == 2:
        return z*z + y*y*z + x**3 - x - 1
    if eq_id == 3:
        return z*z + y*y*z + x**3 + x - 1
    if eq_id == 4:
        return z*z + y*y*z + x**3 + x + 1
    if eq_id == 5:
        return z*z + y*y*z + x**3 - 3
    if eq_id == 6:
        return z*z + y*y*z + x**3 + 3
    if eq_id == 7:
        return z*z + y*y*z + x**3 - x - 2
    if eq_id == 8:
        return z*z + y*y*z + x**3 - x + 2
    if eq_id == 9:
        return z*z + y*y*z - z + x**3 + 2
    raise ValueError(f"unknown equation id: {eq_id}")


def collect_exact_solutions(raw_solutions: object, eq_id: int) -> list[tuple[int, int, int]]:
    if not isinstance(raw_solutions, list):
        return []
    exact = set()
    for item in raw_solutions:
        if not isinstance(item, dict):
            continue
        try:
            x = parse_decimal_int(item["x"])
            y = parse_decimal_int(item["y"])
            z = parse_decimal_int(item["z"])
        except (KeyError, TypeError, ValueError):
            continue
        if residual(eq_id, x, y, z) == 0:
            exact.add((x, y, z))
    return sorted(exact)


def log10_abs_x(x: int) -> float:
    return math.log10(abs(x)) if x else -1.0


def verify_payload(payload: dict, expected_solved: tuple[int, ...] | None = None) -> dict[str, object]:
    solutions_by_equation = payload.get("solutions_by_equation")
    assert isinstance(solutions_by_equation, dict), "payload must contain a solutions_by_equation object"

    solved_equations = []
    total_large_x = 0
    best_equation_large_x_count = 0
    best_equation_max_log10_abs_x = -1.0

    for eq_id in range(1, 10):
        raw = solutions_by_equation.get(str(eq_id), solutions_by_equation.get(eq_id, []))
        exact = collect_exact_solutions(raw, eq_id)
        large = [(x, y, z) for x, y, z in exact if abs(x) > X_THRESHOLD]
        distinct_large_x = {x for x, _, _ in large}
        solved = len(large) >= TARGET_SOLUTIONS_PER_EQUATION and len(distinct_large_x) >= TARGET_SOLUTIONS_PER_EQUATION

        if solved:
            solved_equations.append(eq_id)
        total_large_x += len(large)
        best_equation_large_x_count = max(best_equation_large_x_count, len(large))
        best_equation_max_log10_abs_x = max(
            best_equation_max_log10_abs_x,
            max((log10_abs_x(x) for x, _, _ in exact), default=-1.0),
        )

        if solved:
            assert len(set(large)) == len(large), f"equation {eq_id} has duplicate exact triples"
            print(
                f"Equation {eq_id} verified: {len(large)} large-x solutions, "
                f"min log10(|x|)={min(log10_abs_x(x) for x, _, _ in large):.6f}"
            )

    if expected_solved is not None:
        assert solved_equations == list(expected_solved), (
            f"expected solved equations {expected_solved}, got {solved_equations}"
        )

    metrics = {
        "solved_equations": solved_equations,
        "total_large_x_solutions": total_large_x,
        "best_equation_large_x_count": best_equation_large_x_count,
        "best_equation_max_log10_abs_x": best_equation_max_log10_abs_x,
    }
    print("Verification summary:", metrics)
    return metrics


def require_int(value: Fraction | int) -> int:
    if isinstance(value, Fraction):
        assert value.denominator == 1, f"nonintegral rational with denominator {value.denominator}"
        return value.numerator
    return int(value)


In [3]:
#@title Data
solutions_by_equation = json.loads(r'''
{
  "1": [],
  "2": [],
  "3": [
    {
      "x": "-495125438518540812963667595840878080096011885727262104889200022010606040934980000025475239137864128400018870549478594920000008736367778361000000002311209612800000000000267501356828",
      "y": "39263753900880000000181776638430000000000336623404500000000000311688337500000000000144300219800000000000026722310",
      "z": "-348395759284180599521141430648046451105767163041032532862102927587077431182873371372468375438266900357825342061189050662544946949763680202182999178158449104805648043730074644272427743939037596647404815446182970669494432052062486521796914922913902638000138710630049066659"
    },
    {
      "x": "-495125438518540813003277630922361345134454968663975484772244164130064649482059853908257828053319364717124031409182510255584288690423048830193622911786899723238500549810084065826588",
      "y": "39263753900880000002144964333474000000046871442842580000000512113912539300000002797659337083800000006113403737146",
      "z": "-348395759284180599562948921762148123050564836090153234440649040715293059682061525970090127001005068449973107492844330453682810720141347893385949539184599174255135970287736056990745298705899920560550941206774044904758393614166814756325731094931076145037792516058222729763"
    },
    {
      "x": "-495125438518540813042887666003844610175670754056392693208046072299397175678703125872646269380229471656830967220792138388076101787826019726839820079486106377629405084930200984156588",
      "y": "39263753900880000004108152028518000000171933770082420000003597873336909900000037644415469583800000157548849929322",
      "z": "-348395759284180599604756412876249794999961333161825119933325586789535517534803852995574411061644420417250143371822577083528876714475042415904765808495959920618799166964315264850969897047946948499942174566594295097416660732317882122943697490904514651093484603120252611947"
    }
  ],
  "4": [
    {
      "x": "-108000000000000000000000000000144000000000000000000000000000096000000000000000000000000000032000000000000000000000000000006",
      "y": "36000000000000000000000000000036000000000000000000000000000014000000000000000000000000000002",
      "z": "648000000000000000000000000001296000000000000000000000000001368000000000000000000000000000864000000000000000000000000000362000000000000000000000000000092000000000000000000000000000013"
    },
    {
      "x": "-108000000000000000000000000000576000000000000000000000000001176000000000000000000000000001088000000000000000000000000000386",
      "y": "36000000000000000000000000000144000000000000000000000000000194000000000000000000000000000088",
      "z": "648000000000000000000000000005184000000000000000000000000017568000000000000000000000000032256000000000000000000000000033842000000000000000000000000019248000000000000000000000000004643"
    },
    {
      "x": "-108000000000000000000000000001008000000000000000000000000003552000000000000000000000000005600000000000000000000000000003334",
      "y": "36000000000000000000000000000252000000000000000000000000000590000000000000000000000000000462",
      "z": "648000000000000000000000000009072000000000000000000000000053208000000000000000000000000167328000000000000000000000000297578000000000000000000000000283780000000000000000000000000113389"
    }
  ],
  "5": [],
  "6": [],
  "7": [],
  "8": [],
  "9": [
    {
      "x": "5400000000000000000000000000004320000000000000000000000000001224000000000000000000000000000144000000000000000000000000000004",
      "y": "900000000000000000000000000000540000000000000000000000000000102000000000000000000000000000006",
      "z": "-324000000000000000000000000000388800000000000000000000000000177120000000000000000000000000038016000000000000000000000000003816000000000000000000000000000144000000000000000000000000000002"
    },
    {
      "x": "5400000000000000000000000000025920000000000000000000000000046584000000000000000000000000037152000000000000000000000000011092",
      "y": "900000000000000000000000000003240000000000000000000000000003882000000000000000000000000001548",
      "z": "-324000000000000000000000000002332800000000000000000000000006981120000000000000000000000011114496000000000000000000000009928584000000000000000000000004718304000000000000000000000000931898"
    },
    {
      "x": "5400000000000000000000000000047520000000000000000000000000156744000000000000000000000000229680000000000000000000000000126148",
      "y": "900000000000000000000000000005940000000000000000000000000013062000000000000000000000000009570",
      "z": "-324000000000000000000000000004276800000000000000000000000023505120000000000000000000000068846976000000000000000000000113346792000000000000000000000099451440000000000000000000000036331202"
    }
  ]
}
''')
payload = {"solutions_by_equation": solutions_by_equation}


In [4]:
#@title Verification
metrics = verify_payload(payload, expected_solved=(3, 4, 9))
assert metrics["solved_equations"] == [3, 4, 9]
assert metrics["total_large_x_solutions"] == 9
assert metrics["best_equation_large_x_count"] == 3
print("Payload verification passed.")


Equation 3 verified: 3 large-x solutions, min log10(|x|)=179.694715
Equation 4 verified: 3 large-x solutions, min log10(|x|)=122.033424
Equation 9 verified: 3 large-x solutions, min log10(|x|)=123.732394
Verification summary: {'solved_equations': [3, 4, 9], 'total_large_x_solutions': 9, 'best_equation_large_x_count': 3, 'best_equation_max_log10_abs_x': 179.69471524005482}
Payload verification passed.


### Formula Families

The embedded triples above are generated by three exact integer/rational families.

Equation 3 uses an internal parameter $a = 155 + 1674t$ and $q = (a^4 + 1271) / 837$:

$$
X_3(t) = q - \frac{9q^2}{16}
$$

$$
Y_3(t) = \frac{aq}{4}
$$

$$
Z_3(t) = \frac{2 - q + 5q^2/16 - 27q^3/32 - Y_3(t)^2}{2}
$$

Equation 4 uses $p = 6t + 2$ and $n = (p^2 + 2) / 6$:

$$
X_4(t) = -3n^2 - 2n - 1
$$

$$
Y_4(t) = np
$$

$$
Z_4(t) = 3n^3 + 5n^2 + 4n + 1
$$

Equation 9 uses $s = 5t + 1$ and $n = (6s^2 - 1) / 5$:

$$
X_9(t) = 6n^2 - 2
$$

$$
Y_9(t) = 6ns
$$

$$
Z_9(t) = -2(6n^3 - 6n^2 + 1)
$$

In [5]:
#@title Formula generators
def eq3_family(t: int) -> tuple[int, int, int]:
    a = 155 + 1674 * int(t)
    q = Fraction(a**4 + 1271, 837)
    x = q - Fraction(9, 16) * q**2
    y = Fraction(a, 4) * q
    w = Fraction(2) - q + Fraction(5, 16) * q**2 - Fraction(27, 32) * q**3
    z = (w - y**2) / 2
    return tuple(map(require_int, (x, y, z)))


def eq4_family(t: int) -> tuple[int, int, int]:
    p = 6 * int(t) + 2
    n = (p**2 + 2) // 6
    assert 6 * n == p**2 + 2
    x = -3 * n**2 - 2 * n - 1
    y = n * p
    z = 3 * n**3 + 5 * n**2 + 4 * n + 1
    return x, y, z


def eq9_family(t: int) -> tuple[int, int, int]:
    s = 5 * int(t) + 1
    n = (6 * s**2 - 1) // 5
    assert 5 * n == 6 * s**2 - 1
    x = 6 * n**2 - 2
    y = 6 * n * s
    z = -2 * (6 * n**3 - 6 * n**2 + 1)
    return x, y, z


FAMILY_GENERATORS = {3: eq3_family, 4: eq4_family, 9: eq9_family}
DATA_PARAMETERS = {
    3: [10**20, 10**20 + 1, 10**20 + 2],
    4: [10**30, 10**30 + 1, 10**30 + 2],
    9: [10**30, 10**30 + 1, 10**30 + 2],
}


In [6]:
#@title Formula verification and enumeration
def verify_formula_family(eq_id: int, generator, parameters: list[int], require_large: bool = False) -> None:
    seen_x = set()
    for t in parameters:
        x, y, z = generator(t)
        assert residual(eq_id, x, y, z) == 0, f"equation {eq_id} failed at t={t}"
        if require_large:
            assert abs(x) > X_THRESHOLD, f"equation {eq_id} did not meet large-x threshold at t={t}"
        seen_x.add(x)
        print(
            f"eq {eq_id}, t={t}: residual=0, "
            f"digits(|x|)={len(str(abs(x)))}, large_x={abs(x) > X_THRESHOLD}"
        )
    assert len(seen_x) == len(parameters), f"equation {eq_id} repeated an x-value"


ENUMERATION_PARAMETERS = {
    3: [0, 1, 2, 3, 10, 10**6, 10**20, 10**20 + 1, 10**20 + 2],
    4: [0, 1, 2, 3, 10, 10**6, 10**30, 10**30 + 1, 10**30 + 2],
    9: [0, 1, 2, 3, 10, 10**6, 10**30, 10**30 + 1, 10**30 + 2],
}

for eq_id, params in ENUMERATION_PARAMETERS.items():
    verify_formula_family(eq_id, FAMILY_GENERATORS[eq_id], params, require_large=False)

for eq_id, params in DATA_PARAMETERS.items():
    verify_formula_family(eq_id, FAMILY_GENERATORS[eq_id], params, require_large=True)

for eq_id, params in DATA_PARAMETERS.items():
    generated = [FAMILY_GENERATORS[eq_id](t) for t in params]
    embedded = [
        (parse_decimal_int(row["x"]), parse_decimal_int(row["y"]), parse_decimal_int(row["z"]))
        for row in solutions_by_equation[str(eq_id)]
    ]
    assert generated == embedded, f"embedded data mismatch for equation {eq_id}"

print("Formula enumeration passed, and formula-generated triples match the embedded payload.")


eq 3, t=0: residual=0, digits(|x|)=12, large_x=False
eq 3, t=1: residual=0, digits(|x|)=21, large_x=False
eq 3, t=2: residual=0, digits(|x|)=23, large_x=False
eq 3, t=3: residual=0, digits(|x|)=24, large_x=False
eq 3, t=10: residual=0, digits(|x|)=28, large_x=False
eq 3, t=1000000: residual=0, digits(|x|)=68, large_x=True
eq 3, t=100000000000000000000: residual=0, digits(|x|)=180, large_x=True
eq 3, t=100000000000000000001: residual=0, digits(|x|)=180, large_x=True
eq 3, t=100000000000000000002: residual=0, digits(|x|)=180, large_x=True
eq 4, t=0: residual=0, digits(|x|)=1, large_x=False
eq 4, t=1: residual=0, digits(|x|)=3, large_x=False
eq 4, t=2: residual=0, digits(|x|)=4, large_x=False
eq 4, t=3: residual=0, digits(|x|)=5, large_x=False
eq 4, t=10: residual=0, digits(|x|)=7, large_x=False
eq 4, t=1000000: residual=0, digits(|x|)=27, large_x=False
eq 4, t=1000000000000000000000000000000: residual=0, digits(|x|)=123, large_x=True
eq 4, t=1000000000000000000000000000001: residual=0, d

## 2. Kissing Number in Dimension 11: 593 Points

Problem source: [AlphaEvolve](https://deepmind.google/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/)

The kissing problem asks how many spheres can be arranged tangent to a given sphere, assuming all spheres have the same size and their interiors do not overlap. The maximum such number in $d$ dimensions is called the $d$-dimensional kissing number.

For $d=11$, the best known lower bound was 592 ([Ganzhinov (2022)](https://www.arxiv.org/abs/2207.08266)), and AlphaEvolve improved this to 593.

Below, we use the same verification criterion as AlphaEvolve.

### Results

Although AlphaEvolve published the 593-point configuration for $d=11$, the method by which it was obtained was not disclosed. Open-source variants of AlphaEvolve have not been shown to reproduce this finding. We are also not aware of any AI system other than AlphaEvolve that has been able to reproduce it.

As such, Station is the first open-source AI system that can achieve the same kissing-number lower bound of 593 for $d=11$. In addition, our analysis shows that the configuration found by Station is looser than that of AlphaEvolve, meaning that it has more slack in the pairwise-distance constraints, which is better for this task.

We also note that, unlike the other problems where only the problem statement is given to the agent, this task requires more scaffolding. We had to set up a multi-GPU Ray optimization pipeline in which agents fill in the relevant parts of the code, such as the optimizer, loss function, and hyperparameter ranges.

### Method

The numerical search was carried out on the unit sphere $S^{10}$. In this normalization the kissing constraint is equivalent to $\langle u_i,u_j\rangle \le 1/2$ for every pair $i\ne j$, so the objective was to reduce the largest pairwise inner product. A smooth log-sum-exp surrogate $L_\alpha(u) = \alpha^{-1}\log \sum_{i<j}\exp(\alpha\langle u_i,u_j\rangle)$ was used, with $\alpha$ gradually increased so that the loss changed from a broad average over many close pairs into a sharp approximation of the worst contact.

The first stage used Adam with tangent-space updates on the sphere. For a gradient $g_i$ at a point $u_i$, the radial component was removed by replacing it with $g_i - \langle g_i,u_i\rangle u_i$, and all points were renormalized after each update. Small tangent-space noise was used early to escape shallow local minima. The search also used a state-aware basin-hopping move: it identified geometrically trapped points from their nearest-neighbor contacts, generated replacement candidates on $S^{10}$ from random and antipodal directions, relaxed those candidates by soft repulsion against the existing configuration, inserted the candidate that minimized the worst new contact, and reset nearby optimizer momentum.

The second stage was deterministic polishing. Starting from the best shell configurations found in the first stage, the procedure switched to `float64`, removed noise and basin-hopping replacements, raised the final $\alpha$ so that the loss concentrated almost entirely on active contacts, and ran tangent-projected Adam with a cosine learning-rate schedule decaying to zero. This produced a high-precision real configuration with positive slack. Finally, the real coordinates were multiplied by $10^6$ and rounded to integers; the exact arithmetic verifier below confirms that rounding preserved the kissing inequalities.


In [7]:
#@title verification functions
import itertools
import json
import math
import numpy as np


def compute_squared_norm(point: list[int]) -> int:
    """Returns the squared norm of an integer vector using exact computation."""
    return sum(pow(int(x), 2) for x in point)


def sphere_packing_exact_metrics(sphere_centers: np.ndarray) -> dict[str, object]:
    """Returns exact integer metrics used by the kissing-configuration verifier."""
    sphere_centers = np.around(sphere_centers).astype(np.int64)
    squared_norms = [compute_squared_norm(list(center)) for center in sphere_centers]
    min_squared_norm = min(squared_norms)
    max_squared_norm = max(squared_norms)

    min_squared_distance = None
    min_pair = None
    for i, (a, b) in enumerate(itertools.combinations(range(sphere_centers.shape[0]), 2)):
        d2 = compute_squared_norm(list(sphere_centers[a] - sphere_centers[b]))
        if min_squared_distance is None or d2 < min_squared_distance:
            min_squared_distance = d2
            min_pair = (a, b)

    return {
        "num_spheres": int(sphere_centers.shape[0]),
        "dimension": int(sphere_centers.shape[1]),
        "min_squared_norm": min_squared_norm,
        "max_squared_norm": max_squared_norm,
        "min_squared_distance": min_squared_distance,
        "integer_gap": min_squared_distance - max_squared_norm,
        "min_pair": min_pair,
        "global_max_norm_margin": math.sqrt(min_squared_distance / max_squared_norm) - 1.0,
        "min_scaled_squared_distance": min_squared_distance / max_squared_norm,
    }


def verify_sphere_packing(sphere_centers: np.ndarray) -> dict[str, object]:
    """Checks that after normalizing, the points correspond to a valid sphere packing for kissing numbers.

    Args:
      sphere_centers: the list of sphere centers, of shape [num_spheres, dimension].

    Raises:
      AssertionError: if the sphere packing is not a valid kissing configuration.
    """
    metrics = sphere_packing_exact_metrics(sphere_centers)
    assert metrics["min_squared_norm"] > 1e-6, "Verification failed because the set contains 0."
    assert metrics["min_squared_distance"] >= metrics["max_squared_norm"], (
        f"Verification failed because the minimum squared distance = {metrics['min_squared_distance']} "
        f"< {metrics['max_squared_norm']} = maximum squared norm."
    )
    return metrics


In [8]:
#@title Data
sphere_centers = np.array(json.loads(r'''
[
  [-714380, -10947, 74144, 544495, 112834, 210859, 10453, -346018, 98404, 12785, -26297],
  [-117283, -60646, 220061, -14703, -317090, -147797, 204968, 698589, -361954, 336895, 192316],
  [-478504, -548956, -106772, 151273, 490984, -306145, 159273, -29566, -252824, -55649, -85729],
  [389017, -46901, -101837, 181053, 359498, 544786, -335590, -155236, -72606, -406621, 264487],
  [-290664, 25754, -69231, -128136, 393107, 160188, -300410, 445643, -317372, 302182, 482249],
  [280967, 301700, -114236, -550327, 338870, -486186, -11078, -297981, -230888, -110058, 92620],
  [35360, 96501, -474839, -199506, -278009, 217852, 10171, 436516, 620717, -131672, 78341],
  [-79854, -605638, -282068, -48947, -285279, -447005, 395699, -260895, 110074, 81991, -142086],
  [-66063, -280713, -8311, -259201, 100470, -449533, 95727, -514792, 13921, -596676, -83771],
  [430308, 229370, 232893, 374874, 402326, 255938, -108294, -39249, -341754, 427094, -166161],
  [319874, -308914, 655120, -489398, 115478, -33445, 232076, 146689, -108696, -162933, 73271],
  [-302345, 437509, 11509, -244039, 108744, -25901, 279166, 401276, 444851, -271237, -366841],
  [216332, 9979, 117667, -211826, 352722, 373303, 44070, 613761, -438225, 140468, -200500],
  [-101817, -156535, 227526, 369141, -394129, -246736, -714888, -187683, -18591, 75059, 92800],
  [-405566, -215718, 140731, -251621, -683591, 52240, 102528, -386201, 83558, 67870, -254138],
  [11293, -151563, 80442, -122988, 131696, -324412, -157046, 504674, 559251, -486354, 63689],
  [-348461, 77188, 5052, -135608, 253227, 238421, -59729, -574306, 468735, 405378, 125680],
  [203833, -378033, -366309, 260873, 520039, -29786, 515895, 87196, 24083, 136277, 221530],
  [249644, -436450, 36973, 227286, 52629, -390835, -408561, -234385, 449360, -281030, -189422],
  [-211947, -404036, 47946, 36153, 103577, 117061, 247834, -802234, -76969, 63326, -221046],
  [-214634, -91963, -150268, -110064, -479231, -84011, 149295, -59823, -355548, -615348, -378320],
  [-492458, 89654, 96675, 68064, 65700, 223452, 412881, -305185, -583878, -271927, -52494],
  [96462, -212699, -398232, 347821, -139636, -599032, -275223, -79808, 131087, 420732, -105982],
  [58927, 78440, -29553, 94545, 213827, -222350, 581239, 93106, 221962, 231787, -660228],
  [546823, -236005, -183617, 16139, -129067, 425105, -210407, -51719, 490774, -236888, -264606],
  [122682, -76969, 500487, 448607, 210855, 543035, -19839, 202822, -144890, -286001, -208860],
  [-340771, 442242, 60872, 147179, 463676, -156319, -231241, 363920, 426745, 2829, 235521],
  [76089, 144485, -229942, 206000, 318092, 329214, 279135, 283333, -435814, -565956, -5242],
  [505604, -135114, 227977, -257586, -203939, 449129, -240770, 130488, -338805, -381657, -170379],
  [-628376, 263177, 452071, -21549, -110216, 346952, -189776, -18674, -256565, 23952, 309455],
  [40528, -128345, 294158, 108274, -683917, -33973, -159499, 6179, 589212, -200539, 43257],
  [-85464, -159350, 221859, -217613, -304688, -326294, -293285, -275241, 426873, 572146, -8876],
  [98163, 490414, 348701, 451798, 155539, -153569, 89263, 244314, 63678, -551808, -12822],
  [364497, 189281, -698272, -76798, 365456, 40052, 164433, -201592, -5067, -314547, 189811],
  [-9375, -244876, -137249, 271409, -152737, -189230, 336442, 34554, -807321, 110813, 99382],
  [52240, 363907, -535101, -52657, -190320, 428193, -358516, -113113, 82329, -445334, -98683],
  [9251, -189099, -67787, 125541, 378337, 512910, -402383, 548721, 269265, 44192, -11000],
  [-268422, -341159, -289879, -299795, 404030, 110350, -393482, 238023, -264250, 214328, -367408],
  [-499218, -341900, -407641, -75798, 28952, -207716, -432925, -397348, 199293, 56427, -172552],
  [75563, -134010, 131968, 133328, -579921, 210847, 400368, 51257, -46882, 601765, -182026],
  [-78011, -142462, -617681, 213966, 105990, -233042, 177423, -536385, -255384, 219754, 219239],
  [122715, -5025, -253763, -342096, -675657, -511, -351874, 317704, -109625, -317527, 96878],
  [-258539, -112316, -210799, -416167, -2885, -350334, 53952, -317078, -533968, 178447, -399694],
  [319119, -94594, 77142, -592524, -261185, 521405, 361337, 100704, -69381, 209674, 51151],
  [213599, 106695, 199495, 476697, 269466, -472616, 336976, 170323, 443825, 128213, 154902],
  [38528, -181568, -628594, 397601, 307349, -166536, -108389, 302328, -378159, -47083, -204361],
  [-469392, -410461, -391739, 34708, 98639, 177696, 381322, -271386, 326156, -39139, 297080],
  [510752, 509694, 52470, -47116, -187816, -527338, 203841, -117517, 152142, 199304, -206832],
  [220032, 8795, 217636, -467892, 542997, -104813, -509329, 324635, 60252, 101155, 26861],
  [760047, -207381, -191018, 118770, 203130, 98672, 148116, 12081, -497909, -79447, 37676],
  [265444, 424139, 226830, -4698, 42623, 216530, 498336, -216249, 287737, 515835, 74086],
  [-91714, 418727, -628021, 19592, -170180, 43359, 179327, -110366, -567685, -125787, -90513],
  [499763, -78179, -91017, -59705, -75084, -226328, -401961, 299605, 590666, 266822, 63229],
  [388417, 263673, -228443, 175937, 11139, -409506, -208343, -262696, -584283, 260951, -82169],
  [360228, -127005, 125442, -480126, 229578, 181010, -268739, -219164, -523907, 326267, -145589],
  [-155389, -2300, -813017, 2870, -1138, -133110, 456543, 146069, 155700, 112935, -174240],
  [551438, -109544, -426099, -339798, -241728, 329432, -238278, -258099, -122676, -3561, 285449],
  [-204892, -626798, 271223, -10182, 149557, 16556, -18037, 644048, -78827, -84824, -200761],
  [-161888, -287035, -279362, 70485, 15708, 220041, -399648, -91455, 525240, -470006, 307997],
  [-81990, 228115, -458896, -209267, 279379, 123823, 334925, -90682, 118194, 417704, 533511],
  [-280504, -345434, -333723, -150699, 61933, 422301, 264363, 528792, -22559, 366333, 38995],
  [-257933, -180228, 78822, 429580, 32776, -405860, 532072, -251783, -53815, -174036, 405906],
  [-159044, -114812, -348696, 208677, -133796, 538677, -221985, -607036, -20513, 6386, 264730],
  [-171493, 221257, -309334, 469619, 102261, 152212, -422813, 347297, -81273, -382611, 345539],
  [-6107, 158969, -76425, 128386, -137881, 322380, 164451, -508369, -554432, 483068, -56523],
  [-130281, -53922, -260272, -95993, -94320, -446158, 43468, 5766, 755667, 54791, -345329],
  [655930, -13156, 4550, 295855, -604857, -225644, -171880, -157114, -48450, -89335, 26638],
  [-358549, 585035, -25086, 263014, 340304, 170703, -83337, -135496, -138129, 493527, 162701],
  [-367865, 144706, 279778, 365431, 637264, 54480, -74745, 72706, -340985, -226516, 210697],
  [112260, -31456, -110692, 731107, -333376, 74804, -190561, 31929, -349011, 368000, -168338],
  [-446078, 3704, -127480, -389358, 328720, -643815, -207070, 164411, 28651, -102686, -171234],
  [43818, -240650, -434836, -113649, 42643, 496711, 442783, -167405, -426950, -30045, 286983],
  [151682, 271990, 271194, -81903, -2548, -215800, 384694, 99480, -534973, 477292, -322946],
  [593653, 201326, 40665, 349717, 313484, 147935, 393812, -213355, 273523, -258052, -144628],
  [-86409, -39382, 241783, 577823, 135106, 431462, -438456, -50364, -68586, 226541, 378717],
  [-3470, 226695, 126731, -284864, 169007, 195564, -355939, -25131, 795671, -101976, -117324],
  [501468, 30350, 441464, -217249, 334813, 565285, -71261, -35642, 237660, 97891, -38224],
  [234876, 178100, -94033, -249579, -14670, 485028, 309092, -280281, 443113, -239593, 422687],
  [-537971, 308224, 213505, -595173, -250306, -86137, 254455, 28791, -184141, -205290, -63791],
  [306251, -292377, 290217, 343078, 230087, 277850, 345993, 329606, 238068, 418137, -169680],
  [-120797, -682232, 176061, -173465, 182359, -128001, 90628, 19131, 161290, -369018, 488336],
  [233941, -81469, 502646, 3449, 21210, 100377, 30076, -245927, 358525, -692030, 81329],
  [-623826, 34291, -164293, -29007, -27434, -222286, 454881, -298649, 170902, -216542, -399327],
  [-40364, -447600, 182660, -260660, 428610, 252984, -356705, -4323, 167272, -462674, -282320],
  [-417707, -182778, 156133, 73062, 478223, -213302, 16565, -242145, 595838, -272499, 3975],
  [-260064, 641058, -256958, -301795, 189840, -265672, 372214, -94591, -58796, 259025, -199101],
  [24451, 487908, 248961, 121705, 564689, -512701, -103941, 18222, -174924, 181850, -167375],
  [-309952, 167222, -47085, 13759, -118509, -182556, -316896, -123837, 376264, -652017, -379117],
  [-207016, 90986, -474725, -24461, -42118, -132240, -45651, 238608, -416766, 680952, -85059],
  [423014, -353553, 537462, 231889, -206643, 45033, 28518, 185389, -466184, 210996, -107993],
  [213794, -31159, 262148, -124492, 70183, 109752, 425007, 250007, -72070, -515502, -581430],
  [-96125, 91717, -225281, 136791, 406052, 221816, 349815, 123812, 723727, 190931, -29485],
  [-334177, 54743, -194172, -563567, -167436, 139841, 446623, -50760, 454169, 262720, -71327],
  [-120422, 187314, 32483, -605055, 198038, 248405, 351959, -538573, -207406, 116539, 108661],
  [371148, 88017, 182419, 369866, 4883, 264699, -43897, 342827, 554727, -215954, 374859],
  [67141, -139651, -604963, 449669, 56402, 534054, 147132, 116251, 210057, -187835, 69602],
  [661, -264713, 129223, -449482, -351909, 32901, -62223, 353584, -337615, 215394, -544976],
  [-196221, 135848, 658596, -19491, 7753, -62207, 98470, 320690, -555617, -246542, -151500],
  [6597, 260565, -559992, 31732, 76605, -534745, -52231, -363242, 25688, -187524, -393885],
  [-305721, -309050, 479303, -375948, -48897, -294366, 57830, 60499, 309226, -220958, -446615],
  [-289541, -122519, -427278, 235660, 702858, 152593, -285798, -120651, 90219, 122354, 162515],
  [248776, -439405, 42807, -267008, 212290, -267940, 420869, -394744, -357696, 51546, 302476],
  [43832, -78266, -99850, 140705, -222026, 220012, 708196, 279233, 285716, 132940, 430888],
  [-182847, -80713, 194901, 54115, -158162, 297010, -616590, 479661, -427089, -23110, -112744],
  [34241, 252993, -194030, 244620, 292423, -62070, -720314, 258973, -92150, 363319, -146834],
  [-5172, -66575, -26987, 583239, 374553, -340399, -38566, 20070, -303203, 439605, 333780],
  [150610, -438644, 298536, -291644, -5538, 509461, -4680, -452732, -138859, -149677, 323224],
  [-478782, -304827, -274391, -429794, -332701, -229265, 32036, 84730, 294876, -389746, 86740],
  [-519251, -449886, -87982, 63740, 224936, 517203, -158535, 130820, -249464, -253327, 171547],
  [357402, -479744, -94194, 524160, -169710, 155004, 222968, 134750, -66748, -205897, -436971],
  [-394787, -320131, 729175, 54786, 228530, 177732, 242313, -199131, -10486, -150041, 46428],
  [-38277, 350194, 189278, 197315, 17547, -35513, 133821, 311682, -34424, 29715, 826104],
  [-67368, -295073, -537770, 90341, 238428, -557261, 19375, -31102, 303287, -265991, 282083],
  [164067, 121975, 352702, -203541, 127616, -540788, 229286, 603288, 25279, -9802, -257633],
  [-273460, -386247, 451458, 299986, -77760, -165038, -142618, 135813, -419926, -30929, 482559],
  [-2021, 261026, -145439, 457721, 351934, 7849, 77689, -336787, 379504, -202433, 522320],
  [132843, -110678, 333651, -36598, 236944, 125355, -213281, 505327, -204276, -534850, 396165],
  [50859, -304898, 545893, 326663, -226810, -150055, 403573, 354423, 154706, -313839, 121933],
  [34756, -404810, -447662, -240052, -133505, -384638, -89084, 603325, 25873, 156531, -118393],
  [-220203, 33583, 261744, -233825, -198529, 796, -359895, -728018, -224204, 190214, 204452],
  [-223666, -103206, -23859, 474426, 49417, -11879, -404918, -39449, 680633, 285917, 23787],
  [21384, 37717, 61143, -164186, 299370, -277075, -684960, -290627, -215357, -170395, -414710],
  [369065, -625992, 10129, 338948, 13708, -516746, 68765, 143718, -149922, -65261, 193916],
  [173760, 112110, 357281, 365612, 606586, 47503, -444929, -129736, 298927, -105838, -101792],
  [-76341, 141382, 18772, -82365, -143579, -176362, 80886, -13320, 744383, 174644, 569212],
  [-10336, -377265, 227364, -345764, 399362, 262148, 447752, -80885, 286500, 310749, 269134],
  [-573323, -37883, 91774, -296295, -6155, 309669, -47225, -501112, 148043, -443316, 77012],
  [141182, 311453, 128197, -500170, -180697, -244413, 551969, 57598, -5482, 126799, 447187],
  [-259542, -495187, -333982, 641108, -117638, -53360, -189770, -57708, 1265, -16264, 329499],
  [-389710, -146319, -19152, -538425, -115663, 316471, 171289, 61081, -34890, -23409, 622962],
  [-5365, -102966, -6726, -198144, -541173, -681907, -262327, -144479, 58926, -157806, -272275],
  [-263956, -313284, 481477, -278534, -457613, 173896, -97971, 112724, -112865, -473372, 154939],
  [-682061, 590560, -131708, 159414, 60880, -73059, -63569, 96424, -164825, -282883, -117042],
  [-12431, 92011, 38703, -204615, 296118, -342479, 52641, -426865, -171507, 717556, 117264],
  [-177509, 420286, -529721, 43362, 335081, -425328, -211125, 62901, -232126, 57700, 332327],
  [-306612, 22157, 173187, -59230, 494581, 492725, -302821, -407084, -251696, 51331, -247394],
  [-59425, -437088, 441637, 716944, 116497, -133771, -89340, -98559, 53764, 112419, -177991],
  [-62840, -528419, -26967, 203037, -43606, 37589, -407284, 357946, -57096, 603582, 99911],
  [200746, -109178, 229084, 449971, 454246, -227487, 374898, 275876, -382301, -70657, -258719],
  [-374159, 242010, -153976, 38294, 454638, -78538, -224357, -95501, 258368, 187998, -633912],
  [-77566, -168306, -25192, -719745, 244622, -186241, 198793, 292971, -228402, 414566, 55490],
  [310686, -375277, 290817, 158462, -324992, -407883, 100745, 177758, 307237, 252044, -425792],
  [-192309, -460595, -65937, -184503, -311818, 162709, -532025, 285996, 348355, -11185, -320086],
  [-82746, 77572, -254820, -115886, 666321, 41939, 189232, -62972, -619819, 197190, -14020],
  [41435, -340759, -184524, -211930, -19697, 83456, -140586, -288390, 62653, -40943, -829045],
  [120231, -55812, 250081, -107340, -441269, -234213, -312288, -146862, -700940, -209629, 66636],
  [-70505, -212517, -310249, -418823, 275126, -44474, -200404, -526241, 78103, -95532, 517888],
  [235106, -10634, -500997, 256004, 15749, -193024, -595402, -361009, -50723, -215242, 238404],
  [199607, -16327, -313938, 336604, -489446, 333600, -484877, 117173, 244329, 117557, 273557],
  [377168, -328243, -199241, 263500, -139000, 24163, 240658, -537867, 95771, -413214, 306319],
  [135198, -484351, 290621, 136151, 55595, 26987, 190583, -110333, -499291, -585250, 2071],
  [369708, 496354, 162819, 97188, 313073, 45828, -37979, 398933, 393752, 233108, -333029],
  [-54300, 424223, -210254, 282827, -362113, -289983, 394934, -1672, -31390, 489338, 285063],
  [-544574, 334367, -146126, 244536, -144116, 222220, 378191, 206524, 168919, 371516, -296813],
  [403793, -79729, 93737, 486673, -102439, -238770, 430527, -466202, -68416, 330235, -31907],
  [-166367, 345494, 24008, 146018, -287940, -197235, 689835, 300264, -220913, -302308, 54992],
  [58470, 448057, 444421, 205762, 165359, 443568, 92951, -527478, -967, -154044, 145875],
  [65348, 471333, -56335, -274275, 769805, 179530, -13056, -175643, 192382, 47964, 3414],
  [21090, 126025, 19312, 214869, 521360, 674904, 285629, 132227, -43932, 147497, 295069],
  [-435600, 14370, -47675, 33491, -217995, 308824, -501896, -125782, -66796, 531425, -330398],
  [415206, 456289, -182288, -192059, 157969, 605724, 115891, -150700, -324981, -32070, -121042],
  [-142306, 449924, -291786, 300245, -4562, -512870, 16400, 446395, 146149, 144418, -311804],
  [-184791, -128487, -366216, -377980, -592853, -42856, 429030, 138315, -309185, 113451, 85937],
  [285899, -283390, 141310, -537868, -117885, 285512, -289962, 241721, 355725, -193618, 355995],
  [440955, -53389, -244911, 122763, 95972, -80105, -181775, -104940, -129937, -725009, -354194],
  [323854, -213238, -318860, -418068, -579375, -32435, 8105, -35075, 300006, 259521, -278958],
  [419894, -117567, -179177, -7466, 445524, 333577, 100854, -569122, -21007, 320195, 175795],
  [-48783, 166948, 619627, -429188, -79863, -542456, -120103, -129975, -192405, 175530, -42696],
  [-280820, -301631, 114168, 550456, -338951, 486212, 11170, 297890, 230915, 110012, -92442],
  [-71676, -194520, -10103, -73661, 655535, 236989, 341415, -361630, -55146, -430856, 172012],
  [116571, 199816, 7803, 24981, -70645, 734533, 293632, 332170, 289641, -35173, -345811],
  [-200609, -10826, 350966, -155376, 471017, 357434, -250795, -80710, 278616, -139993, 544120],
  [-205725, 254989, 538360, 269286, 94923, -315866, -15392, -374574, 168624, 267366, 425521],
  [85190, 309118, 23649, 279335, -124206, 441193, -68119, 500414, 3238, 584087, 110911],
  [286898, -267216, -549829, -232390, -194806, -436711, -3994, -88954, -420855, -267748, 67571],
  [-99247, 117053, -523997, 4358, -416302, -42145, 186114, -85442, 126568, -276187, 626526],
  [379128, 130181, 10295, 526509, 129830, -312043, -187377, -52579, 24916, 31036, -638993],
  [248189, -395707, -462985, -506496, -18042, -34710, 375265, 72010, 230393, -53851, 327725],
  [-321979, -145049, -123818, -51852, 101645, -36194, 759340, -168172, -212857, 440929, -32003],
  [422203, 416193, -297770, -89377, 299366, -212587, 181830, 263995, -309631, -122879, -450924],
  [520400, 99912, -220235, 124353, -14245, -399668, 64581, -205179, 22258, 146891, 653418],
  [561646, 257769, 423016, -153992, -199695, -12485, 94680, -405718, -318310, 181166, 260201],
  [48248, -117304, -12450, -86650, -406981, 87075, 348373, -542420, 107347, 345707, 506373],
  [273655, 106406, 82624, -224299, -299364, 495150, -9629, -635277, 183660, 178926, -229012],
  [-305409, 293749, -289570, -342108, -231185, -278167, -344700, -330242, -237251, -418741, 170924],
  [589506, 13274, 178051, -204330, 565930, -209435, 338161, 29858, 9343, 306519, -73675],
  [112615, 116510, -265386, -58089, 110147, -291501, 627869, -474556, 415820, -7259, 102230],
  [19958, -45773, -24256, 106750, 95949, 168974, -112657, 12467, -729534, -151666, -616194],
  [152836, 166520, -263, 670642, -243100, 216722, -195770, -282565, 279683, -432983, -96016],
  [-399458, -111847, 148063, -52650, -142986, 733996, 365633, -242087, 25102, 225290, 18831],
  [-64617, -291728, -367550, -403268, 562031, -174799, 180683, -304144, 185221, 207050, -253457],
  [179894, -159220, -671373, 2174, 11723, 69218, -121569, -309120, 542004, 256995, 128311],
  [-49885, 152911, 90231, -79376, 57851, -650297, -196058, 353538, 97073, 505900, 323526],
  [56472, -135620, -456668, 230601, -20241, 202263, -139975, 328789, 205348, 427444, -568158],
  [114926, 261491, 352472, 398150, -579798, 174890, -135938, 380631, -145111, -191799, 218400],
  [436993, 261924, -108948, 289850, 638686, -66678, -53706, 356356, -53966, -92099, 301525],
  [163341, 217310, -230194, 60104, -305886, -78317, -583795, -317256, 393508, 111589, -401281],
  [127174, -578496, -120256, -344088, -73093, 186920, -153789, 331115, -519643, 34929, 267380],
  [-476354, 160024, -396594, -152317, -185996, 451532, 392920, 149388, 19795, -389117, -3111],
  [-431573, -548789, 311362, -61545, -75114, -5523, -354187, -207880, -357254, -27910, -330297],
  [174789, 168992, -492952, 280287, 332219, -76906, -340036, 171783, 496953, -250086, -220011],
  [-258053, 161897, -353516, 22150, -256380, -409419, -565841, 222703, 363365, -87368, 197328],
  [356165, -432334, -577408, 220485, -346388, 42007, 178928, 107170, -122334, 316718, 153935],
  [304874, -38988, 250157, -143811, 287755, -415755, -56202, 289749, -620599, 97364, 291342],
  [-18740, 20705, 159704, 475651, -15271, 421259, 180652, -295987, 51467, 255080, -617917],
  [135678, 61577, 264404, 101583, 87846, 443959, -35786, -9515, -750748, -58201, 352666],
  [-92737, -514179, 34540, 245794, -739627, -166068, -26090, 199273, -218069, -29279, -44538],
  [-95047, -287504, -384951, 555671, 234444, -132132, 186702, -320099, 346365, -4892, -348732],
  [439612, 324708, -85694, -297480, 268133, 274376, -20502, 339743, 246190, -529642, -36267],
  [-233909, -56805, -48130, 281064, 250932, -508389, 71655, 618257, -154022, -208186, 291342],
  [42887, -270014, -627076, 230532, -460633, -139117, -142825, -14055, 250797, -333106, -229984],
  [539088, -306652, -212690, 596268, 248999, 85625, -252889, -29560, 185206, 204576, 65316],
  [233929, -582936, 277238, 76666, 338486, -169509, 432316, -180728, 246295, -174162, -262419],
  [483948, 69392, 54981, 392581, -294277, 621903, 235089, -132579, -15049, 100232, 217618],
  [607157, 24090, -400559, -103371, 218242, 383308, 134237, 310082, 217904, 309102, 88856],
  [-545562, 238052, 184569, -14708, 127453, -425615, 212344, 50773, -489547, 235981, 266474],
  [192516, 1338, -187892, -98341, 135880, 103087, -86696, -668902, -572272, -322377, 50025],
  [311922, 130042, 115649, 40502, -88517, 40552, -774651, 176242, 203533, -434085, 17275],
  [-388561, -263880, 228354, -176033, -11013, 409611, 208158, 262750, 584135, -260861, 81996],
  [-190217, -256357, -124401, -23067, -479078, 53103, 455595, 345313, 303426, -119251, -465466],
  [118406, -178621, 494234, 251457, -331304, -139221, -281071, 57844, -85351, -444352, -479949],
  [-11506, 363989, 63762, -215791, -598489, 412285, 189859, -273154, -256858, -215773, 255707],
  [-330975, -137237, 732812, 119536, -416883, -55708, -111162, 169039, 37133, 286789, -137355],
  [-36844, -504990, -259715, -135307, -549638, 517779, 86751, -8362, 164546, -174171, 150547],
  [-82112, -90009, -598030, -379692, -256330, 111195, 201954, -548754, 38955, -213132, -128147],
  [310648, -166232, 47444, -13181, 117824, 182240, 317839, 123483, -375566, 651626, 380013],
  [-201066, -263269, -180975, 475892, 147223, 260675, -559395, -45740, -23923, -176706, -441134],
  [429149, 89894, -475492, -369264, -21520, -66348, 403360, -215617, -268510, 385158, -103933],
  [126497, -741874, -488843, 32742, 249583, 174759, -121621, -228813, -77754, -155790, -57324],
  [226195, 356971, -347194, -172432, 196789, 163581, -443878, 89260, 319251, -14571, 546368],
  [-235152, 10545, 501036, -255990, -15783, 193085, 595361, 360958, 50654, 215247, -238435],
  [-13227, -70212, 24888, -497504, -178163, 64470, 344418, 669131, 5413, -364121, 109060],
  [500952, -283917, -452385, 68377, 625, -6279, -223523, 410365, 93485, -369723, 308666],
  [-211164, 94043, -237209, -461474, -441442, 231777, -390049, -268167, 372966, 77650, 244058],
  [148033, -468484, 504116, -76445, -295965, 444180, 164397, -38155, 201379, -34853, -378801],
  [-478579, -179929, 238093, 235902, 224099, 240951, 55653, 236251, -485445, 435390, -183953],
  [437064, -259224, -94069, -123286, 569314, 6953, -47028, -69108, 543854, -170543, 248421],
  [-18183, 495588, -36751, 389537, 97665, -64277, -35747, 445607, -614878, 93598, 24963],
  [49661, -212594, -113558, 667535, 311015, 307927, 164277, -336941, -395821, -69417, 16786],
  [-295291, -230706, 407985, -618966, 84989, 380766, -265931, 58535, 72860, 259679, -104468],
  [269341, 227502, -510568, 341399, 120965, 316554, -153218, -27808, -357700, 273031, 396083],
  [-113003, -29034, 86973, -143441, 702692, -416937, 158193, -14950, 49874, 68894, 508088],
  [-72044, -97573, 18414, -108113, -196828, 228389, -600211, -83187, -234894, -222134, 640833],
  [98468, -680587, -210842, 209450, -38310, 4601, 119155, 267616, 591049, 12145, 44619],
  [355763, 159968, -220682, 461450, -368220, 13135, 218318, 24219, 565044, 244755, -151720],
  [-31928, 189537, -98917, -88843, 539669, -216052, -351473, -85449, 81122, -641512, 241877],
  [24007, 137156, 269862, 363668, -211750, 69109, 127027, 568985, -125346, 134664, -593458],
  [-211377, 426310, -69520, 512693, -239743, -65112, -42422, -326366, -355080, -154397, 431210],
  [189781, -5093, -359254, 143471, -457144, -352718, 234975, 88978, -288400, 147258, -559402],
  [186115, 309243, 144059, -569916, 116067, 291139, -430021, -358182, -8609, -314012, 118524],
  [-337, 50289, -36112, 482735, 195024, -58011, -364161, -658112, -18948, 373238, -127844],
  [-329586, 215274, -178880, 485496, 176835, -262128, 222733, -203475, -396088, 225540, -422585],
  [52702, 772303, -140779, -260381, 100480, -225312, -401215, 81333, 145572, -175670, -180697],
  [-349274, 657234, 5205, -315687, -39507, 509632, -38938, -158791, 168636, 51176, -165204],
  [-369197, 185998, -523114, -338485, -280075, -100241, -250531, 178661, 2352, 42300, -505873],
  [-193774, 34176, -159670, 324349, -372025, -376560, -62857, -597194, 389934, -58094, 185396],
  [-22819, 284380, 450877, 148845, -82796, -501763, -398457, 141606, 444960, 6326, -240191],
  [78335, 142955, 617965, -213559, -106451, 232983, -176911, 536103, 255680, -219984, -218763],
  [-122693, 76827, -500610, -448655, -210778, -542952, 19703, -202788, 144766, 286092, 208809],
  [262001, 357520, 287536, 292788, -387363, -124611, 405476, -232872, 268000, -209859, 367492],
  [-419141, 118635, 179597, 8109, -446294, -333929, -99828, 568717, 21765, -320625, -174813],
  [-174014, -233045, 222009, -71768, 319420, 82770, 568583, 325513, -403395, -104497, 386263],
  [-231667, 370971, 301869, 28103, 36320, -385904, -559403, 225517, -284177, -199527, 286702],
  [41496, -173489, 510856, -97092, 411045, 4197, -153871, 89218, -121011, 235293, -653574],
  [-509593, 171259, -241934, 130812, -5025, 529942, -132231, 149624, 351209, 160930, 406645],
  [-451169, 38485, 236958, -134020, -83116, 84304, 167079, 112797, 120733, 732206, 339853],
  [531800, -305689, 310403, 237017, 121791, 196571, -189287, -506924, -112068, -120110, -312979],
  [624948, -32720, 165111, 30120, 26106, 221783, -453299, 297877, -169839, 215843, 400859],
  [-264716, -326144, -47673, 71863, -320361, 66032, -173163, 595950, 176945, -143201, 521698],
  [454891, 548957, -54295, -28679, -199799, -130121, -73277, 60682, -309837, -444179, 358823],
  [66056, -177525, -252252, -399152, -39137, 230143, 11551, -55368, -151106, -759092, 290546],
  [-759265, 208468, 191466, -118095, -203945, -99036, -147059, -12523, 498683, 78997, -36654],
  [-349724, -587098, 103553, 287851, -234767, 37745, 102590, -214775, 242329, -484708, -180467],
  [-312968, 318991, -650002, 496711, -123913, 30413, -222180, -151680, 115199, 158522, -63406],
  [-85561, 21138, 265432, -9816, -234269, 773632, -406028, -67404, 230553, -210262, 39063],
  [42123, -85198, 90026, 355220, -294954, -374975, -194158, 8346, 54771, -625240, 444280],
  [206579, -212541, -589362, -271864, -63772, 351358, 74573, 319465, -238616, -208203, -394297],
  [-482614, -63826, 176121, -56376, 199034, -100394, 253872, 200536, -25811, -757433, 9144],
  [-204490, 269086, -294155, 237169, 639162, -280054, 371764, 77079, 99754, -293338, -126071],
  [-414468, 365122, -530659, -223030, 196099, -48626, -16397, -191890, 473754, -216423, 119869],
  [255381, 140125, -471866, -83177, 37967, -111306, 381048, 491505, -360248, 17225, 393560],
  [-334669, 190601, 510953, 176945, 261329, 462443, -69902, 128907, 376888, 306752, -139624],
  [12254, 232633, -567964, -89695, -393195, 548234, 147385, -55789, 51287, 363950, -4034],
  [-104537, 211075, -100910, -283511, -258497, 305574, 504107, -158265, -231260, 65944, -597294],
  [153449, 134929, -226911, -116929, -166500, 670480, -53513, 442928, -144713, -177166, 405515],
  [67376, 294960, 537675, -90356, -238383, 557331, -19476, 31128, -303374, 266078, -282090],
  [-414363, 5253, 78931, -212893, -321008, -530719, 293607, 177765, 48263, 423292, -304470],
  [411944, 174283, -160641, -79510, -471038, 215575, -24993, 246618, -601197, 276434, -12298],
  [71551, 74024, 589694, 367860, 270304, -106817, -217813, 557332, -48833, 220687, 112778],
  [317471, -416692, 797, 259875, -127504, 19646, -257548, -412818, -431202, 261384, 387808],
  [-415165, -288754, 110690, 326802, -303569, -286740, 57784, -362963, -223553, 510291, 73346],
  [-446916, -115087, 461447, 349607, 43482, 73925, -428284, 228802, 252028, -373042, 77789],
  [260589, 64652, -120375, 78048, -498758, -490013, 217185, 470905, 261459, -78260, 273959],
  [437648, -284586, 43033, 10468, -456861, 273750, 6669, 640838, 136267, 94509, -60281],
  [-112574, 541531, -211965, -2373, -697935, -171687, 124395, -53559, 251737, -160259, -158589],
  [-583134, -3325, -173129, 211685, -573966, 206839, -329028, -34618, -3495, -310799, 82908],
  [133474, -176941, -357367, 87021, 179093, -122421, 599135, -284580, -329452, -294183, -365378],
  [521435, 134875, -350958, -76215, -446622, 126228, 509485, 44770, 23025, -321120, -17364],
  [-7712, -131203, 240045, 352380, 116685, 96925, 23198, 208769, 610575, -231162, -554583],
  [759746, 261497, -154676, -90817, 314351, -10705, -398744, -190256, 72811, 20467, -149622],
  [-32260, -161070, -77602, 153228, -236664, 365283, -120269, 465655, 129977, -686656, -185426],
  [-61104, -378671, 526605, 41088, 202922, -425185, 343579, 120260, -90513, 451695, 84757],
  [97663, -194520, 55039, -115583, -214710, 508574, -40364, 59470, 613315, 494034, 78076],
  [270513, -626791, 265471, 312785, -202935, 261081, -357527, 86486, 68177, -265846, 213712],
  [-176678, 14342, -271791, 454931, -490447, 175559, 549438, -328578, -245, -69354, -41611],
  [-302857, -284728, -36975, -165926, 223068, -364084, 614430, 330041, 353699, -50707, 43200],
  [2226, -220537, -377938, -116495, 188842, 74609, 333202, -10599, 478191, -592468, -250802],
  [391671, 327463, -202930, -409433, -360875, -68269, -172688, -281250, -282816, -177253, -418723],
  [-150581, 116547, -484707, 317116, -478785, 135736, 64378, 591784, -153647, -11149, -49957],
  [128533, -529747, -75571, 339278, -428455, 139438, -157551, -412313, 264306, 275659, -195807],
  [85653, -21117, -265495, 9860, 234220, -773593, 406055, 67402, -230593, 210272, -38935],
  [-63969, 116629, -67359, -263069, 339603, 410819, 133576, -9827, -47035, 680107, -376515],
  [347919, 412548, 192751, 374367, -282737, 79946, 296197, -196044, -415754, -230877, -304630],
  [-8348, 411328, -157090, -348482, -290717, -545065, -40414, 315343, -446296, 40243, -33704],
  [-125960, 59121, 274353, -250292, 311991, 344423, 378314, -331593, 405176, -116144, -442746],
  [90963, 544066, -263464, 395206, 425522, 87075, -252875, -246944, -260377, -237765, -179471],
  [391220, 19676, -13751, -489280, 441042, 252249, 104776, 62005, -212565, -69571, 532899],
  [109986, -266946, -15844, -124252, -451042, 126557, -449305, -505683, 17745, -463543, -93380],
  [155638, -536567, -228788, -126152, -424579, -284271, -325152, -122475, 220727, 14266, 434614],
  [-270408, -456899, -260866, -50609, -10700, -336268, 219523, 170670, -154726, 334368, 566798],
  [4932, 66256, 26774, -583462, -374265, 340575, 38237, -19890, 302962, -439437, -334108],
  [-108541, -154523, -396688, -436427, 328618, 669559, -82688, -103054, 205321, -8938, -22803],
  [110538, -201652, -40148, 594050, -185606, -244283, -366121, 545881, 198314, -109881, -122691],
  [506213, -51758, 442268, -401875, -220499, -120383, -316483, -187937, 307060, -68647, -293895],
  [-169008, 222110, 225668, -112396, -666253, -332939, -131470, -73672, -21777, 81482, 523428],
  [-214074, 30794, -262374, 124244, -69863, -109564, -425376, -249814, 71809, 515684, 581054],
  [188430, 373533, -200077, -276388, -285908, 1064, 397154, 533879, 87749, 343670, -241318],
  [61142, 178974, 1576, 61927, -642280, -232558, -357068, 370040, 45406, 438373, -187418],
  [-64403, 358211, 30161, -548588, -437549, 18129, -12836, -164581, -166606, 558105, -86085],
  [-133978, -472102, -333612, -407042, -107759, 199082, -53885, -286866, -112530, 574667, 53314],
  [-373247, 468564, -38854, 55071, -189693, -507642, -358917, -273351, -87797, 341604, -122329],
  [549856, -184913, -64624, -528284, 214511, 160472, 222579, -342764, 18788, -293690, -234100],
  [128234, 182828, 390602, 425780, -277973, -637948, 123989, 95282, -308687, 34485, 105338],
  [164565, -8819, -268583, -378369, -355454, 58908, -110002, 370237, 69557, 485131, 488218],
  [-103923, 795450, 474461, 6, -215802, -112311, 143980, 161961, -19092, 144100, 63807],
  [-40484, 275075, -559150, -293444, 232135, 109979, -388319, -399212, -174655, 304172, -157381],
  [264485, 64802, -184710, -371713, 139546, -383322, -246897, 89393, 14455, 493525, -523857],
  [160055, 319304, -91917, -98007, -33377, -87537, -327991, 847032, 27302, -23608, 138755],
  [-279184, -371055, -228872, -400927, 264872, -42014, -262308, 215085, 471996, 297508, 270125],
  [198375, 68988, 138416, 92831, 499527, 90902, -172467, 72322, 341727, 626186, 355611],
  [349252, -430650, -54012, -138302, -474089, 152790, 243243, -370523, -419297, -8208, -223793],
  [-408310, -43765, 1034, 470246, -419277, -244761, -128943, -48703, 196978, 80175, -557052],
  [31468, 392822, -462093, -743651, -79669, 148282, 47766, 121434, -79057, -91905, 134517],
  [59179, 201283, -136548, -73777, 385785, -498913, -413267, -338928, 451480, 174607, 119021],
  [257674, -25887, 93825, 106135, -387068, -177780, 293141, -486871, 306781, -291380, -479006],
  [512377, 502078, 120595, -191154, -481102, 235918, -114753, 104036, 300674, 106393, 146531],
  [-352995, -37272, -296447, 86512, -224889, 440837, -19415, -251090, 576527, -61379, -366270],
  [-355655, 433131, 577801, -219884, 345714, -42190, -178168, -107566, 122852, -317061, -153203],
  [565448, 309971, 379100, 74824, -12359, 111177, 438394, 426849, -144390, -23130, 163050],
  [181294, -204366, -360528, 335134, -376514, 303617, -174417, -25977, -497356, -393229, 126640],
  [52737, -350034, 188212, 387134, 228479, 531068, 97515, -340106, 467682, -62608, 85645],
  [-464647, -408541, -474921, 171995, -199012, 198173, 97924, -65951, -320386, 113546, -392056],
  [117034, -316407, -72036, -118407, 207141, 297143, -772301, -256877, 143399, 222334, 57872],
  [186279, -210146, -207986, -101849, 15431, -266186, 370641, 124459, 385771, 702006, 43795],
  [-108709, -188824, 1455, -15708, 58540, -739101, -281048, -340175, -281706, 28569, 357351],
  [-204481, -111486, 345429, -351916, -72151, -325100, -510771, -226518, 339012, -298954, 274219],
  [-516865, -206272, -42126, -425083, -302450, -213528, -418687, 214968, -229588, 268976, 155543],
  [-412805, -238526, -393410, 91424, -537569, -53001, 17366, 102087, 367041, 402184, 102880],
  [-477699, -505055, 251298, 24266, -224923, 242236, -266977, -217955, 256010, 164890, 363599],
  [386502, 38973, 283227, -216424, -155458, 17057, 477474, 172451, 643540, -114360, -112432],
  [-292099, -59244, -195592, -480704, -230953, 458348, -282369, -148828, -497104, -97534, -141087],
  [-147830, -189840, -282360, 467584, 99981, 335392, 167894, -157466, 126133, 663361, 110996],
  [196337, 519344, 38647, 179863, 321386, -170155, 532476, -243341, -308025, -16511, 296228],
  [402291, 222771, 385260, -103307, 551367, 57335, -33005, -93968, -377016, -395091, -118266],
  [428116, 224030, -250762, -122397, -210230, -271620, -111681, -208912, 532692, -455262, 153458],
  [-386009, -26820, -233754, -173044, 141422, -168091, 342592, 523877, -416207, -63019, -385256],
  [289113, 176817, 800867, 181385, 53886, -163383, 263482, -166026, 127581, 30516, -258973],
  [-134714, -107139, 241690, 137060, 143100, -678575, 80254, -457282, 162550, 164426, -378452],
  [-70818, 166718, 338605, -209028, 23001, -137364, 131653, -397521, -194065, -419475, 632827],
  [-442546, 137975, -129215, -452772, 149695, 210220, -518038, 345567, 109998, -306568, 54336],
  [219261, 163501, -347807, -267827, 273736, 91593, -485208, 216595, -545713, -257772, 50799],
  [-175877, 212220, 364641, -329574, 369774, -305941, 182358, 21928, 502331, 389673, -119156],
  [-259848, 420644, -45492, -240310, -39163, 394337, 393435, 242601, -459351, 288774, 175055],
  [164669, -217825, 255674, 504689, -392300, -129222, 28515, 84820, 208512, 384452, 485253],
  [-44573, -761343, 147308, 268764, -110311, 221969, 412685, -87526, -138379, 170560, 191806],
  [-276124, 328930, -150044, 411484, -97503, 693047, -72620, 38927, -335470, -34365, -115747],
  [374509, -466553, 39800, -53652, 188102, 507133, 360823, 272419, 89005, -342488, 124167],
  [307435, 291308, 40539, 170759, -228682, 362231, -607848, -333317, -349421, 47763, -36846],
  [-143997, 161270, 348787, -98546, -165589, 126913, -614330, 292589, 319555, 301275, 349965],
  [-368892, 463269, 83793, -536400, 184942, -150582, -239474, -124766, 56315, 213254, 419658],
  [-161082, -189897, 106782, 174146, 29207, -546075, -303485, 359032, -432697, 229971, -368904],
  [-433181, -32503, 559035, -307941, 111123, -223496, 27777, 437493, 123057, -44225, 363841],
  [348517, -1493, -3587, -22999, 199078, -286725, 491864, 144076, 81021, -585967, 378865],
  [-232303, -194336, -783139, -127902, -77078, 179137, -358574, 167861, -109273, 25391, 266897],
  [-293880, 285240, -32153, 223187, 409768, 495328, -39465, 12864, 326990, -501015, -92221],
  [186224, -294245, -51489, 550880, 71068, 252245, 87377, 549784, -308532, 27643, 313192],
  [189089, -478836, -167300, 167268, 260068, -89176, 123013, -128079, -351412, 530852, -408136],
  [377868, -552141, 39600, -240079, -365417, -177874, 113445, 121351, 157019, -507792, -133708],
  [410234, 468855, 457906, -180447, 272062, -197621, -74380, -624, 303149, -129107, 377454],
  [38362, -203293, 580613, 16545, -129622, 520921, 109332, 321400, 7508, 159596, 436893],
  [355932, 172617, -115786, -98224, -305152, -198799, -158791, 416736, 291261, -274317, -573460],
  [280230, -128676, 370530, 5267, 221916, 401662, 594269, -239046, -338760, 71475, -163859],
  [-442762, -555390, 73676, 23801, 192498, 155388, 60666, -18178, 327896, 389078, -406693],
  [-554173, 73316, 418445, 151532, -269263, -403615, -52530, -328798, -157311, -343831, -14738],
  [20946, 8191, 432380, -25553, 676606, 96385, 282484, 370753, 277966, -223049, -23085],
  [-218003, -450015, -215856, -14382, 21956, -264935, -517048, 253320, -226953, -499470, 231],
  [358393, -134321, 87630, 52834, 272023, -752, -52821, -332970, 572190, 399965, -409893],
  [108595, 662268, -186965, 157913, -164855, 131300, -110914, -9032, -174222, 379213, -508685],
  [-196155, -312510, 376403, 207940, -239609, -178885, 490013, -116556, -291167, -8494, -500980],
  [106138, 125967, 245783, -515386, -43977, -314441, -229683, 193064, -164064, -631408, -174375],
  [-570002, 155424, 49394, 507591, -189893, -151703, -251597, 360060, -35778, 308714, 205552],
  [-268286, -163490, -122102, 203208, -461570, 387252, -4822, 85583, -375026, 337884, 468959],
  [-307271, -189715, 208134, -530347, 423084, -49170, -212690, -60692, -465610, -245955, 193339],
  [-148363, -156629, 408844, 426212, -443205, 364606, -46521, -395353, -324687, -106711, -16],
  [-617410, -206670, -401463, 193086, 171675, -55369, -135970, 461949, 267487, -86982, -184662],
  [-9294, 567229, -6765, -189574, -281690, 274802, 8068, 422548, -308449, -348444, -303561],
  [248368, 8151, -230804, 268174, 156057, -16114, 403509, 701321, 250718, -212047, -161096],
  [534346, 267173, -214958, 411857, -37992, 357728, -238073, 389455, -118995, -147450, -231500],
  [215827, -750575, 133286, -391688, 127082, -238163, -259335, 1820, 53857, 213260, -174243],
  [254046, 310526, 39406, -83373, 333546, -61617, 157799, -587655, -186529, 150246, -536540],
  [-72068, -512089, 277308, -372927, -450249, -94202, 282173, 233053, 278777, 223781, 208010],
  [-193359, -76007, -354693, -119233, 460803, 186972, 156819, 424621, 211083, -337097, 454343],
  [-436438, 286534, -42140, -9097, 455320, -274235, -4819, -641731, -135100, -95373, 62057],
  [678379, -259066, 305856, 118636, 261513, -76011, -154584, 384614, 124586, -224947, -230110],
  [227659, 207994, -163693, -424817, -76806, 461419, -504234, 274109, 17686, 234095, -307145],
  [347338, -431864, -14692, -160286, -72650, -130605, 643453, 360038, -216624, 139387, -185990],
  [318826, -190818, -146867, 233682, 654253, -403125, -67252, -382873, -152483, -127297, -68078],
  [-9960, 365928, 152954, -429367, 182887, 357958, 149510, 451241, 368778, 201994, 306421],
  [-290337, 410823, -275519, -133688, 298292, 400006, -68533, -192647, -287242, -267221, 456550],
  [-493855, 420981, 162194, -15843, 64966, 118762, 555479, -179758, 240312, -55366, 363749],
  [-230409, 467598, -28525, 287712, -235547, 261779, -393416, 381069, 374448, -64112, -276316],
  [424610, 364597, -699863, -19321, -271451, -192900, -196387, 171667, 38713, 127122, -1079],
  [739283, -451485, 151414, -81302, -129969, 52527, 170727, -132279, 222665, 233759, 222635],
  [-212505, 499703, 23494, -26217, -5345, -692700, 173697, -33913, 231084, -281027, 245642],
  [-16190, 199716, 365784, 101910, -170934, -68027, -353558, 20807, -491435, 601296, 230911],
  [-386353, -82690, -525971, 274928, 152567, 33074, -24805, -306009, -141591, -593907, -16168],
  [113049, 29036, -87090, 143443, -702715, 416998, -158180, 14905, -49898, -68811, -507990],
  [-374024, -100535, 62469, 670939, -129772, -24134, 57971, 235112, -378357, -387015, -164135],
  [-229167, -178989, 353084, 262927, -216032, -86587, 487929, -260751, 551840, 235699, -36504],
  [121518, 184174, -210131, -347180, 367062, 236975, 743805, 170688, 37022, -89299, -63300],
  [-69584, -224575, 207776, -248899, -288495, 60298, 719278, -262169, 76907, -379501, 129258],
  [111960, -384262, 643290, 3988, 143999, -51169, -147538, 95448, 587312, 110824, 120505],
  [214512, -255567, 302411, -226734, -651589, 275571, -357730, -84836, -90727, 286878, 140009],
  [-74268, 275331, -542442, -96762, -46353, -283536, 30947, 389348, -3133, -616148, -18204],
  [82332, -264479, 548876, 105143, 36425, 280249, -19564, -395513, 10288, 610984, 29421],
  [500890, -184060, 234895, -140935, 16156, -527156, 119694, -142845, -358165, -154816, -418399],
  [-137919, 193800, 448957, 597607, -55066, 175824, 494076, 53966, -160698, 232298, 154812],
  [-117968, 128794, 55953, 88392, 480240, -105063, -325724, 527070, -108254, -361894, -436231],
  [309963, -91296, 376248, 313200, -145428, 289244, -518404, 144308, 212200, 306236, -350347],
  [-291069, 280476, 482689, 174567, -376201, 257379, 321242, -35443, 216289, -370552, -277193],
  [114872, -261996, 400093, -353891, 33367, -145214, -178766, -109781, -28104, 345861, 668005],
  [313783, -253480, 47681, -199734, -435848, -502583, 69783, -28151, -308096, 487086, 121368],
  [-10951, -63417, 497358, 225479, 245535, -230202, 27364, -458385, -598937, 113940, -42405],
  [295956, 202132, -394309, 636524, -94825, -367321, 279045, 1162, -53632, -288078, 57100],
  [-118292, 543551, 84022, -328638, 415728, -144078, 171898, 404344, -255057, -282270, 210083],
  [60081, -336425, -83966, 182379, 602573, -350246, -150028, 354992, 263541, 281035, -245292],
  [84029, 271210, 376273, -568234, -220365, 136911, -203027, 328829, -356762, 12514, 332901],
  [-555872, 66314, -268192, 213491, 263469, -432525, 171803, -99741, 293011, 418801, 109067],
  [635843, -251147, -446399, 30204, 100676, -349776, 201068, 13085, 263515, -29244, -298446],
  [327699, -308154, 132461, -129146, 241746, -144341, 169773, 623588, 189203, 161022, 452486],
  [-168932, -58261, 141058, 355920, -102338, 440760, 287985, -21960, 3335, -526061, 506537],
  [58140, -281949, 58646, 228020, 319809, -282795, -574041, 196844, 185918, -30277, 525060],
  [721798, 22863, -68511, -535910, -122275, -213662, 720, 340462, -91456, -18018, 37210],
  [-142888, 140348, 358842, -403694, 203274, 623420, 205731, 118756, -175722, -386683, 36689],
  [91135, 46033, -238114, -572926, -140864, -433365, 445145, 46937, 72920, -229574, -372300],
  [128392, -38661, 794665, -33486, 37158, 148425, -500421, -124099, -182773, -93875, 134003],
  [-699013, -36519, -88131, 65206, 132264, -209057, -174949, -47444, 43825, -240125, 586848],
  [23797, 211975, -58169, -85200, -425530, -455192, 399513, -520394, -333138, -101608, 62093],
  [107034, 45414, -228404, 3326, 330405, 152222, -220143, -690891, 352615, -329954, -207274],
  [-347422, -463882, -140176, -70636, -344880, -57287, 71758, -419705, -373066, -250018, 366415],
  [-573007, -14953, 116635, -93952, 4132, 426754, -76830, 227552, -47772, -166080, -615894],
  [-7931, 348335, -243684, 324281, -375460, -253535, -477638, 97204, -304546, -297792, -299657],
  [-374719, -53560, -249992, 230768, 125101, 23278, -480454, -185800, -661366, 113710, 95303],
  [-164406, -337898, 224667, 305520, 250225, -13623, -359905, -556651, -65545, -362208, 278024],
  [-206104, -20437, 177182, 84960, -119514, -96940, 67064, 679602, 559885, 332107, -68464],
  [126587, -530720, 243639, 71061, 665001, 227856, -153542, 69466, -254770, 175898, 139678],
  [-485565, 88065, -426773, 426637, 193571, 112567, 348740, 172612, -286825, 53029, 325085],
  [-197192, -325650, -152869, 558078, -102118, -286607, 414178, 366833, -1461, 321687, -134469],
  [157483, -440346, -224918, 67137, 122153, 398689, -69003, 24944, 158068, 175324, 700869],
  [97712, 727867, -295067, 295588, -75760, 289668, 285220, 177625, 156941, -122359, 209940],
  [-878, 10412, 527729, -352492, -134921, 108563, 72743, -481969, -291519, -328904, -369332],
  [-399735, -434353, 194205, 208280, -176839, -612532, -93592, 138856, 339317, 21384, 142660],
  [-140477, 559804, 240557, 143281, 404942, 278776, 347822, 111071, -206476, -24918, -412584],
  [-95195, 293196, -384600, 376549, -58753, 138665, 208434, 95167, 46549, -359637, -639203],
  [233228, -388317, -205362, -338118, 549677, -231296, 101687, 330879, -211217, -328090, -102950],
  [-34313, -27778, -443086, 11586, -658616, -89883, -302245, -360199, -291036, 232212, 4267],
  [-201146, 382900, 203652, 312148, -518028, 290288, -75683, -279554, 298659, 362937, 128832],
  [-135525, -509817, -408399, 153777, -135428, -54598, 419107, 314665, -156535, -434019, 149733],
  [-135674, 226783, -5785, 94269, 482393, -111905, 408397, 524667, -41747, 478966, 52929],
  [338232, -92586, -13433, 124098, -239990, -234099, 44528, 582685, -478550, -398400, -140754],
  [7584, 131034, -240120, -352457, -116575, -96804, -23357, -208707, -610709, 231235, 554426],
  [-482175, -135736, 324212, -6684, 445522, -206718, -478754, -56396, 37209, 410260, 24869],
  [246347, 476229, 323113, -655991, 134889, 59863, 169819, 67838, -14802, 25827, -348838],
  [-775853, -285085, 141984, 72719, -293681, 17824, 374568, 202508, -87539, -9892, 125273],
  [-244474, 270180, -448530, 140530, 328175, 404518, 382029, -436969, 13354, 60562, -185041],
  [-116202, 195269, -76850, 789373, 375754, 167568, 2851, 293172, 119223, 129796, -181833],
  [483522, 65147, -175483, 57277, -200092, 99986, -252571, -201142, 26700, 756854, -7907],
  [33871, -330454, -128178, 458402, -217734, -370142, -112575, -474045, -346232, -220619, -269698],
  [18701, -20874, -159850, -475731, 15397, -421151, -180852, 296030, -51592, -254963, 617842],
  [212830, 87159, 15158, -486122, -35527, 16497, 388965, 47823, -690243, -278322, -39347],
  [-65432, 163567, 660235, -399480, -280492, 161144, 122567, -277492, 343127, 49685, 237901],
  [-44269, 220185, 117661, -662042, -317389, -310082, -156757, 333066, 400727, 65945, -9566],
  [298116, 422609, -426537, -270409, 41797, 152441, 180328, -159520, 442697, 11758, -445427],
  [-402662, -343894, 193921, 396831, 374959, 73115, 156219, 289802, 272356, 184892, 402279],
  [-59035, -201205, 136516, 73901, -385857, 498912, 413368, 338846, -451479, -174579, -118880],
  [-79571, -268733, -104241, 231814, 33674, -595596, 180529, 217746, 72082, -427181, -483007],
  [427015, 23383, -563962, 301134, -103355, 225914, -36805, -432897, -128640, 48359, -372847],
  [-175631, -329941, 217977, -265413, -226841, -600583, 312310, 178128, -341026, -265019, 119901],
  [381584, 112657, -56727, -662189, 120122, 21247, -46601, -240782, 385402, 381711, 175282],
  [-76647, 161689, 243669, 387146, 52715, -225575, -27114, 63601, 141259, 766059, -305748],
  [-368301, 112638, 105167, -292944, -590888, 430582, -8008, 425952, 105586, 165952, -7373],
  [495745, -129532, 411826, 174836, 161009, -458110, -364215, -163872, -1784, 375547, 30910],
  [-367512, 402318, -1203, 137496, 98899, 140939, -674955, -344720, 199885, -125362, 156389],
  [209988, -193367, -494299, -555690, 118821, -130284, -451293, -12078, 231317, -220964, -167797],
  [118746, -523936, 556976, -5091, -326071, -311504, 4763, -433191, -9325, -46606, 88596],
  [-400821, 68350, -124558, -102464, -213815, 22514, -11992, 368865, -610849, -367618, 343437],
  [-144320, 608541, 170184, -262297, 100161, 18897, -188438, -227354, -632276, 21538, -115162],
  [-550702, 110644, 426688, 340595, 240778, -329708, 239333, 257555, 123367, 3088, -284418],
  [30933, 34099, -498629, 387825, 92260, -123762, -26711, 454572, 319653, 305744, 414737],
  [178453, -407039, -86289, 82510, -10558, 749766, -158910, 22142, -261288, 289192, -219803],
  [-516565, 260934, 439978, -85232, 18433, 12974, 200837, -397850, -107369, 379266, -331255],
  [189878, 255894, 124114, 22742, 479484, -52894, -456074, -345060, -303756, 119484, 465002],
  [234873, -285474, 439916, -152088, -314934, -401454, -396023, 443903, -21925, -53531, 170599],
  [-133129, 604611, 102413, 427212, 68558, -159264, 215722, -350559, 389017, 16663, -269479],
  [399560, 111846, -148142, 52742, 143137, -733886, -365647, 242123, -25184, -225238, -18703],
  [-429718, 271126, 99581, 131768, -578727, -9715, 58155, 63605, -536952, 165418, -237542],
  [137836, -254119, -48021, -487898, 276530, 175417, 93951, 435192, 382230, 113748, -459975],
  [225656, 382902, 219214, -5436, 76154, 360854, -294976, -133284, 109388, -297682, -642659],
  [380410, 631762, -73617, -251511, 191548, -52682, -55630, 186844, -213963, 461521, 226350],
  [-226827, 593291, -271976, -69264, -347078, 166480, -422301, 175527, -239638, 169601, 272524],
  [-193556, 421079, -556208, -56897, 381805, 321437, -59237, 423081, -74535, 83852, -168626],
  [249309, 112596, 450283, -235632, -700303, -175000, 293331, 108358, -113599, -97293, -160267],
  [53377, -374927, -38942, 536575, 451932, -13462, -3380, 173202, 156501, -550733, 70132],
  [-181868, 498129, 181673, -120539, -267189, 132583, -145020, 110933, 353638, -514474, 399009],
  [344360, -233187, 135029, -5441, -464697, -12544, 288419, 89785, -313139, -185702, 607362],
  [50011, -152850, -90273, 79391, -57862, 650342, 196064, -353590, -97142, -505848, -323427],
  [-227774, -255375, 447972, -343067, -260609, 107317, 258324, -126055, -547357, 290513, 135929],
  [-402985, 236826, 194634, 217611, 81816, 343918, 177746, 705385, -27448, -169674, 112259],
  [233015, -368835, -300838, -26584, -38051, 385348, 561436, -226541, 285459, 198574, -284719],
  [-160256, 441739, -310759, -165089, -21122, -11994, -228060, 131874, 472457, 600312, -42136],
  [37312, -463109, 50626, -366882, -122769, 57042, 65565, -459686, 633561, -107799, 4028],
  [423014, -204807, -178848, -194536, -108050, -350928, -147653, -720702, 46197, 155649, -82968],
  [107111, -208808, 70055, -799272, -363715, -164822, -16001, -285862, -126820, -123564, 168534],
  [-329550, -256727, 7239, 343465, 282188, 342496, 615088, 144155, 59631, -181151, -273949],
  [423372, -347669, -283634, 38198, -197584, -40831, -582742, 116190, -307852, 96841, -346451],
  [-515446, -239540, 230066, -391357, 15122, -365906, 265432, -403447, 136294, 134489, 258537],
  [-318326, 103559, -131819, 459298, -202523, -120055, 292198, 189141, 573376, -373922, 120222],
  [360731, 567449, 123297, 564350, -64446, -142187, -293490, -1057, 130549, 199752, 216172],
  [-247867, 395251, 463502, 506485, 18036, 34369, -375241, -72415, -230541, 54001, -327653],
  [-375552, 389143, 204335, -325027, 143921, -68992, -238599, 471641, -148703, 367040, -313773],
  [273149, 170381, 125958, -198093, 455570, -389339, 11969, -89171, 379631, -341124, -462109],
  [125253, 494305, 400162, -165630, 149000, 58879, -434721, -306647, 146969, 441506, -164929],
  [-245674, -571056, -2656, -488580, 150931, 259631, 359640, -32212, -226545, -130713, -292996],
  [-133017, -10259, 245738, 330853, 689060, 4747, 336952, -309891, 100271, 324677, -111741],
  [-63224, 240644, 277280, -186572, -332437, 111259, 23613, 37172, 452127, 321104, -627727],
  [140037, 678543, -210858, -107339, -138657, -21850, -11276, -584701, 130746, 105332, 272196],
  [125899, -59239, -274335, 250294, -311998, -344362, -378385, 331559, -405254, 116168, 442689],
  [-178858, 222118, 213700, 110552, -24986, 263326, -359361, -130141, -378813, -707228, -32734],
  [333154, -284800, -461220, -195231, 422451, -205343, -264516, 50398, -185943, 397087, 267704],
  [28800, -536233, 22071, 212682, 256398, -281405, 21828, -437221, 327184, 334499, 332262],
  [219922, 133589, -333209, 368089, 52888, 318184, 533312, 215305, -325023, 288471, -252189],
  [-143722, 94463, -342461, 24532, -223172, -120836, 197412, -497153, 194333, 542595, -412099],
  [-217995, 170291, -302341, -458838, 409324, 77059, -5793, -115196, -184066, -443088, -452262],
  [165829, 30803, 332108, 86036, -425038, -170238, -202421, -401366, -239301, 361553, -504144],
  [-156861, 441251, 225411, -66444, -122928, -398897, 69888, -25391, -157490, -175742, -700014],
  [281870, 180489, -48931, -398120, -216732, -318394, -685028, -104328, -104532, 216472, 201984],
  [409864, -154536, 485290, 402832, 277577, 12817, 172968, -158911, -70645, -113458, 510002],
  [-107718, 37917, 114209, -726420, 327913, -76579, 196974, -35171, 353204, -370922, 174546],
  [-53723, 254043, 618195, -242280, 474119, 143585, 127056, 22338, -260601, 340318, 214324],
  [700140, 38305, 89003, -63916, -133715, 208616, 176657, 46593, -42749, 239314, -585194],
  [198619, 271573, -388009, -362204, 385434, -384054, 131820, 368938, 379550, 67580, 82956],
  [-317788, 96694, -76119, 594035, 259485, -521952, -359325, -101700, 70653, -210604, -49193],
  [458648, 394619, 383145, -46356, -85111, -173177, -396626, 279461, -335726, 46365, -312190],
  [-362514, 17741, -419687, -367413, 203815, -271618, 449953, -110792, -260050, -277141, 287500],
  [361169, 30032, 246326, 126217, -163213, 130985, -368708, -502607, 373418, 868, 469309],
  [178087, -211607, 314200, -462675, -110168, -155023, 432199, -352106, 87346, 378360, -336136],
  [39152, 206515, 68958, -280516, 21561, 616897, -242413, -183027, -109665, 457881, 421192],
  [557672, 15038, -103887, 279599, 25349, -303382, 24781, 513733, -162462, 454032, -100253],
  [63423, -240337, -277138, 186832, 332165, -111251, -23304, -37330, -451963, -321237, 628007],
  [-221152, 353427, 353014, -278779, -498094, 36957, -539507, -74014, -39207, -125330, -246151],
  [320160, -74539, 182675, 548738, 184197, -133325, -466208, 60600, -466993, -252810, 52312],
  [297169, 360160, -448524, 317657, 410561, -188948, 148450, -142876, 142460, 448923, -106567],
  [70946, 592613, 275498, 38343, 297332, 449939, -409087, 268113, -117814, -76001, 129084],
  [-367536, -189755, 105474, 85819, 319917, 204415, 140622, -407286, -302914, 282447, 555966],
  [-9112, 502000, -99768, -64315, -13649, 23461, 361527, -367880, 43567, -680128, -50769],
  [-714942, 32684, -38783, -333257, 537662, 157572, 177630, 103344, 51354, 127540, -13452],
  [342959, 15332, 489028, -327049, -93626, -10850, -41889, 344751, 100838, 627487, -51334],
  [-523821, 316312, -304158, -228969, -131486, -199893, 200396, 500888, 119053, 115091, 323893],
  [6283, -205559, 582158, 109517, 369807, -556603, -121002, 41957, -34302, -376511, 30034],
  [-500529, -28832, -440747, 218331, -336028, -565647, 72699, 34938, -236752, -98564, 39613],
  [225035, -125729, -235126, 132035, 669678, 373849, 44614, 34532, -29578, -50864, -515711],
  [-212843, 250826, 28638, -580535, -33847, -239734, -127126, -523539, 283065, -8474, -353375],
  [-320529, 319706, -127099, 137386, -250909, 141679, -158948, -629007, -182417, -165976, -441875],
  [-181087, -14511, 256262, 360580, 374448, -51949, 86944, -357428, -83941, -474237, -511016],
  [194176, 357227, -203554, 285778, 203881, 592305, -284940, -192403, 358149, 252787, -93847],
  [141457, -131232, 476664, -328246, 490656, -133026, -78608, -583542, 144966, 18009, 36524],
  [-189262, 32819, 332644, -323395, 470875, -338665, 506900, -129313, -231614, -128748, -259919],
  [-96170, 196884, -53878, 117292, 212778, -509185, 42638, -60616, -611893, -495099, -75860],
  [-720600, 191439, -343016, -168909, -201855, 98304, 88530, -346474, -165387, 256429, 162776],
  [310760, -208746, 177274, -338760, 39706, -710273, 151627, -70027, 377778, -8045, 192168],
  [-221863, -88864, 506021, 125193, -88923, 95665, -328377, -523498, 391601, -44607, -341859],
  [293758, 349041, 270659, 215688, 13765, -413697, -204219, -577798, -39654, -351460, -10578],
  [564419, -303090, 161664, -221741, 118331, -229207, -348567, -221271, -150386, -385390, 325768]
]
'''), dtype=np.int64)
assert sphere_centers.shape == (593, 11)


In [9]:
#@title Verification
certificate_metrics = verify_sphere_packing(sphere_centers)
assert certificate_metrics["num_spheres"] == 593
assert certificate_metrics["dimension"] == 11
assert certificate_metrics["max_squared_norm"] == 1000001584452
assert certificate_metrics["min_squared_distance"] == 1000008071925
print(
    f"Verified the sphere packing showing kissing number in dimension "
    f"{certificate_metrics['dimension']} is at least {certificate_metrics['num_spheres']}."
)
print(
    f"max_squared_norm={certificate_metrics['max_squared_norm']}, "
    f"min_squared_distance={certificate_metrics['min_squared_distance']}, "
    f"integer_gap={certificate_metrics['integer_gap']}"
)


Verified the sphere packing showing kissing number in dimension 11 is at least 593.
max_squared_norm=1000001584452, min_squared_distance=1000008071925, integer_gap=6487473


In [10]:
#@title AlphaEvolve published 593-point configuration
alpha_evolve_sphere_centers = np.array(json.loads(r'''
[
  [-126140549599, -4345934944399, 470267207045, 263582420739, -214242641164, 6323834592662, -5619609064852, -1309426459793, 402713598698, 2669019955225, -434830723701],
  [4995872033121, -3686427151476, -2854162550811, 1374657172375, -1039798804579, 2439226626247, -3321352498532, -4613529709005, 1873895977070, -2351434851488, -1740061033492],
  [-148878709727, 6009198311530, 3423705484214, -3908379065200, -2039242115428, 740038679089, -1903248798804, -3229808619747, -67448542119, 2608302674396, 3361875535061],
  [-3613653483807, -421965187481, 4386947861356, 4668066279141, 2659643185631, -793481959702, -782545296485, 3044001988734, -210044264644, 732930704465, 5250176454220],
  [2145601851578, 2286907266338, -6008630565540, -951491851263, -1128202168949, -2351035905822, 338686846605, 2773377260, -3167080097124, -2121084647622, 5631442707990],
  [2905871765916, -8266251805063, 212038997927, -422612434475, -2967768531066, 917651742589, -1199305228751, -428118279913, 1046645354339, 629591995938, 3199722319491],
  [4101374281925, 140190714949, 7791359990956, 2617604236962, 2117466506344, 1449617600436, 733498008403, 1078801552552, -1108251520223, -2353519449875, -740114970023],
  [-4114734847133, -711568804977, -1096919699416, -2179625341981, -3277074074174, 3846872693149, -1343845317573, -4029410886259, -760938481950, 4323960137424, 3708537789894],
  [3749853727043, -920834513083, -2932082478667, 453628989161, -3091441942608, 6398413670681, 2921263471507, 652306845488, 3309571127773, -1260515535742, -2071139339072],
  [-5502577676278, -1227448640831, -3806962231330, -700239558766, -634751583528, 218595111082, 36644542589, -4043060252879, -4497661105875, -2852762755592, 2840093077865],
  [6431155805256, 2472183772617, 1852141756766, -3447375436498, -2150831199996, -1035995564180, -582868441525, -7570292152, -2212911701699, 4854953687339, -1645298702536],
  [5457305326303, 1209487773311, 3838331900010, 672841580707, 640009753701, -223039211201, -29010721761, 4023147410765, 4562777370943, 2817042290572, -2858123518221],
  [947994596354, -493063270445, -891551370805, 1943900881453, 4808222709486, 4576541247433, 121556039265, -2928496070412, 2867556918429, 5676016954336, -1090838534345],
  [2487751367449, -4435069885581, 1203527162971, -7039059911283, -2789489469568, -1507124796768, 408730098019, 2199308898207, -2690789476039, -886790999066, 246907513281],
  [1258393365327, 1558236082994, -2901279533502, -4588804764838, 608788356464, 99979762711, -2072852177408, -5237300731547, 2206084299760, 5428740521977, -262884833863],
  [-2167736272270, -3517473061426, -2581444334945, -4875068237304, 1316469763812, -5045504207465, 1421790102552, -1042617987235, -2433031020177, 1273290801609, 3828642503948],
  [-1748341091839, 84738171505, 491018476370, 4552281419561, -872666527835, 3684504527423, -4225299894261, -2607643958170, 1747681580888, -38267500797, 5824587161476],
  [-1190649965834, -4307368301932, -590143473885, 1188487865738, 1457925043503, 5074991372962, -835020531624, 4328426802452, 1938681290116, 658478446157, 5173244484302],
  [1157198248631, -3276093261966, 4564492907322, 71866472319, 6569376668375, 2635857316126, -2535627189471, 666339223157, -370257470349, 2316182402253, 2146785163984],
  [-3008255020828, 3938057143766, -5277192922587, 593373194319, 313114162483, 4123287795940, 1308453155442, -2544545673675, 1482541632927, -605248563516, 4403486139737],
  [2953917709599, -440143619596, -971489179779, 2328335269090, 2051703342736, -1981036966235, 1096402607228, -8032142516776, 1008884833142, 2051765017859, -2374176269185],
  [-260449853094, -1557229191143, -2449904390021, 7358451024369, -307525874663, -2110837486887, -1514682853981, -3642026943603, -341788837820, -3181174130215, -2648178657093],
  [-4525545210765, -4561293261761, -2207465574022, -3656150761762, -1355527319115, -1325108959607, -2606830665241, 4286163870684, -2272423058824, 2270296581664, -1181276273370],
  [-4989184462869, -2211501213310, -3272216512438, 2932576526381, -4053121684822, -741725216911, -1173137805746, 2181007677136, -2973820944500, -3480396984635, -2615745041603],
  [3602451243212, 1363170942462, -5335236817537, -5099315765146, 1671801957653, -733157544910, -1557322490939, 3489905697601, -2568635080802, 2482181047248, 9540974612],
  [-117106507206, -1557986758662, -1613271333904, 3599924179366, -4726770521248, -122491678154, 3003684160051, -2037648443680, -1689648678287, -6017572885613, 2720160888071],
  [656948042076, 5303758392177, 1223385282487, 4208826988200, -2322770619544, -1931938705746, 194867179198, -149785416225, -975234488353, -6149937849011, -2065621641353],
  [-2822884624434, 1200686951368, -740362280165, 2549091748238, 6473313457564, 198867059897, -5879821576926, -497127882332, -583143250935, -1042480487756, 2313887084841],
  [2071972627918, 1689375919671, 2561773181285, -215304283110, 3678351753570, -481570249929, -4708236211267, -713440165548, -2024413488114, -6510658512889, -1821622848642],
  [1540576902788, 553319845306, 5192033407329, 3790472513109, -989994836245, -6751185485237, 63271511119, 1311050643347, -364367110835, -1981107885558, 1912214169188],
  [1043725858252, 1320396911651, -3526919864724, -4284357877757, 1470366412833, 4775346083201, 1368745214981, 3701799302088, 2580249413700, -3202483086523, 2985971881393],
  [2328981318412, 782277770357, 1613319529120, 6708838015807, 2409644663632, -164697695288, -2876391291223, 1405607425611, 2563177561828, -4390790783825, 2102566367890],
  [-2898694569620, -1993213074975, 4767991564340, -549183545871, -1716073975618, -1376827268341, 3962365401068, -1804381043737, 364314410734, -2366368669369, -5921165923758],
  [-2821980692939, -2907092870119, 526484763216, -283219412226, 1816656338961, 4630761564997, 5094816643786, 822612609763, -1628794974160, 4497378787802, -2995063685088],
  [3981893553028, 6726656929267, 4509428321078, -1718973949677, 1235696414272, -3105774701931, -561577844505, 806709024917, 670699852230, -1703554165369, -340483077176],
  [958740882, 492343935025, -4877156942860, 3610017499034, 1303862660779, 2282799096236, -4071872206188, -4319287760002, -3382516964626, 3040863514676, 320653772456],
  [2668842066670, -3328224345221, 1245442050272, 807268381271, 6338371398067, -1028063433204, 2718246800023, 1487663933151, 641493721090, 2991619189786, -4404795224958],
  [491355192733, -1231352019317, 2214609261146, -4246171762475, 3060427833343, -6012660473699, 369697486180, -1926711613647, 1222190747138, -4895424389279, -694078468931],
  [-3936231732764, 3646035008845, 3917837393650, -2302689585385, -631606199075, -1190907291088, -1462454097742, -1026315700574, -3403121373084, 4358068529006, -3870064655165],
  [-891693194815, -3159218172959, -36815057657, -5392159737721, 2695159539664, 4846777170940, -2774317263822, 2475655478805, -3507372987730, -1040335371092, 1477389122043],
  [-1909886271559, -3025194791002, 273889648930, 22368016637, 1769343232414, -6711034204699, 1807271442431, 5498553429397, -2113841429687, -965335059539, -236521072887],
  [1421740442238, -4445189941322, -6335315158518, -671536406806, 2646092406700, -638731456569, -171872290986, -2000918638949, 2251948982557, 2831806435076, -3619165775254],
  [-3348523808774, -3469324536046, -76708109001, 1992777744163, -5049711606642, 581253431088, -3927998298964, 2296310725868, 2561198403600, 3752242898555, 2365461437834],
  [779042434547, -1416461625219, 1452341221883, 1439971242011, 1285662467259, 2679379627727, 1926099800655, -1269351586896, 2663459313927, -8084575368127, -2568350771324],
  [-3980831743069, 3637325407505, -1572468831509, -2438291232144, -3064062866153, -2476525715560, 4274880397145, -2716987458463, -1175083650793, 1660991539240, 4145711223394],
  [6967450268144, 2625304133625, -1401474346935, 1995863610145, 763569629212, -3804173165507, 997773279985, 1860188439504, 1442735815840, 1895879894694, 3664470120161],
  [3015840597743, 1234709422899, -6952463247052, -1298276880914, -1224429458570, -1308306783583, 972871140414, 2450262921520, 5290015394161, -560255643005, -947955712831],
  [-936946977342, 1116679407669, -396124541576, -5766839382952, 2927744211058, -1495168010655, 2861812090255, 3711225318549, -3673521240929, -3839839100081, 1858038850084],
  [1308510697687, -4987098459834, -5143597325210, -1869372147752, -978771635962, 3179840980566, -3422701861744, 3615532701633, 1765018082471, -1744159035660, -1204896731728],
  [-1646730796381, 4421637104002, 37254831639, 4273667291949, -655020690827, -2640710658714, 1397627767521, -1309174476974, 1380616410107, -2725730878362, 6250309303605],
  [-627474373499, -451185628535, 1125092239854, 1228838206359, 26093010133, -2939555208566, 3370434196692, 667250715880, -7209365367712, -1252881093849, 4757673344364],
  [-797420603404, -1227566289334, 3086021511548, -505080489488, 5478728185869, 4126710775002, 391076993732, 139481050671, -3647775845204, -2041750080177, -4835799186849],
  [4092334232882, 729220292791, -1168955935796, -3085250972755, 82524549053, 5275269956420, 3666568911398, 2396726507160, -4862220845505, 358540800263, -1021153633823],
  [1756033032922, -1110377613430, -1249242600393, 2221908240877, -400217238665, -3426298784081, 6127013124122, 4593495905370, 3265502690632, 1973356712550, -2021902581270],
  [-3725619678320, -2329459894461, -5070699197459, 1534301307867, -1037135676352, -3205158454204, 6159879240390, 559487604350, -1262339301196, -862647461628, 826929767550],
  [-2274391587253, -270713035752, -1330852865067, -377690887956, -2008376261801, 4177033325257, -2292376198981, 547802074232, -7458187593121, 2184009598005, -2325794304528],
  [-1452869174536, -1495810745460, 3812294435170, 4056541456396, -1452578046306, -4834849869320, -1310486979894, -3907635677827, -2007229588715, 2874984727132, -3145687074383],
  [1794443008446, -307543586686, 3579316950366, 4062541899933, 930653960952, 1957509312467, -2003233153037, 999980394766, -3454594428395, 5958959150112, -3196146826361],
  [-1460561567534, -916399636812, 4669810481354, -882217985126, 2825371943033, 4288562595538, 4110001703233, 3676245445344, -2776098864617, -863483729868, 3034437958314],
  [-3278999167478, -1273384556037, 681284367484, 4877315854280, 3219855412980, -1975019416928, 259552774384, -5776746289350, -1601605704779, 635789280058, 3563508544421],
  [3932398091349, -3647552677504, -3915156254426, 2300359279437, 632049048201, 1190512933335, 1463079709812, 1024625838059, 3408695534850, -4361112761007, 3868556093990],
  [3405094292821, 1309788195129, 2321854096188, -1882365030199, 4076617425181, 6149102664930, -142554192789, -1909405757679, -914754807934, -3485849481473, 2582854034473],
  [-2463493552091, -6182166095073, 698027052685, 5496165743777, -2223299965981, -3647879802948, 628799524443, 494699815054, -1002913242476, -295445251725, 2243571445095],
  [-729861928381, 580654590161, 736639591977, -1815111899007, -4839349528509, -4562434137639, -151406490905, 3024687926959, -3192175323619, -5507105846599, 1169574501828],
  [-944932369434, -3799932528535, 1269141877026, -1606029533014, -5042197520192, 1753508262775, -5757405933996, 1676387264954, -2659952371999, -2907761426523, 700425645446],
  [-313412211989, -1472067839495, 2014918908248, -2994085643890, -3631601113505, 798675905998, -5151770012594, -4504700612925, 4645803566421, 250361628870, 1550709051811],
  [2806524625245, -4911847282592, 2447813070402, -867370776942, 2711443083233, -1173006619677, 4693314097340, 1244183268233, -3834905781184, -3651786699409, -952863639178],
  [5102102592725, 26765691116, -331028209281, -2142605004855, 884206303205, -6425311667055, -950525299835, 262446895670, -4980233069174, -1053283462981, 562620518775],
  [1098954479782, -1109038396398, 1216752926622, 2017980461121, -7349092644345, 3487054908779, 1652822263220, -2099842431090, 2295686494056, 1022158793060, 3519196027836],
  [-4358075667015, -3089508657759, 4802679891308, -4810354941838, 4228949294611, -1392241901612, 1450152791677, 688600723787, -547780846332, 728656385898, -1423515444254],
  [-3368253979501, -1295226691162, -2347476680864, 1904689227772, -4081008654975, -6145377037947, 136414911035, 1925567171135, 861357305081, 3514943751022, -2568129787920],
  [2231100299994, -4273492332869, -4596691742304, 3642873481075, -5001877658405, -1406691322979, 2412453357834, -1290390423599, 2512433823420, 586628501749, -1104670360982],
  [-2575376667766, -960065509919, -340750042435, -3074326806221, 5178574195494, -1775696830323, -2226386336847, 5177679668223, 209007473419, 1808719719559, 4222211433121],
  [-3578650785329, -760578973239, -3477963098591, -2048910481829, 3686410787928, -3078459201831, 3150278587757, 2610639587537, 384617180163, 5179605695847, -1880968201088],
  [-1453680890271, -673688713741, -4079118060194, 6145074907773, 2493898205845, 1386249452566, 358139860104, 1448264243842, -3561849396434, -1272570622967, 4284699431295],
  [3328068080720, -2292691401219, 1230258674419, -4178966504561, 1952019534393, -5264768742287, 1386218014402, 2997466555571, 2074496226701, 4120699917727, 985876677105],
  [2539767610843, 94765789165, 3854189084765, 1251736924810, 288435856169, -2041205772562, -2845981392459, -4127782598606, 4941323231817, -2767998454739, -3956333511561],
  [1067212670401, -37369907419, 1058370258904, -923482582767, -1672287471130, 1249103492221, -4785063288729, -5636554728192, -1540216098658, 2012650649787, -5607074089976],
  [2693775044582, 2069745791644, 2364479048946, -861476754182, -369013705992, -6661486740994, -1991744606252, -3971515305028, 1993645640090, 3364267727602, 1606968451704],
  [-1060152337718, -1462090647628, 4542812980901, -128973585724, 4479127796943, 2331049535271, 4969221611534, -4599291027477, 1873218159741, -1057368339060, -338512596613],
  [1828308410223, -5297151224782, 4618727669960, -42450901494, 1921798353915, -4281708414571, 722583002058, -2788707550640, -1892464744087, 1898148718778, 3123025242617],
  [-2974029255356, -91718543918, -2789616442708, -417822031141, -4472352298733, 4615977419360, 5600790978158, -2574878657168, -1894939235615, -308842022774, -442698710154],
  [1645018045050, -4422289912096, -36103964862, -4274703871227, 655235768532, 2640591254105, -1397378950407, 1308418198678, -1378166081991, 2724363162782, -6250975307286],
  [433691886277, 2523890861790, 2284499873225, -3703627868154, -518512695555, -4938318927311, 257841134127, 1840194784536, 184975564588, -457909300815, 6793639833180],
  [3325316992112, -2486276617276, 1123273178383, 4294677510387, -1936032844161, -2776149438784, -572327617931, -4803422897516, 2993538631796, -1489999107747, 4125247597048],
  [2393744590379, 1049137728331, 5122417881385, -1088629195634, 5105537095120, -2435589424350, -1707333453883, -3641898816074, -2375980295032, 1693223337093, -3009270903611],
  [28402196897, 1416027777891, 3440429273442, 203961451796, -3940869930791, 2347105807829, 3199355476507, 922594830306, -4951120644270, 4761531968667, 2608391131483],
  [-813227623899, -5357941337067, -1112669738057, -4304097048545, 2345298447370, 1911013266658, -174960144340, 85301205749, 1204772867285, 6020219044345, 1995755808449],
  [996433361992, 3887894764559, -61480148325, 3439374904618, 4912048980888, -5693090961926, -928200494105, -3021190348807, 299266571934, -2320681651943, -233525862680],
  [2453568242579, -1987414549672, -4478808100402, -1731895787954, 5278273305930, -3861728068050, 4222749196955, 1356061448728, 966666015199, -1824003862323, 515695447649],
  [-2664724735182, -1495474258595, 2944595283119, -2919860019644, -2308099678557, -818004793386, 2276432949055, 4387646832659, 2527039579293, 4206496484419, -4353847210838],
  [2103427320648, 3064498166470, -2140607330478, -214240562173, 2185849739279, 3694612769131, -3992827456340, 1545977029970, -1036921145639, 4214246024451, 5095140041841],
  [-4722758966585, 1206959806325, -2218480002577, -2968990772048, 4375690183492, 2762753454305, -688043457918, -1384779569811, 297008499206, -5649005618160, -1154239157355],
  [-2609408155367, 294141147806, 2298399803607, -1460949423253, -6206872086624, 3044769045395, 637152553136, 1635378523205, 2401015430092, -4948485353184, -2135126898687],
  [3867944404697, 2385613461166, 4971200333675, -1447973537461, 1019789091334, 3219226368043, -6182937104131, -497220049095, 1055477569740, 975358261409, -770526134964],
  [768762538555, -861169573712, -3921515618745, -22343685772, -3124204955684, 646496969665, 2154751469802, 5008107566907, -2780174801716, 4768474481990, -3594186429160],
  [-5829157725768, -1845011523627, 5847631650732, -217123128185, -2762905325685, -2720689317598, -2609714245707, 2013419676738, 1074499522941, -866115882344, -754946967281],
  [-1311298215779, -486042628530, -2763893228432, 1526263490041, 2632583689192, -3621344081841, 1368629186733, 20242687756, -3847994704863, -7116704442527, -838416785108],
  [-4079307778929, -2848117033354, 180600291286, 3916227562361, 1526038123223, -1150669460319, 3432721011725, -2556067021564, 4840526733645, 2294106198479, -3035542058866],
  [129849177296, -895608414736, 739412179265, -3313217288439, 1347265236537, -1350474981128, 8576800884753, 753529921447, -1070437463712, 2225825733472, 1946412591514],
  [1022976878190, -45968183236, -4432158106829, -1058813047828, -4104820558761, -36539537494, 952233064667, -7327066689318, 687280832481, -684194666759, 2408791472363],
  [726100104866, -3850927701095, -5622396324275, -2466174786399, -2197077396011, -5197407057482, -329370102200, 1038406981314, -989600011579, -2648022596253, -2435463239897],
  [-165613374687, 552413143883, -1161665499778, -3926523097504, 7762169581077, 3071383417728, 3006518773230, 704634129572, -541373108803, 1699918236493, 705136590918],
  [-2463707289161, 7334104944, -5254000589214, -1875851987073, 575650872497, 193540586190, -4724021792447, 88382347075, -1404280131318, -733098432327, -6132233566685],
  [-1068227955381, 36991913966, -1057698833338, 922844025654, 1672425887977, -1249208449583, 4785238023229, 5636085487623, 1541691946288, -2013439331427, 5606684229854],
  [2220517143322, 198705222749, 247604570500, 1479799498957, 2489049282415, 814044846502, 3030144160090, 4590197712109, -561532095582, 7075362333048, 2300632700445],
  [-156780111086, 2736931326151, 965004659863, -1831034245257, -3726675002694, 5210673474784, 1453882908988, -361014855400, -136594164620, 3121012600730, -5929417932372],
  [-2949947628793, 2988361859580, -212814868864, 903455432497, -2046005627366, -1876329775279, 6503528569315, -277099644684, -1478109831366, 3366507572296, -4231829163727],
  [2045601175700, -1564278540916, 2774622143877, -964561825269, -1253678239635, 552184872386, 3180838585545, -2911804386954, -6180323347514, 2204084547490, -4605681208142],
  [55250421914, -7350188953636, -641569884634, -1083937595197, -1091999530200, -2998210223345, 3830530928145, 490910348930, -1415454125947, 3413215490274, -2374004857697],
  [-5017343821737, 3677836937740, 2869185859152, -1387660752676, 1042242648295, -2441379816081, 3324919305925, 4604118406050, -1842614982590, 2334378964167, 1731413185941],
  [4334119519474, -2595497670494, -1222943617868, 2697227087236, -2416592134269, -1744364781539, -4568500959292, 2410005633708, 5468207643593, 52150443924, 492042756756],
  [2658920838129, 5641550619343, 442931548469, -1551669065028, -5542562030946, 579266633195, -3639887265300, -1186440616563, 1501365270434, -1381312597597, -2936851176766],
  [1437696918439, -3901230133907, 5939123390411, 696495681099, -5307342745969, -526543773849, 1855636404266, 60757669874, -3092578936240, -1971580173004, 1269547930945],
  [-1279198726336, -4494764829706, 1092293047395, 2850892770464, -3897572176262, 1684173370526, -2252708806981, -6118712302944, -2711864726485, -639474370511, 732162729451],
  [2571548192351, 1062970499840, 5931954641540, -4827888529490, 682286003349, -442106562559, 2804703382020, -2023617203399, -3776836195923, -1111938053742, 2374317197047],
  [1987507400483, 2047084168016, -3002059435990, 1244072815805, -2542649156248, 176634825883, 7225040063273, -1155681023105, 3321068587346, -876174069295, 3077163956589],
  [-1461334920223, -5683558990067, 1458287405745, -3140510195587, -547138900823, 2658674539715, 3374598110250, -2852546789839, -4372452460752, 550693577616, 2694030473882],
  [2789684933477, 2418905805336, 3920543907820, -531513340616, 2693654810086, 1969467153943, -1396197508824, 5134579301295, 2611529296474, -1045763809032, 4832372637798],
  [3650744567980, -4644629683627, -3699694543658, 2612553277032, 519883653911, -1881417480910, 943481947691, -3790800920758, -4416251706114, -408269127764, 2417850592918],
  [1436248473698, -6015909485541, -1585609778223, -728279221654, -4061556287510, 2506355530637, 1623620972154, -769589298542, -2935741997700, -3367508117025, -3568797013361],
  [-644923126395, -2161877991026, -2767262539037, -5203279765784, -2774283868680, 612706970955, -3061605161094, 2123312292904, 925203316429, 593138457327, 6084138308239],
  [252405123934, 2206139084474, -4595639419590, 85111351867, -4782930422763, -721735987789, -5272745523900, 4265015464793, -2085605756537, 408606626432, -189359683156],
  [7053112995937, -721686034826, 1665017815644, 3883332184132, -338594544788, 4332759688617, -659180497339, 353063806376, -22371930753, 2334890792379, 2642228581106],
  [-2358290547732, -5112615059945, 1377813744578, -1209079881217, -964930564226, -3896741748046, -1290056495803, -1441252411451, -6180287803578, -2142260186258, -1515589161384],
  [2501097637297, 4123491123748, 4271158567686, 3389529547678, -3347701183653, -578176305504, -2597493200365, 60111207331, 4259394584395, 1251306377062, 3001572968103],
  [-3905847290960, -1488771800312, -2053247525676, -90486946243, -1054900276341, 1382249202756, -2234835133169, -955881342876, 812023295000, 7153556583528, -4188021476324],
  [3894547097085, 2814755872739, 3839420830742, 1947626151590, 1950087300099, -1442382122810, -2803223032330, -1748644554391, -3530905638874, 850911700296, 5327737153655],
  [5123151777017, -404226532300, 3075206595856, -5134854073886, 4638569941373, 465818284017, -707657296426, 3205407676666, -665340777495, -1290658656591, -1774565213611],
  [1916117580566, -4093266494671, -5821590032850, 1263077964827, -497586117275, -2100172023666, -682742738160, 3330924624014, -388213188429, 3880245331412, 3558210516981],
  [3401202788258, 810363437763, -5274150637157, -928677899866, -2090316636540, -4612443348259, 2952211444677, -1956336686949, -1824801816226, 4161313173728, -512573096978],
  [-6969575388595, -1819483463926, -1381333443046, -4002918516902, 798912659548, 2344670318129, 3174898396419, 2144891020890, 1607026711522, -356366181421, 2580164864728],
  [276997024511, 1029528219459, -412748310462, -1893341701947, -764769161531, -2085125184027, -779580331768, -5726504265913, -6450058000710, 3162432669128, 2273938482379],
  [-2006373209666, -2054493934083, 3015218165908, -1255518043090, 2544926372296, -178539759170, -7221911766451, 1147400549023, -3293605634553, 861176827029, -3084659981606],
  [1304027205357, 3324730053896, -243321857757, 5642880845393, -2736638803105, -4801593338556, 2713255478063, -2296124924557, 2924603069862, 1363744529287, -1323946982056],
  [1313958374220, 1081499628340, 4072809931350, -172775867819, 6779271255119, -3685975109542, 3079865241934, 1989147097123, -817455291683, -329359888877, 2592743966517],
  [-1567783876789, -7384170965839, 3311780178305, 1327667060418, 389718199022, 2636863689822, -630307270725, 3399475322545, -3073903523076, 297542749995, -1299777063762],
  [-1504145081860, -538826415777, -5217302414788, -3768713739576, 985738688357, 6755155777720, -69482645562, -1295245807684, 311515406667, 2010148660475, -1897294536706],
  [-2082074589211, -4576770136906, 3060705179466, -1792361754528, -1929011305802, -1520940025193, 2120807677184, 4188855567239, -838713657044, 2987248595378, 4942862485457],
  [-1118160218577, -3932752314296, 145244832515, -3513492678208, -4897566395071, 5677719152947, 949385945492, 2966833768506, -121064747752, 2223158760683, 185123993948],
  [-2885810971379, 3921536918930, 1089916425343, -3072576745427, -1665297542424, 7397430565361, 96402644487, 723506423342, -1998710348122, -790422177651, 1737077257039],
  [4824082055808, 3789854052554, 424544297045, -3281115028812, -2102394783795, 1626782367763, -3185969944244, 2411113323944, -3913025740612, -2319500673832, 2774360181845],
  [-947768904213, 842278199349, 5370428414965, -548373867687, 3148686234987, -737691635251, -2420660381267, -3985317011510, 2700872971740, -3297024906174, 4345599604186],
  [-2483741090108, -4116493995975, -4283394863320, -3379040828527, 3345662921050, 580094565410, 2594282927079, -52488891633, -4284802307248, -1237329360849, -2994423995937],
  [-1098108373298, 3299541851152, -4605910635694, -36129840727, -6576311788859, -2629800174936, 2525791880294, -640513138117, 284228178609, -2269218166260, -2123139014045],
  [-2219471355897, -6471651795388, 353742166342, 171015677393, -1471849306206, -1991285927739, -3589311739944, -1466374989604, 3344329836540, 137984938737, -4546287909462],
  [2073182936613, 466328932539, -2791012405298, -3313588062417, 3530568343429, 2216032040681, 2265865618174, -4346765336882, 5293052453049, -2639133395520, -573740069659],
  [214459878412, 19398313189, 6305390579996, -3608487383784, -1989842234019, 3278350146195, -1223967877264, 3305477784465, -3617311292070, -114403255569, -2635362871908],
  [426321805749, 1137571596648, -934133552682, 6233342815683, -48487549937, -4002542435362, -4889499143122, 703316459258, -975659268269, 3514709109664, 2251081605192],
  [2468565829596, 5155658424646, -1453270930266, 1276928271948, 949302333786, 3903659962438, 1275142291747, 1489697404928, 6018387529292, 2226900271067, 1554937693002],
  [2715506054844, 3482379220419, -1729557792903, -3606698334999, 5937100067475, -1554453997960, -2096963359890, -92230435877, 1000574089023, -2751065007208, 3722722845190],
  [2634743781118, 983127432399, 299209752327, 3110492334599, -5185766104877, 1781713739917, 2216061204070, -5151489900582, -295402636687, -1761199754849, -4197899465723],
  [2077555732930, 283559312757, -2760538621384, 417300252388, 5927503430910, -3533754014174, -1001230518933, -1223879516797, -2575219126684, 4882369112087, 2686317858147],
  [1168466204065, 3822971229690, 5029915887937, -1996991926131, 3051301848435, 1525811987757, 4531837892615, 1633064178647, -970556221271, 2800621530552, -3332129692759],
  [-4333405640177, 2595803503469, 1222444806936, -2696811891176, 2416496360838, 1744487793328, 4568361186596, -2409674071826, -5469243574926, -51579532841, -491734613348],
  [-2467630147606, 1981816676657, 4488642911129, 1723347036133, -5276606227799, 3860287962711, -4220417788713, -1362249656124, -946152098270, 1812822630005, -521332300988],
  [-2241001781198, -40972216213, -4450445816436, -1830900109084, 583181392280, 2373038873262, 2677962053627, 2142841574856, -4906715376677, 3515275831386, 4201176544176],
  [3566116295130, 755631566161, 3486726151340, 2041359806509, -3684933736093, 3077109175398, -3148124103181, -2616165387334, -366438653115, -5189581246319, 1875913477081],
  [1169756207380, 4450182319364, -1017069826463, -2917632011119, 3909587453197, -1692287000160, 2268879223154, 6069084923479, 2871909406721, 554215792900, -771496783680],
  [4545643710456, -2839300538053, 3196872591342, 48408958543, -5617612880939, 2851958971972, -2444934555215, 415011797637, 1397334760407, 814873213931, -3549001395164],
  [1369104443879, 508626624505, 2723357463617, -1490777202574, -2639761015160, 3626693760144, -1377786536700, 5248435782, 3763602126491, 7162511645981, 861345608385],
  [2440131171132, -819355214094, -1455810769125, -601774225686, 415079064241, -101645112079, -2020387519010, 5515392925687, -6019543383458, -2866131618202, -3429160170996],
  [3404935097258, -1475777975123, -1624555070751, -3064900974353, 510268804990, -3238776936905, -1707968738275, 6124575621962, 1783373494878, -3781535254565, 2353218057669],
  [480568627596, 2119861335709, -2131094079761, -6433151784280, -4224803047866, -4467323734356, -2216351640087, -2233676451907, -916228259636, -116451429973, 887544630281],
  [1347340652764, -2425982744085, 1305009012156, 4911066205519, 4558713915155, -3083628705040, -1527150020658, 1092432828781, -5648998324766, -375301281712, -780636116878],
  [-645955447182, -573412844198, 291621049695, -2851751604036, -6178291223918, -2042697299143, -1653484381312, 1144334097639, -3019079742109, 208920449350, -5957391471904],
  [643872500447, -3889012601565, 2040770000899, -5164439015432, 3090436298776, 367803434308, 2592146963002, -244364271528, 3210036084153, -1870466177752, 4832892053998],
  [-3489708056613, -3565689589022, 421784280972, 4382563289631, 822777100831, 1565898196956, 4032550397657, -2172439184083, -4750428726450, -2287378729032, -1953049895133],
  [1510363990797, -5503286576671, -695235405965, 2412916532682, -599696475475, 2886241647929, 526584803957, 3231823250970, 3842006991659, 4548131061588, -2503718301923],
  [1226958249894, -2773201525784, 91343677483, 909406635811, 2053979594022, -3961278384597, -6239413826246, -5274237794401, -1407879540787, -1106055587451, 323498175969],
  [-754260506420, 4933069749976, -59774358666, -581839150403, 5435911970421, 1679010859911, -379642495186, -6244963647764, -688048364149, 979863489122, 1344747317080],
  [-4466010900252, 1515606562093, 561844871854, 3992079077555, 1161311755868, 1990294207776, 6492035972605, -485351519506, -252078871908, 1773771427134, 3256301020790],
  [1800158889991, -4861798591592, -2635881641748, 3375896166871, -2368461074210, -1053116459665, -4421962342629, 680808497648, -3685268727265, 2081875272447, -3182211796350],
  [-700568222425, -2059326140183, 969918550260, -2331659486757, 6358449782368, -702075227607, -515434118465, -3327861842950, -4776211464527, -2744252145422, 2506941608789],
  [-2707039197917, 2328954056365, 4390223123745, -73968588581, 2565737297374, -5510058522172, 906559073621, 430873253591, 3999558820412, 1152761661004, -3562554875280],
  [-1024528772430, 45347036460, 4433248776900, 1057871180179, 4104998210599, 36381024375, -951957791240, 7326387516491, -684996202517, 682937228904, -2409432229150],
  [4067741710999, -2566271397519, -3256492925837, -4386848618314, 1886947461174, 944956097375, -461004655030, -898479697604, -1812451258699, -6184293797316, 121525174369],
  [1315366752583, 800997282555, 4566987593264, -6895175556057, -699226297501, -1451542326893, -328352244017, -1928137156462, 3132392662035, 1114118165728, -3428239828999],
  [2144150939231, 1554529906139, 1257893659262, -353150017642, 2410536308459, -3423149484282, 1256725264007, 506784869501, 7845923503507, -2077267572141, 2457775205201],
  [6273929664447, 1986709087805, -4968720868214, -284932371925, 2539623792742, 2613082443799, 1482794570863, -3206489923520, -1170089623014, 1375141403761, 1703863316471],
  [5113286992062, -262519979148, -1205611478137, 4656492845107, 1575614313201, 2901229953062, 1651661934635, 53279269656, -4041644034623, -4390196272182, -1187317734356],
  [3978464984049, -3638245937238, 1574138884919, 2436835214069, 3064334755163, 2476356395527, -4274505026665, 2715967838611, 1178536371096, -1662873006476, -4146618641126],
  [3350770810383, -6123869174151, -1791629481518, -101508661756, 3779184489497, 5473608843151, 1686970086472, -501827889720, -750568283023, -129282778282, -360452279019],
  [474331919901, 1699069111081, 1367225341153, -3383752095864, 4667453902113, 151303632031, -3058063988967, 2202965005450, 1165308647261, 6289559138375, -2576773702522],
  [-2005176305822, 3377760252313, 2836300312586, 4011574435132, 4826243835769, 1579668182124, 1692136264531, -1004400577170, -2904950274070, -4442988350488, 1610486381879],
  [-1169046415570, -2309917566597, 5637115874876, 3572460874826, 816413649445, -2106133162533, 5137698753257, 2181218129160, -2153312905457, 2606837452574, -1034268872376],
  [2474067050690, -1851274671465, 2426267545725, 1659327951792, -906277412481, 2756227575435, 3811770018044, 4421784098451, -842976716678, -721246831441, -6170974772918],
  [-750066121913, -1354049898449, -3732247107281, 1991239201577, 4538863103650, 1845719188469, -3121715318647, 4478824024069, -3273309473431, 3661450392158, -1332495781935],
  [3433238320095, -3537063600267, -2635906680144, 261302237613, -2759443456705, 3386536019652, 949511068238, -2950096444044, -2047497480251, 5963423843188, -493262884399],
  [-4105722097165, -141913655347, -7788314434300, -2620215156652, -2116962346128, -1450024151373, -732778649936, -1080692149409, 1114563921787, 2350123655098, 738406232725],
  [2200620897542, 705226451779, 2723763190353, 3782000913440, -1482406284152, 3009948542426, 5720928376417, -1025025527459, 2998989283179, 3937668088876, -1844267429759],
  [-4001613157751, 3378934508215, 409158451737, 5494366397472, -3327412136028, 1079984219353, -2631470696356, 4020893842667, 1536948449196, -1726419747723, 1241540389589],
  [888673029462, 2258870927984, 2595865553564, 5350662068447, 2747261226114, -588630358854, 3020228413670, -2016224262166, -1280527406098, -400562929971, -5988204141971],
  [-784448296601, -864748064806, 559798999754, 1688253545921, 240653265587, 1648793052913, -17280348977, 5756292949770, 6805727075975, -3089022346053, -1923969062188],
  [-2732454644794, 1183497496595, 127337376432, -1243692020947, 2061785440351, 2186341094142, -1357024113404, 1082754121094, -1857273260280, -3914021204478, 7665252583820],
  [-1236793825863, -2333318951451, 211305234637, 2283356120529, -2629243609853, -5271440586761, 1723316268126, -415823216033, 4601979636555, -5151650335502, -1485309781336],
  [2821642382102, 2906951566716, -526257070986, 283003410036, -1816610494370, -4630780065571, -5094770360351, -822804971694, 1629306854930, -4497641960600, 2994931491381],
  [5757045691793, 187120190092, 3814983805307, 225282311117, 632342828617, -4181668994943, 5188218310499, -1815462636634, 503375258259, 788144965033, -1800630830432],
  [2678559131102, -555673908285, 1769279437351, 2948585676564, 4394538007670, 2199184553610, 4013515708219, -2604227000232, -4707767382466, 2690870493540, 2061431279089],
  [379190190073, -2714425600161, 5334870609655, -1772639083263, 1735877144851, -1931477068771, -2679193262334, 3669975714046, -3492163017355, -3229146038935, 3297234694602],
  [1518980188259, 4863668733161, 2236438537843, -1739700465146, -2429011831799, -2695460666052, 4238729018848, 1324838215596, 3326618888075, 4511503228801, 1304426815049],
  [-2033226249345, -405083959238, 463453849128, 2230853372887, 1182847258739, -6433452357703, 3685783962573, 116931247361, 1510246758153, 3814439635060, 4158424407346],
  [-951517934441, 1675539718712, -1336285940935, -1645421904301, -2336910100273, -2747786607563, -2184017666043, 1806679730034, -2597175549173, 7708279758434, 2141425780852],
  [-1083879915157, -4640600124494, 4167562887098, -2758177942872, -3592361677113, -279859805331, -2652509510016, -1477927943039, -2277043972392, 4944375002945, -691862540020],
  [-3898439760039, -257945608795, 323292417029, 151022835822, 4501929287046, -634192149356, 5746049858676, 2593260480066, 2269947530578, -3311852390405, -2841763833478],
  [141864048396, -2742792494647, -954616381445, 1821875393772, 3728473011512, -5212023826256, -1451554866067, 354522574070, 158471783785, -3132772093270, 5923484286679],
  [1071769248680, 334438752973, -3760862096385, 3089607262596, 3359666382977, 6606247254319, -2968708276988, 215987412493, 2515190151398, -2201877893616, 292432061488],
  [-1230530900302, 2771727322867, -88812425037, -911548894030, -2053580890639, 3960865394137, 6240083048958, 5272634654412, 1413139988991, 1103161460414, -324993353976],
  [-86998330244, 1058762300965, -5641212497529, -3083499428291, 3245614970178, -3524730716672, -698745775891, -5232054699530, -1592394393518, -1715033802990, -1114791669436],
  [4809659318974, -149233818971, -5314475619759, 2436109542017, 3216650845950, -1668718122847, -4689437464696, 970733506968, -1702018773990, -1901992922741, 295923281742],
  [569079240735, -615142865604, -1688765527873, -3981623417149, -2356414755070, -2577678633195, 5070931344481, -3371149123615, -4372302060906, -3469182077811, -405136618554],
  [-900259136377, 812233153653, -2247324243777, 5222932711560, -2659657768943, 5571312149424, -298641624508, 2116878044993, -1279522203441, 4522091249610, 1195898864850],
  [902141610581, 3298918161640, -3342757931425, -682634882119, 3097205669747, 1059949340361, 1339024616970, -1293379261324, -4136728035009, 5057105421452, -4449093393518],
  [-73353143388, -815510154657, 2852365403161, -1111929579292, 1441350269881, -2025644047242, 3093622515804, -3125572943450, 285987548666, 7905010645558, -1365933247070],
  [2465951952348, 6328200956618, 18797023916, 194486333831, 1417719448242, 1679612858144, 4017966457568, 1466430593215, -2758941377335, 454029325736, 4784493105961],
  [-4109098251814, 4344566738087, 1241576641643, -1336655497978, -2058584846899, 801204721188, -4497140463018, 1970940831292, -3318281163024, -3749067380961, -2618551918968],
  [1648454925486, 4747015799246, -3471514447645, -3561631541054, 201564745507, -2919312907498, -1240784771082, 1963570428022, -488321627252, -5111461208275, -3112577709569],
  [-2551031962307, 5068904330152, 2119147294801, -3762883681579, 2615881186145, 1034576539254, 4104081891625, -547162595789, 2990336877677, -2732468275682, 2771968680612],
  [-3245165420869, -170031800418, 2008342982470, 3956781345125, -1036193413253, -2028762948150, -4275902223348, -1111504962239, -3408998867350, -5018672581355, 2869446580626],
  [2180933012478, -546194667748, 3575643651836, 2455368469823, 389037085827, -2570081736166, 1314204101992, -5721618825168, -3927732407186, -4401729105455, -328809010366],
  [3945590192550, -1756704785784, -137454169152, -1789453902934, -2979714527466, 1252082126175, 2555339462242, 4431903196124, 4945122565474, 1738074490083, 3745956922482],
  [2259619080308, 4590487947210, -2251005923542, -1947039143183, -2494199878457, 3514421935768, 2391274822796, -2570700740949, -561771487022, -5792662736345, 440604120366],
  [682425135818, 587302795151, -317869972891, 2871329834968, 6175415094364, 2049437638613, 1643654972210, -1128116887017, 2969019410103, -180019819214, 5973785736564],
  [-4671784204114, -3636066015725, -4541700480367, 2150041272614, 3184135409938, -4505677577062, -2292223611054, -700095384547, -1667730528470, 331259751111, -792720660805],
  [931616503018, -1118731315337, 399845528585, 5763565676285, -2927098487698, 1494653556681, -2860994615124, -3713538466516, 3681274686970, 3835698208950, -1860120472308],
  [-2660324687047, -191977827658, 4155438805119, 1558555138949, -4256397452858, -2912318234252, -1266917535702, -2756313453107, -1000147214125, 2275249532232, 5586692736924],
  [-3399778343912, 6159145363346, -3063878260556, -4689453897494, 165191722795, -292410107528, -1867435094055, 2612690604473, -333003423324, 77320108261, 2930013664243],
  [531147775146, -2170328967690, -2293201254125, 5367543080577, 121326811130, 4429621544785, 3170478944287, -5098314400427, 1619930635276, -576670726531, 1516748305541],
  [327608737716, 3811429821160, -3932622782162, -826291283558, 4066076816263, 3047328830449, -1581534415805, 320899075464, -5970199045313, -2078284885189, 911827996371],
  [4518785347160, -388700493896, -2490485817584, 5359924247762, 2449270871360, 498109529865, 1367907884520, -424892833282, 5044293942255, -116702307384, -3277333634716],
  [-6525162079463, -1181354204005, 624956783856, 358842171217, 3469986050917, 255099458968, -915621384173, 3284176993775, -5185525820581, -2168287046212, 431412415300],
  [-2941652726062, 3110504473894, 821394292308, -3588079192813, 2753639156092, -2618552523411, -5756456571727, -2919392392152, 2519754256098, -1237747428259, -2035256728565],
  [5048423056420, -2313715010096, -761025759937, -1804248447970, 3211761556085, 726679387348, 1117003513562, 2661551201487, -3278751911642, 339342731636, 5940141443264],
  [-4042579749446, -6750360135928, -4466751606843, 1681946903260, -1228916050969, 3098704888002, 571813546750, -833218627166, -581856657591, 1654660302331, 315409511224],
  [-1035333893018, -1442118298059, -5617578658270, 1864867197505, 3156317086217, 3338066033401, 4743005829326, 2175921776371, 3365096684350, 1379639723178, 500414999183],
  [-11427126637, 7367594985755, 610962666431, 1110263111105, 1087256905897, 3002689697382, -3838425077956, -471264988323, 1351394896230, -3377921610308, 2392062008724],
  [1358649557230, -462924873050, 2962306138089, 4756468859486, 3530257458269, 3657347279216, -3369408194109, -5111547699335, -689873076083, -1041547577939, -1288422971606],
  [-1702441183899, 2379753603282, -1977514619740, -6150487283618, -3941633834821, 2272934930821, 1398178305322, -1214292023464, 5041711426171, 290921398220, 252589314494],
  [-1789898852424, -6045978389703, -2407782197450, -1215663486795, -669694319880, -197516091123, 875316813228, 3264967464244, -2738960314402, -4523497111234, 3617945509846],
  [-690529249041, -551543126829, -1669071497316, -3584852110174, -472677388588, -4932348321410, 4822503098792, -3153350238000, 4686045472618, 1009143113924, -1689075334590],
  [-3403775108391, 1476258452330, 1623721323956, 3065601595098, -510393100108, 3238892394793, 1707768905743, -6124079953293, -1785076817159, 3782462175439, -2352725566758],
  [2396927824505, -1801977200565, 1370352050487, -4811131694918, -3379559646078, -1225803264230, 2281600137297, -3168189949940, 886080762421, 3919090990916, 4655234166250],
  [-1503549869037, 1212735318063, 1076597747714, -2066580829533, 368020916081, 3447110116613, -6157411151982, -4483552814227, -3629149843989, -1778234244327, 2114705819539],
  [4462028326936, -1517183397319, -559064623985, -3994470582582, -1160814586133, -1990740040881, -6491343119880, 483609499542, 257883507284, -1776945142174, -3257932052113],
  [3614730897172, 422382984848, -4387680174406, -4667356932295, -2659814430804, 793535112010, 782431041005, -3043534264706, 208539181470, -732100135520, -5249789136291],
  [-1647666307448, -1710410004391, 3156752345801, 4341228612939, -578114935938, -138079452183, 2126032864910, 5033459758869, -1675582532975, -5703048434848, 83595080804],
  [-71611541693, -2208735633139, -3149735813939, 174588134464, 3916295713013, -1950257452486, -3625851960195, -222009405516, 5048747575851, -4908852220951, -1800386877248],
  [-757479774973, 1842556725253, -5335399910162, 2409191958230, 1896290788129, -4407192826846, 2176298076868, 267030096684, 748547596232, -790242619193, -5722661824073],
  [-1708067341610, 3778225389386, -5739863819085, -849033045991, 5333610382016, 503226217254, -1805456153799, -183182963432, 3486590152969, 1757208882586, -1379371915984],
  [-1372830112462, -26521209511, -2124080741896, 1688948421360, -2531925342951, -7620134073784, 133402939606, -3907749557550, -2114497263376, -1692611313949, 1910221036734],
  [6964785946583, 2532233513343, -655135734296, -587153760045, 750596564141, 1991363355268, -2056984275819, 1289181637869, 3749221382394, -4405034503841, -650008468001],
  [-1729882284524, -1344800787370, -1317512698711, -2649263530170, -2705350082060, -1070400469548, -2711924816032, -4646510085482, 934003050063, -6279053663051, -2955746292466],
  [-1274233595323, 5412938535530, -5424784225187, 2834538558180, -96685163611, -1338972520537, 2243850168311, 2928084938293, -273640652930, 3877623040267, 1044965244792],
  [2898149547620, -2990432543296, 247745768225, -937802201036, 2045083409460, 1891270956134, -6501832935269, 252749135059, 1534982709221, -3375257889106, 4227489575329],
  [-2269865736199, -4594524252363, 2258191837167, 1940818929425, 2495428789378, -3515406037013, -2389572202205, 2566217428711, 576709992011, 5784526702722, -444671457975],
  [-2245342351779, 3384856863440, -3568161851038, -1754563009282, -1186980873045, -3900929284580, 629702048159, 1640587456022, -6769181311722, 545232058360, -1362935439985],
  [-1437312260121, -1847697111240, -3049619556769, -7835377675664, 2880335165688, -1331949009768, -945504108148, 875356400851, 2987731743090, -466280584155, -1718203090590],
  [-278532810318, 2179880920650, -2011458323430, 1781485798113, -2859562978183, 1726766616329, -1351503498302, -2376402753467, 7188032423483, -4181862900095, 403013457950],
  [-4066954318074, 2566602669850, 3255939520713, 4387344278805, -1887028511272, -944891597795, 460897739197, 898848728882, 1811289058759, 6184899102792, -121259486508],
  [2345548484243, 1039005251056, -365378882558, -1226402450103, 2673509261365, -3721614510411, 4030589000797, -5334143996077, -142721473165, -1006686008224, 5005068766952],
  [4341459031588, 4484224767691, 2332866018647, 3544769452216, 1375523081600, 1309149831450, 2636685662476, -4361177622051, 2528756086563, -2417713797695, 1103603852692],
  [750664436098, -2148943522252, 4008663719182, 1762059704738, -1869060518421, 7708557739294, 1945574678840, -1790633954168, -2004669972305, -687062970437, -1116542776830],
  [4329647159964, 1262405360551, 2440979596453, -879866742125, -1927676754522, -2131337489055, 3443513668473, 6450458962922, -3211827601687, 305451932838, 891100160463],
  [2193404209828, -3317090496399, 122821783843, -978804771565, 2462904558619, 2600348067812, -1585390611262, -6654730960632, 297410577503, 1642402629124, 4560722388450],
  [-2409481474165, -1064732547599, 410769153834, 1188086537465, -2665858946672, 3714006215367, -4019149086204, 5305475113973, 235643584566, 955830409373, -5030810211646],
  [-845161422135, 3925130330303, -2077708073922, 5179976208925, -3200810750409, -723860328055, -1555830229190, -337924643751, -3427840044559, 2255422092479, -4752793863350],
  [-1168114860623, -3822826448574, -5030151779254, 1997211921609, -3051362506056, -1525745276917, -4531922239602, -1632910473648, 970029768864, -2800300569351, 3332289694638],
  [-581794081007, -2595079464845, -3123555538763, -6773249377, -6874918744212, -1391281133731, 532315418523, -112614262325, -5048355315029, 1671291716108, 2322738285131],
  [2338012133440, -1336683953743, -3115196655662, 613370202792, -2374543281464, 5833833429157, -1156933975158, -1302463235103, -4390871575893, -1174256566617, 4393255672731],
  [3891039367281, 2404273366108, -564822641231, -1541134362753, -1423497284225, 1936471871239, -454771626662, -2158090210442, 3181298476237, -1668814410250, 7269052382347],
  [-2922513445504, 8259732626733, -200487366516, 412388800769, 2969837437489, -919097665817, 1201968335381, 420914703816, -1022507411409, -642740879182, -3206284945675],
  [413661737289, 1618152381840, 2340126452014, -7269061056510, 290262527334, 2130166416306, 1486826857866, 3708222574137, 117242906970, 3307994981739, 2712013942179],
  [-3403108019382, 5241633649204, -3092714393079, -2995093677834, -2318344914351, 1801341565116, -992301959469, -4822779014794, -2128535474010, 262292422313, -2223912831128],
  [-3171278517329, 1036972010098, -163496287217, -3097558909873, 3999139351372, 601001380339, 1711936961218, -3465559911694, 1956544678212, 1185749576185, -6535813520357],
  [2579929796494, 1485575743088, 4112675668364, 1154406856819, -2756282810637, -860769984091, -4468232066595, 5422364409564, 699865213083, -3564103950269, -1411548383478],
  [1316062228161, -3831074514174, 5731057985101, 3994159989126, -1191850856492, -436276944900, -5541018219297, -41243974686, 181432992600, 610688088654, 1439173634386],
  [-2667525179584, 4965987709141, -2544676139347, 951835957644, -2728676268728, 1186067930727, -4715622352279, -1182564942954, 3632600866661, 3760780591228, 1007304374680],
  [2453277201468, -491631561492, 786969316878, 3033260699249, -4049219737779, -289916809971, -1342406267171, 3788364791102, -2045268192298, -519415590365, 6843451545319],
  [366765670903, -1200692417917, -1429467588701, -1496779860285, 1391348379768, 4134240253425, 1275738234114, -7138858643012, -3162845062064, -2649152656940, -2345184615911],
  [914341258819, -1781328446830, 5225313203447, -2312903454284, -1915917711758, 4421371639718, -2200482167067, -197760517690, -979348654132, 914118035842, 5782922630427],
  [-782180416810, 2052558781591, 2470844921564, -5517471215525, -84460969577, -4465076441973, -3115934462584, 4968069001646, -1239746927806, 356061150341, -1621444586483],
  [2355142884752, -1017634208111, -1736346495382, -3010885033044, 6053769566543, 2670424431904, -4806248354162, -2995681693375, -210057713591, 889905356, -2332347547099],
  [-2645384386648, -3900273217887, 1163765540114, -3009688646456, 666557069730, -688711131228, -1335498699546, 3861250470782, 304172533591, -5656942428963, -4203841385314],
  [-2615974363572, 1031046264966, 468394941374, -4180199374433, 3651122727237, 4505931176975, -3351586434352, -1101201563112, 4177012293650, 1099521763330, 3108680079658],
  [-136897195983, -1081886913472, -657609981089, 6686299501168, -1254690698723, 245381970475, 1924363243205, 1922466896425, 5223373108159, 1821950711408, 3746342618508],
  [2424156496536, -2198726064092, -2884756557450, 903276598819, 975991121091, 758060057008, 7373144182348, -2427240019001, -338080780553, -425095511148, -4251396499489],
  [-6325490988486, 2598656058563, -1207708054039, -1211334362887, -5737679409125, 1245158976971, 244137249672, 2617541998517, -1033672732075, 2668395449981, -859029787814],
  [-2772603497675, -5685522069233, -361553801111, 1482882840208, 5553003984629, -593738427560, 3659694568985, 1136926900543, -1335602128256, 1289653471776, 2887293051407],
  [-4095417183648, 4839098221725, -465435241876, -2479348325824, 1395032968339, 2992221080202, 2556976049731, -799439188010, 1550778200364, 5741502567849, 19751812624],
  [-6908934403500, 1531241071359, -4706719133239, 2128676614342, 1734681290776, 2186163589365, 1764336556458, -387799201584, -1686620354607, 1057206905840, -2867643679802],
  [-2140503593039, 561984352407, -3604049075333, -2430617584736, -394188466297, 2573789311727, -1320334788844, 5739390159704, 3868688028118, 4433547836917, 344374165714],
  [-1893666539574, 1659730122758, 946544030408, -76189385339, 20559221468, 571107687498, 2129920519145, -5678538610919, 6180191631216, 2276775563167, 3502877120755],
  [-2794745255581, 372415077265, 2972006867864, -2348892576132, 3501669213866, 161804686635, 1734915599637, -2037765678905, -1587118565352, 3999728050819, 6303453485896],
  [-968718573149, 5689743280001, 1952593569916, -5300349284827, -3695445383604, -448512674898, 3189704448263, 1223541944931, -2084765359924, -1293321077701, -1798445430447],
  [-1525591131074, -4866300238791, -2231803949435, 1735707161763, 2429785224139, 2694759754916, -4237612395193, -1327733563264, -3316991137642, -4516772902518, -1307125169001],
  [-4466940599372, -1647027909491, 2531558090634, -3028270226670, 910724778055, 192877798160, 344148556492, -1765984879127, 6830022701532, -3059479270272, -1281072994700],
  [1541635846422, 1362353363053, -198072058243, -316239710940, -4095584293555, 5659216425601, -4483688773694, 3175096487836, 3329121932947, -665912022636, 2261940867051],
  [-437105715614, -2149437760686, -3236107456867, 6005716132078, 1287626975401, -1604415904986, 2739810393095, -592828819392, -2493162444723, 5324874624305, -1409804529466],
  [1397647866505, 7312735332159, -3192838943251, -1429753386275, -370962974316, -2656027893092, 659706429203, -3466751520431, 3313054721054, -432560781839, 1234078745832],
  [-293708146266, 4440039470869, 432358239719, -5887353939842, 3500817164191, -4164343477674, 1982663104286, -1491944365746, -1311915602379, 2801491320417, 167661571787],
  [-6163626891441, -136261226569, -2176276951445, 622796697888, -2011615445323, -3964961823186, -1210594725027, 2980786895674, -904406545254, -448590894926, 5072449632079],
  [385073785918, 2021446299101, 936796987436, 3667782702405, -1063153037472, 5814427729415, 2760137975099, 1761773698065, 1632899051403, -3975044593693, 4160269612796],
  [109714879158, -120550740434, 6360099365214, -3047956367250, -457719420703, -1620627096654, 3283373657054, 4116387541021, 2998192254453, -3266540573839, -109342959566],
  [2816298045683, 5843722832103, 2492182163736, 1654309583123, -639788473056, 3284375995947, -433160286357, -2696718584479, -5261469995038, -372036857907, -1579178515081],
  [4664308006134, -395601340473, -1392181786846, 91212707612, -4217859713708, -609179267588, -5649307507006, -2511802374320, -1501802234682, 2648283927022, 3238387930209],
  [4735606214913, 6260817852, 1358663626156, 84369215894, 3699432657352, 2648486372163, 6050758041547, 2492903514265, 2945068298062, -1838695677162, 374376326382],
  [5417322970567, -2065248616302, -519393174767, -5889580242966, -750460081537, 4570124985149, -1835866028015, -901827283530, 1979879462268, 962684788193, 975100689454],
  [-911512194716, 565281354632, -70196043250, 1921790595333, -415630650321, -6556423890214, -4662785919001, 1404684530327, -1785704456781, -2705015278984, -4215613080425],
  [-1418676255514, -2529630289464, 2732865434791, 42265025956, -3126937889025, -229710682698, -1326022638311, 1808071533522, 4989560834424, -3938532880436, 5371430314699],
  [-2154770173062, -1655489775893, 6405363148531, 546322166172, 1272231190292, 2843001042335, -1745559831129, -1136676524727, 3206356279165, 2677931970476, -4449370082130],
  [3167144720180, 3400465286298, 209022404118, -2100076134849, 5075564382316, -609283725349, 3961560836596, -2381212947343, -2292497261332, -3903117524233, -2442422997698],
  [-288840861068, 1594920895700, -1527469002679, 2656358099910, 3564908985902, 4911908294, 5131175474934, 4519124677249, -5082505477121, 280208763309, -1616468280120],
  [2158636040022, 3513858812998, 2587785249793, 4869552631870, -1315394109919, 5044557560708, -1420213024157, 1038640418726, 2446280530658, -1280515133208, -3832271169069],
  [-161207628592, 1424471926574, -2257894915604, -7117922473962, 1060663107915, 2350965116796, 1047783051663, -3719425115065, -1954167589074, -747088199132, 4027537035630],
  [2970021068420, 90202386471, 2792402654131, 415340747262, 4472871111723, -4616263033670, -5600231136442, 2573077535170, 1900778557469, 305707229798, 441162857580],
  [175691387717, 12983192677, 811036497227, -3740551373609, -4422972150623, 1993335857013, 4512377635188, 1617341950259, -1397734151434, -2806884824984, 5382729663013],
  [2260116487747, -31441253112, 4462528514067, 5599611951951, 3850480954428, -2189852968882, 216461989754, -1718012042775, 2842688854909, 3517794520944, 738397699658],
  [3510207103853, -6112307016110, 3003367419328, 4758336321777, -190958098368, 297496767962, 1871066810970, -2557954314180, 169590084297, 4470543314, -2908332380072],
  [-4056853878728, 2088231267532, 2306312239637, 1295837021287, -5357441368825, -1313994440561, 1542310809410, -3065882661537, -5088082843024, -1662287496516, -1150674485511],
  [-4196431697300, -1517019629998, -164153174035, 1261121010131, 4237229602749, 6496602069360, 5994676524, -2510946868028, -2367512895552, 453987039040, 2489010883646],
  [-3812995858287, -446150972528, 1029791281779, 3279436620691, 277503140652, -3743425449490, -4126554994269, -1774179768184, 6042491720568, 293725272166, 1606998591499],
  [-3777648277536, 909754842129, 2951619154550, -470607509255, 3094849665213, -6401365658946, -2916636724929, -664522333345, -3269020243665, 1238465293599, 2060088943158],
  [-4168834440227, 3619474011711, -2374851631283, 1294784688238, 1360399529916, -4471080017925, -235437613868, -4339691111753, 240612553421, 4458017410701, -1241947169126],
  [-3956823318298, -4744591949417, -90763831885, -3724676022218, -5814213810699, -2038890118243, 1502989889968, -1565786109659, 1831121126423, -1007317820938, 954166256105],
  [762150529743, -4708941881, -1237543599039, -2548647433955, 145343016379, 3056651884393, -3645833955480, -334735077049, 6467857952495, 653714211045, -5134497014594],
  [-750814928953, 2148867259446, -4008573099659, -1762178348707, 1869101610402, -7708544182978, -1945580238424, 1790570978275, 2004905471401, 686933180014, 1116499587732],
  [4083148129093, -4439835127532, 5235858794965, -2414215105043, 82444671866, 2444225410125, 3331102481012, 1208773168150, 63373379130, 3437159794961, -128913033634],
  [-773814964513, -222898812318, 3568293909779, -2909424648814, -3401226932222, -6584682394845, 2936928259888, -88014711735, -2948957039760, 2421586839552, -184232961144],
  [-815764090125, 1944969591284, -4012422151591, 1422208542377, 1299999338763, -259239012726, -3562650436673, 3226679983667, 4740167343852, -1370950528653, 5312210159669],
  [1604476820817, -4429392743063, -5529218278565, -4407828402920, -1777715771558, 1355318695197, 4339320904962, -380491261787, 843969667101, 30503134227, 1766372319428],
  [1900004359421, 3193119945072, 879428022823, 2140011849071, 1518589953933, -5905485332377, 915396365000, 2769300395353, -2479123920671, 4561357908670, -2863744936299],
  [-1774755740880, -548200206493, -2969017914696, -3529848401167, 1437887855970, -2989713695603, -5746601802979, 1192653194931, -3584817931168, -3609419357469, 1986008658016],
  [-1625213889932, 4421119860839, 5543654927871, 4395226412743, 1780196769319, -1357444062802, -4335770064592, 371441570966, -813723749688, -46935696425, -1774620718433],
  [3715263790834, -2507441356067, 916933214642, -1140143779240, -1613057939667, 1637675316378, -4053977038224, 5891778174886, -2113162615126, 4002114501913, 924535278642],
  [3559719361406, 3593611015819, -470681462291, -4340318713557, -831229915233, -1558565867977, -4044317435861, 2202917506989, 4648560925856, 2343087643899, 1981376332508],
  [-5149326266331, 425278277647, -2358074079089, -4157768090984, 4060365081707, 363375547590, -3004681046676, -1898453697484, -3150592017271, 3351575609492, 214239081516],
  [-1885574942447, -3414768961929, -2237633145328, 1009180982198, -1258984424464, 83569138486, 2748852970600, -2104550426355, 845115067889, 278675248156, 8024293216243],
  [-2616397571692, -557163011524, 3825473785314, -264303622221, 3632263036396, 1472196243522, 3117970572795, 2829316578089, 5699698566962, 3100518803696, 1718335559607],
  [-2001171217114, 3769109590606, 4843242734316, -2207726731685, 20425655823, 3295100129708, -666418356220, -4366181203628, 719822543059, -3684959001579, -2999207897268],
  [-416440675793, 3966595907281, 5404744038843, 2659058883616, 2152299922905, 5211934991105, 283496230637, -898514893236, 536756726137, 2884298612590, 2553271449435],
  [335206399612, 2429619549291, -1339859634853, -2402137072018, 3268570292751, 2164671951558, -6122786239087, 5227011286623, 1193309401770, -1771237023356, -1295498554102],
  [-4535646281527, 381955225200, 2502373372548, -5370195363497, -2447289461544, -499927932217, -1365080082561, 417493881829, -5019672401578, 103242561889, 3270547937599],
  [1320294581199, 4358886860887, 498976946644, -1109624493511, -1473532016269, -5062368760463, 813784390254, -4271521122302, -2127754988860, -555742065341, -5121863011168],
  [319954682508, 1644930456827, -4048011124793, -3586693983952, 2262601583584, -1140899394730, 2693872741624, 452698386592, 3558061465140, 3516425128220, 5388238523679],
  [576511139734, 1064716969756, 3503843991704, -3344260267626, -7471489976141, -941661781581, -1519077763886, 3002302559918, 663039771013, 1605726864235, 2005922589557],
  [4250970101161, -2011115256118, -2442314255295, -1179417975855, 5333733062342, 1334393134560, -1574038371390, 3150621287536, 4805338266734, 1816225427635, 1226974967377],
  [4983683345524, 1852620815355, -3743000131140, 1791614456828, -567655115089, 2415036182526, -3770153436093, 1353588916043, 865467369408, 4540416849173, -3308391327027],
  [653875695174, -2088800981864, -396296213420, 574777457479, 6775229964712, 1746981249329, -217765778774, 4178116225117, -436521096935, -5077695128414, 1512824196198],
  [6338261765354, 1103907876255, -495193503598, -471410004876, -3451108817434, -276797009173, 953344819002, -3358985667948, 5441258081614, 2023964355435, -508173452306],
  [2536643074779, -3692952713363, -1574451231015, 3508714524187, -2094648436830, 2337704889084, 5790495109123, 2579128237328, -2852311810587, 988057334451, 2447749092585],
  [1107940979252, 4585036193841, 2271843849403, -965363534190, 1620650767048, 3533508575939, 1463699849379, 4695963528445, -1064452003406, -5208530782449, -2021923639670],
  [-5180192026516, -682001819054, 3365130146399, -1146825897072, 833223008031, 3879974708325, -2289723829805, 3278672359678, -1386497537237, 4975083926998, 1283100178528],
  [-1173309723898, 1972452553536, 2451361482700, 640154543412, -1461649408553, 3335640582544, 2487877767457, -5842578928918, -2412034499213, -1801630687007, 5065751601085],
  [-1861323243252, 7134713267597, 1043143150418, 2407895374026, -4374893776181, 2563292203874, 2861480832555, -1695338029534, 1327685478187, 348924414534, 297206271745],
  [-3160682237136, -1293519066942, 7051223326360, 1212818181618, 1240982054689, 1292819826873, -946419879752, -2511751426302, -5074675901170, 439922048758, 885621582510],
  [2188964509271, 3776040312346, 599471719195, -3057340945717, -1053173182133, 1489062314445, 5484454041137, -5695084326043, -279665896719, 2114489660823, -924268607476],
  [-2476893720297, 34883886820, -3998421890342, -3476509194583, -303750020300, -2556222197138, 1420669576580, -1107807790381, 3628549044514, -5341968597365, 3770306885879],
  [5580243107323, 870088036249, -3653724378671, 1377611388532, -854444995382, -3805450560520, 2204562554778, -3085267538823, 792883755047, -4628634029265, -1101316943758],
  [-443073828773, -2527575553027, -2277870089923, 3697915945056, 519660245092, 4937386511160, -256261999794, -1844351342821, -171254223366, 450416274550, -6797384473024],
  [-1614063693797, -5687794257250, 631118897913, 614539461954, 3872879641616, 667496606968, -1141353611169, -4459858887340, -3008967932029, 3376122946463, -2679170226183],
  [-1077503001626, -3179364363585, -6355780313745, -1760217878449, 4672660384934, 1558429221160, -1551256510307, -821218393081, 54578924800, -901856944419, 4132401386776],
  [474012047316, -418492364892, 935216300467, 4101362436092, -7784530923283, -3021692423649, -3072488345516, -563620609805, 92653544193, -1445327407673, -565170540846],
  [-727291774990, -1199846378894, 1522043036747, -6424513445156, 690286592565, 4094735859791, 4355456852779, 48083592644, 762082613769, -3107376393562, -2826339611100],
  [-4785539510058, -195122238138, -87015655469, 1355600434426, -1894721462168, 6343721346916, 895517818241, 69205432358, 5249358217569, 1388620904496, -1044879626466],
  [1403217816097, -2759955368282, 901952695799, -2830822998223, -2369047932369, -1967888992644, -1727162378428, -3656626225376, -2806465725575, -3912460635556, 5706484915651],
  [900933303598, 3782303821132, -1238220147270, 1579435167099, 5047188700148, -1757508394950, 5764458100935, -1695557742093, 2723593858156, 2873081511427, -717836154020],
  [-5591360915532, -2299262997255, 1023283552056, 6123785470364, 542599258027, 1749957055294, -3776886321330, -428384580942, -1507417469899, 1669069049272, -1427924670054],
  [-4320802718315, -4408013786734, -951706198440, -404213484961, 5292587337125, 3116007011938, -1327154063206, 1881795726692, 3074349217745, 540156817422, -2839732225352],
  [-833007262294, 4275286650201, 1266984179126, 3194602883065, -2303239298218, -1413718556553, -3341802901801, -7042332430762, 194373106103, -911851377636, 531440032197],
  [-503498478826, 641644035566, 1642707177561, 4021124148244, 2349026710612, 2585350178490, -5082390229479, 3399573781690, 4277067791291, 3520956838531, 432036574956],
  [-1075519744251, -657866402426, 2279633816118, -1583392676961, -872446027515, 3055025066170, 613936841094, -472056376972, -5999177194784, -6602001415793, 660273253248],
  [5458222620096, 726821298710, 2895790786049, -2423330058281, -2830370552335, -594589717906, 1054714318464, 54724324427, -2323222848422, -5038937807997, -3892710328751],
  [4925348223428, -4238808160264, 1883829333124, -696988168671, 1612785621524, -904493180526, -3575203377554, -2438749496283, 2613419908066, 4676413794434, -1700373938607],
  [789135193341, 876771811315, 2994340714412, 8164545731350, -2280921572686, 485848730350, 1031053925182, -1120268957357, -3550513617260, 344133324190, 1581566277484],
  [1148805594811, -1982179045540, -2434235799568, -655088635527, 1464554557163, -3338144399259, -2483848910744, 5831855229633, 2447698493683, 1782260254513, -5075501792497],
  [759758233706, -5381479911596, 5463015225044, -3259655955371, 419547755608, 1674286216232, -1590599768081, -3251143232941, 385436596648, -3472422706879, -1306550293429],
  [5850570351917, -2844347723006, 1152180391276, -1028321395205, -1803891869981, 2328543877781, 4261580125936, -4060654542183, -83855060725, -2791238368926, 2042396090035],
  [1021382773756, -522249914981, -6452390609, -1854192881588, 401939677286, 6565397550965, 4646202339613, -1356365726256, 1625089296953, 2790761783288, 4257201916257],
  [-916864521284, 4882119279119, -3037114480034, -3598840689545, -1856465954116, -2498825285240, -490199796195, 1425503727696, 1881741783977, 3800615982163, -4816634486056],
  [3228333574343, -3775735676247, -1337034225973, 3272909243213, 1640070794750, -7339972984229, -162276523455, -567635823245, 1489156798299, 1078987798315, -1589561147745],
  [1009310519958, 6374344250331, 1321053132808, 278696672762, -1448952220730, 2205414665864, -1469892187714, 5270798397805, -2122826126532, 3721495116100, -1124741123393],
  [3884090917285, 989485866732, -2756010325512, 2997744873691, 613459920307, 3384135229556, 581056758324, 7411009364290, 161905946025, -432413260354, 223835611333],
  [-6416220723, 62298005769, 3509315492983, -4613741636480, 2127034776469, 5178045787303, -446050563912, -3615014478561, -2191147944382, 4018469496647, -917179569092],
  [-367232182889, -2442285948819, 1361271870123, 2381968867543, -3265462546848, -2167879350239, 6127632329518, -5240087527930, -1146828021226, 1745666144638, 1281263630456],
  [-1251017490073, -4642095748751, -2171086712114, 879411328420, -1608892765072, -3552415305994, -1436676767663, -4762639888399, 1269985074694, 5097038000814, 1962471083642],
  [-864633438705, 2007639737117, 549361299094, -700145530508, -6756070553071, -1777328588218, 244120353697, -4279423278929, 741244752520, 4912354536451, -1603955937975],
  [-2047245747673, 4353208118734, 4466611782840, -3548383600830, 4999839198597, 1454755980125, -2458960259203, 1357907326478, -2777115559538, -417104527389, 1193844659660],
  [3675941328561, -3486332771489, -160820464138, -5684858841984, 3339235986597, -1121182615107, 2683706915249, -4143352587015, -1057443892780, 1469492407342, -1390920364707],
  [993837045276, -438636370991, -8717966456784, 1766684229717, -1069629155048, 2133201807113, 1194040007098, 219210991929, -2630650088855, -1554823751875, -1785927066321],
  [-1367498723704, 6042596120959, 1537745030550, 770188839683, 4053118337687, -2500307022642, -1634141626027, 799918225857, 2835543828405, 3421800261642, 3594511289418],
  [982823525219, 1200708454647, 1945809394940, 4024567441855, 792309730526, 4512350191374, -4653124964118, 3383451539119, -4439864514732, -1275738318566, 1476739467597],
  [1106754610890, -2394320131487, -4897587067212, 292142796785, -1555079481674, 3451029272877, -1532062569004, -1802822589670, 5499495710060, 2971643738652, 3157568531073],
  [-4345262074852, 1492498015734, 2537782846725, 2481763074580, 6679536299018, -19914908484, 942943074417, -290013121900, -1700796117075, 3988954204029, -1377440865793],
  [-5432043423830, -2493218984646, -841654449445, 3599258880566, -348924243872, 484952891758, 2333307556399, 6303940442131, 332828845772, 2211691559452, -265415436677],
  [-6529936089701, -2957302376195, 1140672433647, -3261543281125, -1437066055571, 3684338528247, -1442955524331, -2187329993338, -974580392896, -619358671550, -3582742616297],
  [-1116902797656, -6743508315541, -1817602116827, -434253387542, 2035342670101, -1329262058964, 2502140641844, -4715189368996, 1732358794722, -3514538145381, -170437457561],
  [1029206877767, 1439657260208, 5621866832294, -1868589302411, -3155615871098, -3338652844717, -4742001464113, -2178623887785, -3356167984795, -1384513098671, -502864970606],
  [320429667982, -3946305475847, 930598799560, -1497434135835, -2953541775817, -6590623479233, -4481515560774, 1560827004200, -104091485319, 698423516255, 2457062236060],
  [-2978034195832, 430650162067, 988431484766, -2343115725466, -2048729141708, 1978716107729, -1092585951018, 8021485138991, -973519475706, -2070889786771, 2364691006950],
  [-2712219526408, 3310764168929, -1214934419161, -833709350239, -6332709171084, 1023517680586, -2711222605694, -1506671847696, -578139731924, -3026126981594, 4387288104757],
  [-48903615025, -4144457369607, 3793232874556, -4122724679103, 752987362722, 452285299083, -3863667028717, 3829371623270, 4513875875097, 831222628119, 77709434942],
  [-666721920985, 1088966220636, 1634692132718, 1319565594151, -1376175422408, -4184119051681, -1246328486604, 7021352075837, 3601519529610, 2399648537017, 2230129144183],
  [-450541913644, 2941073593038, 2689672157268, 794082250564, 6644727806254, 2166885399331, -1002974015615, 45546992483, 4739837281813, -2258073274905, -2421035200007],
  [5497457887295, -3331913189351, 2120338954097, -812017651650, 1035748752659, 1874777695388, -3730526907220, -1404709821964, -5727686966612, -471728256119, -120698876988],
  [4191754506567, 1514866414373, 167435490699, -1263952486962, -4236564175885, -6497242769183, -5043047408, 2508947347210, 2374411993924, -457576869117, -2490797587303],
  [-2398785273054, 1801277758244, -1369097835142, 4809980281419, 3379854663624, 1225743869128, -2281363772418, 3167390354183, -883402227426, -3920501243994, -4655889932572],
  [1980737205664, 3053404339992, -323943331253, 20867510076, -1777393188178, 6718044530180, -1819063817558, -5467050226464, 2011280082124, 1021475674295, 265493871800],
  [1599869272571, 3377719330174, 6001091289038, 2081212663557, -4732197244452, -1506882132131, 1470853105028, 1047753312878, -814439480096, 1323626136882, -3917398298387],
  [4083313123547, -3653278918306, 2434503600644, -1346087693035, -1349610752951, 4461562678186, 249213063476, 4301172841954, -116976792089, -4525602358660, 1207228751899],
  [3505121213808, -54964681561, 6330466537123, 936869135621, -2245033441714, 1052802142894, -56606474689, -5736285575286, -98364524252, 2725251134633, 514937353695],
  [-3826788354731, -1549339011126, -2816648506289, 823919117818, 1815495823546, 2433189957751, -3034549349066, -6366145289961, 3649625483909, -303992348639, -1406405494359],
  [-2879258515750, 18437974896, -1788210766892, -2916196539595, -3765367125024, -2883992614280, -4247599018791, 2690967326177, 5052300992980, -2295362186083, -1197661048381],
  [85474414588, -1059316670792, 5642243025391, 3082520650536, -3245404137150, 3524664313803, 698971571389, 5231397282827, 1594622910311, 1713869112672, 1114247507789],
  [-2571844755038, -1063082595609, -5931752930656, 4827706638804, -682261633973, 442064640260, -2804656597975, 2023493164644, 3777253088939, 1111726471262, -2374431254449],
  [-1451865255754, 1434879333664, -222934803148, -6032820997610, -2754518011731, 2875699145837, -5660480943146, 761341028648, -493538755737, 2870569005255, -1557925806967],
  [-3597540747759, -2287267736214, 358447275322, 1718727051754, 1388467521087, -1907068375136, 404805979464, 2287631596342, -3608626971544, 1902172164019, -7151312756131],
  [-3056848776557, 3910743207505, 275583471771, 670911991649, 2757070913723, 5404126904843, -4416514500153, -892044492083, -608187213609, 2020639000540, -3643548954192],
  [-2751876334569, -1550898849626, -3990090800205, -1259031734331, 2776101594701, 846161726660, 4491773842344, -5498344852050, -446646521187, 3420148409171, 1337324771812],
  [2126034257521, -566761087013, -3202512097426, -955255048591, -1957779120038, -98981002933, 5343663038295, 4232773194344, -325280990750, -5643641570750, -1311436791707],
  [2020061716733, 399958811246, -454228609744, -2238824832238, -1181283168353, 6432062026078, -3683641723228, -122700509248, -1491086193600, -3824909829059, -4163691453763],
  [2144395714720, 1048885248121, -2961713568381, 862342851146, -3262508813018, -962393920531, -3358586808125, -2840852389626, -6185303554465, -3784145993324, -1134711364446],
  [-7141283903264, 551834985988, -1228956085633, -3427703495950, -79846573493, -4673140541172, 1013599254678, -369132877622, -117029324101, -2203279605677, -2749847384473],
  [2870850102456, -342530910644, -3025183865207, 2395044282745, -3510929228914, -154488423091, -1747033003302, 2071335827742, 1476515852691, -3939638776374, -6273676579891],
  [5455606189908, 5161643859152, -1963267172191, 1962918350939, -1506569440660, 152020201372, 4080574760052, 1488678826955, -919845174390, 710634322540, -3657197615865],
  [1666038769592, -1144981454845, 2413758433311, -6344060479149, 2341156973304, -1828752695926, -4756556500031, -1554846357901, -1327644699779, 2221633325014, 3045320137186],
  [392718162477, 4676739216630, 77425302885, 1417772730177, 2597757214163, -3343608510592, -2224972537263, 5445498862369, -3671191664282, -1706044551920, 2652838335825],
  [3724160656674, -4061502039499, 1023701547041, -1786358074463, -4415514817647, -4179494048051, -316437645212, -4851662235880, -425996373440, -300935959329, -2126041476570],
  [-5364912915398, 3342606599752, -1736109834737, 750035218692, -1193393702016, -2098009179891, 2967467968446, 1770139858878, 6204110394335, -6003298785, 460376157721],
  [-2393626640652, -1049091665942, -5122478167056, 1088697622782, -5105574070497, 2435637978174, 1707308916357, 3641923304831, 2375835061606, -1693161318275, 3009285224425],
  [-5137934506729, 398416258386, -3064932278532, 5125884313080, -4636778408632, -467235353649, 710025386347, -3211870763345, 686833201600, 1279034186343, 1768743224370],
  [-8499479993195, 2896057304196, 2300168240403, 381215387805, 1211236786556, 912532188414, -1081041844269, -2481562130878, 1138235680116, -391226534827, 1691254723136],
  [4033905237261, -4871081723045, 527943332897, 2438280532272, -1392948478908, -2994883443748, -2550199728912, 785654231548, -1459817677244, -5797650556225, -44856538943],
  [1798605426441, 1782407567411, -127633268326, 2585887676284, -6406453074812, 3248170520160, 1039266846782, 3233776473994, -4249459430586, -2128554974155, -1079650933168],
  [-1524879561131, -1348790227653, 185509040260, 329713680789, 4083642895560, -5661792069994, 4485155389945, -3159021193549, -3361256296350, 675473040821, -2264953045065],
  [415419510304, -3420403757556, 4999987567914, 53920267208, -2835870013600, -2987658367312, 1388260272541, -1118843304886, 6096363081975, 2392770253011, 295213822969],
  [5427952107234, 2491529961799, 844488585689, -3601761663589, 349389474711, -485436373845, -2332590907306, -6305750705375, -326824207595, -2214997798918, 263710502677],
  [5410465593212, 2284182716040, -1830665274215, -2386366134810, 3881010473360, -3743818152518, -734591050674, -1191235830145, 2912744793533, 1133676778387, -3956880585831],
  [-1519966253416, 5499495095313, 701947719793, -2418777804335, 600867197418, -2887137450951, -525054769308, -3236045504713, -3827991322506, -4555732971445, 2499908413479],
  [-220262711735, -21686679736, -6301349010170, 3604978483523, 1990534831616, -3278942072131, 1224944290631, -3308018241872, 3625732944572, 109840321507, 2633054746403],
  [-4610510376437, 2813617129952, -3151596993439, -88494734667, 5624819370040, -2857888666106, 2454279412895, -443157633996, -1303047844300, -865935395631, 3523149627136],
  [1591898023250, 2469941412807, -5931303111509, -3304855274361, -876943572352, 2133776999267, -5194459233467, -1995517495373, 1538603604609, -2276570139335, 1194245096370],
  [-5854130447786, -227585494107, -3740308754695, -288690684276, -617533036803, 4168516334893, -5171221429765, 1767360391392, -361225444154, -867392702819, 1760194993347],
  [-2073846400455, -4864443057694, 3768191015592, 3279233077955, -127179189053, 2890636750283, 1306208403755, -2158360568070, 1156806226339, 4731226850683, 2936725638233],
  [4380402244915, 1718336099159, -2490203993633, 2962792650483, -609296656817, -88183771411, -528514858921, 1835636669616, -6875078311611, 3073438996418, 1462704076511],
  [1808548016756, -7156596836882, -1005793147626, -2439553382253, 4381084164538, -2569424600789, -2852359039920, 1672380191489, -1251321929623, -391277073385, -318876263817],
  [3050479025203, -1112913585673, 583195931072, -2411256428011, -6500725040245, -177287830264, 5844127903559, 597134495908, 253318857497, 1220454761246, -2224137621907],
  [2081026457771, 4576326717134, -3059971157905, 1791758352530, 1929136083966, 1520812361717, -2120604198642, -4189323624596, 840252242315, -2988107401845, -4943288443379],
  [374633385085, 4064005181314, -3643593868621, 3916456117646, 731274504682, -1390998890607, 3891975595409, -3931218738608, -4577567048717, -775483008911, 319402267704],
  [-2835830561561, 6330276368832, 1469030071465, 402575127112, -3894140626988, -5490744757063, -1778186499571, 725501790480, 51664708241, 528979064893, 531180438358],
  [-407377873713, -1054311025402, -2688240679126, -400274214617, 3049297157362, 2934281209053, 6032558300772, -1387844054570, -2051260454929, -4417782970588, 3374173701078],
  [684982537657, -4959087143757, 106581022743, 541496847341, -5426086784485, -1687822946375, 394145178486, 6213307814871, 787828294802, -1036369422923, -1373765538818],
  [2436940108262, -17959317412, 5272745875180, 1859736836422, -572591553586, -196355684278, 4728622193259, -100172273328, 1443302561770, 711844762922, 6121630990387],
  [223238508073, -5979661064274, -3475906742063, 3953573669553, 2030325741149, -732468271069, 1890864460635, 3262533163758, -41044222721, -2549195724778, -3332092252171],
  [-1774327400291, -5768496654472, 2193622125140, 685714364485, -1578315854585, 2649364870676, 5560588530162, 1445106797385, 3397613217391, -2045535048810, 217585806122],
  [4658041545519, 3630655850203, 4551294374768, -2158382540553, -3182453315019, 4504288573550, 2294451726266, 694082709894, 1687704540905, -342007246067, 787331961924],
  [-2465951597827, -1864199930418, -2270856798900, -22595841598, -3664322575097, 410068708466, 4800776928729, 516836974097, 2660325854959, 6212676802611, 1643732824380],
  [359846782248, 1044409315702, -2799990553119, 262881376730, -5481381738941, -4220241657587, -294707117620, -304138737129, 4283597060107, 1718359722968, 4640405561466],
  [276745595187, -4446743546668, -420469848217, 5877043034202, -3498803237174, 4162570629251, -1979809669506, 1484541890021, 1336631902838, -2815013300598, -174470552559],
  [5102172976455, 4259278196381, 646116293517, 2524356648706, 6102964143435, 1254045003892, -957282256921, 1873342983288, -1573270140866, 1489934935322, -1047305543583],
  [-3026034433384, 4955504388403, 4333613435828, -2343097902290, -1131529879827, 1679795382350, -2094050038152, 2926412538950, 4863771372296, 454595688732, -1049686222394],
  [1413569263672, 5664984198517, -1424796239863, 3110895550026, 552999270359, -2663403492361, -3366801478812, 2831708540909, 4442817380487, -589019091899, -2713492019071],
  [-1196383610997, -2317590034549, 488155267722, -1612569113807, 6813504685085, -2793562988534, -651338217079, -3533045594143, 4362308711116, 1960841909849, 591300922110],
  [-6225964858091, 2664169319390, -865309055360, 819974520373, 1830992948396, -2381820387657, -4175153119521, 3871551417818, 656947569754, 2466614045870, -2180885828971],
  [1769560500047, 6037989935725, 2421935205712, 1203216048338, 672174795176, 195496479510, -872012665847, -3273813092151, 2768564775195, 4507389782533, -3626020480703],
  [-5047746784092, 2453874805493, 605654139215, 1972416669127, -2665071988780, -422111933646, -2198438897961, -1931046200821, 3511718736225, -872604552173, -5941804355173],
  [-3328181454099, 2292646681385, -1230196268189, 4178873463660, -1952023152818, 5264778689679, -1386241486348, -2997510223232, -2074354162481, -4120755939342, -985908910127],
  [-2128722363465, 3342442538371, -168399550678, 1018145277613, -2470638175877, -2594108650010, 1574952019260, 6683203855831, -391972896290, -1591148785755, -4535096826577],
  [-5201779311507, -5745047858643, 1148085852099, -1972658788736, 1626021341995, -443420182872, -3687980730355, -1722445336866, 1126867930447, -728342002724, 3676619806419],
  [1025623808061, 2255781933926, 3434448250170, 1588728653730, -1672468315028, 1399827775510, 7540246372905, -415840499655, -2622968673028, -3238462954919, -622524241645],
  [-2330888881958, -783041291864, -1611954800518, -6710003615179, -2409429846717, 164483025058, 2876722005477, -1406424960130, -2560402079163, 4389258928773, -2103339319444],
  [981353914797, 4600570118021, -4095848035320, 2695778349891, 3604971925019, 268230099620, 2669684981719, 1433109276176, 2427347451707, -5026339548167, 650854878249],
  [-577346357670, -1065056354349, -3503260666052, 3343754775146, 7471599453113, 941566908039, 1519206252295, -3002675888721, -661863774295, -1606353643907, -2006230896461],
  [-1530270045251, -1165038214408, -3910191264386, 42757199935, -6749578664506, 3655208406975, -3037805775025, -2087407439549, 1129782270412, 146803346119, -2684898890488],
  [1124648695462, 619841252926, -4343167328282, 5126149833326, -2012444152893, -4492681408815, 228998673850, 4372547039269, 461299896200, -2790084369521, 1344378474112],
  [-4934004008631, 4235433403434, -1877806736947, 691697509341, -1611705070234, 903752864656, 3576555598625, 2434967123936, -2600892774175, -4683201034835, 1697030426423],
  [2586198910446, 161208459786, -4102174412712, -1602983963062, 4264078758861, 2903065212760, 1279685247477, 2722901928486, 1107788946521, -2335008002528, -5616653031119],
  [3230590122770, 164261854428, -1998144233601, -3965645048161, 1037949781618, 2027214434103, 4278313498655, 1105158967135, 3430220607132, 5007148562462, -2875265811886],
  [6761580377743, 1346954619468, 1557368388076, 4516065361338, -1017022327120, -2592367674560, -2820017724509, -1495826289031, -1863906044044, 102998318679, -2866539059950],
  [-1377509159657, 455485182438, -2949168797229, -4767997985688, -3528005486674, -3659205157792, 3372542508170, 5103307954216, 717304332569, 1026608060109, 1280917880731],
  [4576680007827, 3360652753004, 1053724557420, 1053718472281, -5382930598816, -2808844103951, 2350474900212, -2196771652022, -3356114285095, -478194092956, 2613636825032],
  [2852163474361, -1135949180783, -211725018832, 1315891010135, -2075288507285, -2173478161662, 1336495634981, -1030302943854, 1682346975780, 4009979323683, -7616402455926],
  [3857915221829, -4463500003492, -1062016014312, 1187751209208, 2073567876700, -835202429467, 4556863313623, -2101419111944, 3696351907824, 3543026342009, 2502151362963],
  [401382018489, 1051931974710, 2692405464188, 396635193722, -3048558168232, -2934833001129, -6031609308015, 1385264991156, 2059965050099, 4413057494559, -3376567915437],
  [-169167788834, 880019797476, -711861100149, 3289188658280, -1342563601245, 1346475859727, -8570110050068, -770689718251, 1127827874954, -2257027985665, -1961898300042],
  [3930207101923, -4788272198497, 656309023670, -2770544407001, 557594079672, -1221527037189, 1949795656742, 754170867605, 5202682587302, -2373954082924, -3826510589359],
  [-1070179288209, 2415714774160, 4871960243032, -273380831179, 1555236788003, -3439007596191, 1518887854371, 1816423812368, -5553714292496, -2942765660316, -3138907782101],
  [3940018399641, -2506993712381, 1927018684527, 3369813266746, -4298165936640, -3300625203781, 924147333853, 1210577413806, -875664104610, 5450673764923, 978600957183],
  [-4957499786340, -560490955659, -3196530206184, 2729154604869, 2729502871664, 637868740008, -1108714879636, 155258764813, 1587348802021, 5408907770018, 4068977713458],
  [-2489799141264, 4434234188279, -1202064872044, 7037820836620, 2789740578487, 1506913373001, -408374381403, -2200198212345, 2693826864962, 885099416630, -247788096976],
  [-389022934133, 2710548948948, -5327964571766, 1766636537686, -1734693563450, 1930537485456, 2680796992755, -3674273444797, 3506538711694, 3221342484833, -3301123849731],
  [-1316371471221, 4984144857377, 5149034340701, 1864552720515, 979887311061, -3180597467149, 3423896005233, -3618977271654, -1753560603942, 1737977565955, 1201821181321],
  [-3962236789207, 1750259608810, 148960340831, 1779306059384, 2981987855892, -1253539672826, -2552909536048, -4439189885805, -4920945901236, -1751087652709, -3752301685105],
  [-1246240693862, 23064000757, -1867078026750, 4661938680281, 6095716390748, -3242714454691, 273216436183, 4017729577537, 2942885259625, 791597255920, 223038496552],
  [-30649670188, -3871015798898, 704807144470, -2320378138230, -2357264920905, 3134715677961, 2576542245250, -5342097663914, 3860660551080, 1647128407834, -3309975816110],
  [6894364675615, -1536991055335, 4716871019965, -2137498887887, -1732935394892, -2187576148777, -1761947809958, 381412063761, 1707817215756, -1068696752044, 2861883710692],
  [112529695580, 1072157541057, 674722309129, -6700999288825, 1257537007593, -247891340853, -1920166825819, -1933231865155, -5187788661630, -1841296457101, -3756111585811],
  [-3458907956722, -2289333716918, 1197674347308, -2043672756747, -3032049510746, 79839440230, 4925347625983, 3887792418169, -4954357866607, -935659668029, -1785801254471],
  [-1895276345846, 5269942247678, -4571502851511, 573694574, -1913509641237, 4275131330747, -712911549218, 2758847108459, 1991204978412, -1950884204095, -3147761728208],
  [911004404572, -4884438497405, 3041146246306, 3595341898145, 1857144844875, 2498224328604, 491146027412, -1428032983801, -1873303763754, -3805218707120, 4814323097766],
  [2004940182490, -3377861273990, -2836125064246, -4011742049221, -4826198407407, -1579685127785, -1692090129551, 1004315098601, 2905310241165, 4442787437574, -1610585800723],
  [6321039821476, -2600451384479, 1210817283665, 1208651410377, 5738198705036, -1245592945410, -243411859957, -2619475786472, 1040117842899, -2671921119669, 857236814176],
  [1464911252042, -1429635132299, 214089203797, 6040223039937, 2753642908473, -2874026574776, 5657954751029, -756132119662, 475762436227, -2860233689147, 1563968488668],
  [2024222593676, 3280398978682, -776485984108, 3547678058283, -1037485022957, 101765635860, 1222606531210, -3699551759045, -414345348428, 6291285314975, 3991543685779],
  [418325636565, -976628268689, 1960130308613, -3131328225267, 3158424345724, -1238270377957, 224716563958, 3089029508549, -6964846779600, 3950714977405, -71241800321],
  [1489527748759, -1672027828684, 6450993624, 3553007823045, -3447768798791, -4787063826997, 3771831286954, 782420859434, -3615334610695, -1957132225043, -3977276642890],
  [-1828623208758, -278174491750, 2607271135651, 2827484184789, -3362214664270, -2695356375558, -2098354025067, 4762092301817, -5709012503524, 1853552845219, -328174437859],
  [406777144877, 1082675837846, 8160639820723, -1302165462571, 570799717723, -2118686180845, -1205705957770, 1864618075055, 220710737538, 4169787838341, 1785960103011],
  [5431184070137, -309737848981, 2160845782374, 4325354834855, -4093168843177, -335438513349, 2957437208333, 2026067676495, 2741298099837, -3128261655508, -98395198705],
  [-3801575303236, -1326300030816, 2902747383118, -3164052910919, 102088557005, -2691942486225, 232441481562, -7596192477934, 157609214931, 281536406512, -484284223556],
  [870030828371, 4241678915053, 808293549586, -1616247604007, -787092901633, -132753496525, 3552033927118, 233893752987, 5370834839860, -2915968611203, -5227626775815],
  [-506870497010, -2130343626318, 2149499315013, 6417195180367, 4227919514470, 4464540212850, 2220798395012, 2222111890082, 954554218823, 95595711536, -898061942099],
  [-324237004813, 3944786027795, -927906648754, 1495096796314, 2953976428372, 6590277965583, 4482138600796, -1562494879528, 109672701189, -701466848923, -2458541732536],
  [1245292199359, -23444566233, 1867756989833, -4662601885381, -6095537007032, 3242683991266, -273108949196, -4018152292284, -2941412956581, -792344336969, -223379123929],
  [71316212551, 814716464961, -2850928767408, 1110667068218, -1441092269598, 2025468463968, -3093331235645, 3124664682535, -283034304041, -7906608771207, 1365176650977],
  [-1428258126828, 4442614282252, 6339864332681, 667625196526, -2645336875445, 638054729150, 172956580553, 1998043476530, -2242537274146, -2836925815891, 3616561855795],
  [1862005254671, 3405517571734, 2253927404268, -1023599065883, 1261979031615, -85706852298, -2745053539391, 2094330190451, -810787164646, -297173455525, -8033696546314],
  [-1405446876079, 2759059261876, -900391877185, 2829467448571, 2369321616397, 1967666102388, 1727535445022, 3655643684313, 2809717869209, 3910704397889, -5707371709276],
  [-5674064524733, -2501010713368, -2027147460036, 4003059107952, 1467189200434, 1306497371464, 747398537831, -26249408325, 3599012165758, -4470903015833, 2013800548899],
  [-4061251660559, 3970924364989, -826536677181, 1630672788067, 4420910313958, 4159312858739, 360275634236, 4772058270784, 840163189121, 56015589920, 1984957418361],
  [3100369373946, 3618829690233, -3943659757447, -2085962908171, -5284398817946, 2817370743566, 1310074672458, 1099853485048, -660564537771, 3373159711192, 2604825104656],
  [-1599835715474, 1171222746699, -2460125766453, 6384414730274, -2349076021091, 1835123130434, 4745784133927, 1583862847820, 1231102912196, -2169672507598, -3019234328815],
  [4315279050541, -1504471071267, -2516854702418, -2499608888206, -6675867671970, 16443473889, -937797981903, 277048238808, 1744117134227, -4012814526763, 1365286042378],
  [-2576691553872, -4168355303390, 4537168270309, 3289883489257, 4545811286758, -2343742404114, -883746419592, -423757223990, 1797956997202, -3357840962479, -1717670949614],
  [-2770757495943, -2100276262903, -2310355957864, 815198757292, 378008579111, 6654075747375, 2004799154313, 3938478545291, -1882585389522, -3425197886948, -1637953937110],
  [-1957929560953, 632058228489, 3085463734040, 1058548099344, 1935515723895, 113529956234, -5368507010752, -4158778252136, 79583267059, 5775251335043, 1375182887763],
  [1043608214197, 645438379821, -2257458783399, 1563856913321, 876487024162, -3057822807242, -609133909223, 458157961937, 6045544209448, 6576656903033, -672338544894],
  [-4142674083739, 191452447235, 5844843854843, -3328434054873, -2950087353136, 2629530078902, 3529773720027, -1643439769921, 1641203888894, 1727359214850, 1051648120926],
  [-2536425192805, -3412028873887, 1603269936707, 3714143636044, -5958326431609, 1573982825063, 2066214946180, 169698750068, -1260855487109, 2893560055930, -3649617067653],
  [-3370910924887, -2664833577526, -4212224537043, -1664159625471, -2346009074864, 941991345314, 3116568323270, 1337408376752, 3417509333245, -1300488029900, -5455268464385],
  [658394595659, -5845490182112, -1734937401395, 5132881559044, 3731320560325, 406316780591, -3110371669968, -1368398831925, 2549752914186, 1048224195587, 1674960469699],
  [-4069018215699, 2876255152390, 1999107398479, 288371782958, 2154913562127, -3129067806294, -1567216762796, 2815472413652, 2748385177040, -6162870836304, 859237607542],
  [-4081397478017, 4440571166733, -5237098386667, 2415216733177, -82617245215, -2443974120286, -3331468168048, -1207989621760, -65932252178, -3435715154676, 129657560479],
  [4540975362633, -2962280943473, -3727641214052, -3334039256515, 570980206559, -3755688295128, -2042533319394, -2636445564667, 3169252319979, -272285070269, 3149763104314],
  [-1970685170517, -3212396105451, -1122959696365, -2522166126002, -1796870295135, 5719246575631, 184448827929, -3166927469451, 2390181077127, -4324669286196, 2785111405287],
  [200644095783, -1408319683626, 2230072881087, 7141211846352, -1065713117235, -2345629089454, -1055615018478, 3737240815498, 1897700894272, 777942571718, -4010083583079],
  [3207211747233, -3860205876030, 5140569835651, -471207339362, -337229668079, -4107979119479, -1338074993769, 2629126458727, -1774101172768, 760813963139, -4329765470951],
  [2143894337056, 6169806192886, -1781372515887, -719475094542, 1035253435287, -2386252505629, -5559323574118, -1325703604772, -3012161951007, 2175050982007, -642487683129],
  [-1100041253761, -4027322736527, 229361570027, 1637503933953, 608768884422, 512867167535, -3736704991334, 67174202189, -5291870130711, 3555218083799, 4959030247620],
  [1420617715787, 900534716736, -4641612714007, 858580710316, -2821052711788, -4292847420429, -4102875100962, -3693473208841, 2833909355485, 831540662000, -3050318865731],
  [-4069141454652, 2397030807707, -1277633907502, 1139048765540, -81027369382, -511257043713, 4392208537723, -5619448373269, 2158427361261, -4069786076345, -1551431016185],
  [-2187452393712, 2289066386737, 2720374946090, -754365626582, -1007844680787, -738792081057, -7404298713659, 2534366005211, -13257462581, 611993086866, 4338524326161],
  [-2261267990905, 31011633178, -4461713731260, -5600361739838, -3850335661845, 2189742342684, -216324100412, 1717505044705, -2840927392525, -3518702564105, -738885782573],
  [3835162625076, 4018145639189, -3969032215791, 5489480336209, -3320287131143, 1771420196090, -1544616085553, -1452735700983, 93447194857, -1212172345514, 1766865015618],
  [-3082861861208, 3058375781651, 1165336696419, -124017674997, -2636642417196, 2258840895000, 782669515714, 2647187849760, 2623806578654, 2724227790305, 6766735682713],
  [-3924384276199, -1257058873969, 4552529697420, 387469317643, 1545295213894, 5125273879835, -3071156583424, 1850357772371, 1429128901788, -4262910977786, 645241204936],
  [3701699616746, 2383614266719, -1367424358250, 2190400968495, 3001798884658, -56795233975, -4965441320966, -3781381548309, 4601491412457, 1125849679522, 1881052978621],
  [-2189333732830, -3776207341475, -599211629311, 3057114021789, 1053223858244, -1489095677808, -5484378375890, 5694926621678, 280207726420, -2114773711882, 924126154637],
  [2665403576355, -2944060824958, 3157438546205, 1078691117758, 1382923782302, 4459945621588, -505584133266, -1501254246342, 6777564021800, -1030807481720, 1335749241587],
  [4212558911551, 750153481199, 1028258998901, 2239063169609, 3265243301827, -3837339068027, 1328070175434, 4072573976198, 618748560534, -4246823513919, -3669594372664],
  [-93602350411, -1556291309329, 3891657186953, 3723295327173, -2291127866238, 1160236616592, -2729402661166, -356981559810, -3888216948351, -3338076730838, -5303564299718],
  [-1401779354450, 2278330163184, -1845651571694, -265759043783, -1100547800936, 1597321845900, 2809607617862, -311530978355, -4080685213673, -3942194199680, -6739487572218],
  [-2719919362899, -2391373190823, -3969228213732, 574141537015, -2702607686033, -1963447676327, 1386211653292, -5103770747637, -2713538834489, 1100638362345, -4805724744400],
  [2523998535100, 6205802308153, -740529115356, -5459372639947, 2215638830763, 3653337752122, -638121391235, -468164566807, 914489609231, 343023507309, -2219921307491],
  [-2350524571905, 1019407106351, 1732790704046, 3013675873791, -6054456149167, -2669548888846, 4804552750018, 2998031686008, 203759526543, 2877308034, 2335521273375],
  [5028263042894, 2227314771702, 3244450087185, -2909303881378, 4048893324798, 746603634549, 1165671661409, -2163688951411, 2917552225784, 3511807906586, 2632286844261],
  [-4714905505924, 2895973072505, 3843863522651, 3233205036684, -552014538054, 3737708661071, 2071463682670, 2568124973805, -2931446631079, 135373795872, -3221953642295],
  [-399808288793, -2027286498904, -926551441885, -3676835599757, 1065045172846, -5815841324681, -2757928855920, -1768235050712, -1611389885744, 3963506395496, -4165996467535],
  [637654470917, 1884384072534, 3454373034475, -2817649664288, -4340794990017, -2094754538411, 2347442561387, -4183501013377, 3191236618832, -4027802664085, 1876379562685],
  [1169509591568, 888130497109, -2780123094578, 2233387019770, 2466835615100, 826095556511, -1869080414836, -5231726772561, -402940143583, -5836576954281, 3642284968454],
  [1920808507726, -17069658569, -611385986731, -4447880127311, 851659797175, -3668042836712, 4197058334701, 2683661104136, -1998312720929, 174016363969, -5756043708775],
  [26883657634, -4931237083796, -3796335053355, -4723730642380, -1415711385171, 720768095098, -3722199717928, -3871322798266, -2669188419925, 653348669329, -198240881117],
  [1376090713691, 52794369058, 2105420292898, -1682550534643, 2542698783425, 7624383483088, -148779774205, 3913076935775, 2098374493774, 1721371826116, -1882198869441],
  [681296782274, 2051665305196, -956475448024, 2319926209191, -6356153701200, 700058790681, 518621453633, 3319478656765, 4804216979985, 2728945451941, -2514612942808],
  [1618110226052, -2193439357575, 1695044091886, 397656341641, 1073388847499, -1576671686419, -2843336252998, 407189794653, 3766298781170, 4112633385347, 6822906738995],
  [6164789949614, 136739143580, 2175454310590, -622114272459, 2011485522508, 3965112522977, 1210393364110, -2980277210958, 902750629627, 449521686336, -5071953818337],
  [3881218273012, 1479027175218, 2070483335377, 75560330693, 1057802184062, -1384772852312, 2238953203018, 945086030287, -776171777608, -7173099477397, 4178226133483],
  [3235814803974, 1256294864562, -650900085810, -4903438216654, -3214890352777, 1970650069612, -252075430182, 5758156457942, 1664750015810, -670281194160, -3581101717788],
  [3378074137115, -5251527924600, 3110186772230, 2979998202959, 2321062449968, -1804075580466, 996582370415, 4811856309508, 2164952889186, -282367400925, 2213709332985],
  [574905043173, -4386090861270, -1092429566069, -3352980169208, 2327865872888, 1391131336277, 3388203163344, 6943674361107, 166333751023, 709426251370, -645073660217],
  [-5093483115816, -172520412035, -1119927279528, -298914389540, -3684232686653, -2715581429324, -5996134939421, -2654472304753, -2448900299536, 1566386998418, -533468096138],
  [-694829797925, 1077657175865, 2426418664382, -6898838565028, -630728354927, 891813183909, -3004655039870, 712908865280, 1510580937205, -5343119532124, 1820512127979],
  [-2431689551338, -5427049186581, -2338557330124, -815424187913, 981962290892, -3062993059108, 1128316606362, 2535891852174, 6084947796540, 617493934047, 1744232021160],
  [-5435703869515, -2294163621518, 1848370199428, 2371054182793, -3877945120584, 3741103307158, 738838875423, 1180167173884, -2875974256951, -1153626673599, 3946706754350],
  [8219585512141, -3454414521062, -2804725743538, -219356185092, -1174314716502, -448706854993, 982240147012, 2239687163655, -1333597836815, 384385903109, -1761603754385],
  [1591055287346, 5678683783494, -615012178719, -628567142182, -3870188383064, -669880486908, 1145281971929, 4449831881766, 3042481569158, -3394493236261, 2669963658827],
  [-3754248407042, 5294073804022, -296736245623, 2912063924725, -791691392173, 751000806695, -1982989567750, -407567361095, -5153438369097, 2080369983896, 3624255814498],
  [-1109208301621, 1105002101950, -1209577854326, -2024207507329, 7350334972180, -3488128656973, -1651140239034, 2095339824418, -2280768401335, -1030270576505, -3523292766568],
  [-5069180086335, -1838030708503, 3563978996722, -302815316047, 440743640453, -3187355375955, 4347216998557, -1328131210369, 353915357918, -3762804268885, 3582142309006],
  [-3661582264231, -11651104094, -6216800774575, -1027515731357, 2255363625886, -1074441912663, 87463515947, 5662135623408, 329269044501, -2851689332257, -582416329253],
  [56248471539, 4946765000660, 3758738313061, 4784112795759, 1394864083549, -728062773091, 3727632364237, 3905942835080, 2549768126638, -577510027537, 219804846640],
  [5587427890418, 2297669997514, -1020505798020, -6126153847699, -542119172009, -1750346906402, 3777573935242, 426662882168, 1513176033026, -1672209304069, 1426348449094],
  [259175505863, 4398571312812, -563569394904, -182986310141, 198425934183, -6310433112917, 5597938858309, 1367486286215, -595778885575, -2563939288918, 487534331752],
  [-4802134395185, 159246504059, 1550301624727, -4853626304168, -500450446506, -3445176305248, -1086016460941, -198494483359, 3935712232757, 4174986491713, 2166600367239],
  [-2955758367798, 197182840185, 5575429750308, 4821829786482, -2151221731203, 1377147185183, 1369265209861, -2672300951780, 3787808035786, -2629332593910, 296485252008],
  [-907637263048, -2208932588521, -3516691104412, -1517457288318, 1659108833526, -1386977275547, -7560769135554, 467600743208, 2452516555656, 3332072477470, 670257831686],
  [-5343273458688, 2094742859491, 467642153205, 5934014125908, 741852808966, -4562356494190, 1823039549615, 934097259328, -2087710423337, -903735244984, -945273930686],
  [-7240969135778, -2639902878880, 849460636225, 414183625708, -726198001149, -2021521159959, 2097041348792, -1405395811621, -3323825049514, 4169543904047, 535356746516],
  [-2450513019016, 1863099990948, -2437818958899, -1649975566093, 913002479189, -2735144788138, -3826123267412, -4408375983513, 804673891768, 752627625256, 6185218403530],
  [-1335399479000, 3823343667766, -5717460499457, -4005933034396, 1194120926048, 434347561966, 5544143288165, 32763093488, -153238809945, -626002138389, -1446843381290],
  [3384149124219, -3775174772430, -505726264120, -482507757887, -2778435182795, -5352559874474, 4355591684268, 1038732361168, 127898700843, -1750769851358, 3782290455385],
  [3090835195085, -3054792832893, -1170619376721, 128325298949, 2636523183051, -2257212225518, -784444777154, -2644511696348, -2635072545182, -2717234652626, -6763588745282],
  [-3729763407548, 2337035244907, -846339999856, -4541567007554, 1974270756041, 2750510194087, 636060278562, 4678194935433, -2434295636669, 1169387614202, -4270669080080],
  [-182352232618, -15639990365, -806361709654, 3736501768658, 4423778632034, -1993974011012, -4511280703919, -1620258902598, 1407442349625, 2801596965688, -5385383608572]
]
'''), dtype=np.int64)
assert alpha_evolve_sphere_centers.shape == (593, 11)


In [11]:
#@title Verification of AlphaEvolve configuration
alpha_evolve_metrics = verify_sphere_packing(alpha_evolve_sphere_centers)
assert alpha_evolve_metrics["num_spheres"] == 593
assert alpha_evolve_metrics["dimension"] == 11
assert alpha_evolve_metrics["max_squared_norm"] == 100000000000016749244727635
assert alpha_evolve_metrics["min_squared_distance"] == 100000000000048393639962590
print(
    f"Verified the AlphaEvolve sphere packing showing kissing number in dimension "
    f"{alpha_evolve_metrics['dimension']} is at least {alpha_evolve_metrics['num_spheres']}."
)
print(
    f"max_squared_norm={alpha_evolve_metrics['max_squared_norm']}, "
    f"min_squared_distance={alpha_evolve_metrics['min_squared_distance']}, "
    f"integer_gap={alpha_evolve_metrics['integer_gap']}"
)


Verified the AlphaEvolve sphere packing showing kissing number in dimension 11 is at least 593.
max_squared_norm=100000000000016749244727635, min_squared_distance=100000000000048393639962590, integer_gap=31644395234955


In [12]:
#@title Comparison functions and objective analysis
def unit_renormalized_metrics(points: np.ndarray) -> dict[str, float]:
    """Computes scale-free angular metrics after normalizing each vector to unit length."""
    normalized = np.asarray(points, dtype=np.float64)
    normalized = normalized / np.linalg.norm(normalized, axis=1, keepdims=True)
    gram = normalized @ normalized.T
    np.fill_diagonal(gram, -np.inf)
    max_cosine = float(np.max(gram))
    min_scaled_squared_distance = 2.0 - 2.0 * max_cosine
    return {
        "max_cosine": max_cosine,
        "unit_min_scaled_squared_distance": min_scaled_squared_distance,
        "unit_renormalized_margin": math.sqrt(min_scaled_squared_distance) - 1.0,
    }


def compare_kissing_configurations(reference: np.ndarray, candidate: np.ndarray) -> dict[str, dict[str, object]]:
    reference_metrics = verify_sphere_packing(reference) | unit_renormalized_metrics(reference)
    candidate_metrics = verify_sphere_packing(candidate) | unit_renormalized_metrics(candidate)
    return {
        "certificate": reference_metrics,
        "alpha_evolve": candidate_metrics,
        "global_margin_ratio": reference_metrics["global_max_norm_margin"] / candidate_metrics["global_max_norm_margin"],
        "unit_margin_ratio": reference_metrics["unit_renormalized_margin"] / candidate_metrics["unit_renormalized_margin"],
    }


comparison = compare_kissing_configurations(sphere_centers, alpha_evolve_sphere_centers)

print("Certificate normalized metrics:")
for key in ["min_scaled_squared_distance", "global_max_norm_margin", "max_cosine", "unit_renormalized_margin"]:
    print(f"  {key}: {comparison['certificate'][key]}")

print()
print("AlphaEvolve normalized metrics:")
for key in ["min_scaled_squared_distance", "global_max_norm_margin", "max_cosine", "unit_renormalized_margin"]:
    print(f"  {key}: {comparison['alpha_evolve'][key]}")

print()
print(f"Global max-norm margin ratio: {comparison['global_margin_ratio']:.3e}")
print(f"Unit-renormalized margin ratio: {comparison['unit_margin_ratio']:.3e}")


Certificate normalized metrics:
  min_scaled_squared_distance: 1.000006487462721
  global_max_norm_margin: 3.2437260997220108e-06
  max_cosine: 0.49999621474136663
  unit_renormalized_margin: 3.785251469379247e-06

AlphaEvolve normalized metrics:
  min_scaled_squared_distance: 1.0000000000003164
  global_max_norm_margin: 1.580957587066223e-13
  max_cosine: 0.4999999999997728
  unit_renormalized_margin: 2.2715163083830703e-13

Global max-norm margin ratio: 2.052e+07
Unit-renormalized margin ratio: 1.666e+07


The AlphaEvolve configuration also verifies as a 593-point kissing configuration in dimension 11. Its exact integer check gives `max_squared_norm = 100000000000016749244727635` and `min_squared_distance = 100000000000048393639962590`, hence a positive integer gap of `31644395234955`.

The two configurations are distinct in a scale-free sense. Any reordering of points, global rescaling, rotation/reflection, coordinate permutation, or coordinate sign change preserves normalized pairwise distances and normalized Gram/cosine values. These invariants differ: the certificate above has normalized minimum squared distance `1.000006487462721` and maximum cosine `0.4999962147413666`, whereas the AlphaEvolve configuration has normalized minimum squared distance `1.0000000000003164` and maximum cosine `0.4999999999997728`. Thus no simple rigid, scaling, or coordinate transformation carries one configuration to the other.

The certificate above is also looser, which is better for a kissing configuration: larger slack means the closest pair is farther beyond the non-overlap threshold after normalization. Under the global max-norm normalization used by the verifier, the certificate above has margin about `3.2437e-6`, while AlphaEvolve has margin about `1.5810e-13`. After independently renormalizing every vector to unit length, the certificate above still has margin about `3.7853e-6`, while AlphaEvolve has margin about `2.2737e-13`. This gives an objective separation: both certify 593 points, but the certificate above is much farther from the active constraint boundary.


## 3. A Ramsey-style Problem on Hypergraphs

Problem source: [Epoch AI](https://epoch.ai/frontiermath/open-problems/ramsey-hypergraphs)

A hypergraph $(V,H)$ is said to contain a partition of size $m$ if there are sets $D \subset V$ and $P \subset H$ such that $|D| = m$ and every vertex in $D$ is contained in exactly one edge from $P$. Let $H(n)$ be the largest integer $k$ for which there is a hypergraph on $k$ vertices, with no isolated vertices, containing no partition of size greater than $n$.

The known recursive lower bound is given by $k_1 = 1$ and

$$
k_n = \left\lfloor \frac{n}{2}\right\rfloor + k_{\lfloor n/2\rfloor} + k_{\lfloor (n+1)/2\rfloor}.
$$

The goal is to produce witness hypergraphs with more than $k_n$ vertices, already improving the bound at $n = 15$.

### Results

The algorithm below gives verified witnesses for every $1 \le n \le 100$. For every $15 \le n \le 100$, the constructed hypergraph has more than $k_n$ vertices and has maximum partition size exactly $n$. Thus, on the checked range, it proves

$$
H(n) \ge c k_n \quad\text{with}\quad c = \frac{65}{64} = 1.015625.
$$

Note that this problem has also been solved by a number of agents (Claude Opus 4.6, Gemini 3.1 Pro and GPT-5.4) under the scaffold built by Epoch AI.

### Method

The construction is recursive. For a target $n$, choose branch sizes $y_1,\ldots,y_E$ with sum $n$, recursively construct witnesses for those branch sizes, and then add new root vertices. Each root vertex is assigned a nonempty mask $S \subset \{1,\ldots,E\}$. A root vertex with mask $S$ is appended to every edge coming from branch $i$ exactly when $i \in S$.

If a selected edge set uses $t_i$ edges from branch $i$, then a root vertex with mask $S$ is counted in the partition exactly when $\sum_{i\in S} t_i = 1$. Therefore the maximum partition size can be computed exactly by a small dynamic program over the possible selected-edge counts in each branch. This is the verification method used below.

Most values use the standard two-way split $(\lfloor n/2\rfloor, \lceil n/2\rceil)$ with $\lfloor n/2\rfloor$ full-mask root vertices. A finite seed dictionary replaces this split for selected small values up to 22, and values $43$ through $50$ use a fixed late-tail split $(n - 28, 28)$. The seed dictionary was chosen so that the recursive construction stays sparse enough for exact verification while still carrying extra vertices above the known recurrence.


In [13]:
#@title Algorithm
HYPERGRAPH_SEED_CONFIGS = {
    6:  ([2, 2, 2], {3: 1, 5: 1, 6: 1, 7: 2}),
    9:  ([3, 2, 2, 2], {6: 1, 7: 2, 10: 1, 11: 1, 12: 1, 13: 1, 15: 2}),
    10: ([4, 2, 2, 2], {6: 1, 7: 1, 10: 1, 11: 1, 12: 1, 13: 1, 14: 1, 15: 3}),
    11: ([6, 5], {3: 5}),
    12: ([2, 2, 2, 2, 2, 2], {
        7: 1, 9: 1, 14: 1, 19: 1, 20: 1, 26: 1, 29: 1,
        34: 1, 37: 1, 43: 1, 44: 1, 49: 1, 54: 1, 56: 1, 63: 3,
    }),
    13: ([12, 1], {3: 1}),
    14: ([6, 4, 4], {3: 1, 5: 1, 6: 3, 7: 6}),
    15: ([4, 4, 4, 3], {5: 3, 9: 1, 10: 1, 11: 2, 12: 1, 14: 2, 15: 5}),
    16: ([4, 4, 4, 4], {3: 1, 5: 2, 6: 1, 7: 1, 9: 1, 10: 1, 11: 2, 12: 1, 13: 1, 14: 2, 15: 4}),
    17: ([4, 4, 3, 2, 2, 2], {
        6: 1, 11: 1, 24: 1, 25: 1, 28: 1, 31: 2, 40: 1, 44: 1, 45: 1,
        47: 2, 48: 1, 51: 1, 52: 1, 53: 1, 55: 3, 58: 1, 59: 1, 62: 1,
    }),
    18: ([6, 6, 6], {3: 3, 5: 3, 6: 3, 7: 6}),
    19: ([6, 4, 3, 2, 2, 2], {
        3: 1, 14: 1, 22: 1, 23: 1, 24: 1, 28: 1, 29: 1, 31: 1, 40: 1,
        42: 1, 43: 2, 44: 1, 48: 1, 52: 1, 54: 1, 55: 1, 57: 1, 58: 1,
        61: 2, 63: 3,
    }),
    20: ([4, 4, 4, 4, 4], {
        3: 1, 5: 1, 6: 1, 9: 1, 10: 1, 12: 1, 15: 3, 17: 1, 18: 1,
        20: 1, 21: 1, 23: 2, 24: 1, 25: 1, 27: 2, 29: 1, 30: 3, 31: 2,
    }),
    21: ([5, 4, 4, 4, 4], {
        6: 1, 7: 2, 10: 1, 11: 2, 12: 2, 13: 1, 18: 2, 19: 1, 20: 1,
        21: 2, 24: 1, 25: 2, 30: 2, 31: 6,
    }),
    22: ([6, 4, 4, 4, 4], {
        6: 1, 7: 1, 10: 2, 12: 1, 13: 1, 14: 1, 15: 2, 18: 1, 19: 1,
        20: 1, 21: 1, 22: 1, 23: 2, 24: 1, 25: 1, 27: 3, 28: 1, 29: 2,
        30: 1, 31: 2,
    }),
}


def hypergraph_composition(n: int) -> tuple[list[int], dict[int, int]]:
    if n == 1:
        return [], {}
    if 43 <= n <= 50:
        return [n - 28, 28], {3: n - 28}
    if n in HYPERGRAPH_SEED_CONFIGS:
        branch_sizes, root_counts = HYPERGRAPH_SEED_CONFIGS[n]
        return list(branch_sizes), dict(root_counts)
    return [n // 2, (n + 1) // 2], {3: n // 2}


def build_hypergraph(n: int) -> tuple[list[tuple[int, ...]], int]:
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 1:
        raise ValueError("n must be positive")
    if n == 1:
        return [(1,)], 1

    branch_sizes, root_counts = hypergraph_composition(n)
    mapped_branches = []
    next_vertex = 1

    for branch_n in branch_sizes:
        branch_edges, branch_vertex_count = build_hypergraph(branch_n)
        shifted_edges = [
            tuple(vertex + next_vertex - 1 for vertex in edge)
            for edge in branch_edges
        ]
        mapped_branches.append(shifted_edges)
        next_vertex += branch_vertex_count

    roots_by_mask = {}
    for mask, count in sorted(root_counts.items()):
        roots_by_mask[mask] = tuple(range(next_vertex, next_vertex + count))
        next_vertex += count

    edges = []
    for branch_index, branch_edges in enumerate(mapped_branches):
        bit = 1 << branch_index
        appended_roots = tuple(
            vertex
            for mask, roots in roots_by_mask.items()
            if mask & bit
            for vertex in roots
        )
        for edge in branch_edges:
            edges.append(edge + appended_roots)

    return edges, next_vertex - 1


def solution(n: int) -> str:
    edges, _ = build_hypergraph(n)
    return ",".join("{" + ",".join(map(str, edge)) + "}" for edge in edges)


In [14]:
#@title Verification functions
import re
from functools import lru_cache
from itertools import product

HYPERGRAPH_EDGE_RE = re.compile(r"\{([^{}]+)\}")


def k_value(n: int) -> int:
    if n < 1:
        raise ValueError("n must be positive")
    if n == 1:
        return 1
    return n // 2 + k_value(n // 2) + k_value((n + 1) // 2)


def parse_hypergraph_string(text: str) -> tuple[int, tuple[tuple[int, ...], ...]]:
    if not isinstance(text, str):
        raise AssertionError("solution(n) must return a string")
    matches = list(HYPERGRAPH_EDGE_RE.finditer(text))
    if not matches:
        raise AssertionError("hypergraph string must contain at least one edge")

    edges = []
    used_vertices = set()
    for match in matches:
        raw_parts = [part.strip() for part in match.group(1).split(",")]
        if not raw_parts or any(part == "" for part in raw_parts):
            raise AssertionError("edges must be nonempty comma-separated vertex lists")
        edge = []
        local_seen = set()
        for part in raw_parts:
            if not part.isdigit():
                raise AssertionError("vertex labels must be positive integers")
            vertex = int(part)
            if vertex <= 0 or vertex in local_seen:
                raise AssertionError("vertex labels must be positive and unique inside each edge")
            local_seen.add(vertex)
            edge.append(vertex)
        edge_tuple = tuple(sorted(edge))
        edges.append(edge_tuple)
        used_vertices.update(edge_tuple)

    vertex_count = max(used_vertices)
    expected_vertices = set(range(1, vertex_count + 1))
    if used_vertices != expected_vertices:
        raise AssertionError("vertex labels must be exactly 1..|V| with no gaps")

    incidences = [0] * (vertex_count + 1)
    for edge in edges:
        for vertex in edge:
            incidences[vertex] += 1
    if any(incidences[vertex] == 0 for vertex in range(1, vertex_count + 1)):
        raise AssertionError("hypergraph contains an isolated vertex")

    if len(set(edges)) != len(edges):
        raise AssertionError("the construction should not emit duplicate edges")
    return vertex_count, tuple(edges)


@lru_cache(maxsize=None)
def recursive_partition_profile(n: int) -> tuple[int, ...]:
    """profile[t] is the maximum exact-once vertices using exactly t selected edges."""
    if n == 1:
        return (0, 1)

    branch_sizes, root_counts = hypergraph_composition(n)
    child_profiles = [recursive_partition_profile(branch_n) for branch_n in branch_sizes]
    total_edge_count = sum(len(profile) - 1 for profile in child_profiles)
    profile = [-10**18] * (total_edge_count + 1)

    for selected_counts in product(*(range(len(child)) for child in child_profiles)):
        exact_vertices = sum(
            child_profiles[index][selected_count]
            for index, selected_count in enumerate(selected_counts)
        )
        for mask, count in root_counts.items():
            hits = sum(
                selected_count
                for index, selected_count in enumerate(selected_counts)
                if mask & (1 << index)
            )
            if hits == 1:
                exact_vertices += count
        total_selected_edges = sum(selected_counts)
        profile[total_selected_edges] = max(profile[total_selected_edges], exact_vertices)

    return tuple(profile)


@lru_cache(maxsize=None)
def recursive_vertex_count(n: int) -> int:
    if n == 1:
        return 1
    branch_sizes, root_counts = hypergraph_composition(n)
    return sum(recursive_vertex_count(branch_n) for branch_n in branch_sizes) + sum(root_counts.values())


@lru_cache(maxsize=None)
def recursive_edge_count(n: int) -> int:
    if n == 1:
        return 1
    branch_sizes, _ = hypergraph_composition(n)
    return sum(recursive_edge_count(branch_n) for branch_n in branch_sizes)


def brute_force_max_partition_size(edges: tuple[tuple[int, ...], ...], vertex_count: int) -> int:
    best = 0
    edge_count = len(edges)
    for mask in range(1 << edge_count):
        degrees = [0] * (vertex_count + 1)
        for edge_index, edge in enumerate(edges):
            if (mask >> edge_index) & 1:
                for vertex in edge:
                    degrees[vertex] += 1
        best = max(best, sum(1 for vertex in range(1, vertex_count + 1) if degrees[vertex] == 1))
    return best


def verify_hypergraph_algorithm(max_n: int = 100, brute_force_through: int = 15) -> dict[str, object]:
    rows = []
    for n in range(1, max_n + 1):
        witness = solution(n)
        parsed_vertex_count, parsed_edges = parse_hypergraph_string(witness)
        constructed_edges, constructed_vertex_count = build_hypergraph(n)
        assert parsed_vertex_count == constructed_vertex_count == recursive_vertex_count(n)
        assert parsed_edges == tuple(tuple(sorted(edge)) for edge in constructed_edges)
        assert len(parsed_edges) == recursive_edge_count(n) == n

        partition_size = max(recursive_partition_profile(n))
        assert partition_size <= n, f"n={n}: partition size {partition_size} exceeds {n}"
        if n <= brute_force_through:
            brute_partition_size = brute_force_max_partition_size(parsed_edges, parsed_vertex_count)
            assert brute_partition_size == partition_size

        known_bound = k_value(n)
        rows.append({
            "n": n,
            "vertices": parsed_vertex_count,
            "edges": len(parsed_edges),
            "k_n": known_bound,
            "ratio": parsed_vertex_count / known_bound,
            "partition_size": partition_size,
            "slack": n - partition_size,
            "improves_known_bound": parsed_vertex_count > known_bound,
        })

    required_rows = [row for row in rows if row["n"] >= 15]
    min_ratio_row = min(required_rows, key=lambda row: row["ratio"])
    assert all(row["improves_known_bound"] for row in required_rows)
    assert min_ratio_row["ratio"] >= 65 / 64

    return {
        "max_verified_n": max_n,
        "brute_force_checked_through": brute_force_through,
        "min_ratio_n_15_to_100": min_ratio_row["ratio"],
        "worst_ratio_n": min_ratio_row["n"],
        "worst_ratio_vertices": min_ratio_row["vertices"],
        "worst_ratio_k_n": min_ratio_row["k_n"],
        "n_15_vertices": rows[14]["vertices"],
        "n_15_k_n": rows[14]["k_n"],
        "all_rows": rows,
    }


In [15]:
#@title Verification
hypergraph_summary = verify_hypergraph_algorithm(max_n=100, brute_force_through=15)
assert hypergraph_summary["max_verified_n"] == 100
assert hypergraph_summary["brute_force_checked_through"] == 15
assert hypergraph_summary["n_15_vertices"] == 44
assert hypergraph_summary["n_15_k_n"] == 43
assert hypergraph_summary["worst_ratio_n"] == 64
assert hypergraph_summary["worst_ratio_vertices"] == 260
assert hypergraph_summary["worst_ratio_k_n"] == 256
print("Verified recursive hypergraph witnesses for every n = 1..100.")
print(
    "For 15 <= n <= 100, the minimum verified ratio |V|/k_n is "
    f"{hypergraph_summary['min_ratio_n_15_to_100']:.9f} "
    f"at n={hypergraph_summary['worst_ratio_n']}."
)
print(
    "At n=15, the construction has "
    f"|V|={hypergraph_summary['n_15_vertices']} > "
    f"k_15={hypergraph_summary['n_15_k_n']}."
)


Verified recursive hypergraph witnesses for every n = 1..100.
For 15 <= n <= 100, the minimum verified ratio |V|/k_n is 1.015625000 at n=64.
At n=15, the construction has |V|=44 > k_15=43.


## 4. Ramsey Numbers for Book Graphs

Problem source: [Epoch AI](https://epoch.ai/frontiermath/open-problems/ramsey-book-graphs)

This section contains adjacency-string witnesses for the triangular book Ramsey lower-bound task. For each listed integer $n$, the graph has exactly $4n - 2$ vertices, contains no $B_{n-1}$, and its complement contains no $B_n$.

The included collection verifies 24 values in the range $22$ through $50$:

$22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42, 43, 45, 49, 50$.

No witness is included here for $n = 40, 44, 46, 47, 48$.

Adjacency strings use the evaluator order: for each larger endpoint $j = 1,\ldots,N-1$, list bits for pairs $\{0,j\}, \{1,j\}, \ldots, \{j-1,j\}$. A bit 1 means the edge is present in $G$.

### Results

Due to computational limits, we only ran the Station instance for $n$ up to 50. Among the verified results, $n = 22, 23, 24, 26, 28, 29, 30, 32, 33, 34, 35, 36, 38, 42, 43, 50$ are new and were not covered by earlier known human constructions.

Note that, to save computational cost, a single Station instance was given the problem statement for solving all $n$ up to 50.

Independently, [David Turturean](https://epoch.ai/frontiermath/open-problems/ramsey-book-graphs) reported that their scaffold, powered by GPT-5.4 Pro, solved all $n \le 50$ and some additional values of $n$ beyond 50.

### Method

The witnesses below should be read as exact finite certificates, not as output from a single uniform theorem. The construction history splits into one algebraic family and several computer-guided searches inside highly structured graph templates. In every case the final object is just the displayed adjacency string, and the verification code checks the two book-avoidance inequalities directly.

A common template for many entries is a two-layer cyclic graph. For a given $n$, write $m = 2n - 1$, so that $4n - 2 = 2m$, and identify the vertices with two copies of a cyclic group, usually $\mathbb{Z}_m$. Edges inside each layer and between the two layers are then determined by selected difference sets. This makes the search space much smaller than the space of all graphs while still allowing asymmetric choices between the two layers and the cross-layer relation.

**$n = 31, 37, 41, 45, 49$: two-copy Paley-type algebraic constructions.**  These are the cleanest repeated family in the table. The two layers are identified with a finite field of order $q = 2n - 1$ ($q = 61, 73, 81, 89, 97$ respectively), and the edge relations are defined from the quadratic-residue / non-residue structure of that field. Once the field and the residue-class rule are fixed, the graph is deterministic. These five entries are therefore best described as algebraic constructions rather than search-derived examples.

**$n = 22, 23, 24, 25, 26, 27$: cyclic bicirculant search constructions.**  These use the two-copy $\mathbb{Z}_{2n-1}$ template. The graph is invariant under simultaneous cyclic translation of both layers, and the internal and cross-layer edge sets are specified by difference sets. The difference sets were found by stochastic local search in this reduced parameter space. The search is heuristic, but the listed graphs are then verified exactly by computing the maximum common-neighbor count on every edge and non-edge.

**$n = 28, 29, 30, 32, 33, 34, 35$: finite searches in cyclic/bicirculant variants.**  These continue the same basic two-layer philosophy but use more constrained finite-search models. Some runs fix one or both internal-layer relation sets and solve for the cross-layer relation; others optimize complementary or asymmetric cyclic relation sets. The successful instances in this group were found by CP-SAT or simulated annealing over the remaining difference-set variables. They should be regarded as structured computational constructions, not closed-form algebraic families.

**$n = 36, 38, 39$: capacity-guided cyclic or product-cyclic constructions.**  These entries were obtained by first choosing an internal relation set with favorable slack for the two book constraints, then completing the remaining relation set by finite search. The phrase “capacity-guided” means that candidate internal layers were ranked by how much room they leave before any edge or non-edge can violate the common-neighbor bounds. The $n = 39$ example uses a product-cyclic organization rather than a plain prime cyclic group, but the same idea applies: a structured difference-set model is completed computationally and then checked exactly.

**$n = 42, 50$: Hadamard-derived repair constructions.**  These start from highly structured Hadamard-type parent objects. A boundary deletion or projection gives a graph very close to satisfying the two book constraints, and then a small exact repair changes a few edge decisions to remove the remaining violations. These are hybrid constructions: the seed is algebraic/combinatorial, while the final repair is computational and exact.

**$n = 43$: exhaustive search in an asymmetric cyclic bicirculant family.**  This entry also uses a two-layer cyclic model, but the two internal layers and the cross relation are allowed to be asymmetric. The successful graph was found by exhaustive finite enumeration over a reduced orbit parameterization, rather than by a broad random graph search. It is therefore computationally discovered, but within a sharply constrained algebraic ansatz.

The missing values from this notebook are $n = 40, 44, 46, 47, 48$. No witness for those cases is included here.


In [16]:
#@title Verification functions
from __future__ import annotations

import hashlib


def parse_adjacency_string(text: str, expected_vertices: int) -> tuple[int, ...]:
    """Parse a column-major upper-triangle adjacency bitstring into integer masks."""
    bits = ''.join(str(text).split())
    if any(ch not in '01' for ch in bits):
        raise AssertionError('Adjacency string must contain only 0/1 characters.')

    expected_len = expected_vertices * (expected_vertices - 1) // 2
    if len(bits) != expected_len:
        raise AssertionError(
            f'Expected C({expected_vertices}, 2) = {expected_len} bits, got {len(bits)}.'
        )

    masks = [0] * expected_vertices
    index = 0
    for j in range(1, expected_vertices):
        bit_j = 1 << j
        for i in range(j):
            if bits[index] == '1':
                masks[i] |= bit_j
                masks[j] |= 1 << i
            index += 1
    return tuple(masks)


def triangular_book_stats(adjacency: str, n: int) -> dict[str, int | float | bool | str]:
    """Return exact book-obstruction statistics for one triangular-book witness."""
    vertices = 4 * n - 2
    masks = parse_adjacency_string(adjacency, vertices)
    all_mask = (1 << vertices) - 1
    complement_masks = tuple((all_mask ^ (1 << u)) ^ masks[u] for u in range(vertices))

    red_cap = n - 2
    blue_cap = n - 1
    red_max = 0
    blue_max = 0
    red_bad_pairs = 0
    blue_bad_pairs = 0
    edge_count = 0

    for j in range(1, vertices):
        mask_j = masks[j]
        comp_j = complement_masks[j]
        for i in range(j):
            if (mask_j >> i) & 1:
                edge_count += 1
                common = (masks[i] & mask_j).bit_count()
                red_max = max(red_max, common)
                red_bad_pairs += int(common > red_cap)
            else:
                common = (complement_masks[i] & comp_j).bit_count()
                blue_max = max(blue_max, common)
                blue_bad_pairs += int(common > blue_cap)

    total_pairs = vertices * (vertices - 1) // 2
    bits = ''.join(str(adjacency).split())
    return {
        'n': n,
        'vertices': vertices,
        'bit_length': len(bits),
        'edge_count': edge_count,
        'density': edge_count / total_pairs,
        'red_max': red_max,
        'red_cap': red_cap,
        'blue_max': blue_max,
        'blue_cap': blue_cap,
        'red_bad_pairs': red_bad_pairs,
        'blue_bad_pairs': blue_bad_pairs,
        'valid': red_max <= red_cap and blue_max <= blue_cap,
        'sha256': hashlib.sha256(bits.encode()).hexdigest(),
    }


def verify_triangular_book_witness(adjacency: str, n: int) -> dict[str, int | float | bool | str]:
    """Verify that G avoids B_{n-1} and complement(G) avoids B_n."""
    stats = triangular_book_stats(adjacency, n)
    assert stats['red_max'] <= stats['red_cap'], (
        f"n={n}: G contains a red obstruction, red_max={stats['red_max']} > {stats['red_cap']}"
    )
    assert stats['blue_max'] <= stats['blue_cap'], (
        f"n={n}: complement(G) contains a blue obstruction, blue_max={stats['blue_max']} > {stats['blue_cap']}"
    )
    print(
        f"Verified n={n}: |V|={stats['vertices']}, edges={stats['edge_count']}, "
        f"red_max={stats['red_max']}/{stats['red_cap']}, "
        f"blue_max={stats['blue_max']}/{stats['blue_cap']}."
    )
    return stats


def verify_witness_collection(witnesses: dict[int, str]) -> dict[int, dict[str, int | float | bool | str]]:
    """Verify all witnesses in a dictionary keyed by n."""
    all_stats = {}
    for n in sorted(witnesses):
        all_stats[n] = verify_triangular_book_witness(witnesses[n], n)
    print(f'Verified {len(all_stats)} triangular-book Ramsey witnesses.')
    return all_stats


In [17]:
#@title Data
BOOK_RAMSEY_WITNESSES = {
    22: (
        '1111110111001110001111000111010001110010001111001000111010010001111010010001111101001000111111010010'
        '0011111110100100011101111010010001110011110100100011100011110100100011100001111010010001110000011110'
        '1001000111100000111101001000111110000011110100100011101100000111101001000111001100000111101001000111'
        '0001100000111101001000111000011000001111010010001110000011000001111010010001111000001100000111101001'
        '0001111100000110000011110100100011111100000110000011110100100011111110000011000001111010010001110111'
        '1000001100000111101001000111101111000001100000111101001000111010111100000110000011110100100011100101'
        '1110000011000001111010010001111001011110000011000001111010010001110100101111000001100000111101001000'
        '1110010010111100000110000011110100100011100010010111100000110000011110100100011110001001011110000011'
        '0000011110100100011111000100101111000001100000111101001000111111000100101111000001100000111101001000'
        '1110010011100100100010100010111110111011010001100100111001001000101000101111101110110100000100100111'
        '0010010001010001011111011101101000000100100111001001000101000101111101110110100000001001001110010010'
        '0010100010111110111011011000100010010011100100100010100010111110111011011000010001001001110010010001'
        '0100010111110111011111000101000100100111001001000101000101111101110101110001101000100100111001001000'
        '1010001011111011101011100001101000100100111001001000101000101111101111101110001011010001001001110010'
        '0100010100010111110110110111000110110100010010011100100100010100010111110110110111000111011010001001'
        '0011100100100010100010111110010110111000011101101000100100111001001000101000101111100101101110001011'
        '1011010001001001110010010001010001011110001011011100011011101101000100100111001001000101000101110000'
        '1011011100011101110110100010010011100100100010100010111000010110111000111101110110100010010011100100'
        '1000101000101110000101101110001111101110110100010010011100100100010100010111000010110111000011111011'
        '1011010001001001110010010001010001111100001011011100010111110111011010001001001110010010001010001111'
        '1000010110111000010111110111011010001001001110010010001010001111100001011011100000101111101110110100'
        '0100100111001001000101000111110000101101110000001011111011101101000100100111001001000101100111110000'
        '1011011100010001011111011101101000100100111001001000101100111110000101101110000100010111110111011010'
        '0010010011100100100011110011111000010110111000101000101111101110110100010010011100100100011110011111'
        '0000101101110000101000101111101110110100010010011100100100111110011111000010110111000001010001011111'
        '0111011010001001001110010010011111001111100001011011100000010100010111110111011010001001001110010010'
        '0111110011111000010110111000100010100010111110111011010001001001110010000011111001111100001011011100'
        '0010001010001011111011101101000100100111001000001111100111110000101101110000010001010001011111011101'
        '1010001001001110011000011111001111100001011011100010010001010001011111011101101000100100111000100001'
        '1111001111100001011011100001001000101000101111101110110100010010011101010000111110011111000010110111'
        '0000010010001010001011111011101101000100100111110100001111100111110000101101110001001001000101000101'
        '1111011101101000100100110110100001111100111110000101101110001100100100010100010111110111011010001001'
        '0011011010000111110011111000010110111000111001001000101000101111101110110100010010011011010000111110'
        '0111110000101101110000111001001000101000101111101110110100010010111011010000111110011111000010110111'
        '0000011100100100010100010111110111011010001001011101101000011111001111100001011011100010011100100100'
        '0101000101111101110110100010000111011010000111110011111000010110111000010011100100100010100010111110'
        '1110110100010000111011010000111110011111000010110111000'
    ),
    23: (
        '0000001000010001010000101000101010001101010000110101000001101010001001101010001100110101000111001101'
        '0100011110011010100001111001101010000011110011010100000011110011010100010001111001101010001100011110'
        '0110101000011000111100110101000101100011110011010100011011000111100110101000011011000111100110101000'
        '1011011000111100110101000110110110001111001101010000110110110001111001101010000011011011000111100110'
        '1010000001101101100011110011010100010001101101100011110011010100011000110110110001111001101010001110'
        '0011011011000111100110101000111100011011011000111100110101000011110001101101100011110011010100000111'
        '1000110110110001111001101010001001111000110110110001111001101010001100111100011011011000111100110101'
        '0000110011110001101101100011110011010100010110011110001101101100011110011010100001011001111000110110'
        '1100011110011010100010101100111100011011011000111100110101000010101100111100011011011000111100110101'
        '0000010101100111100011011011000111100110101000000101011001111000110110110001111001101010000100011111'
        '0001000010011000011011101011110101000100011111000100001001100001101110101111010111001000111110001000'
        '0100110000110111010111101011010010001111100010000100110000110111010111101111101001000111110001000010'
        '0110000110111010111100111010100100011111000100001001100001101110101111101111010100100011111000100001'
        '0011000011011101011101011111010100100011111000100001001100001101110101110101111110101001000111110001'
        '0000100110000110111010101010111111101010010001111100010000100110000110111010001010111011110101001000'
        '1111100010000100110000110111011001010111101111010100100011111000100001001100001101110110010101110101'
        '1110101001000111110001000010011000011011101100101011110101111010100100011111000100001001100001101100'
        '1100101011111010111101010010001111100010000100110000110100011001010111111010111101010010001111100010'
        '0001001100001100000110010101110111010111101010010001111100010000100110000111000011001010111101110101'
        '1110101001000111110001000010011000011100001100101011111011101011110101001000111110001000010011000011'
        '1000011001010111011011101011110101001000111110001000010011000011100001100101011100110111010111101010'
        '0100011111000100001001100001110000110010101110001101110101111010100100011111000100001001101001110000'
        '1100101011100001101110101111010100100011111000100001001101001110000110010101111000011011101011110101'
        '0010001111100010000100100100111000011001010111110000110111010111101010010001111100010000100100100111'
        '0000110010101110110000110111010111101010010001111100010000100100100111000011001010111001100001101110'
        '1011110101001000111110001000010010010011100001100101011110011000011011101011110101001000111110001000'
        '0100100100111000011001010111010011000011011101011110101001000111110001000110010010011100001100101011'
        '1001001100001101110101111010100100011111000100111001001001110000110010101110001001100001101110101111'
        '0101001000111110001001110010010011100001100101011100001001100001101110101111010100100011111000100111'
        '0010010011100001100101011110000100110000110111010111101010010001111100000011100100100111000011001010'
        '1110100001001100001101110101111010100100011111000000111001001001110000110010101110010000100110000110'
        '1110101111010100100011111010000111001001001110000110010101110001000010011000011011101011110101001000'
        '1111111000011100100100111000011001010111100010000100110000110111010111101010010001111011000011100100'
        '1001110000110010101111100010000100110000110111010111101010010001110011000011100100100111000011001010'
        '1111110001000010011000011011101011110101001000111001100001110010010011100001100101011111110001000010'
        '0110000110111010111101010010001010011000011100100100111000011001010111111110001000010011000011011101'
        '0111101010010001010011000011100100100111000011001010111011111000100001001100001101110101111010100100'
        '0101001100001110010010011100001100101011100111110001000010011000011011101011110101001010101001100001'
        '1100100100111000011001010111000111110001000010011000011011101011110101001110101001100001110010010011'
        '1000011001010111100011111000100001001100001101110101111010100111010100110000111001001001110000110010'
        '10111'
    ),
    24: (
        '0100100010000101000101100010111000101111000101111100010011111000101011111000101101111100010011011111'
        '0001000110111110001000011011111000101000110111110001011000110111110001001100011011111000100011000110'
        '1111100010100110001101111100010010011000110111110001010100110001101111100010110100110001101111100010'
        '0110100110001101111100010101101001100011011111000100101101001100011011111000100010110100110001101111'
        '1000101001011010011000110111110001011001011010011000110111110001001100101101001100011011111000100011'
        '0010110100110001101111100010000110010110100110001101111100010100011001011010011000110111110001011000'
        '1100101101001100011011111000100110001100101101001100011011111000101011000110010110100110001101111100'
        '0101101100011001011010011000110111110001011101100011001011010011000110111110001011110110001100101101'
        '0011000110111110001011111011000110010110100110001101111100010011111011000110010110100110001101111100'
        '0100011111011000110010110100110001101111100010000111110110001100101101001100011011111000101000111110'
        '1100011001011010011000110111110001001000111110110001100101101001100011011111000101001111001000010011'
        '1010001010000101001011111101110011110010000100111010001010000101001011111101011001111001000010011101'
        '0001010000101001011111101101100111100100001001110100010100001010010111111011101100111100100001001110'
        '1000101000010100101111110111101100111100100001001110100010100001010010111111011111011001111001000010'
        '0111010001010000101001011011101111110110011110010000100111010001010000101001010011101111111011001111'
        '0010000100111010001010000101001000011101011111101100111100100001001110100010100001010010000111011011'
        '1111011001111001000010011101000101000010100000001110101011111101100111100100001001110100010100001010'
        '1000001110100101111110110011110010000100111010001010000101010000011101100101111110110011110010000100'
        '1110100010100001000100000111010100101111110110011110010000100111010001010000110010000011101101001011'
        '1111011001111001000010011101000101000011001000001110101010010111111011001111001000010011101000101000'
        '1110010000011101001010010111111011001111001000010011101000101000111001000001110100010100101111110110'
        '0111100100001001110100010100011100100000111010000101001011111101100111100100001001110100010110011100'
        '1000001110110000101001011111101100111100100001001110100010110011100100000111010100001010010111111011'
        '0011110010000100111010001011001110010000011101101000010100101111110110011110010000100111010001011001'
        '1100100000111010101000010100101111110110011110010000100111010001011001110010000011101001010000101001'
        '0111111011001111001000010011101000101100111001000001110100010100001010010111111011001111001000010011'
        '1011001011001110010000011101100010100001010010111111011001111001000010011100100101100111001000001110'
        '1010001010000101001011111101100111100100001001111010010110011100100000111011010001010000101001011111'
        '1011001111001000010011110100101100111001000001110111010001010000101001011111101100111100100001001011'
        '0100101100111001000001110111101000101000010100101111110110011110010000100001101001011001110010000011'
        '1010111010001010000101001011111101100111100100001010011010010110011100100000111010011101000101000010'
        '1001011111101100111100100001110011010010110011100100000111011001110100010100001010010111111011001111'
        '0010000111001101001011001110010000011101010011101000101000010100101111110110011110010000111001101001'
        '0110011100100000111010010011101000101000010100101111110110011110010000111001101001011001110010000011'
        '1010001001110100010100001010010111111011001111001010011100110100101100111001000001110100001001110100'
        '0101000010100101111110110011110010100111001101001011001110010000011101100001001110100010100001010010'
        '1111110110011110000100111001101001011001110010000011101010000100111010001010000101001011111101100111'
        '1000010011100110100101100111001000001110100100001001110100010100001010010111111011001111000010011100'
        '1101001011001110010000011101100100001001110100010100001010010111111011001110000010011100110100101100'
        '1110010000011101110010000100111010001010000101001011111101100111000001001110011010010110011100100000'
        '1110111100100001001110100010100001010010111111011001110000010011100110100101100111001000001110111110'
        '0100001001110100010100001010010111111011001110000010011100110100101100111001000001110101111001000010'
        '0111010001010000101001011111101100111000001001110011010010110011100100000111010011110010000100111010'
        '00101000010100101111110111011100000100111001101001011001110010000011101'
    ),
    25: (
        '1011010101001010001010000101000001011000001011100000101011000001011011000001011101100000101111011000'
        '0010111110110000010101111011000001011011110110000010111011110110000010101101111011000001011011011110'
        '1100000101010110111101100000101001011011110110000010100010110111101100000101100010110111101100000101'
        '1100010110111101100000101011000101101111011000001010011000101101111011000001010001100010110111101100'
        '0001011000110001011011110110000010101000110001011011110110000010110100011000101101111011000001011101'
        '0001100010110111101100000101011010001100010110111101100000101101101000110001011011110110000010111011'
        '0100011000101101111011000001011110110100011000101101111011000001011111011010001100010110111101100000'
        '1010111101101000110001011011110110000010110111101101000110001011011110110000010111011110110100011000'
        '1011011110110000010101101111011010001100010110111101100000101001101111011010001100010110111101100000'
        '1010001101111011010001100010110111101100000101000011011110110100011000101101111011000001010000011011'
        '1101101000110001011011110110000010110000011011110110100011000101101111011000001010100000110111101101'
        '0001100010110111101100000101101000001101111011010001100010110111101100000101010010100000011000010111'
        '0011001111100101110101110001001010000001100001011100110011111001011101011101001001010000001100001011'
        '1001100111110010111010111011001001010000001100001011100110011111001011101010101110010010100000011000'
        '0101110011001111100101110101010011100100101000000110000101110011001111100101110111010101110010010100'
        '0000110000101110011001111100101110111010010111001001010000001100001011100110011111001011111110101010'
        '1110010010100000011000010111001100111110010111111101011010111001001010000001100001011100110011111001'
        '0101111101011101011100100101000000110000101110011001111100100011111010011101011100100101000000110000'
        '1011100110011111001100111110101011101011100100101000000110000101110011001111100010011111010010111010'
        '1110010010100000011000010111001100111110001001111101000101110101110010010100000011000010111001100111'
        '1100010011111010100101110101110010010100000011000010111001100111100001001111101011001011101011100100'
        '1010000001100001011100110011110000100111110101110010111010111001001010000001100001011100110011010000'
        '1001111101011110010111010111001001010000001100001011100110010010000100111110101111100101110101110010'
        '0101000000110000101110011001001000010011111010011111001011101011100100101000000110000101110011001001'
        '0000100111110100011111001011101011100100101000000110000101110011101001000010011111010100111110010111'
        '0101110010010100000011000010111001110100100001001111101011001111100101110101110010010100000011000010'
        '1110011101001000010011111010011001111100101110101110010010100000011000010111001110100100001001111101'
        '0001100111110010111010111001001010000001100001011100111010010000100111110101001100111110010111010111'
        '0010010100000011000010111001110100100001001111101011001100111110010111010111001001010000001100001011'
        '1001110100100001001111101011100110011111001011101011100100101000000110000101110011101001000010011111'
        '0100111001100111110010111010111001001010000001100001011100111010010000100111110101011100110011111001'
        '0111010111001001010000001100001011100111010010000100111110100101110011001111100101110101110010010100'
        '0000110000101110011101001000010011111010001011100110011111001011101011100100101000000110000101110011'
        '1010010000100111110100001011100110011111001011101011100100101000000110100101110011101001000010011111'
        '0100000101110011001111100101110101110010010100000011010010111001110100100001001111101010000101110011'
        '0011111001011101011100100101000000100100101110011101001000010011111010110000101110011001111100101110'
        '1011100100101000000000100101110011101001000010011111010011000010111001100111110010111010111001001010'
        '0000000010010111001110100100001001111101000110000101110011001111100101110101110010010100001000010010'
        '1110011101001000010011111010000110000101110011001111100101110101110010010100001000010010111001110100'
        '1000010011111010000011000010111001100111110010111010111001001010000100001001011100111010010000100111'
        '1101000000110000101110011001111100101110101110010010101001000010010111001110100100001001111101000000'
        '0110000101110011001111100101110101110010010111001000010010111001110100100001001111101010000001100001'
        '0111001100111110010111010111001001011100100001001011100111010010000100111110100100000011000010111001'
        '1001111100101110101110010011111001000010010111001110100100001001111101010100000011000010111001100111'
        '1100101110101110010011111001000010010111001110100100001001111101001010000001100001011100110011111001'
        '0111010111001001111100100001001011100111010010000100111110100010100000011000010111001100111110010111'
        '0101110011011111001000010010111001110100100001001111101010010100000011000010111001100111110010111010'
        '11100010111110010000100101110011101001000010011111010'
    ),
    26: (
        '0100100010000101000100100010101000101101000100110100010001101000100001101000101000110100010110001101'
        '0001001100011010001000110001101000101001100011010001011001100011010001011100110001101000101111001100'
        '0110100010111110011000110100010111111001100011010001001111110011000110100010101111110011000110100010'
        '0101111110011000110100010001011111100110001101000101001011111100110001101000100100101111110011000110'
        '1000101010010111111001100011010001011010010111111001100011010001011101001011111100110001101000101111'
        '0100101111110011000110100010111110100101111110011000110100010111111010010111111001100011010001001111'
        '1101001011111100110001101000100011111101001011111100110001101000101001111110100101111110011000110100'
        '0101100111111010010111111001100011010001001100111111010010111111001100011010001000110011111101001011'
        '1111001100011010001000011001111110100101111110011000110100010100011001111110100101111110011000110100'
        '0101100011001111110100101111110011000110100010011000110011111101001011111100110001101000101011000110'
        '0111111010010111111001100011010001001011000110011111101001011111100110001101000100010110001100111111'
        '0100101111110011000110100010000101100011001111110100101111110011000110100010100010110001100111111010'
        '0101111110011000110100010010001011000110011111101001011111100110001101000100100001010001001011011001'
        '1000010001111101111010011110100001010001001011011001100001000111110111101001111101000010100010010110'
        '1100110000100011111011110100101111010000101000100101101100110000100011111011110100101011101000010100'
        '0100101101100110000100011111011110101101001110100001010001001011011001100001000111110111101111011001'
        '1101000010100010010110110011000010001111101111001110101001110100001010001001011011001100001000111110'
        '1111101110110100111010000101000100101101100110000100011111011101011101110100111010000101000100101101'
        '1001100001000111110110010111011110100111010000101000100101101100110000100011111011001011101111101001'
        '1101000010100010010110110011000010001111101100101110101111010011101000010100010010110110011000010001'
        '1111111001011101101111010011101000010100010010110110011000010001111011100101110111011110100111010000'
        '1010001001011011001100001000111001110010111011110111101001110100001010001001011011001100001000111001'
        '1100101110111110111101001110100001010001001011011001100001000111001110010111011111101111010011101000'
        '0101000100101101100110000100001100111001011101011111011110100111010000101000100101101100110000100001'
        '1001110010111010011111011110100111010000101000100101101100110000100001100111001011101000111110111101'
        '0011101000010100010010110110011000010000110011100101110110001111101111010011101000010100010010110110'
        '0110000000001100111001011101010001111101111010011101000010100010010110110011000000000110011100101110'
        '1001000111110111101001110100001010001001011011001100100000011001110010111010001000111110111101001110'
        '1000010100010010110110011001000000110011100101110100001000111110111101001110100001010001001011011001'
        '1101000000110011100101110110000100011111011110100111010000101000100101101100111010000001100111001011'
        '1011100001000111110111101001110100001010001001011011000110100000011001110010111010110000100011111011'
        '1101001110100001010001001011011010110100000011001110010111010011000010001111101111010011101000010100'
        '0100101101101011010000001100111001011101100110000100011111011110100111010000101000100101101001011010'
        '0000011001110010111011100110000100011111011110100111010000101000100101100001011010000001100111001011'
        '1010110011000010001111101111010011101000010100010010110000101101000000110011100101110110110011000010'
        '0011111011110100111010000101000100101000001011010000001100111001011101110110011000010001111101111010'
        '0111010000101000100100000001011010000001100111001011101011011001100001000111110111101001110100001010'
        '0010011000000101101000000110011100101110110110110011000010001111101111010011101000010100010011000000'
        '1011010000001100111001011101010110110011000010001111101111010011101000010100010011000000101101000000'
        '1100111001011101001011011001100001000111110111101001110100001010001001100000010110100000011001110010'
        '1110110010110110011000010001111101111010011101000010100010011000000101101000000110011100101110101001'
        '0110110011000010001111101111010011101000010100110011000000101101000000110011100101110100100101101100'
        '1100001000111110111101001110100001010111001100000010110100000011001110010111010001001011011001100001'
        '0001111101111010011101000010101110011000000101101000000110011100101110110001001011011001100001000111'
        '1101111010011101000010001110011000000101101000000110011100101110101000100101101100110000100011111011'
        '1101001110100001100111001100000010110100000011001110010111011010001001011011001100001000111110111101'
        '0011101000001001110011000000101101000000110011100101110101010001001011011001100001000111110111101001'
        '1101000101001110011000000101101000000110011100101110100101000100101101100110000100011111011110100111'
        '0100110100111001100000010110100000011001110010111010001010001001011011001100001000111110111101001110'
        '1011101001110011000000101101000000110011100101110100001010001001011011001100001000111110111101001110'
        '1011101001110011000000101101000000110011100101110110000101000100101101100110000100011111011110100111'
        '010111010011100110000001011010000001100111001011101'
    ),
    27: (
        '1111110111001110001110000111000001111000001110100000111001000001111001000001111100100000111011001000'
        '0011100110010000011100011001000001111000110010000011111000110010000011111100011001000001110111000110'
        '0100000111101110001100100000111010111000110010000011110101110001100100000111010101110001100100000111'
        '1010101110001100100000111110101011100011001000001111110101011100011001000001111111010101110001100100'
        '0001110111101010111000110010000011110111101010111000110010000011101011110101011100011001000001111010'
        '1111010101110001100100000111010101111010101110001100100000111101010111101010111000110010000011111010'
        '1011110101011100011001000001111110101011110101011100011001000001110111010101111010101110001100100000'
        '1110011101010111101010111000110010000011100011101010111101010111000110010000011110001110101011110101'
        '0111000110010000011111000111010101111010101110001100100000111011000111010101111010101110001100100000'
        '1110011000111010101111010101110001100100000111100110001110101011110101011100011001000001110100110001'
        '1101010111101010111000110010000011100100110001110101011110101011100011001000001110001001100011101010'
        '1111010101110001100100000111000010011000111010101111010101110001100100000111000001001100011101010111'
        '1010101110001100100000111100000100110001110101011110101011100011001000001111100000100110001110101011'
        '1101010111000110010000011111100000100110001110101011110101011100011001000001110000110001011011010001'
        '0011101010111101101001011000110000001100010110110100010011101010111101101001011000110100000110001011'
        '0110100010011101010111101101001011000100110000011000101101101000100111010101111011010010110000000110'
        '0000110001011011010001001110101011110110100101100100000110000011000101101101000100111010101111011010'
        '0101101100000011000001100010110110100010011101010111101101001011111000100011000001100010110110100010'
        '0111010101111011010010111110001100011000001100010110110100010011101010111101101001011111000011000110'
        '0000110001011011010001001110101011110110100101111100010110001100000110001011011010001001110101011110'
        '1101001011111000010110001100000110001011011010001001110101011110110101101111100000101100011000001100'
        '0101101101000100111010101111011010110111110001001011000110000011000101101101000100111010101111011000'
        '1101111100001001011000110000011000101101101000100111010101111011100110111110001010010110001100000110'
        '0010110110100010011101010111101110011011111000110100101100011000001100010110110100010011101010111101'
        '1100110111110000110100101100011000001100010110110100010011101010111101110011011111000101101001011000'
        '1100000110001011011010001001110101011100111001101111100011011010010110001100000110001011011010001001'
        '1101010110001110011011111000111011010010110001100000110001011011010001001110101011000111001101111100'
        '0111101101001011000110000011000101101101000100111010100100011100110111110000111101101001011000110000'
        '0110001011011010001001110101101000111001101111100010111101101001011000110000011000101101101000100111'
        '0100101000111001101111100001011110110100101100011000001100010110110100010011101101010001110011011111'
        '0001010111101101001011000110000011000101101101000100111001010100011100110111110000101011110110100101'
        '1000110000011000101101101000100111001010100011100110111110001010101111011010010110001100000110001011'
        '0110100010011000101010001110011011111000110101011110110100101100011000001100010110110100010010000101'
        '0100011100110111110001110101011110110100101100011000001100010110110100010010000101010001110011011111'
        '0000111010101111011010010110001100000110001011011010001001000010101000111001101111100000111010101111'
        '0110100101100011000001100010110110100011010000101010001110011011111000100111010101111011010010110001'
        '1000001100010110110100001010000101010001110011011111000010011101010111101101001011000110000011000101'
        '1011010010101000010101000111001101111100000100111010101111011010010110001100000110001011011010010101'
        '0000101010001110011011111000000100111010101111011010010110001100000110001011011010010101000010101000'
        '1110011011111000100010011101010111101101001011000110000011000101101100001010100001010100011100110111'
        '1100001000100111010101111011010010110001100000110001011011100010101000010101000111001101111100010100'
        '0100111010101111011010010110001100000110001011011100010101000010101000111001101111100011010001001110'
        '1010111101101001011000110000011000101101110001010100001010100011100110111110000110100010011101010111'
        '1011010010110001100000110001011011100010101000010101000111001101111100010110100010011101010111101101'
        '0010110001100000110001010011100010101000010101000111001101111100011011010001001110101011110110100101'
        '1000110000011000101001110001010100001010100011100110111110000110110100010011101010111101101001011000'
        '1100000110001110011100010101000010101000111001101111100010110110100010011101010111101101001011000110'
        '0000110000110011100010101000010101000111001101111100001011011010001001110101011110110100101100011000'
        '0011001011001110001010100001010100011100110111110000010110110100010011101010111101101001011000110000'
        '0110110110011100010101000010101000111001101111100000010110110100010011101010111101101001011000110000'
        '0111110110011100010101000010101000111001101111100010001011011010001001110101011110110100101100011000'
        '0011111011001110001010100001010100011100110111110001100010110110100010011101010111101101001011000110'
        '0000111110110011100010101000010101000111001101111100001100010110110100010011101010111101101001011000'
        '1100000111110110011100010101000010101000111001101111100000110001011011010001001110101011110110100101'
        '1000110000011111011001110001010100001010100011100110111110000001100010110110100010011101010111101101'
        '00101100011000001111101100111000101010000101010001110011011111000'
    ),
    28: (
        '1110111011010110010110001011100010110100010110010001011100100010110100100010111010010001011010100100'
        '0101110101001000101111010100100010111110101001000101111110101001000101101111010100100010110011110101'
        '0010001011100111101010010001011110011110101001000101101100111101010010001011001100111101010010001011'
        '0001100111101010010001011100011001111010100100010110100011001111010100100010110010001100111101010010'
        '0010111001000110011110101001000101101001000110011110101001000101100100100011001111010100100010110001'
        '0010001100111101010010001011100010010001100111101010010001011110001001000110011110101001000101101100'
        '0100100011001111010100100010110011000100100011001111010100100010111001100010010001100111101010010001'
        '0111100110001001000110011110101001000101111100110001001000110011110101001000101111110011000100100011'
        '0011110101001000101101111001100010010001100111101010010001011101111001100010010001100111101010010001'
        '0110101111001100010010001100111101010010001011101011110011000100100011001111010100100010110101011110'
        '0110001001000110011110101001000101100101011110011000100100011001111010100100010111001010111100110001'
        '0010001100111101010010001011010010101111001100010010001100111101010010001011001001010111100110001001'
        '0001100111101010010001011000100101011110011000100100011001111010100100010111000100101011110011000100'
        '1000110011110101001000101101000100101011110011000100100011001111010100100010111010001001010111100110'
        '0010010001100111101010010001011110100010010101111001100010010001100111101010010001011100000000101100'
        '1001100111101001111000011110101010111010010000000010110010011001111010011110000111101010101110101010'
        '0000000101100100110011110100111100001111010101011100001010000000010110010011001111010011110000111101'
        '0101011110010101000000001011001001100111101001111000011110101010110100110101000000001011001001100111'
        '1010011110000111101010101101001110101000000001011001001100111101001111000011110101010110100011101010'
        '0000000101100100110011110100111100001111010101111010010111010100000000101100100110011110100111100001'
        '1110101001110100010111010100000000101100100110011110100111100001111010110111010010101110101000000001'
        '0110010011001111010011110000111101011011101000101011101010000000010110010011001111010011110000111101'
        '0110111010010101011101010000000010110010011001111010011110000111101011011101000101010111010100000000'
        '1011001001100111101001111000011110101101110100101010101110101000000001011001001100111101001111000011'
        '1101011011101001101010101110101000000001011001001100111101001111000011010101101110100111010101011101'
        '0100000000101100100110011110100111100001001010110111010011110101010111010100000000101100100110011110'
        '1001111000000010101101110100011110101010111010100000000101100100110011110100111100000001010110111010'
        '0001111010101011101010000000010110010011001111010011110010000101011011101000001111010101011101010000'
        '0000101100100110011110100111101100001010110111010000001111010101011101010000000010110010011001111010'
        '0111101100001010110111010010000111101010101110101000000001011001001100111101001110011000010101101110'
        '1001100001111010101011101010000000010110010011001111010011100110000101011011101001110000111101010101'
        '1101010000000010110010011001111010011100110000101011011101001111000011110101010111010100000000101100'
        '1001100111101001110011000010101101110100011110000111101010101110101000000001011001001100111101001110'
        '0110000101011011101000011110000111101010101110101000000001011001001100111101101110011000010101101110'
        '1001001111000011110101010111010100000000101100100110011110110111001100001010110111010001001111000011'
        '1101010101110101000000001011001001100111101101110011000010101101110100101001111000011110101010111010'
        '1000000001011001001100111101101110011000010101101110100110100111100001111010101011101010000000010110'
        '0100110011110110111001100001010110111010011101001111000011110101010111010100000000101100100110011110'
        '1101110011000010101101110100111101001111000011110101010111010100000000101100100110001110110111001100'
        '0010101101110100011110100111100001111010101011101010000000010110010011000111011011100110000101011011'
        '1010000111101001111000011110101010111010100000000101100100111001110110111001100001010110111010010011'
        '1101001111000011110101010111010100000000101100100111001110110111001100001010110111010011001111010011'
        '1100001111010101011101010000000010110010001100111011011100110000101011011101000110011110100111100001'
        '1110101010111010100000000101100100011001110110111001100001010110111010000110011110100111100001111010'
        '1010111010100000000101100100011001110110111001100001010110111010010011001111010011110000111101010101'
        '1101010000000010110000001100111011011100110000101011011101000100110011110100111100001111010101011101'
        '0100000000101101000011001110110111001100001010110111010000100110011110100111100001111010101011101010'
        '0000000101101000011001110110111001100001010110111010010010011001111010011110000111101010101110101000'
        '0000010110100001100111011011100110000101011011101001100100110011110100111100001111010101011101010000'
        '0000100101000011001110110111001100001010110111010001100100110011110100111100001111010101011101010000'
        '0000110101000011001110110111001100001010110111010010110010011001111010011110000111101010101110101000'
        '0000011010100001100111011011100110000101011011101000101100100110011110100111100001111010101011101010'
        '0000000110101000011001110110111001100001010110111010000101100100110011110100111100001111010101011101'
        '0100000010110101000011001110110111001100001010110111010000010110010011001111010011110000111101010101'
        '1101010000011011010100001100111011011100110000101011011101000000101100100110011110100111100001111010'
        '1010111010100001110110101000011001110110111001100001010110111010000000101100100110011110100111100001'
        '1110101010111010100001110110101000011001110110111001100001010110111010000000010110010011001111010011'
        '1100001111010101011101010010111011010100001100111011011100110000101011011101000000000101100100110011'
        '1101001111000011110101010111010100101110110101000011001110110111001100001010110111010000000000101100'
        '10011001111010011110000111101010101110101001011101101010000110011101101110011000010101101110100'
    ),
    29: (
        '1110110011000111000110100011101000110101000110010100011100101000111100101000110110010100011101100101'
        '0001101011001010001100101100101000110001011001010001100001011001010001110000101100101000110100001011'
        '0010100011001000010110010100011100100001011001010001111001000010110010100011111001000010110010100011'
        '0111001000010110010100011101110010000101100101000111101110010000101100101000111110111001000010110010'
        '1000111111011100100001011001010001111111011100100001011001010001111111101110010000101100101000110111'
        '1110111001000010110010100011101111110111001000010110010100011110111111011100100001011001010001111101'
        '1111101110010000101100101000110111011111101110010000101100101000110011101111110111001000010110010100'
        '0111001110111111011100100001011001010001101001110111111011100100001011001010001100100111011111101110'
        '0100001011001010001100010011101111110111001000010110010100011000010011101111110111001000010110010100'
        '0111000010011101111110111001000010110010100011010000100111011111101110010000101100101000111010000100'
        '1110111111011100100001011001010001111010000100111011111101110010000101100101000110110100001001110111'
        '1110111001000010110010100011001101000010011101111110111001000010110010100011100110100001001110111111'
        '0111001000010110010100011010011010000100111011111101110010000101100101000111010011010000100111011111'
        '1011100100001011001010001101010011010000100111011111101110010000101100101000110010100110100001001110'
        '1111110111001000010110010100011000101001101000010011101111110111001000010110010100011100010100110100'
        '0010011101111110111001000010110010100011110001010011010000100111011111101110010000101100101000111010'
        '0110110010111111000100111001110100000100001110001010111010011011001011111100010011100111010000010000'
        '1110001010001101001101100101111110001001110011101000001000011100010100101101001101100101111110001001'
        '1100111010000010000111000101000101101001101100101111110001001110011101000001000011100011100101011010'
        '0110110010111111000100111001110100000100001110001110001010110100110110010111111000100111001110100000'
        '1000011100011100001010110100110110010111111000100111001110100000100001110101110000010101101001101100'
        '1011111100010011100111010000010000111010111001000101011010011011001011111100010011100111010000010000'
        '1110101110011000101011010011011001011111100010011100111010000010000111010111001110001010110100110110'
        '0101111110001001110011101000001000001101011100011100010101101001101100101111110001001110011101000001'
        '0000011010111000011100010101101001101100101111110001001110011101000001001001101011100000111000101011'
        '0100110110010111111000100111001110100000100100110101110000001110001010110100110110010111111000100111'
        '0011101000001101001101011100100001110001010110100110110010111111000100111001110100000110100110101110'
        '0010000111000101011010011011001011111100010011100111010000111010011010111000010000111000101011010011'
        '0110010111111000100111001110100011110100110101110000010000111000101011010011011001011111100010011100'
        '1110100011110100110101110000001000011100010101101001101100101111110001001110011101010111101001101011'
        '1000000010000111000101011010011011001011111100010011100111011101111010011010111001000001000011100010'
        '1011010011011001011111100010011100111001101111010011010111000100000100001110001010110100110110010111'
        '1110001001110011100110111101001101011100101000001000011100010101101001101100101111110001001110011000'
        '1101111010011010111001101000001000011100010101101001101100101111110001001110011000110111101001101011'
        '1001110100000100001110001010110100110110010111111000100111000100011011110100110101110001110100000100'
        '0011100010101101001101100101111110001001110001000110111101001101011100001110100000100001110001010110'
        '1001101100101111110001001110001000110111101001101011100100111010000010000111000101011010011011001011'
        '1111000100110000100011011110100110101110011001110100000100001110001010110100110110010111111000100100'
        '0001000110111101001101011100111001110100000100001110001010110100110110010111111000100000000100011011'
        '1101001101011100011100111010000010000111000101011010011011001011111100010100000010001101111010011010'
        '1110000111001110100000100001110001010110100110110010111111000101000000100011011110100110101110010011'
        '1001110100000100001110001010110100110110010111111000001000000100011011110100110101110001001110011101'
        '0000010000111000101011010011011001011111100000100000010001101111010011010111000010011100111010000010'
        '0001110001010110100110110010111111010001000000100011011110100110101110000010011100111010000010000111'
        '0001010110100110110010111111110001000000100011011110100110101110010001001110011101000001000011100010'
        '1011010011011001011111011000100000010001101111010011010111001100010011100111010000010000111000101011'
        '0100110110010111110110001000000100011011110100110101110011100010011100111010000010000111000101011010'
        '0110110010111110110001000000100011011110100110101110011110001001110011101000001000011100010101101001'
        '1011001011111011000100000010001101111010011010111001111100010011100111010000010000111000101011010011'
        '0110010111110110001000000100011011110100110101110011111100010011100111010000010000111000101011010011'
        '0110010011110110001000000100011011110100110101110001111110001001110011101000001000011100010101101001'
        '1011001101111011000100000010001101111010011010111001011111100010011100111010000010000111000101011010'
        '0110110001011110110001000000100011011110100110101110001011111100010011100111010000010000111000101011'
        '0100110110001011110110001000000100011011110100110101110000101111110001001110011101000001000011100010'
        '1011010011011100101111011000100000010001101111010011010111001001011111100010011100111010000010000111'
        '0001010110100110111001011110110001000000100011011110100110101110011001011111100010011100111010000010'
        '0001110001010110100110011001011110110001000000100011011110100110101110001100101111110001001110011101'
        '0000010000111000101011010011101100101111011000100000010001101111010011010111001011001011111100010011'
        '1001110100000100001110001010110100101011001011110110001000000100011011110100110101110011011001011111'
        '1000100111001110100000100001110001010110100101011001011110110001000000100011011110100110101110001101'
        '1001011111100010011100111010000010000111000101011010110101100101111011000100000010001101111010011010'
        '1110000110110010111111000100111001110100000100001110001010110111101011001011110110001000000100011011'
        '1101001101011100100110110010111111000100111001110100000100001110001010110011101011001011110110001000'
        '0001000110111101001101011100010011011001011111100010011100111010000010000111000101011001110101100101'
        '11101100010000001000110111101001101011100'
    ),
    30: (
        '1010010001100011100011110001011100011011100010101110001001011100010001011100011000101110001010001011'
        '1000100100010111000110010001011100010100100010111000110100100010111000101010010001011100010010100100'
        '0101110001000101001000101110001100010100100010111000111000101001000101110001011000101001000101110001'
        '1011000101001000101110001110110001010010001011100011110110001010010001011100011111011000101001000101'
        '1100010111101100010100100010111000100111101100010100100010111000110011110110001010010001011100011100'
        '1111011000101001000101110001111001111011000101001000101110001111100111101100010100100010111000101111'
        '0011110110001010010001011100011011110011110110001010010001011100011101111001111011000101001000101110'
        '0010110111100111101100010100100010111000100110111100111101100010100100010111000100011011110011110110'
        '0010100100010111000110001101111001111011000101001000101110001010001101111001111011000101001000101110'
        '0011010001101111001111011000101001000101110001010100011011110011110110001010010001011100010010100011'
        '0111100111101100010100100010111000110010100011011110011110110001010010001011100010100101000110111100'
        '1111011000101001000101110001001001010001101111001111011000101001000101110001000100101000110111100111'
        '1011000101001000101110001100010010100011011110011110110001010010001011100010100010010100011011110011'
        '1101100010100100010111000110100010010100011011110011110110001010010001011100011101000100101000110111'
        '1001111011000101001000101110001111010001001010001101111001111011000101001000101110001011101000100101'
        '0001101111001111011000101001000101110001001110100010010100011011110011110110001010010001011100010001'
        '1101000100101000110111100111101100010100100010111000110001110100010010100011011110011110110001010010'
        '0010111000100111010100001011111011010000100100001110000011001111011011100111010100001011111011010000'
        '1001000011100000110011110110101100111010100001011111011010000100100001110000011001111011010011001110'
        '1010000101111101101000010010000111000001100111101111010110011101010000101111101101000010010000111000'
        '0011001111011110110110011101010000101111101101000010010000111000001100111100111001101100111010100001'
        '0111110110100001001000011100000110011110011101011011001110101000010111110110100001001000011100000110'
        '0111000111011011011001110101000010111110110100001001000011100000110011100011101110110110011101010000'
        '1011111011010000100100001110000011001010001110111101101100111010100001011111011010000100100001110000'
        '0110010100011100111101101100111010100001011111011010000100100001110000011011010001110001111011011001'
        '1101010000101111101101000010010000111000001111101000111010011110110110011101010000101111101101000010'
        '0100001110000010111010001110110011110110110011101010000101111101101000010010000111000001011101000111'
        '0011001111011011001110101000010111110110100001001000011100001101110100011100011001111011011001110101'
        '0000101111101101000010010000111000011011101000111000011001111011011001110101000010111110110100001001'
        '0000111001011011101000111000001100111101101100111010100001011111011010000100100001110010110111010001'
        '1100000011001111011011001110101000010111110110100001001000011110101101110100011101000001100111101101'
        '1001110101000010111110110100001001000011110101101110100011101100000110011110110110011101010000101111'
        '1011010000100100001111010110111010001110111000001100111101101100111010100001011111011010000100100000'
        '1110101101110100011100111000001100111101101100111010100001011111011010000100100000111010110111010001'
        '1100011100000110011110110110011101010000101111101101000010010010011101011011101000111000011100000110'
        '0111101101100111010100001011111011010000100100100111010110111010001110000011100000110011110110110011'
        '1010100001011111011010000100100100111010110111010001110100001110000011001111011011001110101000010111'
        '1101101000010000010011101011011101000111001000011100000110011110110110011101010000101111101101000010'
        '0000100111010110111010001110001000011100000110011110110110011101010000101111101101000011000010011101'
        '0110111010001110100100001110000011001111011011001110101000010111110110100001100001001110101101110100'
        '0111001001000011100000110011110110110011101010000101111101101000011000010011101011011101000111000100'
        '1000011100000110011110110110011101010000101111101101000011000010011101011011101000111000010010000111'
        '0000011001111011011001110101000010111110110100001100001001110101101110100011100000100100001110000011'
        '0011110110110011101010000101111101101000011000010011101011011101000111010000100100001110000011001111'
        '0110110011101010000101111101101000011000010011101011011101000111001000010010000111000001100111101101'
        '1001110101000010111110110100001100001001110101101110100011101010000100100001110000011001111011011001'
        '1101010000101111101001000011000010011101011011101000111011010000100100001110000011001111011011001110'
        '1010000101111101001000011000010011101011011101000111001101000010010000111000001100111101101100111010'
        '1000010111111100100001100001001110101101110100011101011010000100100001110000011001111011011001110101'
        '0000101111111001000011000010011101011011101000111011011010000100100001110000011001111011011001110101'
        '0000101110111001000011000010011101011011101000111011101101000010010000111000001100111101101100111010'
        '1000010111011100100001100001001110101101110100011101111011010000100100001110000011001111011011001110'
        '1010000101010111001000011000010011101011011101000111011111011010000100100001110000011001111011011001'
        '1101010000101010111001000011000010011101011011101000111001111101101000010010000111000001100111101101'
        '1001110101000011101011100100001100001001110101101110100011101011111011010000100100001110000011001111'
        '0110110011101010000011010111001000011000010011101011011101000111001011111011010000100100001110000011'
        '0011110110110011101010001011010111001000011000010011101011011101000111000101111101101000010010000111'
        '0000011001111011011001110101001101101011100100001100001001110101101110100011100001011111011010000100'
        '1000011100000110011110110110011101010111011010111001000011000010011101011011101000111000001011111011'
        '0100001001000011100000110011110110110011101010111011010111001000011000010011101011011101000111010000'
        '1011111011010000100100001110000011001111011011001110101011101101011100100001100001001110101101110100'
        '0111001000010111110110100001001000011100000110011110110110011101010111011010111001000011000010011101'
        '0110111010001110101000010111110110100001001000011100000110011110110110011100010111011010111001000011'
        '0000100111010110111010001110010100001011111011010000100100001110000011001111011011001110001011101101'
        '0111001000011000010011101011011101000111010101000010111110110100001001000011100000110011110110110011'
        '1000101110110101110010000110000100111010110111010001110110101000010111110110100001001000011100000110'
        '0111101101100111000101110110101110010000110000100111010110111010001110111010100001011111011010000100'
        '1000011100000110011110110110011100010111011010111001000011000010011101011011101000111001110101000010'
        '1111101101000010010000111000001100111101101100111000101110110101110010000110000100111010110111010001'
        '110'
    ),
    31: (
        '1011011101111010111010011101000111011000111010100011101001000111011001000111011100100011101111001000'
        '1110111110010001110111111001000111010111110010001110100111110010001110110011111001000111011100111110'
        '0100011101011001111100100011101101100111110010001110101011001111100100011101001011001111100100011101'
        '1001011001111100100011101010010110011111001000111011010010110011111001000111010101001011001111100100'
        '0111010010100101100111110010001110100010100101100111110010001110100001010010110011111001000111010000'
        '0101001011001111100100011101000000101001011001111100100011101100000010100101100111110010001110101000'
        '0001010010110011111001000111011010000001010010110011111001000111010101000000101001011001111100100011'
        '1010010100000010100101100111110010001110110010100000010100101100111110010001110101001010000001010010'
        '1100111110010001110110100101000000101001011001111100100011101110100101000000101001011001111100100011'
        '1010110100101000000101001011001111100100011101001101001010000001010010110011111001000111011001101001'
        '0100000010100101100111110010001110111001101001010000001010010110011111001000111011110011010010100000'
        '0101001011001111100100011101111100110100101000000101001011001111100100011101111110011010010100000010'
        '1001011001111100100011101011111001101001010000001010010110011111001000111010011111001101001010000001'
        '0100101100111110010001110110011111001101001010000001010010110011111001000111010100111110011010010100'
        '0000101001011001111100100011101001001111100110100101000000101001011001111100100011101000100111110011'
        '0100101000000101001011001111100100011101100010011111001101001010000001010010110011111001000111011100'
        '0100111110011010010100000010100101100111110010001110111100010011111001101001010000001010010110011111'
        '0010001110101110001001111100110100101000000101001011001111100100011101101110001001111100110100101000'
        '0001010010110011111001000111010101110001001111100110100101000000101001011001111100100011101101011100'
        '0100111110011010010100000010100101100111110010001110001010111000100111110011010010100000010100101100'
        '1111100100011110101010111000100111110011010010100000010100101100111110010001101011010101110001001111'
        '1001101001010000001010010110011111001000100101110101011100010011111001101001010000001010010110011111'
        '0010000001001110101011100010011111001101001010000001010010110011111001001000100011101010111000100111'
        '1100110100101000000101001011001111100101100010000111010101110001001111100110100101000000101001011001'
        '1111001111000101000111010101110001001111100110100101000000101001011001111100011100010010001110101011'
        '1000100111110011010010100000010100101100111110101110001000100011101010111000100111110011010010100000'
        '0101001011001111111011100010100100011101010111000100111110011010010100000010100101100111101101110001'
        '0110010001110101011100010011111001101001010000001010010110011100110111000101110010001110101011100010'
        '0111110011010010100000010100101100110001101110001011110010001110101011100010011111001101001010000001'
        '0100101100100001101110001011111001000111010101110001001111100110100101000000101001011000000011011100'
        '0100111110010001110101011100010011111001101001010000001010010110100000110111000100011111001000111010'
        '1011100010011111001101001010000001010010111100000110111000101001111100100011101010111000100111110011'
        '0100101000000101001010110000011011100010110011111001000111010101110001001111100110100101000000101001'
        '0001100000110111000100110011111001000111010101110001001111100110100101000000101001100110000011011100'
        '0101011001111100100011101010111000100111110011010010100000010100010011000001101110001001011001111100'
        '1000111010101110001001111100110100101000000101010100110000011011100010001011001111100100011101010111'
        '0001001111100110100101000000101110100110000011011100010100101100111110010001110101011100010011111001'
        '1010010100000010011010011000001101110001001001011001111100100011101010111000100111110011010010100000'
        '0110110100110000011011100010101001011001111100100011101010111000100111110011010010100000001011010011'
        '0000011011100010010100101100111110010001110101011100010011111001101001010000010101101001100000110111'
        '0001000101001011001111100100011101010111000100111110011010010100001101011010011000001101110001000010'
        '1001011001111100100011101010111000100111110011010010100011101011010011000001101110001000001010010110'
        '0111110010001110101011100010011111001101001010011110101101001100000110111000100000010100101100111110'
        '0100011101010111000100111110011010010101111101011010011000001101110001000000010100101100111110010001'
        '1101010111000100111110011010010111111101011010011000001101110001010000001010010110011111001000111010'
        '1011100010011111001101001001111110101101001100000110111000100100000010100101100111110010001110101011'
        '1000100111110011010011011111101011010011000001101110001010100000010100101100111110010001110101011100'
        '0100111110011010001011111101011010011000001101110001001010000001010010110011111001000111010101110001'
        '0011111001101010101111110101101001100000110111000100010100000010100101100111110010001110101011100010'
        '0111110011011101011111101011010011000001101110001010010100000010100101100111110010001110101011100010'
        '0111110011001101011111101011010011000001101110001001001010000001010010110011111001000111010101110001'
        '0011111001110110101111110101101001100000110111000101010010100000010100101100111110010001110101011100'
        '0100111110010101101011111101011010011000001101110001011010010100000010100101100111110010001110101011'
        '1000100111110000101101011111101011010011000001101110001001101001010000001010010110011111001000111010'
        '1011100010011111010010110101111110101101001100000110111000100011010010100000010100101100111110010001'
        '1101010111000100111111100101101011111101011010011000001101110001010011010010100000010100101100111110'
        '0100011101010111000100111101100101101011111101011010011000001101110001011001101001010000001010010110'
        '0111110010001110101011100010011100110010110101111110101101001100000110111000101110011010010100000010'
        '1001011001111100100011101010111000100110001100101101011111101011010011000001101110001011110011010010'
        '1000000101001011001111100100011101010111000100100001100101101011111101011010011000001101110001011111'
        '0011010010100000010100101100111110010001110101011100010000000110010110101111110101101001100000110111'
        '0001001111100110100101000000101001011001111100100011101010111000101000001100101101011111101011010011'
        '0000011011100010001111100110100101000000101001011001111100100011101010111000111000001100101101011111'
        '1010110100110000011011100010100111110011010010100000010100101100111110010001110101011100001100000110'
        '0101101011111101011010011000001101110001001001111100110100101000000101001011001111100100011101010111'
        '0010110000011001011010111111010110100110000011011100010001001111100110100101000000101001011001111100'
        '1000111010101110110110000011001011010111111010110100110000011011100010000100111110011010010100000010'
        '1001011001111100100011101010111111011000001100101101011111101011010011000001101110001010001001111100'
        '1101001010000001010010110011111001000111010101101110110000011001011010111111010110100110000011011100'
        '0101100010011111001101001010000001010010110011111001000111010101001110110000011001011010111111010110'
        '1001100000110111000101110001001111100110100101000000101001011001111100100011101010000111011000001100'
        '1011010111111010110100110000011011100010011100010011111001101001010000001010010110011111001000111010'
        '1100011101100000110010110101111110101101001100000110111000101011100010011111001101001010000001010010'
        '110011111001000111010010001110110000011001011010111111010110100110000011011100010'
    ),
    32: (
        '1011011101011011011011101101011011010011011010001101101100011011011100011011011110001101101111100011'
        '0110111111000110110101111100011011011011111000110110101011111000110110100101111100011011010001011111'
        '0001101101000010111110001101101100001011111000110110101000010111110001101101001000010111110001101101'
        '0001000010111110001101101100010000101111100011011010100010000101111100011011011010001000010111110001'
        '1011011101000100001011111000110110101101000100001011111000110110100110100010000101111100011011010001'
        '1010001000010111110001101101000011010001000010111110001101101100001101000100001011111000110110111000'
        '0110100010000101111100011011010110000110100010000101111100011011011011000011010001000010111110001101'
        '1010101100001101000100001011111000110110100101100001101000100001011111000110110100010110000110100010'
        '0001011111000110110110001011000011010001000010111110001101101010001011000011010001000010111110001101'
        '1010010001011000011010001000010111110001101101000100010110000110100010000101111100011011010000100010'
        '1100001101000100001011111000110110110000100010110000110100010000101111100011011010100001000101100001'
        '1010001000010111110001101101101000010001011000011010001000010111110001101101110100001000101100001101'
        '0001000010111110001101101111010000100010110000110100010000101111100011011011111010000100010110000110'
        '1000100001011111000110110111111010000100010110000110100010000101111100011011010111110100001000101100'
        '0011010001000010111110001101101001111101000010001011000011010001000010111110001101101000111110100001'
        '0001011000011010001000010111110001101101100011111010000100010110000110100010000101111100011011011100'
        '0111110100001000101100001101000100001011111000110110101100011111010000100010110000110100010000101111'
        '1000110110110110001111101000010001011000011010001000010111110001101101110110001111101000010001011000'
        '0110100010000101111100011011010110110001111101000010001011000011010001000010111110001101101101101100'
        '0111110100001000101100001101000100001011111000110110111111001110001000100011000010111011010010000001'
        '0111011011001110011111001110001000100011000010111011010010000001011101101100111010111110011100010001'
        '0001100001011101101001000000101110110110011101101111100111000100010001100001011101101001000000101110'
        '1101100101011101111100111000100010001100001011101101001000000101110110110000100111011111001110001000'
        '1000110000101110110100100000010111011011010010001110111110011100010001000110000101110110100100000010'
        '1110110110100101001110111110011100010001000110000101110110100100000010111011010010010110011101111100'
        '1110001000100011000010111011010010000001011101101001001001100111011111001110001000100011000010111011'
        '0100100000010111011110010010101100111011111001110001000100011000010111011010010000001011101111001001'
        '0110110011101111100111000100010001100001011101101001000000101110011100100100110110011101111100111000'
        '1000100011000010111011010010000001011100111001001010110110011101111100111000100010001100001011101101'
        '0010000001011000111001001011011011001110111110011100010001000110000101110110100100000010100001110010'
        '0101110110110011101111100111000100010001100001011101101001000000100000011100100100111011011001110111'
        '1100111000100010001100001011101101001000000110000011100100101011101101100111011111001110001000100011'
        '0000101110110100100000001000001110010010010111011011001110111110011100010001000110000101110110100100'
        '0001010000011100100100010111011011001110111110011100010001000110000101110110100100001101000001110010'
        '0100001011101101100111011111001110001000100011000010111011010010001110100000111001001000001011101101'
        '1001110111110011100010001000110000101110110100100111101000001110010010000001011101101100111011111001'
        '1100010001000110000101110110100100111101000001110010010000000101110110110011101111100111000100010001'
        '1000010111011010011011110100000111001001010000001011101101100111011111001110001000100011000010111011'
        '0100110111101000001110010010010000001011101101100111011111001110001000100011000010111011010111011110'
        '1000001110010010001000000101110110110011101111100111000100010001100001011101101011101111010000011100'
        '1001010010000001011101101100111011111001110001000100011000010111011010111011110100000111001001001001'
        '0000001011101101100111011111001110001000100011000010111011010111011110100000111001001010100100000010'
        '1110110110011101111100111000100010001100001011101001011101111010000011100100101101001000000101110110'
        '1100111011111001110001000100011000010111010010111011110100000111001001001101001000000101110110110011'
        '1011111001110001000100011000010111110010111011110100000111001001010110100100000010111011011001110111'
        '1100111000100010001100001011111001011101111010000011100100101101101001000000101110110110011101111100'
        '1110001000100011000010111110010111011110100000111001001011101101001000000101110110110011101111100111'
        '0001000100011000010011110010111011110100000111001001001110110100100000010111011011001110111110011100'
        '0100010001100001001111001011101111010000011100100101011101101001000000101110110110011101111100111000'
        '1000100011000010011110010111011110100000111001001001011101101001000000101110110110011101111100111000'
        '1000100011000010011110010111011110100000111001001000101110110100100000010111011011001110111110011100'
        '0100010001100101001111001011101111010000011100100100001011101101001000000101110110110011101111100111'
        '0001000100011011010011110010111011110100000111001001000001011101101001000000101110110110011101111100'
        '1110001000100011111010011110010111011110100000111001001010000101110110100100000010111011011001110111'
        '1100111000100010001011101001111001011101111010000011100100101100001011101101001000000101110110110011'
        '1011111001110001000100010111010011110010111011110100000111001001001100001011101101001000000101110110'
        '1100111011111001110001000100110111010011110010111011110100000111001001000110000101110110100100000010'
        '1110110110011101111100111000100010111011101001111001011101111010000011100100100001100001011101101001'
        '0000001011101101100111011111001110001000111110111010011110010111011110100000111001001010001100001011'
        '1011010010000001011101101100111011111001110001000011110111010011110010111011110100000111001001001000'
        '1100001011101101001000000101110110110011101111100111000100101111011101001111001011101111010000011100'
        '1001000100011000010111011010010000001011101101100111011111001110001001011110111010011110010111011110'
        '1000001110010010000100011000010111011010010000001011101101100111011111001110001001011110111010011110'
        '0101110111101000001110010010100010001100001011101101001000000101110110110011101111100111000000101111'
        '0111010011110010111011110100000111001001001000100011000010111011010010000001011101101100111011111001'
        '1100000010111101110100111100101110111101000001110010010001000100011000010111011010010000001011101101'
        '1001110111110011100000010111101110100111100101110111101000001110010010000100010001100001011101101001'
        '0000001011101101100111011111001111000001011110111010011110010111011110100000111001001010001000100011'
        '0000101110110100100000010111011011001110111110011110000010111101110100111100101110111101000001110010'
        '0101100010001000110000101110110100100000010111011011001110111110011110000010111101110100111100101110'
        '1111010000011100100101110001000100011000010111011010010000001011101101100111011111000111000001011110'
        '1110100111100101110111101000001110010010011100010001000110000101110110100100000010111011011001110111'
        '1100011100000101111011101001111001011101111010000011100100100011100010001000110000101110110100100000'
        '0101110110110011101111110011100000101111011101001111001011101111010000011100100101001110001000100011'
        '0000101110110100100000010111011011001110111101001110000010111101110100111100101110111101000001110010'
        '0101100111000100010001100001011101101001000000101110110110011101110010011100000101111011101001111001'
        '0111011110100000111001001011100111000100010001100001011101101001000000101110110110011101110010011100'
        '0001011110111010011110010111011110100000111001001011110011100010001000110000101110110100100000010111'
        '011011001110101001001110000010111101110100111100101110111101000001110010010'
    ),
    33: (
        '0001001100011000011000001100000011000000011000000001100000000011001000000011001100000001100011000000'
        '0110010110000000110011011000000011001110110000000110011110110000000110011111011000000011000111110110'
        '0000001100101111101100000001100010111110110000000110010101111101100000001100010101111101100000001100'
        '1010101111101100000001100110101011111011000000011000110101011111011000000011001011010101111101100000'
        '0011001101101010111110110000000110011101101010111110110000000110001110110101011111011000000011000011'
        '1011010101111101100000001100000111011010101111101100000001100000011101101010111110110000000110010000'
        '1110110101011111011000000011001100001110110101011111011000000011001110000111011010101111101100000001'
        '1000111000011101101010111110110000000110010111000011101101010111110110000000110011011100001110110101'
        '0111110110000000110001101110000111011010101111101100000001100101101110000111011010101111101100000001'
        '1000101101110000111011010101111101100000001100101011011100001110110101011111011000000011000101011011'
        '1000011101101010111110110000000110010101011011100001110110101011111011000000011001101010110111000011'
        '1011010101111101100000001100111010101101110000111011010101111101100000001100111101010110111000011101'
        '1010101111101100000001100111110101011011100001110110101011111011000000011000111110101011011100001110'
        '1101010111110110000000110010111110101011011100001110110101011111011000000011001101111101010110111000'
        '0111011010101111101100000001100011011111010101101110000111011010101111101100000001100001101111101010'
        '1101110000111011010101111101100000001100000110111110101011011100001110110101011111011000000011000000'
        '1101111101010110111000011101101010111110110000000110000000110111110101011011100001110110101011111011'
        '0000000110000000011011111010101101110000111011010101111101100000001100000000011011111010101101110000'
        '1110110101011111011000000011001000000011011111010101101110000111011010101111101100000001100110000000'
        '1101111101010110111000011101101010111110110000000110001100000001101111101010110111000011101101010111'
        '1101100000001100001100000001101111101010110111000011101101010111110110000000110011101100011100010101'
        '0010011110110100001100100111001010101100100011111011000111000101010010011110110100001100100111001010'
        '1011001000101111011000111000101010010011110110100001100100111001010101100100110011110110001110001010'
        '1001001111011010000110010011100101010110010011000111101100011100010101001001111011010000110010011100'
        '1010101100100111000111101100011100010101001001111011010000110010011100101010110010011010001111011000'
        '1110001010100100111101101000011001001110010101011011001100100011110110001110001010100100111101101000'
        '0110010011100101010111110011100100011110110001110001010100100111101101000011001001110010101011111001'
        '1110010001111011000111000101010010011110110100001100100111001010101111100110110010001111011000111000'
        '1010100100111101101000011001001110010101111111001110110010001111011000111000101010010011110110100001'
        '1001001110010101111111001101011001000111101100011100010101001001111011010000110010011100101011111110'
        '0111010110010001111011000111000101010010011110110100001100100111001000111111100110101011001000111101'
        '1000111000101010010011110110100001100100111001100111111100111010101100100011110110001110001010100100'
        '1111011010000110010011100010011111110011010101011001000111101100011100010101001001111011010000110010'
        '0111000100111111100110010101011001000111101100011100010101001001111011010000110010011100010011111110'
        '0111001010101100100011110110001110001010100100111101101000011001001100001001111111001111001010101100'
        '1000111101100011100010101001001111011010000110010010000010011111110011111001010101100100011110110001'
        '1100010101001001111011010000110010010000010011111110011011100101010110010001111011000111000101010010'
        '0111101101000011001001000001001111111001100111001010101100100011110110001110001010100100111101101000'
        '0110011010000010011111110011100111001010101100100011110110001110001010100100111101101000011000101000'
        '0010011111110011010011100101010110010001111011000111000101010010011110110100001101010100000100111111'
        '1001100100111001010101100100011110110001110001010100100111101101000011010101000001001111111001110010'
        '0111001010101100100011110110001110001010100100111101101000010010101000001001111111001111001001110010'
        '1010110010001111011000111000101010010011110110100001001010100000100111111100110110010011100101010110'
        '0100011110110001110001010100100111101101000010010101000001001111111001100110010011100101010110010001'
        '1110110001110001010100100111101101000010010101000001001111111001100011001001110010101011001000111101'
        '1000111000101010010011110110100001001010100000100111111100110000110010011100101010110010001111011000'
        '1110001010100100111101101100010010101000001001111111001110000110010011100101010110010001111011000111'
        '0001010100100111101101100010010101000001001111111001101000011001001110010101011001000111101100011100'
        '0101010010011110111110001001010100000100111111100111010000110010011100101010110010001111011000111000'
        '1010100100111101111100010010101000001001111111001111010000110010011100101010110010001111011000111000'
        '1010100100111100111100010010101000001001111111001101101000011001001110010101011001000111101100011100'
        '0101010010011110011110001001010100000100111111100111011010000110010011100101010110010001111011000111'
        '0001010100100111000111100010010101000001001111111001111011010000110010011100101010110010001111011000'
        '1110001010100100111000111100010010101000001001111111001111101101000011001001110010101011001000111101'
        '1000111000101010010010100011110001001010100000100111111100111111011010000110010011100101010110010001'
        '1110110001110001010100100001000111100010010101000001001111111001101111011010000110010011100101010110'
        '0100011110110001110001010100101001000111100010010101000001001111111001100111101101000011001001110010'
        '1010110010001111011000111000101010010100100011110001001010100000100111111100111001111011010000110010'
        '0111001010101100100011110110001110001010100101001000111100010010101000001001111111001101001111011010'
        '0001100100111001010101100100011110110001110001010100101001000111100010010101000001001111111001100100'
        '1111011010000110010011100101010110010001111011000111000101011010100100011110001001010100000100111111'
        '1001110010011110110100001100100111001010101100100011110110001110001010010101001000111100010010101000'
        '0010011111110011010010011110110100001100100111001010101100100011110110001110001010010101001000111100'
        '0100101010000010011111110011101001001111011010000110010011100101010110010001111011000111000100001010'
        '1001000111100010010101000001001111111001101010010011110110100001100100111001010101100100011110110001'
        '1100010000101010010001111000100101010000010011111110011101010010011110110100001100100111001010101100'
        '1000111101100011100000000101010010001111000100101010000010011111110011010101001001111011010000110010'
        '0111001010101100100011110110001110010000010101001000111100010010101000001001111111001100101010010011'
        '1101101000011001001110010101011001000111101100011100100000101010010001111000100101010000010011111110'
        '0110001010100100111101101000011001001110010101011001000111101100011100100000101010010001111000100101'
        '0100000100111111100111000101010010011110110100001100100111001010101100100011110110001110010000010101'
        '0010001111000100101010000010011111110011110001010100100111101101000011001001110010101011001000111101'
        '1000111001000001010100100011110001001010100000100111111100111110001010100100111101101000011001001110'
        '0101010110010001111011000111001000001010100100011110001001010100000100111111100110111000101010010011'
        '1101101000011001001110010101011001000111101100111100100000101010010001111000100101010000010011111110'
        '0110011100010101001001111011010000110010011100101010110010001111011011111001000001010100100011110001'
        '0010101000001001111111001100011100010101001001111011010000110010011100101010110010001111011111111001'
        '0000010101001000111100010010101000001001111111001110001110001010100100111101101000011001001110010101'
        '0110010001111011111111001000001010100100011110001001010100000100111111100111100011100010101001001111'
        '0110100001100100111001010101100100011110011111110010000010101001000111100010010101000001001111111001'
        '1011000111000101010010011110110100001100100111001010101100100011110011111110010000010101001000111100'
        '0100101010000010011111110011101100011100010101001001111011010000110010011100101010110010001111001111'
        '1110010000010101001000111100010010101000001001111111001111011000111000101010010011110110100001100100'
        '1110010101011001000111100111111100100000101010010001111000100101010000010011111110011'
    ),
    34: (
        '1111110111001110001110000111000001111000001110100000111001000001111001000001111100100000111011001000'
        '0011100110010000011110011001000001110100110010000011100100110010000011110010011001000001110100100110'
        '0100000111101001001100100000111010100100110010000011100101001001100100000111000101001001100100000111'
        '1000101001001100100000111010001010010011001000001111010001010010011001000001111101000101001001100100'
        '0001111110100010100100110010000011111110100010100100110010000011101111010001010010011001000001111011'
        '1101000101001001100100000111110111101000101001001100100000111111011110100010100100110010000011111110'
        '1111010001010010011001000001110111101111010001010010011001000001111011110111101000101001001100100000'
        '1111101111011110100010100100110010000011111101111011110100010100100110010000011111110111101111010001'
        '0100100110010000011101111011110111101000101001001100100000111101111011110111101000101001001100100000'
        '1110101111011110111101000101001001100100000111001011110111101111010001010010011001000001110001011110'
        '1111011110100010100100110010000011110001011110111101111010001010010011001000001110100010111101111011'
        '1101000101001001100100000111101000101111011110111101000101001001100100000111010100010111101111011110'
        '1000101001001100100000111001010001011110111101111010001010010011001000001111001010001011110111101111'
        '0100010100100110010000011101001010001011110111101111010001010010011001000001110010010100010111101111'
        '0111101000101001001100100000111100100101000101111011110111101000101001001100100000111110010010100010'
        '1111011110111101000101001001100100000111011001001010001011110111101111010001010010011001000001110011'
        '0010010100010111101111011110100010100100110010000011110011001001010001011110111101111010001010010011'
        '0010000011101001100100101000101111011110111101000101001001100100000111001001100100101000101111011110'
        '1111010001010010011001000001110001001100100101000101111011110111101000101001001100100000111000010011'
        '0010010100010111101111011110100010100100110010000011100000100110010010100010111101111011110100010100'
        '1001100100000111100000100110010010100010111101111011110100010100100110010000011111000001001100100101'
        '0001011110111101111010001010010011001000001111110000010011001001010001011110111101111010001010010011'
        '0010000011111111110100101000100110001111000110110000100101011100001000111100100111111101001010001001'
        '1000111100011011000010010101110000100011110010101111111010010100010011000111100011011000010010101110'
        '0001000111100000101111111010010100010011000111100011011000010010101110000100011110000001011111110100'
        '1010001001100011110001101100001001010111000010001111100010010111111101001010001001100011110001101100'
        '0010010101110000100011111000110010111111101001010001001100011110001101100001001010111000010001111100'
        '0111001011111110100101000100110001111000110110000100101011100001000111110001111001011111110100101000'
        '1001100011110001101100001001010111000010001111100001111001011111110100101000100110001111000110110000'
        '1001010111000010001111100000111100101111111010010100010011000111100011011000010010101110000101011111'
        '0000001111001011111110100101000100110001111000110110000100101011100001110111110001000111100101111111'
        '0100101000100110001111000110110000100101011100000110111110000100011110010111111101001010001001100011'
        '1100011011000010010101110000011011111000001000111100101111111010010100010011000111100011011000010010'
        '1011100100110111110000001000111100101111111010010100010011000111100011011000010010101110110011011111'
        '0000000100011110010111111101001010001001100011110001101100001001010111011001101111100010000100011110'
        '0101111111010010100010011000111100011011000010010101110110011011111000110000100011110010111111101001'
        '0100010011000111100011011000010010101110110011011111000111000010001111001011111110100101000100110001'
        '1110001101100001001010011011001101111100001110000100011110010111111101001010001001100011110001101100'
        '0010010110110110011011111000101110000100011110010111111101001010001001100011110001101100001001001011'
        '0110011011111000010111000010001111001011111110100101000100110001111000110110000100110101101100110111'
        '1100010101110000100011110010111111101001010001001100011110001101100001001101011011001101111100001010'
        '1110000100011110010111111101001010001001100011110001101100001011101011011001101111100000101011100001'
        '0001111001011111110100101000100110001111000110110000101110101101100110111110001001010111000010001111'
        '0010111111101001010001001100011110001101100001011101011011001101111100001001010111000010001111001011'
        '1111101001010001001100011110001101100001011101011011001101111100000100101011100001000111100101111111'
        '0100101000100110001111000110110000101110101101100110111110000001001010111000010001111001011111110100'
        '1010001001100011110001101100001011101011011001101111100000001001010111000010001111001011111110100101'
        '0001001100011110001101100001011101011011001101111100010000100101011100001000111100101111111010010100'
        '0100110001111000110110000101110101101100110111110001100001001010111000010001111001011111110100101000'
        '1001100011110001100100001011101011011001101111100001100001001010111000010001111001011111110100101000'
        '1001100011110001100100001011101011011001101111100010110000100101011100001000111100101111111010010100'
        '0100110001111000100010000101110101101100110111110001101100001001010111000010001111001011111110100101'
        '0001001100011110000000100001011101011011001101111100001101100001001010111000010001111001011111110100'
        '1010001001100011110010000100001011101011011001101111100000110110000100101011100001000111100101111111'
        '0100101000100110001111001000010000101110101101100110111110000001101100001001010111000010001111001011'
        '1111101001010001001100011110010000100001011101011011001101111100010001101100001001010111000010001111'
        '0010111111101001010001001100011100010000100001011101011011001101111100011000110110000100101011100001'
        '0001111001011111110100101000100110001100001000010000101110101101100110111110001110001101100001001010'
        '1110000100011110010111111101001010001001100011000010000100001011101011011001101111100011110001101100'
        '0010010101110000100011110010111111101001010001001100001000010000100001011101011011001101111100001111'
        '0001101100001001010111000010001111001011111110100101000100110010100001000010000101110101101100110111'
        '1100000111100011011000010010101110000100011110010111111101001010001001101101000010000100001011101011'
        '0110011011111000000111100011011000010010101110000100011110010111111101001010001001111101000010000100'
        '0010111010110110011011111000100011110001101100001001010111000010001111001011111110100101000100101110'
        '1000010000100001011101011011001101111100011000111100011011000010010101110000100011110010111111101001'
        '0100010010111010000100001000010111010110110011011111000011000111100011011000010010101110000100011110'
        '0101111111010010100010010111010000100001000010111010110110011011111000001100011110001101100001001010'
        '1110000100011110010111111101001010001101011101000010000100001011101011011001101111100010011000111100'
        '0110110000100101011100001000111100101111111010010100011010111010000100001000010111010110110011011111'
        '0000100110001111000110110000100101011100001000111100101111111010010100011010111010000100001000010111'
        '0101101100110111110000010011000111100011011000010010101110000100011110010111111101001010101101011101'
        '0000100001000010111010110110011011111000000100110001111000110110000100101011100001000111100101111111'
        '0100101110110101110100001000010000101110101101100110111110001000100110001111000110110000100101011100'
        '0010001111001011111110100100110110101110100001000010000101110101101100110111110000100010011000111100'
        '0110110000100101011100001000111100101111111010010011011010111010000100001000010111010110110011011111'
        '0001010001001100011110001101100001001010111000010001111001011111110100100110110101110100001000010000'
        '1011101011011001101111100001010001001100011110001101100001001010111000010001111001011111110101100110'
        '1101011101000010000100001011101011011001101111100000101000100110001111000110110000100101011100001000'
        '1111001011111110101100110110101110100001000010000101110101101100110111110001001010001001100011110001'
        '1011000010010101110000100011110010111111101011001101101011101000010000100001011101011011001101111100'
        '0010010100010011000111100011011000010010101110000100011110010111111111011001101101011101000010000100'
        '0010111010110110011011111000101001010001001100011110001101100001001010111000010001111001011111111101'
        '1001101101011101000010000100001011101011011001101111100011010010100010011000111100011011000010010101'
        '1100001000111100101111111110110011011010111010000100001000010111010110110011011111000111010010100010'
        '0110001111000110110000100101011100001000111100101111111110110011011010111010000100001000010111010110'
        '1100110111110001111010010100010011000111100011011000010010101110000100011110010111011111011001101101'
        '0111010000100001000010111010110110011011111000111110100101000100110001111000110110000100101011100001'
        '0001111001011001111101100110110101110100001000010000101110101101100110111110001111110100101000100110'
        '0011110001101100001001010111000010001111001010001111101100110110101110100001000010000101110101101100'
        '11011111000'
    ),
    35: (
        '1011011101011010011011001101010011010010011011001001101110010011010110010011011011001001101010110010'
        '0110110101100100110111010110010011011110101100100110101110101100100110100111010110010011010001110101'
        '1001001101000011101011001001101100001110101100100110101000011101011001001101001000011101011001001101'
        '0001000011101011001001101000010000111010110010011011000010000111010110010011010100001000011101011001'
        '0011011010000100001110101100100110111010000100001110101100100110111101000010000111010110010011011111'
        '0100001000011101011001001101011110100001000011101011001001101101111010000100001110101100100110111011'
        '1101000010000111010110010011010110111101000010000111010110010011011011011110100001000011101011001001'
        '1011101101111010000100001110101100100110111101101111010000100001110101100100110111110110111101000010'
        '0001110101100100110101111011011110100001000011101011001001101101111011011110100001000011101011001001'
        '1010101111011011110100001000011101011001001101001011110110111101000010000111010110010011010001011110'
        '1101111010000100001110101100100110100001011110110111101000010000111010110010011011000010111101101111'
        '0100001000011101011001001101010000101111011011110100001000011101011001001101001000010111101101111010'
        '0001000011101011001001101000100001011110110111101000010000111010110010011010000100001011110110111101'
        '0000100001110101100100110110000100001011110110111101000010000111010110010011011100001000010111101101'
        '1110100001000011101011001001101111000010000101111011011110100001000011101011001001101011100001000010'
        '1111011011110100001000011101011001001101101110000100001011110110111101000010000111010110010011010101'
        '1100001000010111101101111010000100001110101100100110110101110000100001011110110111101000010000111010'
        '1100100110111010111000010000101111011011110100001000011101011001001101011010111000010000101111011011'
        '1101000010000111010110010011010011010111000010000101111011011110100001000011101011001001101100110101'
        '1100001000010111101101111010000100001110101100100110101001101011100001000010111101101111010000100001'
        '1101011001001101001001101011100001000010111101101111010000100001110101100100110110010011010111000010'
        '0001011110110111101000010000111010110010011011100100110101110000100001011110110111101000010000111010'
        '1100100110101100100110101110000100001011110110111101000010000111010110010011011011001001101011100001'
        '0000101111011011110100001000011101011001001101101100011010000000110100000100111001110100010111100011'
        '1010101110011111101100011010000000110100000100111001110100010111100011101010111001110111011000110100'
        '0000011010000010011100111010001011110001110101011100111011110110001101000000011010000010011100111010'
        '0010111100011101010111001010111110110001101000000011010000010011100111010001011110001110101011100001'
        '0011111011000110100000001101000001001110011101000101111000111010101110100100011111011000110100000001'
        '1010000010011100111010001011110001110101011111001010011111011000110100000001101000001001110011101000'
        '1011110001110101011011001011001111101100011010000000110100000100111001110100010111100011101010110110'
        '0101110011111011000110100000001101000001001110011101000101111000111010101101100100111001111101100011'
        '0100000001101000001001110011101000101111000111010101101100101011100111110110001101000000011010000010'
        '0111001110100010111100011101000110110010010111001111101100011010000000110100000100111001110100010111'
        '1000111011001101100101010111001111101100011010000000110100000100111001110100010111100011100100110110'
        '0100101011100111110110001101000000011010000010011100111010001011110001111010011011001010101011100111'
        '1101100011010000000110100000100111001110100010111100011010100110110010110101011100111110110001101000'
        '0000110100000100111001110100010111100010010100110110010111010101110011111011000110100000001101000001'
        '0011100111010001011110000001010011011001001110101011100111110110001101000000011010000010011100111010'
        '0010111100100010100110110010001110101011100111110110001101000000011010000010011100111010001011110110'
        '0010100110110010000111010101110011111011000110100000001101000001001110011101000101111111000101001101'
        '1001010001110101011100111110110001101000000011010000010011100111010001011111110001010011011001011000'
        '1110101011100111110110001101000000011010000010011100111010001011011110001010011011001011100011101010'
        '1110011111011000110100000001101000001001110011101000101101111000101001101100101111000111010101110011'
        '1110110001101000000011010000010011100111010001011011110001010011011001001111000111010101110011111011'
        '0001101000000011010000010011100111010001111011110001010011011001010111100011101010111001111101100011'
        '0100000001101000001001110011101000111101111000101001101100100101111000111010101110011111011000110100'
        '0000011010000010011100111010001111011110001010011011001000101111000111010101110011111011000110100000'
        '0011010000010011100111010101111011110001010011011001000010111100011101010111001111101100011010000000'
        '1101000001001110011101010111101111000101001101100101000101111000111010101110011111011000110100000001'
        '1010000010011100111000101111011110001010011011001001000101111000111010101110011111011000110100000001'
        '1010000010011100111000101111011110001010011011001010100010111100011101010111001111101100011010000000'
        '1101000001001110011000010111101111000101001101100101101000101111000111010101110011111011000110100000'
        '0011010000010011100110000101111011110001010011011001011101000101111000111010101110011111011000110100'
        '0000011010000010011100010000101111011110001010011011001001110100010111100011101010111001111101100011'
        '0100000001101000001001110001000010111101111000101001101100100011101000101111000111010101110011111011'
        '0001101000000011010000010011110010000101111011110001010011011001010011101000101111000111010101110011'
        '1110110001101000000011010000010011010010000101111011110001010011011001011001110100010111100011101010'
        '1110011111011000110100000001101000001001001001000010111101111000101001101100101110011101000101111000'
        '1110101011100111110110001101000000011010000010000010010000101111011110001010011011001001110011101000'
        '1011110001110101011100111110110001101000000011010000010000010010000101111011110001010011011001000111'
        '0011101000101111000111010101110011111011000110100000001101000001100001001000010111101111000101001101'
        '1001010011100111010001011110001110101011100111110110001101000000011010000001000010010000101111011110'
        '0010100110110010010011100111010001011110001110101011100111110110001101000000011010000101000010010000'
        '1011110111100010100110110010001001110011101000101111000111010101110011111011000110100000001101000110'
        '1000010010000101111011110001010011011001000010011100111010001011110001110101011100111110110001101000'
        '0000110100111010000100100001011110111100010100110110010000010011100111010001011110001110101011100111'
        '1101100011010000000110101111010000100100001011110111100010100110110010000001001110011101000101111000'
        '1110101011100111110110001101000000011010111101000010010000101111011110001010011011001010000010011100'
        '1110100010111100011101010111001111101100011010000000110101111010000100100001011110111100010100110110'
        '0100100000100111001110100010111100011101010111001111101100011010000000111101111010000100100001011110'
        '1111000101001101100101010000010011100111010001011110001110101011100111110110001101000000011110111101'
        '0000100100001011110111100010100110110010110100000100111001110100010111100011101010111001111101100011'
        '0100000001111011110100001001000010111101111000101001101100100110100000100111001110100010111100011101'
        '0101110011111011000110100000001111011110100001001000010111101111000101001101100100011010000010011100'
        '1110100010111100011101010111001111101100011010000000111101111010000100100001011110111100010100110110'
        '0100001101000001001110011101000101111000111010101110011111011000110100000001111011110100001001000010'
        '1111011110001010011011001000001101000001001110011101000101111000111010101110011111011000110100010001'
        '1110111101000010010000101111011110001010011011001000000110100000100111001110100010111100011101010111'
        '0011111011000110100010001111011110100001001000010111101111000101001101100100000001101000001001110011'
        '1010001011110001110101011100111110110001101010100011110111101000010010000101111011110001010011011001'
        '0000000011010000010011100111010001011110001110101011100111110110001101010100011110111101000010010000'
        '1011110111100010100110110010100000001101000001001110011101000101111000111010101110011111011000110001'
        '0100011110111101000010010000101111011110001010011011001001000000011010000010011100111010001011110001'
        '1101010111001111101100011100101000111101111010000100100001011110111100010100110110010101000000011010'
        '0000100111001110100010111100011101010111001111101100011100101000111101111010000100100001011110111100'
        '0101001101100101101000000011010000010011100111010001011110001110101011100111110110000110010100011110'
        '1111010000100100001011110111100010100110110010011010000000110100000100111001110100010111100011101010'
        '1110011111011001011001010001111011110100001001000010111101111000101001101100100011010000000110100000'
        '1001110011101000101111000111010101110011111011011011001010001111011110100001001000010111101111000101'
        '0011011001000011010000000110100000100111001110100010111100011101010111001111101101101100101000111101'
        '1110100001001000010111101111000101001101100101000110100000001101000001001110011101000101111000111010'
        '1011100111110100110110010100011110111101000010010000101111011110001010011011001011000110100000001101'
        '0000010011100111010001011110001110101011100111110100110110010100011110111101000010010000101111011110'
        '0010100110110010011000110100000001101000001001110011101000101111000111010101110011111010011011001010'
        '00111101111010000100100001011110111100010100110110010'
    ),
    36: (
        '0000000000100001100000110000001100000001100001000110000010001100001010001100000101000110000101010001'
        '1000011010100011000001101010001100001011010100011000011011010100011000011101101010001100000111011010'
        '1000110000101110110101000110000110111011010100011000011101110110101000110000011101110110101000110000'
        '0011101110110101000110000000111011101101010001100000000111011101101010001100001000011101110110101000'
        '1100001100001110111011010100011000001100001110111011010100011000010110000111011101101010001100000101'
        '1000011101110110101000110000001011000011101110110101000110000100101100001110111011010100011000011001'
        '0110000111011101101010001100001110010110000111011101101010001100001111001011000011101110110101000110'
        '0000111100101100001110111011010100011000000111100101100001110111011010100011000010011110010110000111'
        '0111011010100011000001001111001011000011101110110101000110000101001111001011000011101110110101000110'
        '0001101001111001011000011101110110101000110000011010011110010110000111011101101010001100000011010011'
        '1100101100001110111011010100011000000011010011110010110000111011101101010001100000000110100111100101'
        '1000011101110110101000110000100001101001111001011000011101110110101000110000110000110100111100101100'
        '0011101110110101000110000111000011010011110010110000111011101101010001100000111000011010011110010110'
        '0001110111011010100011000010111000011010011110010110000111011101101010001100001101110000110100111100'
        '1011000011101110110101000110000111011100001101001111001011000011101110110101000110000011101110000110'
        '1001111001011000011101110110101000110000101110111000011010011110010110000111011101101010001100001101'
        '1101110000110100111100101100001110111011010100011000001101110111000011010011110010110000111011101101'
        '0100011000010110111011100001101001111001011000011101110110101000110000010110111011100001101001111001'
        '0110000111011101101010001100001010110111011100001101001111001011000011101110110101000110000010101101'
        '1101110000110100111100101100001110111011010100011000000101011011101110000110100111100101100001110111'
        '0110101000110000000101011011101110000110100111100101100001110111011010100011000010001010110111011100'
        '0011010011110010110000111011101101010001100001100010101101110111000011010011110010110000111011101101'
        '0100011000001100010101101110111000011010011110010110000111011101101010001100000011000101011011101110'
        '0001101001111001011000011101110110101000110000000110001010110111011100001101001111001011000011101110'
        '1101010001100000000110001010110111011100001101001111001011000011101110110101000110000101011110010001'
        '0011111101100100000011100010101101001011011000011001101001010111100100010011111101100100000011100010'
        '1011010010110110000110011011101010111100100010011111101100100000011100010101101001011011000011001101'
        '1010101011110010001001111110110010000001110001010110100101101100001100111111010101011110010001001111'
        '1101100100000011100010101101001011011000011001111111010101011110010001001111110110010000001110001010'
        '1101001011011000011000111101101010101111001000100111111011001000000111000101011010010110110000110001'
        '1110011010101011110010001001111110110010000001110001010110100101101100001110011111001101010101111001'
        '0001001111110110010000001110001010110100101101100001110011111100110101010111100100010011111101100100'
        '0000111000101011010010110110000111001111011001101010101111001000100111111011001000000111000101011010'
        '0101101100001110011110011001101010101111001000100111111011001000000111000101011010010110110010111001'
        '1110001100110101010111100100010011111101100100000011100010101101001011011001011100111100001100110101'
        '0101111001000100111111011001000000111000101011010010110111010111001111100001100110101010111100100010'
        '0111111011001000000111000101011010010110101010111001111110000110011010101011110010001001111110110010'
        '0000011100010101101001011000101011100111101100001100110101010111100100010011111101100100000011100010'
        '1011010010111001010111001111101100001100110101010111100100010011111101100100000011100010101101001010'
        '1001010111001111110110000110011010101011110010001001111110110010000001110001010110100100010010101110'
        '0111101101100001100110101010111100100010011111101100100000011100010101101001000100101011100111110110'
        '1100001100110101010111100100010011111101100100000011100010101101001000100101011100111101011011000011'
        '0011010101011110010001001111110110010000001110001010110100100010010101110011110010110110000110011010'
        '1010111100100010011111101100100000011100010101101001000100101011100111110010110110000110011010101011'
        '1100100010011111101100100000011100010101100001000100101011100111101001011011000011001101010101111001'
        '0001001111110110010000001110001010111000100010010101110011111010010110110000110011010101011110010001'
        '0011111101100100000011100010101110001000100101011100111111010010110110000110011010101011110010001001'
        '1111101100100000011100010101110001000100101011100111101101001011011000011001101010101111001000100111'
        '1110110010000001110001011111000100010010101110011111011010010110110000110011010101011110010001001111'
        '1101100100000011100010011110001000100101011100111101011010010110110000110011010101011110010001001111'
        '1101100100000011100010011110001000100101011100111110101101001011011000011001101010101111001000100111'
        '1110110010000001110001001111000100010010101110011110101011010010110110000110011010101011110010001001'
        '1111101100100000011100010011110001000100101011100111100101011010010110110000110011010101011110010001'
        '0011111101100100000011101010011110001000100101011100111100010101101001011011000011001101010101111001'
        '0001001111110110010000001111101001111000100010010101110011111000101011010010110110000110011010101011'
        '1100100010011111101100100000011011010011110001000100101011100111111000101011010010110110000110011010'
        '1010111100100010011111101100100000010011010011110001000100101011100111111100010101101001011011000011'
        '0011010101011110010001001111110110010000000001101001111000100010010101110011110111000101011010010110'
        '1100001100110101010111100100010011111101100100000000011010011110001000100101011100111100111000101011'
        '0100101101100001100110101010111100100010011111101100100001000011010011110001000100101011100111100011'
        '1000101011010010110110000110011010101011110010001001111110110010001100001101001111000100010010101110'
        '0111100001110001010110100101101100001100110101010111100100010011111101100100011000011010011110001000'
        '1001010111001111000001110001010110100101101100001100110101010111100100010011111101100101011000011010'
        '0111100010001001010111001111000000111000101011010010110110000110011010101011110010001001111110110010'
        '1011000011010011110001000100101011100111110000001110001010110100101101100001100110101010111100100010'
        '0111111011000010110000110100111100010001001010111001111010000001110001010110100101101100001100110101'
        '0101111001000100111111011010010110000110100111100010001001010111001111001000000111000101011010010110'
        '1100001100110101010111100100010011111101111001011000011010011110001000100101011100111110010000001110'
        '0010101101001011011000011001101010101111001000100111111011110010110000110100111100010001001010111001'
        '1111100100000011100010101101001011011000011001101010101111001000100111111011110010110000110100111100'
        '0100010010101110011110110010000001110001010110100101101100001100110101010111100100010011111101111001'
        '0110000110100111100010001001010111001111101100100000011100010101101001011011000011001101010101111001'
        '0001001111100111100101100001101001111000100010010101110011111101100100000011100010101101001011011000'
        '0110011010101011110010001001111000111100101100001101001111000100010010101110011111110110010000001110'
        '0010101101001011011000011001101010101111001000100111100011110010110000110100111100010001001010111001'
        '1111111011001000000111000101011010010110110000110011010101011110010001001101000111100101100001101001'
        '1110001000100101011100111111111011001000000111000101011010010110110000110011010101011110010001001001'
        '0001111001011000011010011110001000100101011100111111111101100100000011100010101101001011011000011001'
        '1010101011110010001000001000111100101100001101001111000100010010101110011110111111011001000000111000'
        '1010110100101101100001100110101010111100100010100010001111001011000011010011110001000100101011100111'
        '1001111110110010000001110001010110100101101100001100110101010111100100010100010001111001011000011010'
        '0111100010001001010111001111100111111011001000000111000101011010010110110000110011010101011110010000'
        '0100010001111001011000011010011110001000100101011100111101001111110110010000001110001010110100101101'
        '1000011001101010101111001001001000100011110010110000110100111100010001001010111001111001001111110110'
        '0100000011100010101101001011011000011001101010101111001001001000100011110010110000110100111100010001'
        '0010101110011110001001111110110010000001110001010110100101101100001100110101010111100110100100010001'
        '1110010110000110100111100010001001010111001111100010011111101100100000011100010101101001011011000011'
        '0011010101011110001010010001000111100101100001101001111000100010010101110011110100010011111101100100'
        '0000111000101011010010110110000110011010101011110101010010001000111100101100001101001111000100010010'
        '1011100111100100010011111101100100000011100010101101001011011000011001101010101111110101001000100011'
        '1100101100001101001111000100010010101110011111001000100111111011001000000111000101011010010110110000'
        '1100110101010111111010100100010001111001011000011010011110001000100101011100111111001000100111111011'
        '0010000001110001010110100101101100001100110101010110111010100100010001111001011000011010011110001000'
        '1001010111001111111001000100111111011001000000111000101011010010110110000110011010101010011101010010'
        '0010001111001011000011010011110001000100101011100111111110010001001111110110010000001110001010110100'
        '1011011000011001101010101001110101001000100011110010110000110100111100010001001010111001111011110010'
        '0010011111101100100000011100010101101001011011000011001101010111001110101001000100011110010110000110'
        '1001111000100010010101110011111011110010001001111110110010000001110001010110100101101100001100110101'
        '0111001110101001000100011110010110000110100111100010001001010111001111010111100100010011111101100100'
        '0000111000101011010010110110000110011010111110011101010010001000111100101100001101001111000100010010'
        '10111001111'
    ),
    37: (
        '1111111111011111011110101111101011111101011110110101111001101011111001101011110100110101111001001101'
        '0111100010011010111110001001101011110100010011010111110100010011010111111010001001101011110110100010'
        '0110101111001101000100110101111000110100010011010111110001101000100110101111110001101000100110101111'
        '1110001101000100110101111011100011010001001101011111011100011010001001101011110101110001101000100110'
        '1011110010111000110100010011010111100010111000110100010011010111100001011100011010001001101011111000'
        '0101110001101000100110101111010000101110001101000100110101111001000010111000110100010011010111110010'
        '0001011100011010001001101011111100100001011100011010001001101011111110010000101110001101000100110101'
        '1111111001000010111000110100010011010111101111001000010111000110100010011010111100111100100001011100'
        '0110100010011010111110011110010000101110001101000100110101111010011110010000101110001101000100110101'
        '1110010011110010000101110001101000100110101111000100111100100001011100011010001001101011110000100111'
        '1001000010111000110100010011010111110000100111100100001011100011010001001101011110100001001111001000'
        '0101110001101000100110101111101000010011110010000101110001101000100110101111110100001001111001000010'
        '1110001101000100110101111111010000100111100100001011100011010001001101011110111010000100111100100001'
        '0111000110100010011010111100111010000100111100100001011100011010001001101011110001110100001001111001'
        '0000101110001101000100110101111100011101000010011110010000101110001101000100110101111110001110100001'
        '0011110010000101110001101000100110101111011000111010000100111100100001011100011010001001101011111011'
        '0001110100001001111001000010111000110100010011010111101011000111010000100111100100001011100011010001'
        '0011010111100101100011101000010011110010000101110001101000100110101111000101100011101000010011110010'
        '0001011100011010001001101011111000101100011101000010011110010000101110001101000100110101111010001011'
        '0001110100001001111001000010111000110100010011010111100100010110001110100001001111001000010111000110'
        '1000100110101111100100010110001110100001001111001000010111000110100010011010111111001000101100011101'
        '0000100111100100001011100011010001001101011110110010001011000111010000100111100100001011100011010001'
        '0011010111110110010001011000111010000100111100100001011100011010001001101011110101100100010110001110'
        '1000010011110010000101110001101000100110101111101011001000101100011101000010011110010000101110001101'
        '0001001101011111101011001000101100011101000010011110010000101110001101000100110101111111010110010001'
        '0110001110100001001111001000010111000110100010011010111111110101100100010110001110100001001111001000'
        '0101110001101000100110101111011110101100100010110001110100001001111001000010111000110100010011010111'
        '1101111010110010001011000111010000100111100100001011100011010001001101011101101111010110010001011000'
        '1110100001001111001000010111000110100010011010110011101111010110010001011000111010000100111100100001'
        '0111000110100010011010100011110111101011001000101100011101000010011110010000101110001101000100110100'
        '0000111101111010110010001011000111010000100111100100001011100011010001001101100001011110111101011001'
        '0001011000111010000100111100100001011100011010001001100100000101111011110101100100010110001110100001'
        '0011110010000101110001101000100111010000101011110111101011001000101100011101000010011110010000101110'
        '0011010001001010100001101011110111101011001000101100011101000010011110010000101110001101000100001010'
        '0000110101111011110101100100010110001110100001001111001000010111000110100010100101000000110101111011'
        '1101011001000101100011101000010011110010000101110001101000111001010000100110101111011110101100100010'
        '1100011101000010011110010000101110001101000011001010000010011010111101111010110010001011000111010000'
        '1001111001000010111000110100101100101000000100110101111011110101100100010110001110100001001111001000'
        '0101110001101011011001010000000100110101111011110101100100010110001110100001001111001000010111000110'
        '1111011001010000100010011010111101111010110010001011000111010000100111100100001011100011001110110010'
        '1000001000100110101111011110101100100010110001110100001001111001000010111000111011101100101000010100'
        '0100110101111011110101100100010110001110100001001111001000010111000101011101100101000011010001001101'
        '0111101111010110010001011000111010000100111100100001011100000101110110010100000110100010011010111101'
        '1110101100100010110001110100001001111001000010111001001011101100101000000110100010011010111101111010'
        '1100100010110001110100001001111001000010111011001011101100101000000011010001001101011110111101011001'
        '0001011000111010000100111100100001011111100101110110010100001000110100010011010111101111010110010001'
        '0110001110100001001111001000010110111001011101100101000011000110100010011010111101111010110010001011'
        '0001110100001001111001000010100111001011101100101000011100011010001001101011110111101011001000101100'
        '0111010000100111100100001000011100101110110010100000111000110100010011010111101111010110010001011000'
        '1110100001001111001000011000111001011101100101000010111000110100010011010111101111010110010001011000'
        '1110100001001111001000001000111001011101100101000001011100011010001001101011110111101011001000101100'
        '0111010000100111100100010100011100101110110010100000010111000110100010011010111101111010110010001011'
        '0001110100001001111001001101000111001011101100101000000010111000110100010011010111101111010110010001'
        '0110001110100001001111001011101000111001011101100101000000001011100011010001001101011110111101011001'
        '0001011000111010000100111100111110100011100101110110010100001000010111000110100010011010111101111010'
        '1100100010110001110100001001111000111101000111001011101100101000001000010111000110100010011010111101'
        '1110101100100010110001110100001001111010111101000111001011101100101000000100001011100011010001001101'
        '0111101111010110010001011000111010000100111111011110100011100101110110010100001001000010111000110100'
        '0100110101111011110101100100010110001110100001001110110111101000111001011101100101000011001000010111'
        '0001101000100110101111011110101100100010110001110100001001100110111101000111001011101100101000011100'
        '1000010111000110100010011010111101111010110010001011000111010000100100011011110100011100101110110010'
        '1000011110010000101110001101000100110101111011110101100100010110001110100001000000110111101000111001'
        '0111011001010000011110010000101110001101000100110101111011110101100100010110001110100001010000110111'
        '1010001110010111011001010000001111001000010111000110100010011010111101111010110010001011000111010000'
        '1110000110111101000111001011101100101000010011110010000101110001101000100110101111011110101100100010'
        '1100011101000001100001101111010001110010111011001010000010011110010000101110001101000100110101111011'
        '1101011001000101100011101000101100001101111010001110010111011001010000001001111001000010111000110100'
        '0100110101111011110101100100010110001110100110110000110111101000111001011101100101000000010011110010'
        '0001011100011010001001101011110111101011001000101100011101011101100001101111010001110010111011001010'
        '0000000100111100100001011100011010001001101011110111101011001000101100011101111101100001101111010001'
        '1100101110110010100001000010011110010000101110001101000100110101111011110101100100010110001110011110'
        '1100001101111010001110010111011001010000010000100111100100001011100011010001001101011110111101011001'
        '0001011000111101111011000011011110100011100101110110010100001010000100111100100001011100011010001001'
        '1010111101111010110010001011000110101111011000011011110100011100101110110010100001101000010011110010'
        '0001011100011010001001101011110111101011001000101100010010111101100001101111010001110010111011001010'
        '0001110100001001111001000010111000110100010011010111101111010110010001011000000101111011000011011110'
        '1000111001011101100101000001110100001001111001000010111000110100010011010111101111010110010001011001'
        '0001011110110000110111101000111001011101100101000000111010000100111100100001011100011010001001101011'
        '1101111010110010001011011000101111011000011011110100011100101110110010100000001110100001001111001000'
        '0101110001101000100110101111011110101100100010111110001011110110000110111101000111001011101100101000'
        '0100011101000010011110010000101110001101000100110101111011110101100100010101110001011110110000110111'
        '1010001110010111011001010000110001110100001001111001000010111000110100010011010111101111010110010001'
        '0001110001011110110000110111101000111001011101100101000001100011101000010011110010000101110001101000'
        '1001101011110111101011001000110011100010111101100001101111010001110010111011001010000101100011101000'
        '0100111100100001011100011010001001101011110111101011001000010011100010111101100001101111010001110010'
        '1110110010100000101100011101000010011110010000101110001101000100110101111011110101100100101001110001'
        '0111101100001101111010001110010111011001010000001011000111010000100111100100001011100011010001001101'
        '0111101111010110010110100111000101111011000011011110100011100101110110010100000001011000111010000100'
        '1111001000010111000110100010011010111101111010110011110100111000101111011000011011110100011100101110'
        '1100101000010001011000111010000100111100100001011100011010001001101011110111101011000111010011100010'
        '1111011000011011110100011100101110110010100000100010110001110100001001111001000010111000110100010011'
        '0101111011110101101011101001110001011110110000110111101000111001011101100101000000100010110001110100'
        '0010011110010000101110001101000100110101111011110101111011101001110001011110110000110111101000111001'
        '0111011001010000100100010110001110100001001111001000010111000110100010011010111101111010101101110100'
        '1110001011110110000110111101000111001011101100101000011001000101100011101000010011110010000101110001'
        '1010001001101011110111101000110111010011100010111101100001101111010001110010111011001010000011001000'
        '1011000111010000100111100100001011100011010001001101011110111101100110111010011100010111101100001101'
        '1110100011100101110110010100001011001000101100011101000010011110010000101110001101000100110101111011'
        '1100100110111010011100010111101100001101111010001110010111011001010000010110010001011000111010000100'
        '1111001000010111000110100010011010111101111101001101110100111000101111011000011011110100011100101110'
        '1100101000010101100100010110001110100001001111001000010111000110100010011010111101110101001101110100'
        '1110001011110110000110111101000111001011101100101000011010110010001011000111010000100111100100001011'
        '1000110100010011010111101100101001101110100111000101111011000011011110100011100101110110010100001110'
        '1011001000101100011101000010011110010000101110001101000100110101111010001010011011101001110001011110'
        '1100001101111010001110010111011001010000111101011001000101100011101000010011110010000101110001101000'
        '1001101011110000010100110111010011100010111101100001101111010001110010111011001010000'
    ),
    38: (
        '0100101010010100010100001010100010100100010100010001010100100010101100100010101110010001010011100100'
        '0101000111001000101000011100100010100000111001000101000000111001000101010000011100100010101100000111'
        '0010001010011000001110010001010101100000111001000101011011000001110010001010111011000001110010001010'
        '1111011000001110010001010111110110000011100100010100111110110000011100100010101011111011000001110010'
        '0010100101111101100000111001000101010101111101100000111001000101011010111110110000011100100010100110'
        '1011111011000001110010001010101101011111011000001110010001010010110101111101100000111001000101000101'
        '1010111110110000011100100010100001011010111110110000011100100010101000101101011111011000001110010001'
        '0101100010110101111101100000111001000101001100010110101111101100000111001000101000110001011010111110'
        '1100000111001000101000011000101101011111011000001110010001010100011000101101011111011000001110010001'
        '0100100011000101101011111011000001110010001010101000110001011010111110110000011100100010101101000110'
        '0010110101111101100000111001000101001101000110001011010111110110000011100100010101011010001100010110'
        '1011111011000001110010001010010110100011000101101011111011000001110010001010101011010001100010110101'
        '1111011000001110010001010110101101000110001011010111110110000011100100010101110101101000110001011010'
        '1111101100000111001000101011110101101000110001011010111110110000011100100010101111101011010001100010'
        '1101011111011000001110010001010011111010110100011000101101011111011000001110010001010101111101011010'
        '0011000101101011111011000001110010001010110111110101101000110001011010111110110000011100100010100110'
        '1111101011010001100010110101111101100000111001000101000110111110101101000110001011010111110110000011'
        '1001000101000011011111010110100011000101101011111011000001110010001010000011011111010110100011000101'
        '1010111110110000011100100010100000011011111010110100011000101101011111011000001110010001010100000110'
        '1111101011010001100010110101111101100000111001000101011000001101111101011010001100010110101111101100'
        '0001110010001010111000001101111101011010001100010110101111101100000111001000101001110000011011111010'
        '1101000110001011010111110110000011100100010100011100000110111110101101000110001011010111110110000011'
        '1001000101010011100000110111110101101000110001011010111110110000011100100010100100111000001101111101'
        '0110100011000101101011111011000001110010001010001001110000011011111010110100011000101101011111011000'
        '0011100100010100001001110000011011111010110100011000101101011111011000001110010001010100010011100000'
        '1101111101011010001100010110101111101100000111001000101001000100111000001101111101011010001100010110'
        '1011111011000001110010001010101000100111000001101111101011010001100010110101111101100000111001000101'
        '0010100010011100000110111110101101000110001011010111110110000011100100010101001101100000010101011110'
        '0101000011111100101111001010000100011100110011011001001101100000010101011110010100001111110010111100'
        '1010000100011100110011011110100110110000001010101111001010000111111001011110010100001000111001100110'
        '1011101001101100000010101011110010100001111110010111100101000010001110011001101010110100110110000001'
        '0101011110010100001111110010111100101000010001110011001101011011010011011000000101010111100101000011'
        '1111001011110010100001000111001100110101110110100110110000001010101111001010000111111001011110010100'
        '0010001110011001101010110110100110110000001010101111001010000111111001011110010100001000111001101110'
        '1010011011010011011000000101010111100101000011111100101111001010000100011100110111010110011011010011'
        '0110000001010101111001010000111111001011110010100001000111001101110101110011011010011011000000101010'
        '1111001010000111111001011110010100001000111001101110101011001101101001101100000010101011110010100001'
        '1111100101111001010000100011100110111010100110011011010011011000000101010111100101000011111100101111'
        '0010100001000111001101110101100110011011010011011000000101010111100101000011111100101111001010000100'
        '0110001101110101110011001101101001101100000010101011110010100001111110010111100101000010001100011011'
        '1010111100110011011010011011000000101010111100101000011111100101111001010000100011000110111010101110'
        '0110011011010011011000000101010111100101000011111100101111001010000100111000110111010100111001100110'
        '1101001101100000010101011110010100001111110010111100101000010111100011011101010001110011001101101001'
        '1011000000101010111100101000011111100101111001010000111111000110111010110001110011001101101001101100'
        '0000101010111100101000011111100101111001010000011111000110111010101000111001100110110100110110000001'
        '0101011110010100001111110010111100101000001111100011011101010010001110011001101101001101100000010101'
        '0111100101000011111100101111001010010011111000110111010100010001110011001101101001101100000010101011'
        '1100101000011111100101111001010010011111000110111010100001000111001100110110100110110000001010101111'
        '0010100001111110010111100101001001111100011011101011000010001110011001101101001101100000010101011110'
        '0101000011111100101111001000010011111000110111010101000010001110011001101101001101100000010101011110'
        '0101000011111100101111001000010011111000110111010110100001000111001100110110100110110000001010101111'
        '0010100001111110010111100000001001111100011011101010101000010001110011001101101001101100000010101011'
        '1100101000011111100101111010000010011111000110111010100101000010001110011001101101001101100000010101'
        '0111100101000011111100101111010000010011111000110111010110010100001000111001100110110100110110000001'
        '0101011110010100001111110010111101000001001111100011011101011100101000010001110011001101101001101100'
        '0000101010111100101000011111100101101010000010011111000110111010111100101000010001110011001101101001'
        '1011000000101010111100101000011111100101001010000010011111000110111010111110010100001000111001100110'
        '1101001101100000010101011110010100001111110010100101000001001111100011011101010111100101000010001110'
        '0110011011010011011000000101010111100101000011111100101001010000010011111000110111010110111100101000'
        '0100011100110011011010011011000000101010111100101000011111100101001010000010011111000110111010101011'
        '1100101000010001110011001101101001101100000010101011110010100001111110110100101000001001111100011011'
        '1010100101111001010000100011100110011011010011011000000101010111100101000011111111101001010000010011'
        '1110001101110101100101111001010000100011100110011011010011011000000101010111100101000011111011101001'
        '0100000100111110001101110101110010111100101000010001110011001101101001101100000010101011110010100001'
        '1110011101001010000010011111000110111010111100101111001010000100011100110011011010011011000000101010'
        '1111001010000111100111010010100000100111110001101110101111100101111001010000100011100110011011010011'
        '0110000001010101111001010000111100111010010100000100111110001101110101111110010111100101000010001110'
        '0110011011010011011000000101010111100101000011110011101001010000010011111000110111010111111100101111'
        '0010100001000111001100110110100110110000001010101111001010000011100111010010100000100111110001101110'
        '1010111111001011110010100001000111001100110110100110110000001010101111001010001011100111010010100000'
        '1001111100011011101010011111100101111001010000100011100110011011010011011000000101010111100101000101'
        '1100111010010100000100111110001101110101000111111001011110010100001000111001100110110100110110000001'
        '0101011110010100010111001110100101000001001111100011011101010000111111001011110010100001000111001100'
        '1101101001101100000010101011110010110010111001110100101000001001111100011011101011000011111100101111'
        '0010100001000111001100110110100110110000001010101111001001001011100111010010100000100111110001101110'
        '1010100001111110010111100101000010001110011001101101001101100000010101011110011010010111001110100101'
        '0000010011111000110111010110100001111110010111100101000010001110011001101101001101100000010101011110'
        '0010100101110011101001010000010011111000110111010101010000111111001011110010100001000111001100110110'
        '1001101100000010101011110001010010111001110100101000001001111100011011101010010100001111110010111100'
        '1010000100011100110011011010011011000000101010111100010100101110011101001010000010011111000110111010'
        '1100101000011111100101111001010000100011100110011011010011011000000101010111000010100101110011101001'
        '0100000100111110001101110101110010100001111110010111100101000010001110011001101101001101100000010101'
        '0110000010100101110011101001010000010011111000110111010111100101000011111100101111001010000100011100'
        '1100110110100110110000001010101100000101001011100111010010100000100111110001101110101111100101000011'
        '1111001011110010100001000111001100110110100110110000001010100100000101001011100111010010100000100111'
        '1100011011101010111100101000011111100101111001010000100011100110011011010011011000000101010010000010'
        '1001011100111010010100000100111110001101110101101111001010000111111001011110010100001000111001100110'
        '1101001101100000010101001000001010010111001110100101000001001111100011011101010101111001010000111111'
        '0010111100101000010001110011001101101001101100000010111001000001010010111001110100101000001001111100'
        '0110111010110101111001010000111111001011110010100001000111001100110110100110110000001011100100000101'
        '0010111001110100101000001001111100011011101010101011110010100001111110010111100101000010001110011001'
        '1011010011011000000111110010000010100101110011101001010000010011111000110111010110101011110010100001'
        '1111100101111001010000100011100110011011010011011000000111110010000010100101110011101001010000010011'
        '1110001101110101010101011110010100001111110010111100101000010001110011001101101001101100000011111001'
        '0000010100101110011101001010000010011111000110111010100101010111100101000011111100101111001010000100'
        '0111001100110110100110110000001111100100000101001011100111010010100000100111110001101110101000101010'
        '1111001010000111111001011110010100001000111001100110110100110110000001111100100000101001011100111010'
        '0101000001001111100011011101010000101010111100101000011111100101111001010000100011100110011011010011'
        '0110010001111100100000101001011100111010010100000100111110001101110101000001010101111001010000111111'
        '0010111100101000010001110011001101101001101101100011111001000001010010111001110100101000001001111100'
        '0110111010100000010101011110010100001111110010111100101000010001110011001101101001101101100011111001'
        '0000010100101110011101001010000010011111000110111010110000001010101111001010000111111001011110010100'
        '0010001110011001101101001101101100011111001000001010010111001110100101000001001111100011011101011100'
        '0000101010111100101000011111100101111001010000100011100110011011010011011011000111110010000010100101'
        '1100111010010100000100111110001101110101011000000101010111100101000011111100101111001010000100011100'
        '1100110110100111110110001111100100000101001011100111010010100000100111110001101110101101100000010101'
        '0111100101000011111100101111001010000100011100110011011010010111011000111110010000010100101110011101'
        '0010100000100111110001101110101110110000001010101111001010000111111001011110010100001000111001100110'
        '1101001011101100011111001000001010010111001110100101000001001111100011011101010110110000001010101111'
        '0010100001111110010111100101000010001110011001101101001011101100011111001000001010010111001110100101'
        '0000010011111000110111010100110110000001010101111001010000111111001011110010100001000111001100110110'
        '110101110110001111100100000101001011100111010010100000100111110001101110101'
    ),
    39: (
        '0000000000000000000000011110000111101000111001100011000111000100001111000000000111100000000011001100'
        '1111010110010001111011011001000111000110110110001100000110111110001000010011011111000000001100110011'
        '1100000000011000101100110011110101100010110010001111001011001101100100011100001011001101101100011000'
        '0001011001101111100010000100010110011011111000000001100010110011001111000000001100101011000101100110'
        '0111101110010101100010110010001111001110010101100110110010001110010111000010110011011011000110000101'
        '1100001011001101111100010000001011110001011001101111100000000100101111000101100110011110000000010010'
        '1111001010110001011001100111101100101111001010110001011001000111101110010011100101011001101100100011'
        '1000111001101110000101100110110110001100010111000101110000101100110111110001000001011100010111100010'
        '1100110111110000000000101111001011110001011001100111100000000111010010010111100101011000101100110011'
        '1100111010110010111100101011000101100100011110001110111100100111001010110011011001000111001001110011'
        '1001101110000101100110110110001100001001111011100010111000010110011011111000100001010011010111000101'
        '1110001011001101111100000000110100100101111001011110001011001100111100000000110100111101001001011110'
        '0101011000101100110011110111010001110101100101111001010110001011001000111100111010001110111100100111'
        '0010101100110110010001110000111011001110011100110111000010110011011011000110001001110010011110111000'
        '1011100001011001101111100010000010011110100110101110001011110001011001101111100000000101001111010010'
        '0101111001011110001011001100111100000000010001111010011110100100101111001010110001011001100111101010'
        '0011110100011101011001011110010101100010110010001111011010000111010001110111100100111001010110011011'
        '0010001110001101000011101100111001110011011100001011001101101100011000001101010011100100111101110001'
        '0111000010110011011111000100000001101010011110100110101110001011110001011001101111100000000100011010'
        '1001111010010010111100101111000101100110011110000000001100110100011110100111101001001011110010101100'
        '0101100110011110101100110100011110100011101011001011110010101100010110010001111011011001101000011101'
        '0001110111100100111001010110011011001000111000110110011010000111011001110011100110111000010110011011'
        '0110001100000110110011010100111001001111011100010111000010110011011111000100001001101000110101001111'
        '0100110101110001011110001011001101111100000000110011010001101010011110100100101111001011110001011001'
        '1001111000000000011110011001101000111101001111010010010111100101011000101100110011110000111110110011'
        '0100011110100011101011001011110010101100010110010001111010001111101100110100001110100011101111001001'
        '1100101011001101100100011100110001101101100110100001110110011100111001101110000101100110110110001100'
        '0111000100110110011010100111001001111011100010111000010110011011111000100001111000100110100011010100'
        '1111010011010111000101111000101100110111110000000001111001100110100011010100111101001001011110010111'
        '1000101100110011110000000010101111000110000111001100001111110000100100010000101010100101001111011101'
        '1101101011010001100001110011000011111110001000000100001010101001011011110011011111110101101000110000'
        '1100011001011111010001000000101001010101001001011111011011111111010110100011000010000110110111100100'
        '0100000010100101010100110101111101101111011110101101001110000000001111101111001000100000010100101010'
        '1001101011111011011111011110001101001110001000001111101101001000100000010100101010101110101011101111'
        '1110101111000110100111001100000111110100100100010000101010000101011111010101110111111111011101010111'
        '1000110000111001100001111110000100100010000101010100101001111011100001011011111010110100011000011100'
        '1100001111111000100000010000101010100101101111011100001101101111101011010001100001100011001011111010'
        '0010000001010010101010010010111101110001111011011111010110100011000010000110110111100100010000001010'
        '0101010100110101110011100111111011001111010110100111000000000111110111100100010000001010010101010011'
        '0101100011101111011101110111100011010011100010000011111011010010001000000101001010101011101010000111'
        '1111110111010101111000110100111001100000111110100100100010000101010000101011111010100001111111101111'
        '0111011101010111100011000011100110000111111000010010001000010101010010101001100110000110111100110111'
        '1101011010001100001110011000011111110001000000100001010101001010100110111000010101111101101111101011'
        '0100011000011000110010111110100010000001010010101010010001001101110001110101111101101111101011010001'
        '1000010000110110111100100010000001010010101010011001001001110011111010111110110011110101101001110000'
        '0000011111011110010001000000101001010101001100100000111011111110101011101110111100011010011100010000'
        '0111110110100100010000001010010101010011001000001111111111110101011101010111100011010011100110000011'
        '1110100100100010000101010000101010011001100001111111110010100111101110111010101111000110000111001100'
        '0011111100001001000100001010101001110100110011000010100101101111001101111101011010001100001110011000'
        '0111111100010000001000010101010011101001101110000110100100101111101101111101011010001100001100011001'
        '0111110100010000001010010101010011001001101110001101010011010111110110111110101101000110000100001101'
        '1011110010001000000101001011101001100100100111001111010100110101111101100111101011010011100000000011'
        '1110111100100010000001010010111010011001000001110111101010101110101011101110111100011010011100010000'
        '0111110110100100010000001010010111010011001000001111111100101011111010101110101011110001101001110011'
        '0000011111010010010001000010101000011101001100110000111111110101010100101001111011101110101011110001'
        '1000011100110000111111000010010001000001101010011101001100110000100101010100101101111001101111101011'
        '0100011000011100110000111111100010000001000001101010011101001101110000110010101010010010111110110111'
        '1101011010001100001100011001011111010001000000101000110101001100100110111000110100101010100110101111'
        '1011011111010110100011000010000110110111100100010000001010001111010011001001001110011110100101010100'
        '1101011111011001111010110100111000000000111110111100100010000001010001111010011001000001110111101010'
        '0101010101110101011101110111100011010011100010000011111011010010001000001101000011101001100100000111'
        '1111110101000010101111101010111010101111000110100111001100000111110100100100010000011010000111010011'
        '0011000011111111000100001010101001010011110111011101010111100011000011100110000111111000010010110100'
        '0011010100111010011001100001000010000101010100101101111001101111101011010001100001110011000011111110'
        '0010000110100001101010011101001101110000100000101001010101001001011111011011111010110100011000011000'
        '1100101111101000100001101100011010100110010011011100011000000101001010101001101011111011011111010110'
        '1000110000100001101101111001000110001100100011110100110010010011100111100000010100101010100110101111'
        '1011001111010110100111000000000111110111100100001000111010001111010011001000001110111101000000101001'
        '0101010111010101110111011110001101001110001000001111101101001001010001110100001110100110010000011111'
        '1110010000101010000101011111010101110101011110001101001110011000001111101001001011010000110100001110'
        '1001100110000111111110001001000100001010101001010011110111011101010111100011000011100110000111111000'
        '0101101101000011010100111010011001100001100010000001000010101010010110111100110111110101101000110000'
        '1110011000011111110001010011010000110101001110100110111000010100010000001010010101010010010111110110'
        '1111101011010001100001100011001011111110001000011011000110101001100100110111000110010001000000101001'
        '0101010011010111110110111110101101000110000100001101101111011000110001100100011110100110010010011100'
        '1111001000100000010100101010100110101111101100111101011010011100000000011111011110110000100011101000'
        '1111010011001000001110111101001000100000010100101010101110101011101110111100011010011100010000011111'
        '0110101100101000111010000111010011001000001111111100100100010000101010000101011111010101110101011110'
        '0011010011100110000011111010010110110100001101000011101001100110000111111111111110000100100010000101'
        '0101001010011110111011101010111100011000011100110000001011000010110110100001101010011101001100110000'
        '1011111110001000000100001010101001011011110011011111010110100011000011100110000001011100010100110100'
        '0011010100111010011011100001101111101000100000010100101010100100101111101101111101011010001100001100'
        '0110010001011100010000110110001101010011001001101110001111011110010001000000101001010101001101011111'
        '0110111110101101000110000100001101100010011000110001100100011110100110010010011100111111011110010001'
        '0000001010010101010011010111110110011110101101001110000000001101100011011000010001110100011110100110'
        '0100000111011111111011010010001000000101001010101011101010111011101111000110100111000100000110110000'
        '1011001010001110100001110100110010000011111111111110100100100010000101010000101011111010101110101011'
        '1100011010011100110000001011000010110110100001101000011101001100110000111111110110000111111000010010'
        '0010000101010100101001111011101110101011110001100001110101110000101100001011011010000110101001110100'
        '1100110000100110000111111100010000001000010101010010110111100110111110101101000110000111010111000010'
        '1110001010011010000110101001110100110111000010001100101111101000100000010100101010100100101111101101'
        '1111010110100011000011001011110001011100010000110110001101010011001001101110001100001101101111001000'
        '1000000101001010101001101011111011011111010110100011000011001011110001001100011000110010001111010011'
        '0010010011100111000001111101111001000100000010100101010100110101111101100111101011010011100001100101'
        '0110001101100001000111010001111010011001000001110111110000011111011010010001000000101001010101011101'
        '0101110111011110001101001110001110010101100001011001010001110100001110100110010000011111111110000011'
        '1110100100100010000101010000101011111010101110101011110001101001110001110010101100001011011010000110'
        '1000011101001100110000111111110001110011000011111100001001000100001010101001010011110111011101010111'
        '1000110100110010111000010110000101101101000011010100111010011001100001000011100110000111111100010000'
        '0010000101010100101101111001101111101011010001101001100101110000101110001010011010000110101001110100'
        '1101110000110000110001100101111101000100000010100101010100100101111101101111101011010001001001100101'
        '1110001011100010000110110001101010011001001101110001111000010000110110111100100010000001010010101010'
        '0110101111101101111101011010001001001100101111000100110001100011001000111101001100100100111001111110'
        '0000000011111011110010001000000101001010101001101011111011001111010110100110010011001010110001101100'
        '0010001110100011110100110010000011101111011100010000011111011010010001000000101001010101011101010111'
        '0111011110001101001100101110010101100001011001010001110100001110100110010000011111111001110011000001'
        '1111010010010001000010101000010101111101010111010101111000110100110010111001010110000101101101000011'
        '0100001110100110011000011111111100011000011100110000111111000010010001000010101010010100111101110111'
        '0101011111000011001100101110000101100001011011010000110101001110100110011000010100011000011100110000'
        '1111111000100000010000101010100101101111001101111101011111000001001100101110000101110001010011010000'
        '1101010011101001101110000110100011000011000110010111110100010000001010010101010010010111110110111110'
        '1010111000001001100101111000101110001000011011000110101001100100110111000111101000110000100001101101'
        '1110010001000000101001010101001101011111011011111010001110010010011001011110001001100011000110010001'
        '1110100110010010011100111011010011100000000011111011110010001000000101001010101001101011111011001111'
        '0100011101100100110010101100011011000010001110100011110100110010000011101111001101001110001000001111'
        '1011010010001000000101001010101011101010111011101111000001110110010111001010110000101100101000111010'
        '0001110100110010000011111111000110100111001100000111110100100100010000101010000101011111010101110101'
        '011111000011001100101110010101100001011011010000110100001110100110011000011111111'
    ),
    41: (
        '1111100111101111011101100111011101111001110010101111001001111010110011001111101000101110111010001111'
        '1011111100110010111011110001011001111110100010111011110001011110011100101000111101011110010011001110'
        '0111101011111100010001100111110111010001100010111011111100110001000111110111010111100111001100101110'
        '0011110101111000101100111100111001111010001011101111100000001101101000110101010110000000110110101010'
        '0111011010000001011010011101000111100011000010011011010101101011000001100001001101111010100101110001'
        '0100000110110101111010010111000000110110100110010101011101110000000011011010011001110101110011100000'
        '0101101001101100011110011101110110101011100000001101101001001110011010011100110000000110110100101111'
        '0011101000111010000001011010010011110101110101101000011000010011011000110011111011010100100001100001'
        '0011011100010111011101111010000010100000110110101000111110111010101011000000110110100110111001100101'
        '1100011101010000000110110100111111000101100111100011110000000101101001101111010001011101111101101000'
        '1101010111000000010001011110011100101101101010100111001100000001000111101011110011011010011101000111'
        '0100000000110011100111101011100110110101011010000110000111100010001100111110010011011110101001000011'
        '0001110100011000101110111001101101011110100000101000111001100010001111101111101001100101010110000001'
        '1001011110011100110010111001101001100111010100000001100111101011110001011001111010011011000111100000'
        '0010110011100111101000101110111101000000011110001101100101110000000110110100011010101110000000101011'
        '1001100101100110000000110110101010011101011000000110101010011001011101000000101101001110100011110001'
        '0100000101111010110110000011000010011011010101101011000011000010010101111011001000001100001001101111'
        '0101001011100001100001011010101101100100010100000110110101111010010111000000101110001011100101101000'
        '0001101101001100101010111011100000001100111001010101101100000000110110100110011101011100111000000011'
        '1010101100010110110000001011010011011000111100111011110110010110100000001111000101101010111000000011'
        '0110100100111001110010110110000000101011100101001110011000000011011010010111100101100101101100000011'
        '0101010110100011101000000101101001001111010111011011000001010000010111101010110100001100001001101100'
        '0110011111011011001000011000010010101111010100100001100001001101110001011101110110110010000110000101'
        '1010101111010000010100000110110101000111110111100101101000000101110001011010101011000000110110100110'
        '1110011001011100101101100000001100111001010011101010000000110110100111111000101100111001011011000000'
        '0111010101101000111100000001011010011011110100010111011101111000110110010110100000011011010001101010'
        '1110000000100010111100111001101011100110010110110000000011011010101001110011000000010001111010111100'
        '1110101010011001011011000000101101001110100011101000000001100111001111010110010111101011011000001010'
        '0010011011010101101000011000011110001000110011111010010101111011001000011000001001101111010100100001'
        '1000111010001100010111011101011010101101100100001100000110110101111010000010100011100110001000111110'
        '1111100010111001011010000001011101001100101010110000001100101111001110011001011100111001010101101100'
        '0000011001101001100111010100000001100111101011110001011001111010101100010110110000000111010011011000'
        '1111000000010110011100111101000101110111011101110100010111100111001101000000011110001101100101110000'
        '0001101101000110101011011100110100011110101111001100000001010111001100101100110000000110110101010011'
        '1001100111010011001110011110100110000001101010100110010111010000001011010011101000110011001110111110'
        '0010001100111000101000001011110101101100000110000100110110101011010001011101110111010001100010111000'
        '1100001001010111101100100000110000100110111101010011000101110011111001100010001111000011000010110101'
        '0110110010001010000011011010111101000100010111001101011110011100110000000010111000101110010110100000'
        '0110110100110010101011010001110011101001111010111100010000000110011100101010110110000000011011010011'
        '0011101010011000011101110100111001111010001000000011101010110001011011000000101101001101100011110100'
        '0100010011100101110111010001011110110010110100000001111000101101010111000000011011010001100011001011'
        '1100101110011010001111110010110110000000101011100101001110011000000011011010101000011000111101011001'
        '1101001100111011001011011000000110101010110100011101000000101101001110000101000011001111100111011111'
        '0001010110110000010100000101111010101101000011000010011011011001100000110001011101110111011101000111'
        '0110010000110000100101011110101001000011000010011011011101000100001000111110111001111100110001101100'
        '1000011000010110101011110100000101000001101101101110000010001110011001011100110101111001001011010000'
        '0010111000101101010101100000011011010011000011001101000111110001011001110100111101001011011000000011'
        '0011100101001110101000000011011010011000011101001100011101000101110111010011100100101101100000001110'
        '1010110100011110000000101101001101000101110100010001000101111001110010111011100111100011011001011010'
        '0000011011010001101010111000000001110100001100011001000111101011110010111001110101110011001011011000'
        '0000011011010101001110011000000101110000101000011000110011100111101011001110111010101001100101101100'
        '0000101101001110100011101000000110011000110000101001111000100011001111100111010010111101011011000001'
        '0100010011011010101101000011000000001110111001100000111101000110001011101110111010010101111011001000'
        '0110000010011011110101001000011000000101110011101000100011100110001000111110111001101011010101101100'
        '1000011000001101101011110100000101000000110011101110000010000101111001110011001011100111100010111001'
        '0110100000010111010011001010101100000011010100001100011001101000100111101011110001011001110101110010'
        '1010110110000000110011010011001110101000000011110000101000011101001100010011100111101000101110111010'
        '1010110001011011000000011101001101100011110000000101011000110000101110100010001100000001101101000110'
        '1010101110111010001011110011100110100000001111000110110010100111111100100101110010101001100000001101'
        '1010101001110101110011010001111010111100110000000101011100110010110100111111100100101010110001010100'
        '0000101101001110100011110011101001100111001111010011000000110101010011001011010111111010010110001011'
        '1000000011000010011011010101101011001110111110001000110011100010100000101111010110110011100111101100'
        '1001010100101001000011000010011011110101001011101110111010001100010111000110000100101011110110010111'
        '1001111011001000010101101000000101000001101101011110100101110011111001100010001111000011000010110101'
        '0110110011110101111100100101000010110100000000011011010011001010101110111001101011110011100110000000'
        '0101110001011100101101111111001001011001101010100010001000000011011010011001110101110011101001111010'
        '1111000100000001100111001010101101101111111001001011001100010100011000000000101101001101100011110011'
        '1011101001110011110100010000000111010101100010110111111110100101100100111000011000100001101010111000'
        '0000110110100100111001011101110100010111101100101101000000011110001100101010001111111001001011011000'
        '1101010011100110000000110110100101111001011100110100011111100101101100000001010111000101100011001111'
        '1110010010110100001101101000111010000001011010010011110101100111010011001110110010110110000001101010'
        '1000101110001011111101001011011000010100101011010000110000100110110001100111110011101111100010101101'
        '1000001010000010111100101001011110011110110010011100110000011101010010000110000100110111000101110111'
        '0111011101000111011001000011000010010101100101011011110011110110010001110100010000111101000001010000'
        '0110110101000111110111001111100110001101100100001100001011010110000101111101011111001001010111000001'
        '0000101010110000001101101001101110011001011100110101111001001011010000001011100010111010101001111110'
        '0100101100100011001101000100111010100000001101101001111110001011001110100111101001011011000000011001'
        '1100101110001010111111100100101100000011101001100010001111000000010110100110111101000101110111010011'
        '1001001011011000000011101010110011100001111111010010110010000101110100010001101101000110101011100000'
        '0010001011110011100101110111001111000110110010110100000000100101110010101000111111101110100001100011'
        '0011011010101001110011000000010001111010111100101110011101011100110010110110000000100100101010110001'
        '1001111111011100001010000110101101001110100011101000000001100111001111010110011101110101010011001011'
        '0110000000100101100010111000101111111100110001100001010010011011010101101000011000011110001000110011'
        '1110011101001011110101101100000101000011001001010100101111001111000011101110011000001010011011110101'
        '0010000110001110100011000101110111011101001010111101100100001100001011001000010101101111001110001011'
        '1001110100010000011011010111101000001010001110011000100011111011100110101101010110110010000110001100'
        '1001010000101111101011100011001110111000001000110100110010101011000000110010111100111001100101110011'
        '1100010111001011010000001010010110011010101001111110011010000110001100110100010110100110011101010000'
        '0001100111101011110001011001110101110010101011011000000011010010110011000101011111110011000010100001'
        '1101001100010100110110001111000000010110011100111101000101110111010101011000101101100000001101011001'
        '0011100001111111010011000110000101110100010001010000000111100011011001011100000001101101000110101010'
        '1110111010001011110011100101011111110000111001001101000111111100100101110010101011000000010101110011'
        '0010110011000000011011010101001110101110011010001111010111100001111111010100011001101001100111111100'
        '1001010101100010011000000110101010011001011101000000101101001110100011110011101001100111001111010100'
        '1111110010101011001101000101111110100101100010111000000010100000101111010110110000011000010011011010'
        '1011010110011101111100010001100111111010111110100001010010011111001111011001001010100101001000110000'
        '1001010111101100100000110000100110111101010010111011101110100011000101111110011110110101000010011011'
        '1110011110110010000101011010000000110000101101010110110010001010000011011010111101001011100111110011'
        '0001000111111110011110100101010010011011101011111001001010000101101000000000101110001011100101101000'
        '0001101101001100101010111011100110101111001110011001111110100011101000110100101111110010010110011010'
        '1010001000100000011001110010101011011000000001101101001100111010111001110100111101011110001011111100'
        '1100011010101001001111111100100101100110001010001100000000001110101011000101101100000010110100110110'
        '0011110011101110100111001111010001111111100010101001110100100111111010010110010011100001100010001011'
        '0010110100000001111000101101010111000000011011010010011100101110111010001011101001101001011111110000'
        '1110100101010001111111001001011011000110110010110110000000101011100101001110011000000011011010010111'
        '1001011100110100011110011010010011111110101000110101100011001111111001001011010000110011001011011000'
        '0001101010101101000111010000001011010010011110101100111010011001111001101001001111110010101010010111'
        '0001011111101001011011000010100101101100000101000001011110101011010000110000100110110001100111110011'
        '1011111000100100100111110101111101000010101001011110011110110010011100110000011101100100001100001001'
        '0101111010100100001100001001101110001011101110111011101000100100110111100111101101010000101011011110'
        '0111101100100011101000100001101100100001100001011010101111010000010100000110110101000111110111001111'
        '1001100100100110111100111101001010100001011111010111110010010101110000010001001011010000001011100010'
        '1101010101100000011011010011011100110010111001101011110001101001011111101000111010010101010011111100'
        '1001011001000110011010001010110110000000110011100101001110101000000011011010011111100010110011101001'
        '1110101010010011111110011000110101100010101111111001001011000000111010011000001011011000000011101010'
        '1101000111100000001011010011011110100010111011101001110011101001001111111000101010010111000011111110'
        '1001011001000010111010001000011110001101100101101000000110110100011010101110000000100010111100111001'
        '0111011101000011100100110100101111110010010111001010100011111110111010000110001101010111001100101101'
        '1000000001101101010100111001100000001000111101011110010111001101010001100110100100111111110010010101'
        '0110001100111111101110000101000011011010101001100101101100000010110100111010001110100000000110011100'
        '1111010110011101001010101100110100100111111010010110001011100010111111110011000110000101000010111101'
        '0110110000010100010011011010101101000011000011110001000110011111001110111010000101001001111101011101'
        '1001001010100101111001111000011101110011000001100101011110110010000110000010011011110101001000011000'
        '1110100011000101110111011100110101000010011011110011111011001000010101101111001110001011100111010001'
        '0000101101010110110010000110000011011010111101000001010001110011000100011111011100111010010101001001'
        '1011110011111001001010000101111101011100011001110111000001000110001011100101101000000101110100110010'
        '1010110000001100101111001110011001011100110011101000110100101111110100010110011010101001111110011010'
        '0001100011001101000101110010101011011000000011001101001100111010100000001100111101011110001011001110'
        '1100011010101001001111111001100101100110001010111111100110000101000011101001100010101011000101101100'
        '0000011101001101100011110000000101100111001111010001011101110010101001110100100111111100010110010011'
        '10000111111101001100011000010111010001000'
    ),
    42: (
        '0100111100011111101000001111101101000110011111100110100011110011111101001101000001111001111101101001'
        '1010000000111100111110101101001101000110000111100111111001011010011010001111000011110011111101001011'
        '0100110100011111100001111001111110101001011010011010000011111100001111001111101101010010110100110100'
        '0000011111100001111001111101011010100101101001101000000000111111000011110011111010101101010010110100'
        '1101000000000001111110000111100111110101010110101001011010011010000000000000111111000011110011111010'
        '1010101101010010110100110100011000000000011111100001111001111110010101010110101001011010011010000011'
        '0000000000111111000011110011111011001010101011010100101101001101000110011000000000011111100001111001'
        '1111100110010101010110101001011010011010000011001100000000001111110000111100111110110011001010101011'
        '0101001011010011010001100110011000000000011111100001111001111110011001100101010101101010010110100110'
        '1000111100110011000000000011111100001111001111110100110011001010101011010100101101001101000001111001'
        '1001100000000001111110000111100111110110100110011001010101011010100101101001101000110011110011001100'
        '0000000011111100001111001111110011010011001100101010101101010010110100110100000110011110011001100000'
        '0000011111100001111001111101100110100110011001010101011010100101101001101000110011001111001100110000'
        '0000001111110000111100111111001100110100110011001010101011010100101101001101000001100110011110011001'
        '1000000000011111100001111001111101100110011010011001100101010101101010010110100110100000001100110011'
        '1100110011000000000011111100001111001111101011001100110100110011001010101011010100101101001101000000'
        '0001100110011110011001100000000001111110000111100111110101011001100110100110011001010101011010100101'
        '1010011010000000000011001100111100110011000000000011111100001111001111101010101100110011010011001100'
        '1010101011010100101101001101000000000000011001100111100110011000000000011111100001111001111101010101'
        '0110011001101001100110010101010110101001011010011010001100000000001100110011110011001100000000001111'
        '1100001111001111110010101010110011001101001100110010101010110101001011010011010001111000000000011001'
        '1001111001100110000000000111111000011110011111101001010101011001100110100110011001010101011010100101'
        '1010011010001111110000000000110011001111001100110000000000111111000011110011111101010010101010110011'
        '0011010011001100101010101101010010110100110100000111111000000000011001100111100110011000000000011111'
        '1000011110011111011010100101010101100110011010011001100101010101101010010110100110100000001111110000'
        '0000001100110011110011001100000000001111110000111100111110101101010010101010110011001101001100110010'
        '1010101101010010110100110100011000011111100000000001100110011110011001100000000001111110000111100111'
        '1110010110101001010101011001100110100110011001010101011010100101101001101000111100001111110000000000'
        '1100110011110011001100000000001111110000111100111111010010110101001010101011001100110100110011001010'
        '1010110101001011010011010000011110000111111000000000011001100111100110011000000000011111100001111001'
        '1111011010010110101001010101011001100110100110011001010101011010100101101001101000110011110000111111'
        '0000000000110011001111001100110000000000111111000011110011111100110100101101010010101010110011001101'
        '0011001100101010101101010010110100110100011110011110000111111000000000011001100111100110011000000000'
        '0111111000011110011111101001101001011010100101010101100110011010011001100101010101101010010110100110'
        '1000111111111111111111111111111111111111111111111111111111111111111111111111111111111101011110011110'
        '0001111110000000000110011001111001100110000000000111111000011110011110100101001101001011010100101010'
        '1011001100110100110011001010101011010100101101001101001011101111001111000011111100000000001100110011'
        '1100110011000000000011111100001111001100011000101001101001011010100101010101100110011010011001100101'
        '0101011010100101101001100011011111011110011110000111111000000000011001100111100110011000000000011111'
        '1000011110000000110100010100110100101101010010101010110011001101001100110010101010110101001011010010'
        '0101100011111011110011110000111111000000000011001100111100110011000000000011111100001111011000010110'
        '1000101001101001011010100101010101100110011010011001100101010101101010010110100100101101100111110111'
        '1001111000011111100000000001100110011110011001100000000001111110000110001100001100110100010100110100'
        '1011010100101010101100110011010011001100101010101101010010110001100101101111001111101111001111000011'
        '1111000000000011001100111100110011000000000011111100000000011000011010011010001010011010010110101001'
        '0101010110011001101001100110010101010110101001010010110010110001111001111101111001111000011111100000'
        '0000011001100111100110011000000000011111100011000011000010110100110100010100110100101101010010101010'
        '1100110011010011001100101010101101010010100101100101100000111100111110111100111100001111110000000000'
        '1100110011110011001100000000001111110111100001100001010110100110100010100110100101101010010101010110'
        '0110011010011001100101010101101010010100101100101101100001111001111101111001111000011111100000000001'
        '1001100111100110011000000000011110001111000011000011001011010011010001010011010010110101001010101011'
        '0011001101001100110010101010110100011010010110010110111100001111001111101111001111000011111100000000'
        '0011001100111100110011000000000011000001111000011000011010010110100110100010100110100101101010010101'
        '0101100110011010011001100101010101100010110100101100101101111110000111100111110111100111100001111110'
        '0000000001100110011110011001100000000000000000111100001100001101010010110100110100010100110100101101'
        '0100101010101100110011010011001100101010101001010110100101100101100011111100001111001111101111001111'
        '0000111111000000000011001100111100110011000000000110000001111000011000010110101001011010011010001010'
        '0110100101101010010101010110011001101001100110010101010100101011010010110010110000011111100001111001'
        '1111011110011110000111111000000000011001100111100110011000000011110000001111000011000010101101010010'
        '1101001101000101001101001011010100101010101100110011010011001100101010101001010110100101100101100000'
        '0011111100001111001111101111001111000011111100000000001100110011110011001100000111111000000111100001'
        '1000010101011010100101101001101000101001101001011010100101010101100110011010011001100101010101001010'
        '1101001011001011000000000111111000011110011111011110011110000111111000000000011001100111100110011000'
        '1111111100000011110000110000101010101101010010110100110100010100110100101101010010101010110011001101'
        '0011001100101010101001010110100101100101100000000000111111000011110011111011110011110000111111000000'
        '0000110011001111001100110111111111100000011110000110000101010101011010100101101001101000101001101001'
        '0110101001010101011001100110100110011001010101010010101101001011001011011000000000011111100001111001'
        '1111011110011110000111111000000000011001100111100110000011111111110000001111000011000011001010101011'
        '0101001011010011010001010011010010110101001010101011001100110100110010011010101010010101101001011001'
        '0110001100000000001111110000111100111110111100111100001111110000000000110011001111001101100111111111'
        '1000000111100001100001011001010101011010100101101001101000101001101001011010100101010101100110011010'
        '0110010011010101010010101101001011001011011001100000000001111110000111100111110111100111100001111110'
        '0000000001100110011110000011001111111111000000111100001100001100110010101010110101001011010011010001'
        '0100110100101101010010101010110011001101001001100110101010100101011010010110010110001100110000000000'
        '1111110000111100111110111100111100001111110000000000110011001111011001100111111111100000011110000110'
        '0001011001100101010101101010010110100110100010100110100101101010010101010110011001101001001100110101'
        '0101001010110100101100101101100110011000000000011111100001111001111101111001111000011111100000000001'
        '1001100110001100110011111111110000001111000011000011001100110010101010110101001011010011010001010011'
        '0100101101010010101010110011001100011001100110101010100101011010010110010110111100110011000000000011'
        '1111000011110011111011110011110000111111000000000011001100000001100110011111111110000001111000011000'
        '0110100110011001010101011010100101101001101000101001101001011010100101010101100110010010110011001101'
        '0101010010101101001011001011000111100110011000000000011111100001111001111101111001111000011111100000'
        '0000011001101100001100110011111111110000001111000011000010110100110011001010101011010100101101001101'
        '0001010011010010110101001010101011001100100101100110011010101010010101101001011001011011001111001100'
        '1100000000001111110000111100111110111100111100001111110000000000110000011000011001100111111111100000'
        '0111100001100001100110100110011001010101011010100101101001101000101001101001011010100101010101100100'
        '1100101100110011010101010010101101001011001011000110011110011001100000000001111110000111100111110111'
        '1001111000011111100000000001101100110000110011001111111111000000111100001100001011001101001100110010'
        '1010101101010010110100110100010100110100101101010010101010110010011001011001100110101010100101011010'
        '0101100101101100110011110011001100000000001111110000111100111110111100111100001111110000000000000110'
        '0110000110011001111111111000000111100001100001100110011010011001100101010101101010010110100110100010'
        '1001101001011010100101010101001100110010110011001101010101001010110100101100101100011001100111100110'
        '0110000000000111111000011110011111011110011110000111111000000000110011001100001100110011111111110000'
        '0011110000110000101100110011010011001100101010101101010010110100110100010100110100101101010010101010'
        '1001100110010110011001101010101001010110100101100101100000110011001111001100110000000000111111000011'
        '1100111110111100111100001111110000000111100110011000011001100111111111100000011110000110000101011001'
        '1001101001100110010101010110101001011010011010001010011010010110101001010101010011001100101100110011'
        '0101010100101011010010110010110000000110011001111001100110000000000111111000011110011111011110011110'
        '0001111110000011111100110011000011001100111111111100000011110000110000101010110011001101001100110010'
        '1010101101010010110100110100010100110100101101010010101010100110011001011001100110101010100101011010'
        '0101100101100000000011001100111100110011000000000011111100001111001111101111001111000011111100011111'
        '1110011001100001100110011111111110000001111000011000010101010110011001101001100110010101010110101001'
        '0110100110100010100110100101101010010101010100110011001011001100110101010100101011010010110010110000'
        '0000000110011001111001100110000000000111111000011110011111011110011110000111111011111111110011001100'
        '0011001100111111111100000011110000110000101010101011001100110100110011001010101011010100101101001101'
        '0001010011010010110101001010101010011001100101100110011010101010010101101001011001011011000000000011'
        '0011001111001100110000000000111111000011110011111011110011110000111100011111111110011001100001100110'
        '0111111111100000011110000110000110010101010110011001101001100110010101010110101001011010011010001010'
        '0110100101101000110101010100110011001011001100110101010100101011010010110010110111100000000001100110'
        '0111100110011000000000011111100001111001111101111001111000011000001111111111001100110000110011001111'
        '1111110000001111000011000011010010101010110011001101001100110010101010110101001011010011010001010011'
        '0100101100010110101010100110011001011001100110101010100101011010010110010110111111000000000011001100'
        '1111001100110000000000111111000011110011111011110011110000000000011111111110011001100001100110011111'
        '1111100000011110000110000110101001010101011001100110100110011001010101011010100101101001101000101001'
        '1010010100101011010101010011001100101100110011010101010010101101001011001011000111111000000000011001'
        '1001111001100110000000000111111000011110011111011110011110001100000011111111110011001100001100110011'
        '1111111100000011110000110000101101010010101010110011001101001100110010101010110101001011010011010001'
        '0100110100101001010110101010100110011001011001100110101010100101011010010110010110000011111100000000'
        '0011001100111100110011000000000011111100001111001111101111001111011110000001111111111001100110000110'
        '0110011111111110000001111000011000010101101010010101010110011001101001100110010101010110101001011010'
        '0110100010100110100101001010110101010100110011001011001100110101010100101011010010110010110110000111'
        '1110000000000110011001111001100110000000000111111000011110011111011110011000111100000011111111110011'
        '0011000011001100111111111100000011110000110000110010110101001010101011001100110100110011001010101011'
        '0101001011010011010001010011000110100101011010101010011001100101100110011010101010010101101001011001'
        '0110111100001111110000000000110011001111001100110000000000111111000011110011111011110000000111100000'
        '0111111111100110011000011001100111111111100000011110000110000110100101101010010101010110011001101001'
        '1001100101010101101010010110100110100010100100101101001010110101010100110011001011001100110101010100'
        '1010110100101100101100011110000111111000000000011001100111100110011000000000011111100001111001111101'
        '1110110000111100000011111111110011001100001100110011111111110000001111000011000010110100101101010010'
        '1010101100110011010011001100101010101101010010110100110100010100100101101001010110101010100110011001'
        '0110011001101010101001010110100101100101101100111100001111110000000000110011001111001100110000000000'
        '1111110000111100111110110001100001111000000111111111100110011000011001100111111111100000011110000110'
        '0001100110100101101010010101010110011001101001100110010101010110101001011010011010001000110010110100'
        '1010110101010100110011001011001100110101010100101011010010110010110111100111100001111110000000000110'
        '0110011110011001100000000001111110000111100111110000001100001111000000111111111100110011000011001100'
        '1111111111000000111100001100001101001101001011010100101010101100110011010011001100101010101101010010'
        '11010011010000010110010110100101011010101010011001100101100110011010101010010101101001011001011'
    ),
    43: (
        '0001000100001001001001100100011001000011001000001100100100011001001100011001000110001100100101100011'
        '0010011011000110010001101100011001001011011000110010001011011000110010000101101100011001000001011011'
        '0001100100000010110110001100100100001011011000110010011000010110110001100100111000010110110001100100'
        '1111000010110110001100100011110000101101100011001001011110000101101100011001001101111000010110110001'
        '1001001110111100001011011000110010011110111100001011011000110010011111011110000101101100011001000111'
        '1101111000010110110001100100001111101111000010110110001100100000111110111100001011011000110010010001'
        '1111011110000101101100011001000100011111011110000101101100011001001010001111101111000010110110001100'
        '1000101000111110111100001011011000110010010101000111110111100001011011000110010001010100011111011110'
        '0001011011000110010010101010001111101111000010110110001100100010101010001111101111000010110110001100'
        '1000010101010001111101111000010110110001100100100101010100011111011110000101101100011001000100101010'
        '1000111110111100001011011000110010010100101010100011111011110000101101100011001000101001010101000111'
        '1101111000010110110001100100101010010101010001111101111000010110110001100100010101001010101000111110'
        '1111000010110110001100100101010100101010100011111011110000101101100011001000101010100101010100011111'
        '0111100001011011000110010000101010100101010100011111011110000101101100011001000001010101001010101000'
        '1111101111000010110110001100100100010101010010101010001111101111000010110110001100100110001010101001'
        '0101010001111101111000010110110001100100111000101010100101010100011111011110000101101100011001001111'
        '0001010101001010101000111110111100001011011000110010011111000101010100101010100011111011110000101101'
        '1000110010001111100010101010010101010001111101111000010110110001100100101111100010101010010101010001'
        '1111011110000101101100011001001101111100010101010010101010001111101111000010110110001100100111011111'
        '0001010101001010101000111110111100001011011000110010011110111110001010101001010101000111110111100001'
        '0110110001100100011110111110001010101001010101000111110111100001011011000110010000111101111100010101'
        '0100101010100011111011110000101101100011001000001111011111000101010100101010100011111011110000101101'
        '1000110010000001111011111000101010100101010100011111011110000101101100011001001000011110111110001010'
        '1010010101010001111101111000010110110001100100010000111101111100010101010010101010001111101111000010'
        '1101100011001001010000111101111100010101010010101010001111101111000010110110001100100110100001111011'
        '1110001010101001010101000111110111100001011011000110010001101000011110111110001010101001010101000111'
        '1101111000010110110001100100101101000011110111110001010101001010101000111110111100001011011000110010'
        '0110110100001111011111000101010100101010100011111011110000101101100011001000110110100001111011111000'
        '1010101001010101000111110111100001011011000110010000110110100001111011111000101010100101010100011111'
        '0111100001011011000110010000011011010000111101111100010101010010101010001111101111000010110110001100'
        '1001000110110100001111011111000101010100101010100011111011110000101101100011001001100011011010000111'
        '1011111000101010100101010100011111011110000101101100011001000110001101101000011110111110001010101001'
        '0101010001111101111000010110110001100100001100011011010000111101111100010101010010101010001111101111'
        '0000101101100011001001001100011011010000111101111100010101010010101010001111101111000010110110001100'
        '1000100110001101101000011110111110001010101001010101000111110111100001011011000110010000100110001101'
        '1010000111101111100010101010010101010001111101111000010110110001100100010010000010011111110000011000'
        '0001000110101001110100111011101011100110101101111000110001001000001001111111000001100000010001101010'
        '0111010011101110101110011010110111100011110010010000010011111110000011000000100011010100111010011101'
        '1101011100110101101111000111110010010000010011111110000011000000100011010100111010011101110101110011'
        '0101101111000011011001001000001001111111000001100000010001101010011101001110111010111001101011011110'
        '0101100110010010000010011111110000011000000100011010100111010011101110101110011010110111101101100011'
        '0010010000010011111110000011000000100011010100111010011101110101110011010110111101101110001100100100'
        '0001001111111000001100000010001101010011101001110111010111001101011011100110111100011001001000001001'
        '1111110000011000000100011010100111010011101110101110011010110111001101111100011001001000001001111111'
        '0000011000000100011010100111010011101110101110011010110111001101111110001100100100000100111111100000'
        '1100000010001101010011101001110111010111001101011011100110110111100011001001000001001111111000001100'
        '0000100011010100111010011101110101110011010110111001101110111100011001001000001001111111000001100000'
        '0100011010100111010011101110101110011010100111001101111011110001100100100000100111111100000110000001'
        '0001101010011101001110111010111001101010011100110110110111100011001001000001001111111000001100000010'
        '0011010100111010011101110101110011010100111001101110110111100011001001000001001111111000001100000010'
        '0011010100111010011101110101110011000100111001101101011011110001100100100000100111111100000110000001'
        '0001101010011101001110111010111001110010011100110111010110111100011001001000001001111111000001100000'
        '0100011010100111010011101110101110010100100111001101111010110111100011001001000001001111111000001100'
        '0000100011010100111010011101110101110010100100111001101101101011011110001100100100000100111111100000'
        '1100000010001101010011101001110111010111011010010011100110110011010110111100011001001000001001111111'
        '0000011000000100011010100111010011101110101111110100100111001101110011010110111100011001001000001001'
        '1111110000011000000100011010100111010011101110101111110100100111001101111001101011011110001100100100'
        '0001001111111000001100000010001101010011101001110111010101111010010011100110111110011010110111100011'
        '0010010000010011111110000011000000100011010100111010011101110100011110100100111001101101110011010110'
        '1111000110010010000010011111110000011000000100011010100111010011101110100011110100100111001101110111'
        '0011010110111100011001001000001001111111000001100000010001101010011101001110111000001111010010011100'
        '1101101011100110101101111000110010010000010011111110000011000000100011010100111010011101111000011110'
        '1001001110011011101011100110101101111000110010010000010011111110000011000000100011010100111010011101'
        '1010000111101001001110011011110101110011010110111100011001001000001001111111000001100000010001101010'
        '0111010011101001000011110100100111001101111101011100110101101111000110010010000010011111110000011000'
        '0001000110101001110100111000010000111101001001110011011011101011100110101101111000110010010000010011'
        '1111100000110000001000110101001110100111000010000111101001001110011011101110101110011010110111100011'
        '0010010000010011111110000011000000100011010100111010011000001000011110100100111001101111011101011100'
        '1101011011110001100100100000100111111100000110000001000110101001110100110000010000111101001001110011'
        '0111110111010111001101011011110001100100100000100111111100000110000001000110101001110100110000010000'
        '1111010010011100110110111011101011100110101101111000110010010000010011111110000011000000100011010100'
        '1110101110000010000111101001001110011011001110111010111001101011011110001100100100000100111111100000'
        '1100000010001101010011101011100000100001111010010011100110111001110111010111001101011011110001100100'
        '1000001001111111000001100000010001101010011101011100000100001111010010011100110110100111011101011100'
        '1101011011110001100100100000100111111100000110000001000110101001110101110000010000111101001001110011'
        '0111010011101110101110011010110111100011001001000001001111111000001100000010001101010011101011100000'
        '1000011110100100111001101111010011101110101110011010110111100011001001000001001111111000001100000010'
        '0011010100101010111000001000011110100100111001101111101001110111010111001101011011110001100100100000'
        '1001111111000001100000010001101010010101011100000100001111010010011100110110111010011101110101110011'
        '0101101111000110010010000010011111110000011000000100011010100101010111000001000011110100100111001101'
        '1001110100111011101011100110101101111000110010010000010011111110000011000000100011010110101010111000'
        '0010000111101001001110011011100111010011101110101110011010110111100011001001000001001111111000001100'
        '0000100011010110101010111000001000011110100100111001101101001110100111011101011100110101101111000110'
        '0100100000100111111100000110000001000110101101010101110000010000111101001001110011011101001110100111'
        '0111010111001101011011110001100100100000100111111100000110000001000110101101010101110000010000111101'
        '0010011100110110101001110100111011101011100110101101111000110010010000010011111110000011000000100011'
        '0101101010101110000010000111101001001110011011101010011101001110111010111001101011011110001100100100'
        '0001001111111000001100000010001101011010101011100000100001111010010011100110111101010011101001110111'
        '0101110011010110111100011001001000001001111111000001100000010000101011010101011100000100001111010010'
        '0111001101101101010011101001110111010111001101011011110001100100100000100111111100000110000001001010'
        '1011010101011100000100001111010010011100110110011010100111010011101110101110011010110111100011001001'
        '0000010011111110000011000000100101010110101010111000001000011110100100111001101100011010100111010011'
        '1011101011100110101101111000110010010000010011111110000011000000110101010110101010111000001000011110'
        '1001001110011011100011010100111010011101110101110011010110111100011001001000001001111111000001100000'
        '0110101010110101010111000001000011110100100111001101101000110101001110100111011101011100110101101111'
        '0001100100100000100111111100000110000011101010101101010101110000010000111101001001110011011001000110'
        '1010011101001110111010111001101011011110001100100100000100111111100000110000011101010101101010101110'
        '0000100001111010010011100110110001000110101001110100111011101011100110101101111000110010010000010011'
        '1111100000110000011101010101101010101110000010000111101001001110011011000010001101010011101001110111'
        '0101110011010110111100011001001000001001111111000001100000111010101011010101011100000100001111010010'
        '0111001101100000100011010100111010011101110101110011010110111100011001001000001001111111000001100000'
        '1110101010110101010111000001000011110100100111001101100000010001101010011101001110111010111001101011'
        '0111100011001001000001001111111000001100000111010101011010101011100000100001111010010011100110111000'
        '0001000110101001110100111011101011100110101101111000110010010000010011111110000011000001110101010110'
        '1010101110000010000111101001001110011011110000001000110101001110100111011101011100110101101111000110'
        '0100100000100111111100000010000011101010101101010101110000010000111101001001110011011011000000100011'
        '0101001110100111011101011100110101101111000110010010000010011111110000001000001110101010110101010111'
        '0000010000111101001001110011011001100000010001101010011101001110111010111001101011011110001100100100'
        '0001001111111000000100000111010101011010101011100000100001111010010011100110110001100000010001101010'
        '0111010011101110101110011010110111100011001001000001001111111000000100000111010101011010101011100000'
        '1000011110100100111001101100001100000010001101010011101001110111010111001101011011110001100100100000'
        '1001111111010000100000111010101011010101011100000100001111010010011100110110000011000000100011010100'
        '1110100111011101011100110101101111000110010010000010011111111100001000001110101010110101010111000001'
        '0000111101001001110011011100000110000001000110101001110100111011101011100110101101111000110010010000'
        '0100111111111000010000011101010101101010101110000010000111101001001110011011110000011000000100011010'
        '1001110100111011101011100110101101111000110010010000010011111111100001000001110101010110101010111000'
        '0010000111101001001110011011111000001100000010001101010011101001110111010111001101011011110001100100'
        '1000001001111011110000100000111010101011010101011100000100001111010010011100110111111000001100000010'
        '0011010100111010011101110101110011010110111100011001001000001001111011110000100000111010101011010101'
        '0111000001000011110100100111001101111111000001100000010001101010011101001110111010111001101011011110'
        '0011001001000001001101011110000100000111010101011010101011100000100001111010010011100110111111110000'
        '0110000001000110101001110100111011101011100110101101111000110010010000010010010111100001000001110101'
        '0101101010101110000010000111101001001110011011111111100000110000001000110101001110100111011101011100'
        '1101011011110001100100100000100100101111000010000011101010101101010101110000010000111101001001110011'
        '0110111111100000110000001000110101001110100111011101011100110101101111000110010010000010010010111100'
        '0010000011101010101101010101110000010000111101001001110011011001111111000001100000010001101010011101'
        '0011101110101110011010110111100011001001000001001001011110000100000111010101011010101011100000100001'
        '1110100100111001101110011111110000011000000100011010100111010011101110101110011010110111100011001001'
        '0000010010010111100001000001110101010110101010111000001000011110100100111001101101001111111000001100'
        '0000100011010100111010011101110101110011010110111100011001001000011001001011110000100000111010101011'
        '0101010111000001000011110100100111001101100100111111100000110000001000110101001110100111011101011100'
        '1101011011110001100100100011100100101111000010000011101010101101010101110000010000111101001001110011'
        '0110001001111111000001100000010001101010011101001110111010111001101011011110001100100100011100100101'
        '1110000100000111010101011010101011100000100001111010010011100110110000100111111100000110000001000110'
        '1010011101001110111010111001101011011110001100100100011100100101111000010000011101010101101010101110'
        '0000100001111010010011100110110000010011111110000011000000100011010100111010011101110101110011010110'
        '1111000110010011001110010010111100001000001110101010110101010111000001000011110100100111001101110000'
        '0100111111100000110000001000110101001110100111011101011100110101101111000110010011001110010010111100'
        '0010000011101010101101010101110000010000111101001001110011011010000010011111110000011000000100011010'
        '1001110100111011101011100110101101111000110010011001110010010111100001000001110101010110101010111000'
        '0010000111101001001110011011001000001001111111000001100000010001101010011101001110111010111001101011'
        '0111100011001101100111001001011110000100000111010101011010101011100000100001111010010011100110111001'
        '0000010011111110000011000000100011010100111010011101110101110011010110111100011001101100111001001011'
        '11000010000011101010101101010101110000010000111101001001110011011'
    ),
    45: (
        '1110111011110110110110011011100110111100110111110011011111100110110111100110110011110011011000111100'
        '1101100001111001101110000111100110111100001111001101111100001111001101101110000111100110111011100001'
        '1110011011110111000011110011011111011100001111001101101110111000011110011011001110111000011110011011'
        '1001110111000011110011011010011101110000111100110110010011101110000111100110110001001110111000011110'
        '0110110000100111011100001111001101100000100111011100001111001101100000010011101110000111100110111000'
        '0001001110111000011110011011010000001001110111000011110011011101000000100111011100001111001101101010'
        '0000010011101110000111100110111010100000010011101110000111100110110101010000001001110111000011110011'
        '0110010101000000100111011100001111001101110010101000000100111011100001111001101111001010100000010011'
        '1011100001111001101101100101010000001001110111000011110011011101100101010000001001110111000011110011'
        '0110101100101010000001001110111000011110011011101011001010100000010011101110000111100110111101011001'
        '0101000000100111011100001111001101101101011001010100000010011101110000111100110111011010110010101000'
        '0001001110111000011110011011010110101100101010000001001110111000011110011011101011010110010101000000'
        '1001110111000011110011011110101101011001010100000010011101110000111100110110110101101011001010100000'
        '0100111011100001111001101100110101101011001010100000010011101110000111100110111001101011010110010101'
        '0000001001110111000011110011011010011010110101100101010000001001110111000011110011011101001101011010'
        '1100101010000001001110111000011110011011010100110101101011001010100000010011101110000111100110111010'
        '1001101011010110010101000000100111011100001111001101101010100110101101011001010100000010011101110000'
        '1111001101100101010011010110101100101010000001001110111000011110011011000101010011010110101100101010'
        '0000010011101110000111100110110000101010011010110101100101010000001001110111000011110011011000001010'
        '1001101011010110010101000000100111011100001111001101100000010101001101011010110010101000000100111011'
        '1000011110011011100000010101001101011010110010101000000100111011100001111001101101000000101010011010'
        '1101011001010100000010011101110000111100110110010000001010100110101101011001010100000010011101110000'
        '1111001101110010000001010100110101101011001010100000010011101110000111100110111100100000010101001101'
        '0110101100101010000001001110111000011110011011111001000000101010011010110101100101010000001001110111'
        '0000111100110110111001000000101010011010110101100101010000001001110111000011110011011101110010000001'
        '0101001101011010110010101000000100111011100001111001101111011100100000010101001101011010110010101000'
        '0001001110111000011110011011111011100100000010101001101011010110010101000000100111011100001111001101'
        '1011101110010000001010100110101101011001010100000010011101110000111100110110011101110010000001010100'
        '1101011010110010101000000100111011100001111001101100011101110010000001010100110101101011001010100000'
        '0100111011100001111001101100001110111001000000101010011010110101100101010000001001110111000011110011'
        '0111000011101110010000001010100110101101011001010100000010011101110000111100110111100001110111001000'
        '0001010100110101101011001010100000010011101110000111100110111110000111011100100000010101001101011010'
        '1100101010000001001110111000011110011011111100001110111001000000101010011010110101100101010000001001'
        '1101110000111100110110111100001110111001000000101010011010110101100101010000001001110111000011110011'
        '0110011110000111011100100000010101001101011010110010101000000100111011100001111001101110011110000111'
        '0111001000000101010011010110101100101010000001001110111000011110011011110011110000111011100100000010'
        '1010011010110101100101010000001001110111000011110011011011001111000011101110010000001010100110101101'
        '0110010101000000100111011100001111001101110110011110000111011100100000010101001101011010110010101000'
        '0001001110111000011110011011110110011110000111011100100000010101001101011010110010101000000100111011'
        '1000011110011011011011001111000011101110010000001010100110101101011001010100000010011101110000111100'
        '1101110110110011110000111011100100000010101001101011010110010101000000100111011100001111001101011011'
        '0110011110000111011100100000010101001101011010110010101000000100111011100001111001100001101101100111'
        '1000011101110010000001010100110101101011001010100000010011101110000111100111001011011011001111000011'
        '1011100100000010101001101011010110010101000000100111011100001111001010011011011011001111000011101110'
        '0100000010101001101011010110010101000000100111011100001111000010001101101101100111100001110111001000'
        '0001010100110101101011001010100000010011101110000111101001000011011011011001111000011101110010000001'
        '0101001101011010110010101000000100111011100001111110010010011011011011001111000011101110010000001010'
        '1001101011010110010101000000100111011100001110110010011001101101101100111100001110111001000000101010'
        '0110101101011001010100000010011101110000110011001001110011011011011001111000011101110010000001010100'
        '1101011010110010101000000100111011100001000110010011110011011011011001111000011101110010000001010100'
        '1101011010110010101000000100111011100000000110010001111001101101101100111100001110111001000000101010'
        '0110101101011001010100000010011101110001000011001000011110011011011011001111000011101110010000001010'
        '1001101011010110010101000000100111011100110000110010000011110011011011011001111000011101110010000001'
        '0101001101011010110010101000000100111011101110000110010000001111001101101101100111100001110111001000'
        '0001010100110101101011001010100000010011101111111000011001001000011110011011011011001111000011101110'
        '0100000010101001101011010110010101000000100111011011110000110010011000011110011011011011001111000011'
        '1011100100000010101001101011010110010101000000100111010011110000110010011100001111001101101101100111'
        '1000011101110010000001010100110101101011001010100000010011100001111000011001000111000011110011011011'
        '0110011110000111011100100000010101001101011010110010101000000100111100011110000110010010111000011110'
        '0110110110110011110000111011100100000010101001101011010110010101000000100110100011110000110010011011'
        '1000011110011011011011001111000011101110010000001010100110101101011001010100000010010010001111000011'
        '0010011101110000111100110110110110011110000111011100100000010101001101011010110010101000000100000100'
        '0111100001100100011101110000111100110110110110011110000111011100100000010101001101011010110010101000'
        '0001010001000111100001100100001110111000011110011011011011001111000011101110010000001010100110101101'
        '0110010101000000111000100011110000110010010011101110000111100110110110110011110000111011100100000010'
        '1010011010110101100101010000000110001000111100001100100010011101110000111100110110110110011110000111'
        '0111001000000101010011010110101100101010000010110001000111100001100100001001110111000011110011011011'
        '0110011110000111011100100000010101001101011010110010101000011011000100011110000110010000010011101110'
        '0001111001101101101100111100001110111001000000101010011010110101100101010001110110001000111100001100'
        '1000000100111011100001111001101101101100111100001110111001000000101010011010110101100101010011110110'
        '0010001111000011001000000010011101110000111100110110110110011110000111011100100000010101001101011010'
        '1100101010111110110001000111100001100100000000100111011100001111001101101101100111100001110111001000'
        '0001010100110101101011001010111111101100010001111000011001001000000100111011100001111001101101101100'
        '1111000011101110010000001010100110101101011001010011111101100010001111000011001000100000010011101110'
        '0001111001101101101100111100001110111001000000101010011010110101100101101111110110001000111100001100'
        '1001010000001001110111000011110011011011011001111000011101110010000001010100110101101011001001011111'
        '1011000100011110000110010001010000001001110111000011110011011011011001111000011101110010000001010100'
        '1101011010110011010111111011000100011110000110010010101000000100111011100001111001101101101100111100'
        '0011101110010000001010100110101101011000101011111101100010001111000011001000101010000001001110111000'
        '0111100110110110110011110000111011100100000010101001101011010110101010111111011000100011110000110010'
        '0001010100000010011101110000111100110110110110011110000111011100100000010101001101011010111101010111'
        '1110110001000111100001100100100101010000001001110111000011110011011011011001111000011101110010000001'
        '0101001101011010101101010111111011000100011110000110010011001010100000010011101110000111100110110110'
        '1100111100001110111001000000101010011010110100011010101111110110001000111100001100100011001010100000'
        '0100111011100001111001101101101100111100001110111001000000101010011010110110011010101111110110001000'
        '1111000011001001011001010100000010011101110000111100110110110110011110000111011100100000010101001101'
        '0110010011010101111110110001000111100001100100010110010101000000100111011100001111001101101101100111'
        '1000011101110010000001010100110101110100110101011111101100010001111000011001001010110010101000000100'
        '1110111000011110011011011011001111000011101110010000001010100110101010100110101011111101100010001111'
        '0000110010011010110010101000000100111011100001111001101101101100111100001110111001000000101010011010'
        '0010100110101011111101100010001111000011001000110101100101010000001001110111000011110011011011011001'
        '1110000111011100100000010101001101100101001101010111111011000100011110000110010010110101100101010000'
        '0010011101110000111100110110110110011110000111011100100000010101001100100101001101010111111011000100'
        '0111100001100100010110101100101010000001001110111000011110011011011011001111000011101110010000001010'
        '1001110100101001101010111111011000100011110000110010010101101011001010100000010011101110000111100110'
        '1101101100111100001110111001000000101010010101001010011010101111110110001000111100001100100110101101'
        '0110010101000000100111011100001111001101101101100111100001110111001000000101010000101001010011010101'
        '1111101100010001111000011001000110101101011001010100000010011101110000111100110110110110011110000111'
        '0111001000000101010100101001010011010101111110110001000111100001100100001101011010110010101000000100'
        '1110111000011110011011011011001111000011101110010000001010111001010010100110101011111101100010001111'
        '0000110010010011010110101100101010000001001110111000011110011011011011001111000011101110010000001010'
        '0110010100101001101010111111011000100011110000110010001001101011010110010101000000100111011100001111'
        '0011011011011001111000011101110010000001011011001010010100110101011111101100010001111000011001001010'
        '0110101101011001010100000010011101110000111100110110110110011110000111011100100000010010110010100101'
        '0011010101111110110001000111100001100100010100110101101011001010100000010011101110000111100110110110'
        '1100111100001110111001000000110101100101001010011010101111110110001000111100001100100101010011010110'
        '1011001010100000010011101110000111100110110110110011110000111011100100000001010110010100101001101010'
        '1111110110001000111100001100100010101001101011010110010101000000100111011100001111001101101101100111'
        '1000011101110010000010101011001010010100110101011111101100010001111000011001000010101001101011010110'
        '0101010000001001110111000011110011011011011001111000011101110010000110101011001010010100110101011111'
        '1011000100011110000110010000010101001101011010110010101000000100111011100001111001101101101100111100'
        '0011101110010001110101011001010010100110101011111101100010001111000011001000000101010011010110101100'
        '1010100000010011101110000111100110110110110011110000111011100100111101010110010100101001101010111111'
        '0110001000111100001100100000001010100110101101011001010100000010011101110000111100110110110110011110'
        '0001110111001011111010101100101001010011010101111110110001000111100001100100000000101010011010110101'
        '1001010100000010011101110000111100110110110110011110000111011100111111101010110010100101001101010111'
        '1110110001000111100001100100100000010101001101011010110010101000000100111011100001111001101101101100'
        '1111000011101110001111110101011001010010100110101011111101100010001111000011001000100000010101001101'
        '0110101100101010000001001110111000011110011011011011001111000011101110101111110101011001010010100110'
        '1010111111011000100011110000110010000100000010101001101011010110010101000000100111011100001111001101'
        '1011011001111000011101111101111110101011001010010100110101011111101100010001111000011001001001000000'
        '1010100110101101011001010100000010011101110000111100110110110110011110000111011011011111101010110010'
        '1001010011010101111110110001000111100001100100110010000001010100110101101011001010100000010011101110'
        '0001111001101101101100111100001110100110111111010101100101001010011010101111110110001000111100001100'
        '1001110010000001010100110101101011001010100000010011101110000111100110110110110011110000111000011011'
        '1111010101100101001010011010101111110110001000111100001100100011100100000010101001101011010110010101'
        '0000001001110111000011110011011011011001111000011110001101111110101011001010010100110101011111101100'
        '0100011110000110010010111001000000101010011010110101100101010000001001110111000011110011011011011001'
        '1110000110100011011111101010110010100101001101010111111011000100011110000110010011011100100000010101'
        '0011010110101100101010000001001110111000011110011011011011001111000010010001101111110101011001010010'
        '1001101010111111011000100011110000110010011101110010000001010100110101101011001010100000010011101110'
        '0001111001101101101100111100000001000110111111010101100101001010011010101111110110001000111100001100'
        '1000111011100100000010101001101011010110010101000000100111011100001111001101101101100111100010001000'
        '1101111110101011001010010100110101011111101100010001111000011001000011101110010000001010100110101101'
        '0110010101000000100111011100001111001101101101100111100110001000110111111010101100101001010011010101'
        '1111101100010001111000011001000001110111001000000101010011010110101100101010000001001110111000011110'
        '0110110110110011110111000100011011111101010110010100101001101010111111011000100011110000110010000001'
        '1101110010000001010100110101101011001010100000010011101110000111100110110110110011111111000100011011'
        '1111010101100101001010011010101111110110001000111100001100100100001110111001000000101010011010110101'
        '1001010100000010011101110000111100110110110110011101111000100011011111101010110010100101001101010111'
        '1110110001000111100001100100110000111011100100000010101001101011010110010101000000100111011100001111'
        '0011011011011001100111100010001101111110101011001010010100110101011111101100010001111000011001001110'
        '0001110111001000000101010011010110101100101010000001001110111000011110011011011011001000111100010001'
        '1011111101010110010100101001101010111111011000100011110000110010011110000111011100100000010101001101'
        '0110101100101010000001001110111000011110011011011011000000111100010001101111110101011001010010100110'
        '1010111111011000100011110000110010001111000011101110010000001010100110101101011001010100000010011101'
        '1100001111001101101101101000011110001000110111111010101100101001010011010101111110110001000111100001'
        '1001000011110000111011100100000010101001101011010110010101000000100111011100001111001101101101111000'
        '0111100010001101111110101011001010010100110101011111101100010001111000011001001001111000011101110010'
        '0000010101001101011010110010101000000100111011100001111001101101101011000011110001000110111111010101'
        '1001010010100110101011111101100010001111000011001001100111100001110111001000000101010011010110101100'
        '1010100000010011101110000111100110110110001100001111000100011011111101010110010100101001101010111111'
        '0110001000111100001100100011001111000011101110010000001010100110101101011001010100000010011101110000'
        '1111001101101110011000011110001000110111111010101100101001010011010101111110110001000111100001100100'
        '1011001111000011101110010000001010100110101101011001010100000010011101110000111100110110101001100001'
        '1110001000110111111010101100101001010011010101111110110001000111100001100100110110011110000111011100'
        '1000000101010011010110101100101010000001001110111000011110011011000100110000111100010001101111110101'
        '01100101001010011010101111110110001000111100001100100'
    ),
    49: (
        '...'
        '111010111001001010000'
    ),
    50: (
        '...'
        '011'
    ),
}

EXPECTED_SOLVED_N = [22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42, 43, 45, 49, 50]
MISSING_IN_LOCAL_AUDIT = [40, 44, 46, 47, 48]
PROVENANCE_SUMMARY = {
    'task': 'Triangular Book Ramsey Lower Bound',
    'audit_date': '2026-05-27',
    'verified_count': 24,
    'algebraic_n': [31, 37, 41, 45, 49],
    'computational_search_n': [22, 23, 24, 25, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 38, 39, 42, 43, 50],
}


In [18]:
#@title Verification
stats = verify_witness_collection(BOOK_RAMSEY_WITNESSES)
assert sorted(stats) == EXPECTED_SOLVED_N
assert all(row['valid'] for row in stats.values())
print('Solved n values:', EXPECTED_SOLVED_N)
print('Not present as solved in the local audit:', MISSING_IN_LOCAL_AUDIT)


Verified n=22: |V|=86, edges=1806, red_max=20/20, blue_max=21/21.
Verified n=23: |V|=90, edges=1980, red_max=21/21, blue_max=22/22.
Verified n=24: |V|=94, edges=2162, red_max=22/22, blue_max=23/23.
Verified n=25: |V|=98, edges=2352, red_max=23/23, blue_max=24/24.
Verified n=26: |V|=102, edges=2550, red_max=24/24, blue_max=25/25.
Verified n=27: |V|=106, edges=2756, red_max=25/25, blue_max=26/26.
Verified n=28: |V|=110, edges=2970, red_max=26/26, blue_max=27/27.
Verified n=29: |V|=114, edges=3192, red_max=27/27, blue_max=28/28.
Verified n=30: |V|=118, edges=3422, red_max=28/28, blue_max=29/29.
Verified n=31: |V|=122, edges=3660, red_max=29/29, blue_max=30/30.
Verified n=32: |V|=126, edges=3906, red_max=30/30, blue_max=31/31.
Verified n=33: |V|=130, edges=4160, red_max=31/31, blue_max=32/32.
Verified n=34: |V|=134, edges=4422, red_max=32/32, blue_max=33/33.
Verified n=35: |V|=138, edges=4692, red_max=33/33, blue_max=34/34.
Verified n=36: |V|=142, edges=4970, red_max=34/34, blue_max=35/35.

## 5. Explicit Deformations of Algebras

Problem source: [Epoch AI](https://epoch.ai/frontiermath/open-problems/explicit-deformations)

The problem asks for an explicit flat one-parameter deformation whose special fiber is the monomial spider algebra

$$
A_0 = k[x,y,z]/(x^8,y^8,z^8,xy,xz,yz)
$$

and whose generic fiber is the curvilinear algebra $k[t]/(t^{22})$. The construction below gives such a family over any field of characteristic zero.

### Results

The Station was able to solve this problem, which was independently solved by GPT-5.2 Pro with David Turturean’s scaffold. Further inspection shows that the Station solution is an alternative presentation of the same underlying Möbius-generator construction. The
problem was also subsequently delisted from Epoch AI as it was deemed not to meet their bar for a publishable result. Nonetheless, since our attempt was done before the delisting and the construction is still nontrivial, we include it here for completeness.

### Method

Let $a_1=1$, $a_2=2$, and $a_3=3$, and let $u_1,u_2,u_3$ be the three coordinate functions. Define three quadratic relations

$$
(a_j-a_i)u_i u_j - \epsilon(u_j-u_i)=0 \qquad (1\le i<j\le 3).
$$

These are

$$
u_1u_2=\epsilon(u_2-u_1),\qquad
2u_1u_3=\epsilon(u_3-u_1),\qquad
u_2u_3=\epsilon(u_3-u_2).
$$

To close the algebra at length 22, add three Hermite interpolation relations. For a triple $d=(d_1,d_2,d_3)$ with $\max d_i=8$, set

$$
C_d = \sum_{i=1}^3\sum_{r=0}^{d_i-1}
\frac{1}{r!}
\left.
\frac{d^r}{d\lambda^r}
\prod_{j\ne i}(\lambda-a_j)^{-d_j}
\right|_{\lambda=a_i}
\epsilon^{8-d_i+r} u_i^{d_i-r}.
$$

After clearing the constant denominators, take

$$
I = (Q_{12},Q_{13},Q_{23},C_{(8,7,7)},C_{(7,8,7)},C_{(7,7,8)})
\subset k[u_1,u_2,u_3,\epsilon],
$$

where $Q_{ij}=(a_j-a_i)u_i u_j-\epsilon(u_j-u_i)$. This is the desired deformation ideal.

For $\epsilon\ne0$, the relations are modeled by the substitution

$$
u_i = \frac{\epsilon t}{1-a_i t}\quad\text{in}\quad k(\epsilon)[t]/(t^{22}).
$$

The quadratic relations hold identically under this substitution. The three Hermite relations are exactly the principal-part identities expressing $u_i^8$ in the 22-element basis

$$
1,
 u_1,u_1^2,\ldots,u_1^7,
 u_2,u_2^2,\ldots,u_2^7,
 u_3,u_3^2,\ldots,u_3^7.
$$

This basis is a confluent Vandermonde/Hermite basis after expanding in powers of $t$, so the generic fiber is $k(\epsilon)[t]/(t^{22})$. In particular, $u_1$ is a curvilinear generator because

$$
t = \frac{u_1}{\epsilon + a_1u_1},
$$

and the denominator is a unit in the generic Artin fiber.

At $\epsilon=0$, the quadratic relations become $u_i u_j=0$ for $i\ne j$, while the three closure relations reduce to nonzero scalar multiples of $u_1^8$, $u_2^8$, and $u_3^8$. Thus the special fiber is the spider algebra. The displayed 22 monomials form a free $k[\epsilon]$-basis, giving flatness.


In [19]:
#@title Verification of the deformation construction
# This cell constructs the six generators and checks the two fibers symbolically.
# It uses SymPy only for exact rational arithmetic and formal power-series expansion.

import math
import sympy as sp

lam, eps, tau = sp.symbols("lambda eps tau")
u1, u2, u3 = sp.symbols("u1 u2 u3")
roots = (sp.Integer(1), sp.Integer(2), sp.Integer(3))
u = (u1, u2, u3)


def clear_constant_denominators(expr):
    """Return a polynomial proportional to expr, with integer coefficients."""
    num, _den = sp.together(sp.expand(expr)).as_numer_denom()
    return sp.expand(num)


def hermite_closure(degrees):
    """The closure relation C_degrees from the formula above."""
    D = max(degrees)
    rel = sp.Integer(0)
    for i, (a_i, d_i) in enumerate(zip(roots, degrees)):
        principal_part_denominator = sp.Integer(1)
        for j, a_j in enumerate(roots):
            if i != j:
                principal_part_denominator *= (lam - a_j) ** (-degrees[j])
        for r in range(d_i):
            coeff = sp.diff(principal_part_denominator, lam, r).subs(lam, a_i) / math.factorial(r)
            rel += coeff * eps ** (D - d_i + r) * u[i] ** (d_i - r)
    return clear_constant_denominators(rel)


quadratic_relations = [
    (roots[j] - roots[i]) * u[i] * u[j] - eps * (u[j] - u[i])
    for i in range(3)
    for j in range(i + 1, 3)
]
closure_relations = [
    hermite_closure((8, 7, 7)),
    hermite_closure((7, 8, 7)),
    hermite_closure((7, 7, 8)),
]
I_gens = quadratic_relations + closure_relations

# Check that the displayed generators really have integer coefficients.
for g in I_gens:
    poly = sp.Poly(g, u1, u2, u3, eps)
    assert all(c.is_Integer for c in poly.coeffs())

# Special fiber: mixed products vanish and the closures become u_i^8 up to nonzero scalars.
special_quadratics = [sp.factor(g.subs(eps, 0)) for g in quadratic_relations]
assert special_quadratics == [u1 * u2, 2 * u1 * u3, u2 * u3]

for relation, variable in zip(closure_relations, u):
    special = sp.Poly(sp.expand(relation.subs(eps, 0)), u1, u2, u3)
    terms = special.terms()
    assert len(terms) == 1
    monomial, coefficient = terms[0]
    assert coefficient != 0
    assert monomial == tuple(8 if v == variable else 0 for v in u)

# Generic fiber: verify the map u_i = eps*t/(1-a_i*t) kills all relations modulo t^22.
series_substitution = {
    u[i]: eps * tau / (1 - roots[i] * tau)
    for i in range(3)
}
for g in I_gens:
    image = sp.series(g.subs(series_substitution), tau, 0, 22).removeO().expand()
    assert image == 0

# The expected 22 basis elements are independent after expanding at eps=1.
basis = [sp.Integer(1)]
for variable in u:
    basis.extend(variable ** power for power in range(1, 8))

basis_images = [
    sp.series(b.subs(series_substitution).subs(eps, 1), tau, 0, 22).removeO().expand()
    for b in basis
]
coefficient_matrix = sp.Matrix([
    [image.coeff(tau, degree) for image in basis_images]
    for degree in range(22)
])
assert coefficient_matrix.rank() == 22

print("Constructed", len(I_gens), "generators for the deformation ideal.")
print("Special fiber: spider algebra with basis 1 and three length-7 legs.")
print("Generic fiber: verified 22-dimensional curvilinear basis in k(eps)[t]/(t^22).")
print("Closure relation term counts:", [len(sp.Poly(c, u1, u2, u3, eps).terms()) for c in closure_relations])


Constructed 6 generators for the deformation ideal.
Special fiber: spider algebra with basis 1 and three length-7 legs.
Generic fiber: verified 22-dimensional curvilinear basis in k(eps)[t]/(t^22).
Closure relation term counts: [22, 18, 22]


### Relation to the Divided-Difference Rees Presentation

The construction above is an alternative presentation of the same Möbius-generator construction used in the divided-difference/Rees approach described [here](https://arxiv.org/abs/2603.00886). Both start from the same three elements

$$
u_a=\frac{t}{1-at}\quad (a=1,2,3)
$$

inside $k[t]/(t^{22})$, together with the identity

$$
u_b-u_a=(b-a)u_a u_b.
$$

The presentation above keeps $u_1,u_2,u_3$ as the three coordinates and closes the algebra by Hermite interpolation. The divided-difference presentation instead uses the linear change of generators

$$
x=u_1,
\qquad
 y=u_2-u_1,
\qquad
 z=u_3-2u_2+u_1,
$$

with inverse

$$
u_1=x,
\qquad
u_2=x+y,
\qquad
u_3=x+2y+z.
$$

Thus, on the generic fiber, the two presentations generate the same subalgebra of $k[t]/(t^{22})$. The difference is the chosen degeneration: the divided-difference presentation homogenizes the relations using the $t$-adic orders of $x,y,z$, while the presentation above degenerates directly in the Möbius-generator coordinates. Consequently the displayed ideals are not literally the same polynomial ideal, but they are two presentations of the same underlying Möbius-generator deformation mechanism.
